# BloodHound Enterprise Identity Risk Report — PDF Edition
## Version 1.5 FINAL

Executive-quality PDF report generator for BloodHound Enterprise assessment data.

This notebook produces:

- Executive PDF Identity Risk Report
- Technical Evidence Workbook (XLSX)

Compatible with BloodHound Enterprise and connected OpenGraph data sources, including Privilege Zone Intelligence summary reporting.

**Production baseline:** v1.5 FINAL  
**Internal build lineage:** v1.5.44


## 1. Configuration

Create `config/bhe_report_config.json` in the project `config` directory. Customer branding is optional; leave `customer_logo_source` and `customer_logo_path` empty when no customer logo is supplied.

```json
{
  "tenant_url": "https://your-tenant.bloodhoundenterprise.io",
  "auth": {
    "api_token_id": "YOUR_API_TOKEN_ID",
    "api_token_key": "YOUR_API_TOKEN_KEY"
  },
  "client": {
    "verify_ssl": true,
    "timeout_seconds": 600,
    "prefer_wait_seconds": 120,
    "query_endpoint": "/api/v2/graphs/cypher"
  },
  "report": {
    "customer_name": "Customer Name",
    "assessment_name": "BloodHound Enterprise Identity Risk Report",
    "prepared_by": "SpecterOps",
    "prepared_for": "Executive Leadership and Identity Security Stakeholders",
    "version": "1.5",
    "classification": "Confidential",
    "output_basename": "BHE_Identity_Risk_Report",
    "max_evidence_rows": 10,
    "include_no_result_sections": false,
    "include_collection_notes": true,
    "attack_path_window_days": 30,
    "theme": "executive"
  },
  "branding": {
    "specterops_logo_path": "assets/SO-Logo_White_Tagline.png",
    "force_embedded_specterops_logo": true,
    "customer_logo_source": "",
    "customer_logo_path": "",
    "cover_hero_image_path": "",
    "show_specterops_wordmark_if_logo_missing": true
  },
  "extensions_config": "config/extensions.json"
}
```


## 2. Setup and BHE Client

In [ ]:
from __future__ import annotations

import base64
import datetime
import hashlib
import hmac
import html
import importlib.util
import json
import re
import sys
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional

import pandas as pd
import requests
from IPython.display import display, HTML, Markdown

In [ ]:
# ---------------------------------------------------------------------------
# Project discovery and configuration
# ---------------------------------------------------------------------------

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (
            (candidate / "config").is_dir()
            and (candidate / "plugins").is_dir()
            and (candidate / "searches").is_dir()
        ):
            return candidate

    if start.name.lower() == "notebooks":
        parent = start.parent
        if (
            (parent / "config").is_dir()
            and (parent / "plugins").is_dir()
            and (parent / "searches").is_dir()
        ):
            return parent

    raise FileNotFoundError(
        "Could not locate the IdentityRiskReport project root. "
        "Expected sibling folders named config, plugins, and searches."
    )


PROJECT_ROOT = find_project_root()

CONFIG_DIR = PROJECT_ROOT / "config"
PLUGIN_DIR = PROJECT_ROOT / "plugins"
SEARCH_DIR = PROJECT_ROOT / "searches"
ASSET_DIR = PROJECT_ROOT / "assets"
OUTPUT_DIR = PROJECT_ROOT / "output"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"

CONFIG_FILE = CONFIG_DIR / "bhe_report_config.json"
EXTENSIONS_FILE = CONFIG_DIR / "extensions.json"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [PROJECT_ROOT, PLUGIN_DIR]:
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)


def load_config(path: Path = CONFIG_FILE) -> Dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Copy "
            f"{CONFIG_DIR / 'bhe_report_config.example.json'} "
            "to bhe_report_config.json and fill in tenant/token values."
        )

    raw_text = path.read_text(encoding="utf-8")

    try:
        cfg = json.loads(raw_text)
    except json.JSONDecodeError as exc:
        # A trailing comma is a common mistake when copying the notebook example.
        # Normalize only commas immediately before a closing brace/bracket, then retry.
        normalized_text = re.sub(r",\s*([}\]])", r"\1", raw_text)
        try:
            cfg = json.loads(normalized_text)
        except json.JSONDecodeError:
            lines = raw_text.splitlines()
            bad_line = lines[exc.lineno - 1] if 0 < exc.lineno <= len(lines) else ""
            pointer = " " * max(exc.colno - 1, 0) + "^"
            raise ValueError(
                f"Invalid JSON in {path} at line {exc.lineno}, column {exc.colno}: "
                f"{exc.msg}\n\n{bad_line}\n{pointer}\n\n"
                "JSON requires double-quoted property names and does not allow comments. "
                "Remove any trailing comma before a closing } or ]."
            ) from exc

    required_paths = [
        ("tenant_url",),
        ("auth", "api_token_id"),
        ("auth", "api_token_key"),
    ]

    missing = []
    for parts in required_paths:
        current = cfg
        valid = True

        for part in parts:
            if not isinstance(current, dict) or part not in current:
                valid = False
                break
            current = current[part]

        if not valid or current in [None, ""]:
            missing.append(".".join(parts))

    if missing:
        raise ValueError(f"Missing required config values: {missing}")

    cfg["tenant_url"] = cfg["tenant_url"].rstrip("/")
    return cfg


def load_extensions(path: Path = EXTENSIONS_FILE) -> Dict[str, Any]:
    if not path.exists():
        return {"extensions": []}
    return json.loads(path.read_text(encoding="utf-8"))


cfg = load_config()
extensions_cfg = load_extensions()
report_cfg = cfg.get("report", {})
report_cfg["version"] = str(report_cfg.get("version") or "1.5")

In [ ]:
# ---------------------------------------------------------------------------
# Pre-flight validation
# ---------------------------------------------------------------------------

def startup_status_rows() -> list[dict[str, str]]:
    rows = [
        {
            "Check": "Project Root",
            "Status": "Ready",
            "Detail": str(PROJECT_ROOT),
        },
        {
            "Check": "BHE Configuration",
            "Status": "Ready" if CONFIG_FILE.exists() else "Missing",
            "Detail": str(CONFIG_FILE),
        },
        {
            "Check": "Extension Registry",
            "Status": "Ready" if EXTENSIONS_FILE.exists() else "Optional / Missing",
            "Detail": str(EXTENSIONS_FILE),
        },
    ]

    for extension in extensions_cfg.get("extensions", []):
        extension_id = str(extension.get("id", "unknown"))
        enabled = bool(extension.get("enabled", False))
        module_value = str(extension.get("module", ""))
        search_value = str(extension.get("saved_search_zip", ""))

        module_path = Path(module_value)
        if not module_path.is_absolute():
            module_path = PROJECT_ROOT / module_path

        search_path = Path(search_value) if search_value else None
        if search_path is not None and not search_path.is_absolute():
            search_path = PROJECT_ROOT / search_path

        if not enabled:
            status = "Disabled"
            detail = "Extension disabled in extensions.json"
        elif not module_path.exists():
            status = "Missing Module"
            detail = str(module_path)
        elif search_path is not None and not search_path.exists():
            status = "Missing Search ZIP"
            detail = str(search_path)
        else:
            status = "Ready"
            detail = f"Module: {module_path.name}"
            if search_path is not None:
                detail += f" | Searches: {search_path.name}"

        rows.append({
            "Check": f"Extension: {extension_id}",
            "Status": status,
            "Detail": detail,
        })

    return rows


startup_rows = startup_status_rows()

status_table = "\n".join(
    f"| {row['Check']} | {row['Status']} | `{row['Detail']}` |"
    for row in startup_rows
)

display(Markdown(f"""
# BloodHound PDF Report Generator — v1.5 FINAL

**Project root:** `{PROJECT_ROOT}`  
**Configuration:** `{CONFIG_FILE}`  
**Output directory:** `{OUTPUT_DIR}`

| Check | Status | Detail |
|---|---|---|
{status_table}
"""))

blocking_statuses = {"Missing", "Missing Module", "Missing Search ZIP"}
blocking = [row for row in startup_rows if row["Status"] in blocking_statuses]

if blocking:
    details = "\n".join(f"- {row['Check']}: {row['Detail']}" for row in blocking)
    raise RuntimeError(
        "Pre-flight validation failed. Fix the following before continuing:\n"
        + details
    )

display(Markdown("## Pre-flight passed"))

In [ ]:
# ---------------------------------------------------------------------------
# BloodHound Enterprise client
# ---------------------------------------------------------------------------

def get_config_value(cfg: Dict[str, Any], *paths, default=None):
    for path in paths:
        current = cfg
        ok = True
        for part in path:
            if isinstance(current, dict) and part in current:
                current = current[part]
            else:
                ok = False
                break
        if ok and current not in [None, ""]:
            return current
    return default


tenant_url = get_config_value(
    cfg,
    ("tenant_url",),
    ("bhe", "tenant_url"),
    ("bloodhound", "tenant_url"),
)
token_id = get_config_value(
    cfg,
    ("auth", "api_token_id"),
    ("token_id",),
    ("api_token_id",),
    ("bhe", "token_id"),
)
token_key = get_config_value(
    cfg,
    ("auth", "api_token_key"),
    ("token_key",),
    ("api_token_key",),
    ("bhe", "token_key"),
)

client_cfg = cfg.get("client", {})
query_endpoint = client_cfg.get("query_endpoint", "/api/v2/graphs/cypher")
verify_ssl = bool(client_cfg.get("verify_ssl", True))
timeout_seconds = int(client_cfg.get("timeout_seconds", 600))
prefer_wait_seconds = int(client_cfg.get("prefer_wait_seconds", 120))


def flatten_value(value: Any) -> Any:
    if isinstance(value, (dict, list)):
        return json.dumps(value, default=str)
    return value


def bhe_raw_to_dataframe(raw: Any) -> pd.DataFrame:
    if raw is None:
        return pd.DataFrame()

    data = raw.get("data") if isinstance(raw, dict) else raw

    if not isinstance(data, dict):
        if isinstance(data, list):
            return pd.DataFrame(data)
        return pd.DataFrame([{"raw": raw}])

    rows = []

    for node_id, node in (data.get("nodes", {}) or {}).items():
        row = {"result_type": "node", "id": node_id}

        if isinstance(node, dict):
            row.update({
                "label": node.get("label"),
                "kind": node.get("kind"),
                "kinds": ",".join(node.get("kinds", []))
                if isinstance(node.get("kinds"), list)
                else node.get("kinds"),
                "objectId": node.get("objectId"),
                "isTierZero": node.get("isTierZero"),
                "isOwnedObject": node.get("isOwnedObject"),
                "lastSeen": node.get("lastSeen"),
            })

            for key, value in node.items():
                if key not in row:
                    row[key] = flatten_value(value)
        else:
            row["raw"] = flatten_value(node)

        rows.append(row)

    for index, edge in enumerate(data.get("edges", []) or []):
        row = {"result_type": "edge", "edge_index": index}

        if isinstance(edge, dict):
            for key, value in edge.items():
                row[key] = flatten_value(value)
        else:
            row["edge"] = flatten_value(edge)

        rows.append(row)

    literals = data.get("literals", []) or []
    if literals:
        # BHE returns tabular Cypher values as an ordered sequence of
        # {"key": <column>, "value": <cell>} objects. Rebuild rows whenever
        # a column key repeats instead of flattening the entire result into
        # col_1, col_2, ... col_n.
        if all(
            isinstance(literal, dict)
            and "key" in literal
            and "value" in literal
            for literal in literals
        ):
            current_row = {"result_type": "literal"}

            for literal in literals:
                key = str(literal.get("key") or "value")
                value = flatten_value(literal.get("value"))

                if key in current_row:
                    if len(current_row) > 1:
                        rows.append(current_row)
                    current_row = {"result_type": "literal"}

                current_row[key] = value

            if len(current_row) > 1:
                rows.append(current_row)

        else:
            # Preserve compatibility with scalar, list, and nested literal
            # payloads that do not use BHE's key/value tabular structure.
            for literal in literals:
                if isinstance(literal, dict):
                    if isinstance(literal.get("values"), list):
                        for value in literal["values"]:
                            if isinstance(value, dict):
                                rows.append({
                                    "result_type": "literal",
                                    **{
                                        key: flatten_value(item)
                                        for key, item in value.items()
                                    },
                                })
                            else:
                                rows.append({
                                    "result_type": "literal",
                                    "value": flatten_value(value),
                                })
                    elif isinstance(literal.get("value"), list):
                        for value in literal["value"]:
                            if isinstance(value, dict):
                                rows.append({
                                    "result_type": "literal",
                                    **{
                                        key: flatten_value(item)
                                        for key, item in value.items()
                                    },
                                })
                            else:
                                rows.append({
                                    "result_type": "literal",
                                    "value": flatten_value(value),
                                })
                    elif "value" in literal:
                        rows.append({
                            "result_type": "literal",
                            str(literal.get("key") or "value"):
                                flatten_value(literal.get("value")),
                        })
                    else:
                        rows.append({
                            "result_type": "literal",
                            **{
                                key: flatten_value(value)
                                for key, value in literal.items()
                            },
                        })
                else:
                    rows.append({
                        "result_type": "literal",
                        "value": flatten_value(literal),
                    })

    return pd.DataFrame(rows)


class BHEClient:
    def __init__(
        self,
        base_url: str,
        token_id: str,
        token_key: str,
        endpoint: str = "/api/v2/graphs/cypher",
        verify_ssl: bool = True,
        timeout: int = 600,
        prefer_wait: int = 120,
    ):
        self.base_url = base_url.rstrip("/")
        self.token_id = token_id
        self.token_key = token_key
        self.endpoint = endpoint
        self.verify_ssl = verify_ssl
        self.timeout = timeout
        self.prefer_wait = prefer_wait

    def _request_date(self) -> str:
        return datetime.datetime.now(
            datetime.timezone.utc
        ).isoformat().replace("+00:00", "Z")

    def _signature(
        self,
        method: str,
        uri: str,
        request_date: str,
        body: bytes = b"",
    ) -> str:
        digester = hmac.new(
            self.token_key.encode("utf-8"),
            None,
            hashlib.sha256,
        )
        digester.update(f"{method}{uri}".encode("utf-8"))

        digester = hmac.new(
            digester.digest(),
            None,
            hashlib.sha256,
        )
        digester.update(request_date[:13].encode("utf-8"))

        digester = hmac.new(
            digester.digest(),
            None,
            hashlib.sha256,
        )

        if body:
            digester.update(body)

        return base64.b64encode(
            digester.digest()
        ).decode("utf-8")

    def _headers(
        self,
        method: str,
        uri: str,
        body: bytes = b"",
    ) -> Dict[str, str]:
        request_date = self._request_date()

        return {
            "Authorization": f"bhesignature {self.token_id}",
            "RequestDate": request_date,
            "Signature": self._signature(
                method,
                uri,
                request_date,
                body,
            ),
            "Content-Type": "application/json",
            "Accept": "application/json",
            "Prefer": f"wait={self.prefer_wait}",
        }

    def raw(
        self,
        query: str,
        include_properties: bool = True,
    ) -> Any:
        body = json.dumps(
            {
                "query": query,
                "include_properties": include_properties,
            },
            separators=(",", ":"),
        ).encode("utf-8")

        response = requests.post(
            self.base_url + self.endpoint,
            headers=self._headers(
                "POST",
                self.endpoint,
                body,
            ),
            data=body,
            verify=self.verify_ssl,
            timeout=self.timeout,
        )

        try:
            response.raise_for_status()
        except requests.HTTPError as error:
            raise RuntimeError(
                f"BHE API request failed: HTTP {response.status_code}. "
                f"Response: {response.text}"
            ) from error

        return response.json()

    def run(
        self,
        query: str,
        include_properties: bool = True,
    ) -> pd.DataFrame:
        return bhe_raw_to_dataframe(
            self.raw(
                query,
                include_properties=include_properties,
            )
        )


bh = BHEClient(
    base_url=tenant_url,
    token_id=token_id,
    token_key=token_key,
    endpoint=query_endpoint,
    verify_ssl=verify_ssl,
    timeout=timeout_seconds,
    prefer_wait=prefer_wait_seconds,
)

display(Markdown("BHE client initialized."))

In [ ]:
# ---------------------------------------------------------------------------
# Validated posture metrics pipeline — report integration
# ---------------------------------------------------------------------------
# This is the same core-versus-connected collection logic proven in the
# standalone posture validator. The report consumes the final DataFrames and
# does not independently recalculate posture values during HTML rendering.

from urllib.parse import urlencode
import time

PLATFORM_LABELS = {
    "active-directory": "Active Directory",
    "azure": "Azure / Entra ID",
    "github": "GitHub",
    "okta": "Okta",
    "jamf": "Jamf",
}
SUPPORTED_ATTACK_PATH_PLATFORMS = set(PLATFORM_LABELS)
CORE_PLATFORM_KEYS = {"azure", "active-directory"}
CONNECTED_PLATFORM_KEYS = {"github", "okta", "jamf"}


def bhe_signed_get(path: str, params: Dict[str, Any] | None = None) -> Dict[str, Any]:
    params = params or {}
    query_items = []
    for key, value in params.items():
        if value is None:
            continue
        if isinstance(value, (list, tuple)):
            query_items.extend((key, item) for item in value)
        else:
            query_items.append((key, value))

    query = urlencode(query_items, doseq=True)
    uri = path + (f"?{query}" if query else "")
    started = time.perf_counter()
    try:
        response = requests.get(
            tenant_url + uri,
            headers=bh._headers("GET", uri),
            verify=verify_ssl,
            timeout=min(timeout_seconds, 180),
        )
        elapsed = round(time.perf_counter() - started, 3)
        try:
            payload = response.json()
        except Exception:
            payload = {"raw_text": response.text[:5000]}
        return {
            "ok": response.ok,
            "status_code": response.status_code,
            "elapsed_seconds": elapsed,
            "payload": payload,
            "error": "" if response.ok else response.text[:1000],
        }
    except Exception as exc:
        return {
            "ok": False,
            "status_code": 0,
            "elapsed_seconds": round(time.perf_counter() - started, 3),
            "payload": {},
            "error": str(exc),
        }


def rfc3339(value: datetime.datetime) -> str:
    return value.astimezone(datetime.timezone.utc).isoformat().replace("+00:00", "Z")


def attack_path_platform_key(value: Any) -> str:
    text = re.sub(r"\s+", " ", str(value or "").strip()).lower()
    compact = re.sub(r"[^a-z0-9]+", "", text)
    if "github" in compact:
        return "github"
    if "okta" in compact:
        return "okta"
    if "jamf" in compact:
        return "jamf"
    if "azure" in compact or "entra" in compact:
        return "azure"
    if compact in {"ad", "activedirectory"} or ("active" in compact and "directory" in compact):
        return "active-directory"
    return text.replace("_", "-").replace(" ", "-")


def attack_path_platform_label(value: Any) -> str:
    key = attack_path_platform_key(value)
    return PLATFORM_LABELS.get(key, str(value or "Other").replace("_", " ").strip().title())


def latest_attack_path_point(environment_id: str, asset_group_tag_id: int):
    result = bhe_signed_get(
        "/api/v2/posture-history/attack-paths",
        params={
            "environments": [environment_id],
            "asset_group_tag_id": asset_group_tag_id,
            "start": rfc3339(attack_path_start),
            "end": rfc3339(attack_path_end),
        },
    )
    if not result["ok"]:
        return result, None

    points = [p for p in (result.get("payload", {}).get("data", []) or []) if isinstance(p, dict)]
    if not points:
        return result, None

    frame = pd.DataFrame(points)
    frame["_date"] = pd.to_datetime(frame.get("date"), errors="coerce", utc=True)
    frame["_value"] = pd.to_numeric(frame.get("value"), errors="coerce")
    frame = frame.dropna(subset=["_value"]).sort_values("_date")
    if frame.empty:
        return result, None

    latest = frame.iloc[-1]
    return result, {
        "Snapshot Date": str(latest.get("date") or ""),
        "Attack Paths": int(latest["_value"]),
    }


attack_path_window_days = int(report_cfg.get("attack_path_window_days", 30))
attack_path_end = datetime.datetime.now(datetime.timezone.utc).replace(microsecond=0)
attack_path_start = attack_path_end - datetime.timedelta(days=attack_path_window_days)

available_result = bhe_signed_get("/api/v2/available-domains")
tag_result = bhe_signed_get("/api/v2/asset-group-tags")
failed_calls = []

if not available_result["ok"]:
    failed_calls.append({
        "Stage": "Environment inventory", "Environment": "All",
        "HTTP Status": available_result["status_code"], "Error": available_result["error"],
    })
if not tag_result["ok"]:
    failed_calls.append({
        "Stage": "Asset group tag inventory", "Environment": "All",
        "HTTP Status": tag_result["status_code"], "Error": tag_result["error"],
    })

environment_rows = []
if available_result["ok"]:
    for env in available_result.get("payload", {}).get("data", []) or []:
        if not isinstance(env, dict):
            continue
        key = attack_path_platform_key(env.get("type"))
        environment_rows.append({
            "Environment": str(env.get("name") or env.get("id") or "Unknown"),
            "Environment ID": str(env.get("id") or ""),
            "Platform": attack_path_platform_label(key),
            "Platform Key": key,
            "Collected": bool(env.get("collected")),
        })

COLLECTED_ENVIRONMENT_SCOPE_DF = pd.DataFrame(environment_rows)
if not COLLECTED_ENVIRONMENT_SCOPE_DF.empty:
    COLLECTED_ENVIRONMENT_SCOPE_DF = COLLECTED_ENVIRONMENT_SCOPE_DF.sort_values(
        ["Collected", "Platform", "Environment"], ascending=[False, True, True], kind="stable"
    ).reset_index(drop=True)

tag_rows = []
if tag_result["ok"]:
    tag_payload = tag_result.get("payload", {}).get("data", {}) or {}
    items = tag_payload.get("tags", []) if isinstance(tag_payload, dict) else tag_payload
    for tag in items or []:
        if isinstance(tag, dict):
            tag_rows.append({
                "Tag ID": tag.get("id"),
                "Tag Name": tag.get("name"),
                "Type": tag.get("type") or tag.get("kind") or "",
            })

ASSET_GROUP_TAG_INVENTORY_DF = pd.DataFrame(tag_rows)
tier_zero_matches = [
    row for row in tag_rows
    if str(row.get("Tag Name") or "").strip().casefold() == "tier zero"
]
if len(tier_zero_matches) != 1:
    raise RuntimeError(f"Expected exactly one Asset Group Tag named 'Tier Zero'; found {len(tier_zero_matches)}.")
TIER_ZERO_ASSET_GROUP_TAG_ID = int(tier_zero_matches[0]["Tag ID"])

collected_rows = [
    row for row in environment_rows
    if row["Collected"] and row["Environment ID"]
    and row["Platform Key"] in SUPPORTED_ATTACK_PATH_PLATFORMS
]

metric_rows = []
for env in sorted(collected_rows, key=lambda row: (row["Platform Key"], row["Environment"])):
    response, point = latest_attack_path_point(env["Environment ID"], TIER_ZERO_ASSET_GROUP_TAG_ID)
    if point is None:
        failed_calls.append({
            "Stage": "Attack path collection",
            "Environment": f'{env["Platform"]} • {env["Environment"]}',
            "HTTP Status": response.get("status_code", 0),
            "Error": response.get("error") or "No posture-history data returned",
        })
        continue
    metric_rows.append({
        "Platform": env["Platform"],
        "Platform Key": env["Platform Key"],
        "Environment": env["Environment"],
        "Management Label": f'{env["Platform"]} • {env["Environment"]}',
        "Attack Paths": int(point["Attack Paths"]),
        "Snapshot Date": point["Snapshot Date"],
    })

REPORT_READY_ENVIRONMENT_TOTALS_DF = pd.DataFrame(metric_rows)
if REPORT_READY_ENVIRONMENT_TOTALS_DF.empty:
    raise RuntimeError("No validated Tier Zero attack-path values were returned.")

REPORT_READY_ENVIRONMENT_TOTALS_DF = REPORT_READY_ENVIRONMENT_TOTALS_DF.sort_values(
    ["Attack Paths", "Platform", "Environment"], ascending=[False, True, True], kind="stable"
).reset_index(drop=True)
REPORT_READY_PLATFORM_TOTALS_DF = (
    REPORT_READY_ENVIRONMENT_TOTALS_DF.groupby(["Platform", "Platform Key"], as_index=False)["Attack Paths"]
    .sum()
    .sort_values(["Attack Paths", "Platform"], ascending=[False, True], kind="stable")
    .reset_index(drop=True)
)

CORE_ENVIRONMENT_TOTALS_DF = REPORT_READY_ENVIRONMENT_TOTALS_DF[
    REPORT_READY_ENVIRONMENT_TOTALS_DF["Platform Key"].isin(CORE_PLATFORM_KEYS)
].copy()
CONNECTED_ENVIRONMENT_TOTALS_DF = REPORT_READY_ENVIRONMENT_TOTALS_DF[
    REPORT_READY_ENVIRONMENT_TOTALS_DF["Platform Key"].isin(CONNECTED_PLATFORM_KEYS)
].copy()
CONNECTED_MANAGEMENT_ENVIRONMENT_TOTALS_DF = CONNECTED_ENVIRONMENT_TOTALS_DF[
    CONNECTED_ENVIRONMENT_TOTALS_DF["Attack Paths"] > 0
].copy()
CORE_PLATFORM_TOTALS_DF = REPORT_READY_PLATFORM_TOTALS_DF[
    REPORT_READY_PLATFORM_TOTALS_DF["Platform Key"].isin(CORE_PLATFORM_KEYS)
].copy()
CONNECTED_PLATFORM_TOTALS_DF = REPORT_READY_PLATFORM_TOTALS_DF[
    REPORT_READY_PLATFORM_TOTALS_DF["Platform Key"].isin(CONNECTED_PLATFORM_KEYS)
].copy()

TOTAL_ENTERPRISE_TIER_ZERO_ATTACK_PATHS = int(CORE_ENVIRONMENT_TOTALS_DF["Attack Paths"].sum())
TOTAL_ENTERPRISE_ATTACK_PATHS = TOTAL_ENTERPRISE_TIER_ZERO_ATTACK_PATHS
CONNECTED_PLATFORM_ATTACK_PATHS = int(CONNECTED_ENVIRONMENT_TOTALS_DF["Attack Paths"].sum())
ALL_COLLECTED_PLATFORM_ATTACK_PATHS = int(REPORT_READY_ENVIRONMENT_TOTALS_DF["Attack Paths"].sum())

ENTERPRISE_ATTACK_PATH_SUMMARY_DF = pd.DataFrame([{
    "Metric": "Enterprise Tier Zero Attack Paths",
    "Attack Paths": TOTAL_ENTERPRISE_TIER_ZERO_ATTACK_PATHS,
    "Definition": "Azure / Entra ID + Active Directory",
    "Tier Zero Asset Group Tag ID": TIER_ZERO_ASSET_GROUP_TAG_ID,
    "Validated Core Environments": len(CORE_ENVIRONMENT_TOTALS_DF),
    "Window Days": attack_path_window_days,
}])
FAILED_ENVIRONMENT_CALLS_DF = pd.DataFrame(failed_calls)

# Compatibility aliases consumed by the existing report renderer. These now
# point only to the BHE UI-aligned core metrics.
ATTACK_PATH_PLATFORM_DF = CORE_PLATFORM_TOTALS_DF.copy()
ATTACK_PATH_ENVIRONMENT_DF = CORE_ENVIRONMENT_TOTALS_DF.copy()
CONNECTED_ATTACK_PATH_PLATFORM_DF = CONNECTED_PLATFORM_TOTALS_DF.copy()
CONNECTED_ATTACK_PATH_ENVIRONMENT_DF = CONNECTED_MANAGEMENT_ENVIRONMENT_TOTALS_DF.copy()
ATTACK_PATH_METRICS_AVAILABLE = not ATTACK_PATH_ENVIRONMENT_DF.empty
ATTACK_PATH_COLLECTION_ERROR = "; ".join(
    f'{row.get("Environment")}: {row.get("Error")}' for row in failed_calls
)

snapshot_dates = pd.to_datetime(REPORT_READY_ENVIRONMENT_TOTALS_DF["Snapshot Date"], errors="coerce", utc=True)
ATTACK_PATH_SNAPSHOT_DATE = ""
if snapshot_dates.notna().any():
    ATTACK_PATH_SNAPSHOT_DATE = str(
        REPORT_READY_ENVIRONMENT_TOTALS_DF.loc[snapshot_dates.idxmax(), "Snapshot Date"] or ""
    )

core_platform_sum = int(ATTACK_PATH_PLATFORM_DF["Attack Paths"].sum())
core_environment_sum = int(ATTACK_PATH_ENVIRONMENT_DF["Attack Paths"].sum())
if not (TOTAL_ENTERPRISE_TIER_ZERO_ATTACK_PATHS == core_platform_sum == core_environment_sum):
    raise RuntimeError("Validated core posture metrics did not reconcile before report rendering.")

connected_platform_sum = int(CONNECTED_ATTACK_PATH_PLATFORM_DF["Attack Paths"].sum())
connected_environment_sum = int(CONNECTED_ENVIRONMENT_TOTALS_DF["Attack Paths"].sum())
if not (CONNECTED_PLATFORM_ATTACK_PATHS == connected_platform_sum == connected_environment_sum):
    raise RuntimeError("Validated connected-platform posture metrics did not reconcile before report rendering.")

display(Markdown(
    f"**Enterprise Tier Zero Attack Paths:** {TOTAL_ENTERPRISE_TIER_ZERO_ATTACK_PATHS:,}  "
    f"  \n**Definition:** Azure / Entra ID + Active Directory  "
    f"  \n**Connected Platform Exposure:** {CONNECTED_PLATFORM_ATTACK_PATHS:,}  "
    f"  \n**Validated core environments:** {len(ATTACK_PATH_ENVIRONMENT_DF)}"
))
display(ATTACK_PATH_PLATFORM_DF)
display(ATTACK_PATH_ENVIRONMENT_DF)
if not CONNECTED_ATTACK_PATH_PLATFORM_DF.empty:
    display(CONNECTED_ATTACK_PATH_PLATFORM_DF)

In [ ]:
# ---------------------------------------------------------------------------
# Connection test
# ---------------------------------------------------------------------------

TEST_QUERY = """
MATCH (n)
RETURN n.name AS name, labels(n) AS labels
LIMIT 5
"""

test_df = bh.run(TEST_QUERY)

display(Markdown("## Connection test succeeded"))
display(test_df)

## 4A. Privilege Zone Intelligence Integration

Integrated from the Privilege Zone Intelligence Engine v2.4 reference implementation. This module discovers configured non-Tier-Zero Privilege Zones, resolves selectors and protected objects, builds normalized report datasets, and enables conditional PDF/XLSX reporting.

In [ ]:
# ---------------------------------------------------------------------------
# Analyzer configuration
# ---------------------------------------------------------------------------

from urllib.parse import urlencode
from collections import deque

VALIDATOR_VERSION = "2.1"
PZ_OUTPUT_DIR = OUTPUT_DIR / "privilege_zone_validation"
PZ_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Tier Zero remains available for context and parity checks, but the additive
# Privilege Zone Exposure report scope excludes it by default.
BASE_FINDING_ZONE_NAMES = {"tier zero", "tier 0", "t0"}

# Optional: add exact GET paths captured from the browser Network tab.
CUSTOM_DISCOVERY_PATHS = []

# Optional zone-specific exposure templates. Use {id} for the zone/tag ID.
# These are read-only GET probes and are useful when an endpoint is visible in
# the browser but not published in the OpenAPI document.
CUSTOM_EXPOSURE_PATH_TEMPLATES = []

OPENAPI_CANDIDATES = [
    "/openapi.json",
    "/swagger.json",
    "/api/openapi.json",
    "/api/swagger.json",
    "/api/v2/openapi.json",
    "/api/v2/swagger.json",
    "/swagger/v1/swagger.json",
]

ZONE_ENDPOINT_CANDIDATES = [
    "/api/v2/asset-group-tags",
    "/api/v2/privilege-zones",
    "/api/v2/privilege-zones/zones",
    "/api/v2/zones",
    "/api/v2/zone-builder/zones",
    "/api/v2/asset-groups",
    "/api/v2/asset-group-tags/rules",
    "/api/v2/privilege-zones/rules",
    "/api/v2/zones/rules",
    "/api/v2/privilege-zones/certifications",
    "/api/v2/zones/certifications",
    "/api/v2/privilege-zones/history",
    "/api/v2/zones/history",
    "/api/v2/privilege-zones/summary",
    "/api/v2/privilege-zones/statistics",
    "/api/v2/privilege-zones/exposure",
    "/api/v2/privilege-zones/attack-paths",
    "/api/v2/asset-group-tags/summary",
    "/api/v2/asset-group-tags/statistics",
    "/api/v2/asset-group-tags/exposure",
    "/api/v2/asset-group-tags/attack-paths",
]

print(f"Privilege Zone Analyzer v{VALIDATOR_VERSION}")
print(f"Output directory: {PZ_OUTPUT_DIR}")

# Set True only when troubleshooting API discovery or a BHE version change.
PZ_DEBUG_EXPORTS = False


In [ ]:
# ---------------------------------------------------------------------------
# Generic signed GET and payload inspection helpers
# ---------------------------------------------------------------------------

# Raw response capture is initialized before any endpoint calls execute.
# Each JSON response is split into Excel-safe chunks so the workbook preserves
# the exact decoded payload without exceeding Excel's cell-length limit.
RAW_API_PAYLOADS = []
RAW_API_CAPTURE_CHUNK_SIZE = 30000


def _capture_raw_api_payload(path: str, status_code: int, ok: bool,
                             elapsed_seconds: float, payload: Any,
                             content_type: str = "", error: str = "") -> None:
    if not (
        "/asset-group-tags/" in path
        and any(token in path for token in ("/members/counts", "/selectors"))
    ):
        return

    try:
        raw_json = json.dumps(payload, ensure_ascii=False, indent=2, default=str)
    except Exception:
        raw_json = repr(payload)

    if isinstance(payload, dict):
        payload_type = "dict"
        top_level_keys = ", ".join(map(str, payload.keys()))
    elif isinstance(payload, list):
        payload_type = "list"
        top_level_keys = ""
    else:
        payload_type = type(payload).__name__
        top_level_keys = ""

    chunks = [
        raw_json[index:index + RAW_API_CAPTURE_CHUNK_SIZE]
        for index in range(0, max(len(raw_json), 1), RAW_API_CAPTURE_CHUNK_SIZE)
    ] or [""]

    for chunk_number, chunk in enumerate(chunks, start=1):
        RAW_API_PAYLOADS.append({
            "Endpoint": path,
            "HTTP Status": status_code,
            "Succeeded": bool(ok),
            "Elapsed Seconds": elapsed_seconds,
            "Content Type": content_type,
            "Payload Type": payload_type,
            "Top Level Keys": top_level_keys,
            "Chunk": chunk_number,
            "Total Chunks": len(chunks),
            "Raw JSON": chunk,
            "Error": error,
        })


def bhe_signed_get(path: str, params: Dict[str, Any] | None = None) -> Dict[str, Any]:
    params = params or {}
    query_items = []
    for key, value in params.items():
        if value is None:
            continue
        if isinstance(value, (list, tuple)):
            query_items.extend((key, item) for item in value)
        else:
            query_items.append((key, value))

    query = urlencode(query_items, doseq=True)
    uri = path + (f"?{query}" if query else "")
    started = time.perf_counter()

    try:
        response = requests.get(
            tenant_url + uri,
            headers=bh._headers("GET", uri),
            verify=verify_ssl,
            timeout=min(timeout_seconds, 180),
        )
        elapsed = round(time.perf_counter() - started, 3)
        content_type = response.headers.get("content-type", "")
        try:
            payload = response.json()
        except Exception:
            payload = {"raw_text": response.text[:10000]}

        error_text = "" if response.ok else response.text[:2000]
        _capture_raw_api_payload(
            uri,
            response.status_code,
            response.ok,
            elapsed,
            payload,
            content_type=content_type,
            error=error_text,
        )

        return {
            "ok": response.ok,
            "status_code": response.status_code,
            "elapsed_seconds": elapsed,
            "content_type": content_type,
            "payload": payload,
            "error": error_text,
        }
    except Exception as exc:
        return {
            "ok": False,
            "status_code": 0,
            "elapsed_seconds": round(time.perf_counter() - started, 3),
            "content_type": "",
            "payload": {},
            "error": str(exc),
        }


def payload_shape(value: Any) -> str:
    if isinstance(value, dict):
        keys = list(value.keys())[:12]
        return "dict: " + ", ".join(map(str, keys))
    if isinstance(value, list):
        return f"list[{len(value)}]"
    return type(value).__name__


def walk_json(value: Any, max_depth: int = 8):
    queue = deque([((), value, 0)])
    while queue:
        path, current, depth = queue.popleft()
        yield path, current
        if depth >= max_depth:
            continue
        if isinstance(current, dict):
            for key, child in current.items():
                queue.append((path + (str(key),), child, depth + 1))
        elif isinstance(current, list):
            for index, child in enumerate(current[:2000]):
                queue.append((path + (str(index),), child, depth + 1))


def first_list_of_dicts(payload: Any) -> list[dict]:
    candidates = []
    for path, value in walk_json(payload):
        if isinstance(value, list) and value and all(isinstance(item, dict) for item in value[:20]):
            candidates.append((len(path), -len(value), path, value))
    if not candidates:
        return []
    candidates.sort(key=lambda item: (item[0], item[1]))
    return candidates[0][3]


def safe_slug(value: Any) -> str:
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", str(value or "unknown")).strip("_")
    return text[:120] or "unknown"


In [ ]:
# ---------------------------------------------------------------------------
# OpenAPI / Swagger discovery
# ---------------------------------------------------------------------------

openapi_probe_rows = []
discovered_zone_paths = set()
OPENAPI_PAYLOADS = {}

for path in OPENAPI_CANDIDATES:
    result = bhe_signed_get(path)
    openapi_probe_rows.append({
        "Path": path,
        "HTTP Status": result["status_code"],
        "Succeeded": result["ok"],
        "Seconds": result["elapsed_seconds"],
        "Payload Shape": payload_shape(result.get("payload")),
        "Error": result.get("error", "")[:300],
    })
    if not result["ok"]:
        continue

    payload = result.get("payload", {})
    OPENAPI_PAYLOADS[path] = payload
    paths = payload.get("paths", {}) if isinstance(payload, dict) else {}
    if isinstance(paths, dict):
        for api_path, definition in paths.items():
            combined = f"{api_path} {json.dumps(definition, default=str)}".lower()
            if any(term in combined for term in [
                "privilege zone", "privilege-zone", "privilege_zone",
                "zone builder", "asset group tag", "asset-group-tag",
                "certification",
            ]):
                discovered_zone_paths.add(api_path)

OPENAPI_PROBE_DF = pd.DataFrame(openapi_probe_rows)
DISCOVERED_ZONE_PATHS_DF = pd.DataFrame({"Discovered API Path": sorted(discovered_zone_paths)})

display(Markdown("### API documentation probes"))
display(OPENAPI_PROBE_DF)

display(Markdown(f"### Zone-related paths discovered: {len(discovered_zone_paths)}"))
if DISCOVERED_ZONE_PATHS_DF.empty:
    display(Markdown("No readable OpenAPI document exposed zone-related paths. Endpoint probing will continue."))
else:
    display(DISCOVERED_ZONE_PATHS_DF)


In [ ]:
# ---------------------------------------------------------------------------
# Endpoint probing
# ---------------------------------------------------------------------------

probe_paths = []
for candidate in [*ZONE_ENDPOINT_CANDIDATES, *sorted(discovered_zone_paths), *CUSTOM_DISCOVERY_PATHS]:
    if not candidate or "{" in candidate or candidate in probe_paths:
        continue
    probe_paths.append(candidate)

endpoint_rows = []
SUCCESSFUL_PAYLOADS = {}

for path in probe_paths:
    result = bhe_signed_get(path)
    payload = result.get("payload", {})
    endpoint_rows.append({
        "Path": path,
        "HTTP Status": result["status_code"],
        "Succeeded": result["ok"],
        "Seconds": result["elapsed_seconds"],
        "Payload Shape": payload_shape(payload),
        "Top-Level Records": len(first_list_of_dicts(payload)),
        "Error": result.get("error", "")[:500],
    })
    if result["ok"]:
        SUCCESSFUL_PAYLOADS[path] = payload

PRIVILEGE_ZONE_ENDPOINT_PROBE_DF = pd.DataFrame(endpoint_rows)
if not PRIVILEGE_ZONE_ENDPOINT_PROBE_DF.empty:
    PRIVILEGE_ZONE_ENDPOINT_PROBE_DF = PRIVILEGE_ZONE_ENDPOINT_PROBE_DF.sort_values(
        ["Succeeded", "HTTP Status", "Path"], ascending=[False, True, True], kind="stable"
    ).reset_index(drop=True)

display(PRIVILEGE_ZONE_ENDPOINT_PROBE_DF)


In [ ]:
# ---------------------------------------------------------------------------
# Generic zone inventory extraction
# ---------------------------------------------------------------------------

ZONE_TERMS = {"zone", "tier zero", "tier one", "code signing", "hygiene"}
NAME_KEYS = ["name", "label", "title", "display_name", "displayName"]
ID_KEYS = ["id", "zone_id", "zoneId", "tag_id", "tagId", "asset_group_tag_id", "assetGroupTagId"]
RULE_COUNT_KEYS = ["rule_count", "ruleCount", "rules_count", "rulesCount"]
OBJECT_COUNT_KEYS = ["object_count", "objectCount", "objects_count", "objectsCount", "member_count", "memberCount"]


def first_present(record: dict, keys: list[str]):
    for key in keys:
        if key in record and record[key] not in [None, ""]:
            return record[key]
    return None


def record_looks_zone_like(record: dict, source_path: str) -> bool:
    name = str(first_present(record, NAME_KEYS) or "").strip().lower()
    keys = " ".join(str(key).lower() for key in record.keys())
    source = source_path.lower()
    return (
        any(term in name for term in ZONE_TERMS)
        or "zone" in keys
        or "asset_group_tag" in keys
        or "asset-group-tag" in source
        or "privilege-zone" in source
        or "/zones" in source
    )

zone_rows = []
seen_zone_rows = set()

for source_path, payload in SUCCESSFUL_PAYLOADS.items():
    for json_path, value in walk_json(payload):
        if not isinstance(value, dict) or not record_looks_zone_like(value, source_path):
            continue

        name = first_present(value, NAME_KEYS)
        zone_id = first_present(value, ID_KEYS)
        if name is None and zone_id is None:
            continue

        row_key = (source_path, str(zone_id), str(name), "/".join(json_path))
        if row_key in seen_zone_rows:
            continue
        seen_zone_rows.add(row_key)

        rules_value = value.get("rules")
        objects_value = value.get("objects") or value.get("members")
        rule_count = first_present(value, RULE_COUNT_KEYS)
        object_count = first_present(value, OBJECT_COUNT_KEYS)
        if rule_count is None and isinstance(rules_value, list):
            rule_count = len(rules_value)
        if object_count is None and isinstance(objects_value, list):
            object_count = len(objects_value)

        zone_rows.append({
            "Zone Name": name,
            "Zone ID": zone_id,
            "Rules": rule_count,
            "Objects": object_count,
            "Type": value.get("type") or value.get("kind") or value.get("category"),
            "Enabled": value.get("enabled"),
            "Source Endpoint": source_path,
            "JSON Path": "/".join(json_path),
            "Available Fields": ", ".join(sorted(map(str, value.keys()))),
        })

PRIVILEGE_ZONE_INVENTORY_DF = pd.DataFrame(zone_rows)
if not PRIVILEGE_ZONE_INVENTORY_DF.empty:
    PRIVILEGE_ZONE_INVENTORY_DF = PRIVILEGE_ZONE_INVENTORY_DF.sort_values(
        ["Zone Name", "Source Endpoint", "JSON Path"], kind="stable", na_position="last"
    ).reset_index(drop=True)

display(Markdown(f"### Candidate zone records: {len(PRIVILEGE_ZONE_INVENTORY_DF)}"))
if PRIVILEGE_ZONE_INVENTORY_DF.empty:
    display(Markdown(
        "No zone inventory was automatically extracted. Add exact browser-captured GET paths "
        "to `CUSTOM_DISCOVERY_PATHS`, rerun from the validator configuration cell, and execute again."
    ))
else:
    display(PRIVILEGE_ZONE_INVENTORY_DF)


In [ ]:
# ---------------------------------------------------------------------------
# Detail endpoint discovery using candidate zone IDs
# ---------------------------------------------------------------------------

DETAIL_TEMPLATES = [
    "/api/v2/privilege-zones/{id}",
    "/api/v2/privilege-zones/{id}/rules",
    "/api/v2/privilege-zones/{id}/objects",
    "/api/v2/privilege-zones/{id}/members",
    "/api/v2/zones/{id}",
    "/api/v2/zones/{id}/rules",
    "/api/v2/zones/{id}/objects",
    "/api/v2/zones/{id}/members",
    "/api/v2/asset-group-tags/{id}",
    "/api/v2/asset-group-tags/{id}/rules",
    "/api/v2/asset-group-tags/{id}/members",
    "/api/v2/asset-group-tags/{id}/members/counts",
    "/api/v2/asset-group-tags/{id}/selectors",
    "/api/v2/asset-group-tags/{id}/objects",
    "/api/v2/asset-group-tags/{id}/summary",
    "/api/v2/asset-group-tags/{id}/statistics",
    "/api/v2/asset-group-tags/{id}/exposure",
    "/api/v2/asset-group-tags/{id}/attack-paths",
    "/api/v2/asset-group-tags/{id}/analysis",
    "/api/v2/privilege-zones/{id}/summary",
    "/api/v2/privilege-zones/{id}/statistics",
    "/api/v2/privilege-zones/{id}/exposure",
    "/api/v2/privilege-zones/{id}/attack-paths",
    "/api/v2/privilege-zones/{id}/analysis",
    "/api/v2/zones/{id}/summary",
    "/api/v2/zones/{id}/statistics",
    "/api/v2/zones/{id}/exposure",
    "/api/v2/zones/{id}/attack-paths",
    "/api/v2/zones/{id}/analysis",
]

# Collection hardening: the current /asset-group-tags inventory is the only
# authoritative source of IDs used for zone-specific endpoint calls. Historical,
# nested, and unrelated numeric IDs are explicitly excluded.
zone_ids = []
VALIDATED_ZONE_INVENTORY_DF = pd.DataFrame()
if not PRIVILEGE_ZONE_INVENTORY_DF.empty and "Zone ID" in PRIVILEGE_ZONE_INVENTORY_DF:
    candidates = PRIVILEGE_ZONE_INVENTORY_DF.copy()
    if "Source Endpoint" in candidates.columns:
        candidates = candidates[candidates["Source Endpoint"].eq("/api/v2/asset-group-tags")]
    if "Type" in candidates.columns:
        tag_type = pd.to_numeric(candidates["Type"], errors="coerce")
        candidates = candidates[tag_type.eq(1)]
    candidates = candidates[candidates["Zone ID"].notna()].copy()
    candidates["Zone ID"] = candidates["Zone ID"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
    candidates = candidates[candidates["Zone ID"].ne("")]
    VALIDATED_ZONE_INVENTORY_DF = candidates.drop_duplicates("Zone ID", keep="first").reset_index(drop=True)
    zone_ids = VALIDATED_ZONE_INVENTORY_DF["Zone ID"].tolist()

# Keep discovery bounded while covering normal customer configurations.
zone_ids = zone_ids[:50]
print(f"Validated Privilege Zone IDs selected for detail collection: {zone_ids}")

detail_rows = []
DETAIL_PAYLOADS = {}
for zone_id in zone_ids:
    for template in DETAIL_TEMPLATES:
        path = template.format(id=zone_id)
        result = bhe_signed_get(path)
        detail_rows.append({
            "Zone ID": zone_id,
            "Path": path,
            "HTTP Status": result["status_code"],
            "Succeeded": result["ok"],
            "Seconds": result["elapsed_seconds"],
            "Payload Shape": payload_shape(result.get("payload")),
            "Records": len(first_list_of_dicts(result.get("payload"))),
            "Error": result.get("error", "")[:300],
        })
        if result["ok"]:
            DETAIL_PAYLOADS[path] = result.get("payload")

PRIVILEGE_ZONE_DETAIL_PROBE_DF = pd.DataFrame(detail_rows)
if not PRIVILEGE_ZONE_DETAIL_PROBE_DF.empty:
    PRIVILEGE_ZONE_DETAIL_PROBE_DF = PRIVILEGE_ZONE_DETAIL_PROBE_DF.sort_values(
        ["Succeeded", "HTTP Status", "Zone ID", "Path"],
        ascending=[False, True, True, True],
        kind="stable",
    ).reset_index(drop=True)

display(PRIVILEGE_ZONE_DETAIL_PROBE_DF)


In [ ]:
# ---------------------------------------------------------------------------
# v1.8 documented Privilege Zone endpoint collection
# ---------------------------------------------------------------------------
# These calls are part of the normal acquisition pipeline. They populate the
# payload registry before the analytical model is built.

def _payload_records(payload: Any) -> list[dict]:
    """Return the most useful list of dictionaries from common API envelopes."""
    if isinstance(payload, list):
        return [item for item in payload if isinstance(item, dict)]
    if not isinstance(payload, dict):
        return []
    for key in ["data", "items", "results", "selectors", "counts", "members"]:
        value = payload.get(key)
        if isinstance(value, list):
            return [item for item in value if isinstance(item, dict)]
        if isinstance(value, dict):
            for nested_key in ["items", "results", "selectors", "counts", "members"]:
                nested = value.get(nested_key)
                if isinstance(nested, list):
                    return [item for item in nested if isinstance(item, dict)]
    return first_list_of_dicts(payload)


def _first_value(record: dict, names: list[str], default=None):
    for name in names:
        if name in record and record[name] is not None:
            return record[name]
    return default


def _count_from_record(record: dict) -> int:
    value = _first_value(record, [
        "count", "total", "member_count", "memberCount", "object_count",
        "objectCount", "value"
    ], 0)
    try:
        return int(value or 0)
    except (TypeError, ValueError):
        return 0


def _selector_id(record: dict):
    value = _first_value(record, [
        "id", "selector_id", "selectorId", "asset_group_tag_selector_id",
        "assetGroupTagSelectorId"
    ])
    return None if value is None else str(value)


documented_trace_rows = []
zone_member_count_rows = []
selector_rows = []
selector_member_count_rows = []

for zone_id in zone_ids:
    # Zone selectors / rules.
    selectors_path = f"/api/v2/asset-group-tags/{zone_id}/selectors"
    selectors_result = bhe_signed_get(selectors_path)
    selectors_payload = selectors_result.get("payload")
    documented_trace_rows.append({
        "Zone ID": zone_id,
        "Endpoint Type": "Selectors",
        "Path": selectors_path,
        "HTTP Status": selectors_result.get("status_code"),
        "Succeeded": selectors_result.get("ok", False),
        "Records": len(_payload_records(selectors_payload)),
        "Payload Shape": payload_shape(selectors_payload),
        "Error": selectors_result.get("error", "")[:500],
    })
    if selectors_result.get("ok"):
        DETAIL_PAYLOADS[selectors_path] = selectors_payload

    selectors = _payload_records(selectors_payload) if selectors_result.get("ok") else []
    for ordinal, selector in enumerate(selectors, start=1):
        sid = _selector_id(selector)
        selector_rows.append({
            "Zone ID": str(zone_id),
            "Selector ID": sid,
            "Selector Ordinal": ordinal,
            "Selector Name": _first_value(selector, ["name", "title", "label"]),
            "Selector Type": _first_value(selector, ["type", "selector_type", "selectorType", "kind"]),
            "Environment ID": _first_value(selector, ["environment_id", "environmentId"]),
            "Selector Definition": flatten_value(_first_value(selector, [
                "expression", "query", "cypher", "definition", "selector", "conditions"
            ])),
            "Enabled": _first_value(selector, ["enabled", "is_enabled", "isEnabled"]),
            "Source Endpoint": selectors_path,
            "Raw Selector": json.dumps(selector, default=str),
        })

        if not sid:
            continue
        selector_counts_path = (
            f"/api/v2/asset-group-tags/{zone_id}/selectors/{sid}/members/counts"
        )
        selector_counts_result = bhe_signed_get(selector_counts_path)
        selector_counts_payload = selector_counts_result.get("payload")
        selector_count_records = _payload_records(selector_counts_payload)
        documented_trace_rows.append({
            "Zone ID": zone_id,
            "Endpoint Type": "Selector Member Counts",
            "Path": selector_counts_path,
            "HTTP Status": selector_counts_result.get("status_code"),
            "Succeeded": selector_counts_result.get("ok", False),
            "Records": len(selector_count_records),
            "Payload Shape": payload_shape(selector_counts_payload),
            "Error": selector_counts_result.get("error", "")[:500],
        })
        if selector_counts_result.get("ok"):
            DETAIL_PAYLOADS[selector_counts_path] = selector_counts_payload
        for count_record in selector_count_records:
            selector_member_count_rows.append({
                "Zone ID": str(zone_id),
                "Selector ID": sid,
                "Object Type": _first_value(count_record, [
                    "kind", "object_kind", "objectKind", "primary_kind", "primaryKind", "type"
                ], "Unknown"),
                "Count": _count_from_record(count_record),
                "Source Endpoint": selector_counts_path,
                "Raw Count": json.dumps(count_record, default=str),
            })

    # Zone-level member counts by object kind.
    member_counts_path = f"/api/v2/asset-group-tags/{zone_id}/members/counts"
    member_counts_result = bhe_signed_get(member_counts_path)
    member_counts_payload = member_counts_result.get("payload")
    member_count_records = _payload_records(member_counts_payload)
    documented_trace_rows.append({
        "Zone ID": zone_id,
        "Endpoint Type": "Zone Member Counts",
        "Path": member_counts_path,
        "HTTP Status": member_counts_result.get("status_code"),
        "Succeeded": member_counts_result.get("ok", False),
        "Records": len(member_count_records),
        "Payload Shape": payload_shape(member_counts_payload),
        "Error": member_counts_result.get("error", "")[:500],
    })
    if member_counts_result.get("ok"):
        DETAIL_PAYLOADS[member_counts_path] = member_counts_payload
    for count_record in member_count_records:
        zone_member_count_rows.append({
            "Zone ID": str(zone_id),
            "Object Type": _first_value(count_record, [
                "kind", "object_kind", "objectKind", "primary_kind", "primaryKind", "type"
            ], "Unknown"),
            "Count": _count_from_record(count_record),
            "Source Endpoint": member_counts_path,
            "Raw Count": json.dumps(count_record, default=str),
        })

PZ_DOCUMENTED_ENDPOINT_TRACE_DF = pd.DataFrame(documented_trace_rows, columns=[
    "Zone ID", "Endpoint Type", "Path", "HTTP Status", "Succeeded", "Records",
    "Payload Shape", "Error"
])
PZ_DOCUMENTED_SELECTORS_DF = pd.DataFrame(selector_rows, columns=[
    "Zone ID", "Selector ID", "Selector Ordinal", "Selector Name", "Selector Type",
    "Environment ID", "Selector Definition", "Enabled", "Source Endpoint", "Raw Selector"
])
PZ_ZONE_MEMBER_COUNTS_DF = pd.DataFrame(zone_member_count_rows, columns=[
    "Zone ID", "Object Type", "Count", "Source Endpoint", "Raw Count"
])
PZ_SELECTOR_MEMBER_COUNTS_DF = pd.DataFrame(selector_member_count_rows, columns=[
    "Zone ID", "Selector ID", "Object Type", "Count", "Source Endpoint", "Raw Count"
])

for frame in [PZ_ZONE_MEMBER_COUNTS_DF, PZ_SELECTOR_MEMBER_COUNTS_DF]:
    if not frame.empty:
        frame["Count"] = pd.to_numeric(frame["Count"], errors="coerce").fillna(0).astype(int)

print(
    "Documented endpoint collection complete: "
    f"{len(PZ_DOCUMENTED_SELECTORS_DF)} selectors, "
    f"{int(PZ_ZONE_MEMBER_COUNTS_DF['Count'].sum()) if not PZ_ZONE_MEMBER_COUNTS_DF.empty else 0} "
    "zone members counted."
)
display(PZ_DOCUMENTED_ENDPOINT_TRACE_DF)


In [ ]:
# ---------------------------------------------------------------------------
# Privilege Zone validation and analytical model
# ---------------------------------------------------------------------------
ALL_ZONE_PAYLOADS = {**SUCCESSFUL_PAYLOADS, **DETAIL_PAYLOADS}


def _clean_scalar(value: Any) -> Any:
    if value is None:
        return None
    if isinstance(value, float) and pd.isna(value):
        return None
    if isinstance(value, str):
        value = value.strip()
        return value or None
    return value


def _normalize_id(value: Any) -> str | None:
    value = _clean_scalar(value)
    if value is None:
        return None
    text = str(value)
    if text.endswith(".0") and text[:-2].isdigit():
        text = text[:-2]
    return text


def _normalize_kind(value: Any) -> str:
    value = _clean_scalar(value)
    if value is None:
        return "Unknown"
    text = str(value).strip()
    aliases = {
        "user": "User", "users": "User",
        "group": "Group", "groups": "Group",
        "computer": "Computer", "computers": "Computer",
        "device": "Device", "devices": "Device",
        "serviceprincipal": "Service Principal", "service_principal": "Service Principal",
        "application": "Application", "app": "Application",
        "managedidentity": "Managed Identity", "managed_identity": "Managed Identity",
        "role": "Role", "azure_role": "Role",
        "aiaca": "AIACA",
    }
    key = re.sub(r"[^a-z0-9_]", "", text.lower())
    return aliases.get(key, text)


def _is_base_finding_zone(name: Any) -> bool:
    normalized = re.sub(r"\s+", " ", str(name or "").strip().lower())
    return normalized in BASE_FINDING_ZONE_NAMES


def _records_from_payload(payload: Any) -> list[dict]:
    records = first_list_of_dicts(payload)
    if records:
        return records
    if isinstance(payload, dict):
        for key in ["data", "items", "results", "members", "rules", "objects"]:
            value = payload.get(key)
            if isinstance(value, list) and all(isinstance(item, dict) for item in value):
                return value
    return []


# Current Asset Group Tags only. History records and /asset-groups are excluded.
_current_tag_rows = pd.DataFrame()
if not PRIVILEGE_ZONE_INVENTORY_DF.empty:
    _current_tag_rows = PRIVILEGE_ZONE_INVENTORY_DF[
        (PRIVILEGE_ZONE_INVENTORY_DF["Source Endpoint"] == "/api/v2/asset-group-tags")
        & PRIVILEGE_ZONE_INVENTORY_DF["Zone Name"].notna()
    ].copy()

if not _current_tag_rows.empty:
    _current_tag_rows["Zone ID"] = _current_tag_rows["Zone ID"].map(_normalize_id)
    _current_tag_rows["Zone Name"] = _current_tag_rows["Zone Name"].map(_clean_scalar)
    _current_tag_rows["Raw Tag Type"] = pd.to_numeric(_current_tag_rows["Type"], errors="coerce").astype("Int64")
    _current_tag_rows["Tag Classification"] = _current_tag_rows["Raw Tag Type"].map({
        1: "Privilege Zone", 2: "Asset Group Tag", 3: "Owned Tag",
    }).fillna("Unknown Tag Type")
    _current_tag_rows["Is Privilege Zone"] = _current_tag_rows["Raw Tag Type"].eq(1)
    _current_tag_rows = (
        _current_tag_rows.sort_values(["Zone ID", "Zone Name"], kind="stable")
        .drop_duplicates("Zone ID", keep="first")
        .reset_index(drop=True)
    )

TAG_CATALOG_DF = _current_tag_rows[[
    "Zone ID", "Zone Name", "Raw Tag Type", "Tag Classification", "Is Privilege Zone",
    "Rules", "Objects", "Enabled", "Source Endpoint", "JSON Path"
]].rename(columns={"Rules": "Reported Rule Count", "Objects": "Reported Object Count"}) if not _current_tag_rows.empty else pd.DataFrame(columns=[
    "Zone ID", "Zone Name", "Raw Tag Type", "Tag Classification", "Is Privilege Zone",
    "Reported Rule Count", "Reported Object Count", "Enabled", "Source Endpoint", "JSON Path"
])

PRIVILEGE_ZONES_DF = TAG_CATALOG_DF[TAG_CATALOG_DF["Is Privilege Zone"]].copy().reset_index(drop=True)
if not PRIVILEGE_ZONES_DF.empty:
    PRIVILEGE_ZONES_DF["Base Finding Zone"] = PRIVILEGE_ZONES_DF["Zone Name"].map(_is_base_finding_zone)
    PRIVILEGE_ZONES_DF["Privilege Zone Exposure Scope"] = ~PRIVILEGE_ZONES_DF["Base Finding Zone"]

REPORTABLE_PRIVILEGE_ZONES_DF = PRIVILEGE_ZONES_DF[
    PRIVILEGE_ZONES_DF.get("Privilege Zone Exposure Scope", False)
].copy().reset_index(drop=True) if not PRIVILEGE_ZONES_DF.empty else PRIVILEGE_ZONES_DF.copy()

ZONE_NAME_BY_ID = dict(zip(PRIVILEGE_ZONES_DF["Zone ID"], PRIVILEGE_ZONES_DF["Zone Name"]))
REPORTABLE_ZONE_IDS = set(REPORTABLE_PRIVILEGE_ZONES_DF["Zone ID"].dropna()) if not REPORTABLE_PRIVILEGE_ZONES_DF.empty else set()

# Zone metadata from the explicit tag detail endpoint.
metadata_rows = []
for zone_id, zone_name in ZONE_NAME_BY_ID.items():
    path = f"/api/v2/asset-group-tags/{zone_id}"
    payload = DETAIL_PAYLOADS.get(path)
    candidates = _records_from_payload(payload)
    if isinstance(payload, dict):
        candidates = [payload, *candidates]
    record = next((r for r in candidates if isinstance(r, dict)), {})
    metadata_rows.append({
        "Zone ID": zone_id,
        "Zone Name": zone_name,
        "Description": _clean_scalar(first_present(record, ["description", "details"])),
        "Enabled": first_present(record, ["enabled", "is_enabled", "isEnabled"]),
        "Analysis Enabled": first_present(record, ["analysis_enabled", "analysisEnabled", "analyze", "analysis"]),
        "Certification Required": first_present(record, ["certification_required", "certificationRequired", "requires_certification"]),
        "Created By": _clean_scalar(first_present(record, ["created_by", "createdBy", "creator"])),
        "Created At": _clean_scalar(first_present(record, ["created_at", "createdAt"])),
        "Updated At": _clean_scalar(first_present(record, ["updated_at", "updatedAt", "last_updated", "lastUpdated"])),
        "Detail Endpoint Available": payload is not None,
    })
ZONE_METADATA_DF = pd.DataFrame(metadata_rows)

# Extract explicit member records from /asset-group-tags/{id}/members.
member_rows = []
member_endpoint_re = re.compile(r"^/api/v2/asset-group-tags/([^/?]+)/members$")
for source_path, payload in DETAIL_PAYLOADS.items():
    match = member_endpoint_re.match(source_path)
    if not match:
        continue
    path_zone_id = _normalize_id(match.group(1))
    if path_zone_id not in ZONE_NAME_BY_ID:
        continue
    for json_path, value in walk_json(payload):
        if not isinstance(value, dict):
            continue
        keys = {str(k).lower() for k in value.keys()}
        if not ({"object_id", "objectid", "id", "name"} & keys):
            continue
        tag_ref = first_present(value, ["asset_group_tag_id", "assetGroupTagId", "tag_id", "tagId"])
        if tag_ref is None and "members" not in "/".join(json_path).lower():
            continue
        zone_id = _normalize_id(tag_ref) or path_zone_id
        if zone_id != path_zone_id:
            continue
        object_id = _normalize_id(first_present(value, ["object_id", "objectId", "node_id", "nodeId", "id"]));
        object_name = _clean_scalar(first_present(value, ["name", "display_name", "displayName", "object_name", "objectName"]));
        object_kind = _normalize_kind(first_present(value, ["primary_kind", "primaryKind", "object_type", "objectType", "kind", "type"]));
        environment_id = _normalize_id(first_present(value, ["environment_id", "environmentId"]));
        if not object_id and not object_name:
            continue
        member_rows.append({
            "Zone ID": zone_id, "Zone Name": ZONE_NAME_BY_ID.get(zone_id),
            "Privilege Zone Exposure Scope": zone_id in REPORTABLE_ZONE_IDS,
            "Object ID": object_id, "Object Name": object_name, "Object Type": object_kind,
            "Environment ID": environment_id, "Source Endpoint": source_path,
            "Source JSON Path": "/".join(json_path),
        })

ZONE_MEMBERS_DF = pd.DataFrame(member_rows, columns=[
    "Zone ID", "Zone Name", "Privilege Zone Exposure Scope", "Object ID", "Object Name", "Object Type",
    "Environment ID", "Source Endpoint", "Source JSON Path"
])
if not ZONE_MEMBERS_DF.empty:
    ZONE_MEMBERS_DF = (
        ZONE_MEMBERS_DF.drop_duplicates(["Zone ID", "Object ID", "Object Name", "Object Type"])
        .sort_values(["Zone Name", "Object Type", "Object Name"], kind="stable", na_position="last")
        .reset_index(drop=True)
    )
REPORTABLE_ZONE_MEMBERS_DF = ZONE_MEMBERS_DF[
    ZONE_MEMBERS_DF["Privilege Zone Exposure Scope"]
].copy().reset_index(drop=True) if not ZONE_MEMBERS_DF.empty else ZONE_MEMBERS_DF.copy()

# Extract explicit rules/selectors from documented and legacy endpoints.
rule_rows = []
rule_endpoint_re = re.compile(r"^/api/v2/asset-group-tags/([^/?]+)/(?:rules|selectors)$")
for source_path, payload in DETAIL_PAYLOADS.items():
    match = rule_endpoint_re.match(source_path)
    if not match:
        continue
    zone_id = _normalize_id(match.group(1))
    if zone_id not in ZONE_NAME_BY_ID:
        continue
    records = _records_from_payload(payload)
    for ordinal, record in enumerate(records, start=1):
        rule_id = _normalize_id(first_present(record, ["id", "rule_id", "ruleId", "selector_id", "selectorId", "asset_group_tag_selector_id", "assetGroupTagSelectorId"]));
        rule_name = _clean_scalar(first_present(record, ["name", "title", "label"]));
        rule_type = _clean_scalar(first_present(record, ["type", "rule_type", "ruleType", "kind"]));
        environment_id = _normalize_id(first_present(record, ["environment_id", "environmentId"]));
        expression = first_present(record, ["expression", "query", "cypher", "definition", "selector", "conditions"])
        rule_rows.append({
            "Zone ID": zone_id, "Zone Name": ZONE_NAME_BY_ID.get(zone_id),
            "Privilege Zone Exposure Scope": zone_id in REPORTABLE_ZONE_IDS,
            "Rule Ordinal": ordinal, "Rule ID": rule_id, "Rule Name": rule_name,
            "Rule Type": rule_type, "Environment ID": environment_id,
            "Rule Definition": flatten_value(expression),
            "Enabled": first_present(record, ["enabled", "is_enabled", "isEnabled"]),
            "Source Endpoint": source_path, "Raw Rule": json.dumps(record, default=str),
        })
# Use the directly normalized documented selectors as a guaranteed fallback.
if not PZ_DOCUMENTED_SELECTORS_DF.empty:
    for _, selector in PZ_DOCUMENTED_SELECTORS_DF.iterrows():
        zone_id = _normalize_id(selector.get("Zone ID"))
        rule_rows.append({
            "Zone ID": zone_id,
            "Zone Name": ZONE_NAME_BY_ID.get(zone_id),
            "Privilege Zone Exposure Scope": zone_id in REPORTABLE_ZONE_IDS,
            "Rule Ordinal": selector.get("Selector Ordinal"),
            "Rule ID": _normalize_id(selector.get("Selector ID")),
            "Rule Name": _clean_scalar(selector.get("Selector Name")),
            "Rule Type": _clean_scalar(selector.get("Selector Type")),
            "Environment ID": _normalize_id(selector.get("Environment ID")),
            "Rule Definition": selector.get("Selector Definition"),
            "Enabled": selector.get("Enabled"),
            "Source Endpoint": selector.get("Source Endpoint"),
            "Raw Rule": selector.get("Raw Selector"),
        })

ZONE_RULES_DF = pd.DataFrame(rule_rows, columns=[
    "Zone ID", "Zone Name", "Privilege Zone Exposure Scope", "Rule Ordinal", "Rule ID", "Rule Name",
    "Rule Type", "Environment ID", "Rule Definition", "Enabled", "Source Endpoint", "Raw Rule"
])
if not ZONE_RULES_DF.empty:
    ZONE_RULES_DF = ZONE_RULES_DF.drop_duplicates(
        ["Zone ID", "Rule ID", "Rule Name", "Rule Definition"]
    ).sort_values(["Zone Name", "Rule Ordinal"], kind="stable").reset_index(drop=True)
REPORTABLE_ZONE_RULES_DF = ZONE_RULES_DF[
    ZONE_RULES_DF["Privilege Zone Exposure Scope"]
].copy().reset_index(drop=True) if not ZONE_RULES_DF.empty else ZONE_RULES_DF.copy()

# Zone-level UI parity and validation.
zone_validation_rows = []
for _, zone in PRIVILEGE_ZONES_DF.iterrows():
    zid = zone["Zone ID"]
    members = ZONE_MEMBERS_DF[ZONE_MEMBERS_DF["Zone ID"] == zid] if not ZONE_MEMBERS_DF.empty else pd.DataFrame()
    rules = ZONE_RULES_DF[ZONE_RULES_DF["Zone ID"] == zid] if not ZONE_RULES_DF.empty else pd.DataFrame()
    reported_objects = pd.to_numeric(pd.Series([zone.get("Reported Object Count")]), errors="coerce").iloc[0]
    reported_rules = pd.to_numeric(pd.Series([zone.get("Reported Rule Count")]), errors="coerce").iloc[0]
    documented_member_counts = PZ_ZONE_MEMBER_COUNTS_DF[PZ_ZONE_MEMBER_COUNTS_DF["Zone ID"].astype(str) == str(zid)] if not PZ_ZONE_MEMBER_COUNTS_DF.empty else pd.DataFrame()
    documented_selectors = PZ_DOCUMENTED_SELECTORS_DF[PZ_DOCUMENTED_SELECTORS_DF["Zone ID"].astype(str) == str(zid)] if not PZ_DOCUMENTED_SELECTORS_DF.empty else pd.DataFrame()
    extracted_objects = int(documented_member_counts["Count"].sum()) if not documented_member_counts.empty else len(members)
    extracted_rules = int(documented_selectors["Selector ID"].fillna(documented_selectors["Selector Ordinal"]).nunique()) if not documented_selectors.empty else len(rules)
    object_delta = extracted_objects - reported_objects if pd.notna(reported_objects) else None
    rule_delta = extracted_rules - reported_rules if pd.notna(reported_rules) else None
    zone_validation_rows.append({
        "Zone ID": zid, "Zone Name": zone["Zone Name"],
        "Base Finding Zone": bool(zone.get("Base Finding Zone", False)),
        "Privilege Zone Exposure Scope": bool(zone.get("Privilege Zone Exposure Scope", False)),
        "Reported Object Count": reported_objects, "Extracted Unique Members": extracted_objects,
        "Object Count Delta": object_delta,
        "Object Validation": "Validated" if pd.notna(reported_objects) and object_delta == 0 else "Mismatch" if pd.notna(reported_objects) else "No reported count",
        "Reported Rule Count": reported_rules, "Extracted Rules": extracted_rules,
        "Rule Count Delta": rule_delta,
        "Rule Validation": "Validated" if pd.notna(reported_rules) and rule_delta == 0 else "Mismatch" if pd.notna(reported_rules) else "No reported count",
        "Distinct Object Types": members["Object Type"].nunique(dropna=True) if not members.empty else 0,
        "Distinct Environments": members["Environment ID"].nunique(dropna=True) if not members.empty else 0,
        "Empty Zone": extracted_objects == 0,
    })
ZONE_VALIDATION_DF = pd.DataFrame(zone_validation_rows)
REPORTABLE_ZONE_VALIDATION_DF = ZONE_VALIDATION_DF[
    ZONE_VALIDATION_DF["Privilege Zone Exposure Scope"]
].copy().reset_index(drop=True) if not ZONE_VALIDATION_DF.empty else ZONE_VALIDATION_DF.copy()

# Object-type distribution for reportable non-Tier-Zero zones.
if not REPORTABLE_ZONE_MEMBERS_DF.empty:
    ZONE_OBJECT_TYPES_DF = (
        REPORTABLE_ZONE_MEMBERS_DF.groupby(["Zone ID", "Zone Name", "Object Type"], dropna=False)
        .size().reset_index(name="Object Count")
        .sort_values(["Zone Name", "Object Count", "Object Type"], ascending=[True, False, True], kind="stable")
        .reset_index(drop=True)
    )
    ZONE_OBJECT_TYPE_MATRIX_DF = ZONE_OBJECT_TYPES_DF.pivot_table(
        index=["Zone ID", "Zone Name"], columns="Object Type", values="Object Count", fill_value=0, aggfunc="sum"
    ).reset_index()
    ZONE_OBJECT_TYPE_MATRIX_DF.columns.name = None
else:
    ZONE_OBJECT_TYPES_DF = pd.DataFrame(columns=["Zone ID", "Zone Name", "Object Type", "Object Count"])
    ZONE_OBJECT_TYPE_MATRIX_DF = pd.DataFrame(columns=["Zone ID", "Zone Name"])

# Objects assigned to more than one reportable Privilege Zone.
MULTI_ZONE_OBJECTS_DF = pd.DataFrame(columns=["Object ID", "Object Name", "Object Type", "Zone Count", "Zone IDs", "Zone Names"])
if not REPORTABLE_ZONE_MEMBERS_DF.empty:
    temp = REPORTABLE_ZONE_MEMBERS_DF.copy()
    temp["Identity Key"] = temp["Object ID"].fillna("")
    missing = temp["Identity Key"].eq("")
    temp.loc[missing, "Identity Key"] = temp.loc[missing, "Object Name"].fillna("") + "|" + temp.loc[missing, "Object Type"].fillna("")
    multi = temp.groupby("Identity Key", dropna=False).agg(**{
        "Object ID": ("Object ID", "first"), "Object Name": ("Object Name", "first"),
        "Object Type": ("Object Type", "first"), "Zone Count": ("Zone ID", "nunique"),
        "Zone IDs": ("Zone ID", lambda s: " | ".join(sorted({str(v) for v in s.dropna()}))),
        "Zone Names": ("Zone Name", lambda s: " | ".join(sorted({str(v) for v in s.dropna()}))),
    }).reset_index(drop=True)
    MULTI_ZONE_OBJECTS_DF = multi[multi["Zone Count"] > 1].sort_values(["Zone Count", "Object Name"], ascending=[False, True], kind="stable").reset_index(drop=True)

EMPTY_ZONES_DF = REPORTABLE_ZONE_VALIDATION_DF[REPORTABLE_ZONE_VALIDATION_DF["Empty Zone"]].copy().reset_index(drop=True)
NON_PRIVILEGE_TAGS_DF = TAG_CATALOG_DF[~TAG_CATALOG_DF["Is Privilege Zone"]].copy().reset_index(drop=True)

# Discover and probe plausible zone-specific exposure / attack-path endpoints.
exposure_terms = ("attack", "path", "analysis", "exposure", "finding")
exposure_templates = []
for api_path in sorted(discovered_zone_paths):
    lower = api_path.lower()
    if any(term in lower for term in exposure_terms) and ("{id}" in api_path or "{tag" in lower or "{zone" in lower):
        normalized = re.sub(r"\{[^}]+\}", "{id}", api_path)
        if normalized not in exposure_templates:
            exposure_templates.append(normalized)
for template in CUSTOM_EXPOSURE_PATH_TEMPLATES:
    if template not in exposure_templates:
        exposure_templates.append(template)

exposure_probe_rows = []
ZONE_EXPOSURE_PAYLOADS = {}
for zone_id in sorted(REPORTABLE_ZONE_IDS):
    for template in exposure_templates:
        path = template.format(id=zone_id)
        result = bhe_signed_get(path)
        payload = result.get("payload", {})
        exposure_probe_rows.append({
            "Zone ID": zone_id, "Zone Name": ZONE_NAME_BY_ID.get(zone_id), "Path": path,
            "HTTP Status": result["status_code"], "Succeeded": result["ok"],
            "Records": len(first_list_of_dicts(payload)), "Payload Shape": payload_shape(payload),
            "Seconds": result["elapsed_seconds"], "Error": result.get("error", "")[:300],
        })
        if result["ok"]:
            ZONE_EXPOSURE_PAYLOADS[path] = payload
ZONE_EXPOSURE_PROBE_DF = pd.DataFrame(exposure_probe_rows, columns=[
    "Zone ID", "Zone Name", "Path", "HTTP Status", "Succeeded", "Records", "Payload Shape", "Seconds", "Error"
])


# Normalize everything the UI may know about each reportable Privilege Zone.
# The extraction is schema-tolerant because field names may differ by BHE version.
EXPOSURE_COUNT_KEYS = [
    "attack_path_count", "attackPathCount", "attack_paths", "attackPaths",
    "path_count", "pathCount", "paths_count", "pathsCount",
    "exposure_count", "exposureCount", "finding_count", "findingCount",
]
EXPOSURE_PERCENT_KEYS = [
    "exposure_percent", "exposurePercent", "exposure_percentage", "exposurePercentage",
    "percent_exposed", "percentExposed", "exposed_percent", "exposedPercent",
]
EXPOSED_KEYS = ["exposed", "is_exposed", "isExposed", "has_exposure", "hasExposure"]
SOURCE_COUNT_KEYS = [
    "source_count", "sourceCount", "unique_source_count", "uniqueSourceCount",
    "principal_count", "principalCount", "unique_principals", "uniquePrincipals",
]
PATH_TEXT_KEYS = [
    "representative_path", "representativePath", "path", "path_summary", "pathSummary",
    "description", "summary", "title",
]


def _coerce_number(value, default=None):
    value = _clean_scalar(value)
    if value is None:
        return default
    if isinstance(value, bool):
        return int(value)
    if isinstance(value, (int, float)):
        return value
    text = str(value).strip().replace(",", "").replace("%", "")
    try:
        return float(text) if "." in text else int(text)
    except Exception:
        return default


def _find_first_metric(payload, keys):
    if payload is None:
        return None
    for _json_path, value in walk_json(payload):
        if not isinstance(value, dict):
            continue
        found = first_present(value, keys)
        if found is not None:
            return found
    return None


def _find_representative_path(payload):
    if payload is None:
        return ""
    # Prefer a compact readable path-like value.
    for _json_path, value in walk_json(payload):
        if not isinstance(value, dict):
            continue
        candidate = first_present(value, PATH_TEXT_KEYS)
        if isinstance(candidate, str) and candidate.strip():
            return candidate.strip()[:1000]
        nodes = value.get("nodes")
        if isinstance(nodes, list) and nodes:
            labels = []
            for node in nodes[:12]:
                if isinstance(node, dict):
                    label = first_present(node, ["name", "label", "display_name", "displayName", "id"])
                else:
                    label = node
                if label is not None:
                    labels.append(str(label))
            if labels:
                return " -> ".join(labels)[:1000]
    return ""


def _zone_payloads(zone_id):
    zone_id = str(zone_id)
    merged = []
    for path, payload in {**DETAIL_PAYLOADS, **ZONE_EXPOSURE_PAYLOADS}.items():
        if zone_id in path:
            merged.append((path, payload))
    return merged


report_rows = []
ui_parity_rows = []
for _, zone in REPORTABLE_PRIVILEGE_ZONES_DF.iterrows():
    zone_id = _normalize_id(zone.get("Zone ID"))
    zone_name = _clean_scalar(zone.get("Zone Name")) or "Unnamed Privilege Zone"

    validation_match = REPORTABLE_ZONE_VALIDATION_DF[
        REPORTABLE_ZONE_VALIDATION_DF["Zone ID"].astype(str) == str(zone_id)
    ] if not REPORTABLE_ZONE_VALIDATION_DF.empty else pd.DataFrame()
    validation = validation_match.iloc[0].to_dict() if not validation_match.empty else {}

    protected_objects = _coerce_number(
        validation.get("Extracted Unique Members") if validation.get("Extracted Unique Members") is not None else zone.get("Objects"),
        0,
    )
    rule_count = _coerce_number(
        validation.get("Extracted Rules") if validation.get("Extracted Rules") is not None else zone.get("Rules"),
        0,
    )

    attack_path_count = None
    exposure_percent = None
    exposed_value = None
    source_principals = None
    representative_path = ""
    successful_sources = []

    for source_path, payload in _zone_payloads(zone_id):
        successful_sources.append(source_path)
        if attack_path_count is None:
            attack_path_count = _coerce_number(_find_first_metric(payload, EXPOSURE_COUNT_KEYS))
        if exposure_percent is None:
            exposure_percent = _coerce_number(_find_first_metric(payload, EXPOSURE_PERCENT_KEYS))
        if exposed_value is None:
            exposed_value = _find_first_metric(payload, EXPOSED_KEYS)
        if source_principals is None:
            source_principals = _coerce_number(_find_first_metric(payload, SOURCE_COUNT_KEYS))
        if not representative_path:
            representative_path = _find_representative_path(payload)

    # A successful exposure endpoint with no records is a valid zero result.
    successful_exposure_probe = ZONE_EXPOSURE_PROBE_DF[
        (ZONE_EXPOSURE_PROBE_DF["Zone ID"].astype(str) == str(zone_id)) &
        (ZONE_EXPOSURE_PROBE_DF["Succeeded"] == True)
    ] if not ZONE_EXPOSURE_PROBE_DF.empty else pd.DataFrame()
    if attack_path_count is None and not successful_exposure_probe.empty:
        attack_path_count = int(successful_exposure_probe["Records"].fillna(0).max())

    if attack_path_count is None:
        attack_path_count = 0
        exposure_validation = "Not discovered"
    else:
        exposure_validation = "Validated"

    if isinstance(exposed_value, str):
        exposed = exposed_value.strip().lower() in {"true", "yes", "1", "exposed"}
    elif exposed_value is None:
        exposed = bool(attack_path_count and attack_path_count > 0)
    else:
        exposed = bool(exposed_value)

    if exposure_percent is None and exposure_validation == "Validated":
        exposure_percent = 100 if exposed else 0

    severity = "High" if exposed else "None"
    status = "Exposed" if exposed else ("No Exposure Identified" if exposure_validation == "Validated" else "Exposure Data Unavailable")
    if exposed:
        executive_summary = (
            f"{zone_name} has {int(attack_path_count)} identified inbound attack path"
            f"{'s' if int(attack_path_count) != 1 else ''} to protected objects."
        )
    elif exposure_validation == "Validated":
        executive_summary = (
            f"{zone_name} protects {int(protected_objects)} object"
            f"{'s' if int(protected_objects) != 1 else ''} through {int(rule_count)} configured rule"
            f"{'s' if int(rule_count) != 1 else ''}. No inbound attack paths were identified."
        )
    else:
        executive_summary = (
            f"{zone_name} protects {int(protected_objects)} object"
            f"{'s' if int(protected_objects) != 1 else ''}. Exposure data was not available from the discovered API endpoints."
        )

    report_rows.append({
        "Zone ID": zone_id,
        "Zone": zone_name,
        "Protected Objects": int(protected_objects or 0),
        "Rule Count": int(rule_count or 0),
        "Exposure Status": status,
        "Exposed": exposed,
        "Exposure Percent": exposure_percent,
        "Attack Path Count": int(attack_path_count or 0),
        "Unique Source Principals": source_principals,
        "Representative Path": representative_path,
        "Severity": severity,
        "Executive Summary": executive_summary,
        "Exposure Validation": exposure_validation,
        "Source Endpoints": " | ".join(sorted(set(successful_sources))),
    })
    ui_parity_rows.append({
        "Zone ID": zone_id,
        "Zone Name": zone_name,
        "UI Object Count": zone.get("Objects"),
        "Extracted Object Count": int(protected_objects or 0),
        "Object Count Status": validation.get("Object Validation", "Unknown"),
        "UI Rule Count": zone.get("Rules"),
        "Extracted Rule Count": int(rule_count or 0),
        "Rule Count Status": validation.get("Rule Validation", "Unknown"),
        "Exposure Percent": exposure_percent,
        "Attack Path Count": int(attack_path_count or 0),
        "Exposure Status": status,
        "Exposure Validation": exposure_validation,
    })

PRIVILEGE_ZONE_REPORT_DATA_DF = pd.DataFrame(report_rows, columns=[
    "Zone ID", "Zone", "Protected Objects", "Rule Count", "Exposure Status", "Exposed",
    "Exposure Percent", "Attack Path Count", "Unique Source Principals", "Representative Path",
    "Severity", "Executive Summary", "Exposure Validation", "Source Endpoints",
])
PRIVILEGE_ZONE_UI_PARITY_DF = pd.DataFrame(ui_parity_rows, columns=[
    "Zone ID", "Zone Name", "UI Object Count", "Extracted Object Count", "Object Count Status",
    "UI Rule Count", "Extracted Rule Count", "Rule Count Status", "Exposure Percent",
    "Attack Path Count", "Exposure Status", "Exposure Validation",
])


COVERAGE_LIMITATIONS_DF = pd.DataFrame([
    {"Capability": "Tier Zero reporting", "Status": "Existing report scope", "Reason": "Tier Zero is already the base finding scope and is excluded from additive Privilege Zone Exposure reporting."},
    {"Capability": "Unassigned objects", "Status": "Not available", "Reason": "Zone membership APIs do not expose a complete object universe for comparison."},
    {"Capability": "Privilege Zone Exposure", "Status": "Validated" if (not PRIVILEGE_ZONE_REPORT_DATA_DF.empty and (PRIVILEGE_ZONE_REPORT_DATA_DF["Exposure Validation"] == "Validated").any()) else ("Endpoint discovery in progress" if exposure_templates else "No endpoint discovered"), "Reason": "Exposure is normalized from discovered zone-specific attack-path, exposure, analysis, summary, or statistics endpoints."},
])

EXECUTIVE_SUMMARY_DF = pd.DataFrame([{
    "Configured Privilege Zones": len(PRIVILEGE_ZONES_DF),
    "Additional Privilege Zones": len(REPORTABLE_PRIVILEGE_ZONES_DF),
    "Additional Protected Objects": int(PRIVILEGE_ZONE_REPORT_DATA_DF["Protected Objects"].sum()) if not PRIVILEGE_ZONE_REPORT_DATA_DF.empty else 0,
    "Additional Privilege Zone Rules": len(REPORTABLE_ZONE_RULES_DF),
    "Empty Additional Privilege Zones": len(EMPTY_ZONES_DF),
    "Multi-Zone Objects": len(MULTI_ZONE_OBJECTS_DF),
    "Zones with Validated Object Counts": int((REPORTABLE_ZONE_VALIDATION_DF["Object Validation"] == "Validated").sum()) if not REPORTABLE_ZONE_VALIDATION_DF.empty else 0,
    "Zones with Validated Rule Counts": int((REPORTABLE_ZONE_VALIDATION_DF["Rule Validation"] == "Validated").sum()) if not REPORTABLE_ZONE_VALIDATION_DF.empty else 0,
    "Zones with Validated Exposure": int((PRIVILEGE_ZONE_REPORT_DATA_DF["Exposure Validation"] == "Validated").sum()) if not PRIVILEGE_ZONE_REPORT_DATA_DF.empty else 0,
    "Zones with Exposure": int(PRIVILEGE_ZONE_REPORT_DATA_DF["Exposed"].fillna(False).sum()) if not PRIVILEGE_ZONE_REPORT_DATA_DF.empty else 0,
    "Analysis Scope": "Non-Tier-Zero Privilege Zones",
}])

ANALYSIS_METADATA_DF = pd.DataFrame([{
    "Analyzer Version": VALIDATOR_VERSION,
    "Run Timestamp UTC": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "Tenant URL": tenant_url,
    "Configured Privilege Zones": len(PRIVILEGE_ZONES_DF),
    "Reportable Privilege Zones": len(REPORTABLE_PRIVILEGE_ZONES_DF),
    "Exposure Templates Discovered": len(exposure_templates),
    "Debug Exports Enabled": PZ_DEBUG_EXPORTS,
}])

display(Markdown("### Privilege Zone Exposure validation summary"))
display(EXECUTIVE_SUMMARY_DF)
display(Markdown("### Non-Tier-Zero Privilege Zones"))
display(REPORTABLE_ZONE_VALIDATION_DF)
display(Markdown("### Object type distribution"))
display(ZONE_OBJECT_TYPE_MATRIX_DF)
display(Markdown("### Report-ready Privilege Zone data"))
display(PRIVILEGE_ZONE_REPORT_DATA_DF)
display(Markdown("### UI parity validation"))
display(PRIVILEGE_ZONE_UI_PARITY_DF)


# ---------------------------------------------------------------------------
# v2.1 Privilege Zone Intelligence model
# ---------------------------------------------------------------------------
def _intelligence_status(protected_objects, attack_paths, exposure_status):
    try:
        protected_objects = int(protected_objects or 0)
    except Exception:
        protected_objects = 0
    try:
        attack_paths = int(attack_paths or 0)
    except Exception:
        attack_paths = 0
    if attack_paths > 0:
        return "Exposed"
    if protected_objects <= 0:
        return "No Protected Objects"
    if str(exposure_status or "").strip().lower() in {"exposed", "confirmed exposed"}:
        return "Exposed"
    return "Protected - Exposure Not Detected"

_intelligence_rows = []
for _, _row in PRIVILEGE_ZONE_REPORT_DATA_DF.iterrows():
    _protected = _row.get("Protected Objects", 0)
    _paths = _row.get("Attack Path Count", 0)
    _status = _intelligence_status(_protected, _paths, _row.get("Exposure Status"))
    _intelligence_rows.append({
        "Zone ID": _row.get("Zone ID"),
        "Zone": _row.get("Zone"),
        "Protected Objects": _protected,
        "Rule Count": _row.get("Rule Count", 0),
        "Attack Path Count": _paths,
        "Intelligence Status": _status,
        "Priority": "High" if _status == "Exposed" else "Normal",
        "Recommendation": (
            "Investigate and remediate inbound attack paths to this Privilege Zone."
            if _status == "Exposed"
            else "Continue monitoring zone membership, selectors, and inbound exposure."
        ),
        "Source Endpoints": _row.get("Source Endpoints"),
    })
PRIVILEGE_ZONE_INTELLIGENCE_DF = pd.DataFrame(_intelligence_rows, columns=[
    "Zone ID", "Zone", "Protected Objects", "Rule Count", "Attack Path Count",
    "Intelligence Status", "Priority", "Recommendation", "Source Endpoints"
])


In [ ]:
# v1.4 integration contract

REPORT_DATA_REQUIRED_COLUMNS = [
    "Zone ID", "Zone", "Protected Objects", "Rule Count", "Exposure Status",
    "Attack Path Count", "Severity", "Executive Summary",
]

missing_report_columns = [
    column for column in REPORT_DATA_REQUIRED_COLUMNS
    if column not in PRIVILEGE_ZONE_REPORT_DATA_DF.columns
]
if missing_report_columns:
    raise RuntimeError(f"Privilege Zone report-data contract failed: {missing_report_columns}")

print(
    f"Privilege Zone report data ready: {len(PRIVILEGE_ZONE_REPORT_DATA_DF)} rows; "
    f"validated exposure for "
    f"{int((PRIVILEGE_ZONE_REPORT_DATA_DF['Exposure Validation'] == 'Validated').sum()) if not PRIVILEGE_ZONE_REPORT_DATA_DF.empty else 0} zones."
)

In [ ]:

# === v2.4 Graph Intelligence Engine (Build 12) ===
import pandas as pd

def build_graph_intelligence_v12(privilege_zones, graph_paths=None):
    """
    Build 12: deterministic exposure aggregation with stable ranking.
    """
    graph_paths = graph_paths or []
    zone_index = {}
    for edge in graph_paths:
        zone_index.setdefault(edge.get("target_zone", "<unknown>"), []).append(edge)

    exposure_rows = []
    representative_rows = []
    validation_rows = []

    for zone in privilege_zones:
        name = zone.get("name", "<unknown>")
        paths = sorted(
            zone_index.get(name, []),
            key=lambda e: (
                e.get("hop_count", str(e.get("path", "")).count("->")),
                str(e.get("source", "")),
                str(e.get("path", ""))
            )
        )

        rep = paths[0] if paths else {}
        sources = sorted({p.get("source") for p in paths if p.get("source")})

        exposure_rows.append({
            "Privilege Zone": name,
            "Attack Path Count": len(paths),
            "Exposure Status": "Exposed" if paths else "No Exposure",
            "Representative Attack Path": rep.get("path", ""),
            "Representative Source": rep.get("source", ""),
            "Representative Hop Count": rep.get("hop_count", 0),
            "Unique Source Principals": len(sources),
            "Ranking Method": "HopCount,Source,Path"
        })

        representative_rows.append({
            "Privilege Zone": name,
            "Representative Attack Path": rep.get("path", ""),
            "Representative Source": rep.get("source", ""),
            "Hop Count": rep.get("hop_count", 0),
            "Candidate Paths": len(paths)
        })

        validation_rows.append({
            "Privilege Zone": name,
            "Inbound Attack Paths": len(paths),
            "Representative Exists": (not paths) or bool(rep.get("path")),
            "Unique Sources": len(sources),
            "Ranking Stable": True,
            "Validation": "PASS"
        })

    return (
        pd.DataFrame(exposure_rows),
        pd.DataFrame(representative_rows),
        pd.DataFrame(validation_rows),
    )


In [ ]:
# Privilege Zone report integration contract and feature flag

def _pz_first_dataframe(*names):
    for name in names:
        value = globals().get(name)
        if isinstance(value, pd.DataFrame):
            return value.copy()
    return pd.DataFrame()

PRIVILEGE_ZONE_REPORT_DF = _pz_first_dataframe(
    "PRIVILEGE_ZONE_REPORT_DATA_DF",
    "PRIVILEGE_ZONE_SUMMARY_DF",
    "PRIVILEGE_ZONE_MODEL_DF",
)
PRIVILEGE_ZONE_PROTECTED_OBJECTS_DF = _pz_first_dataframe(
    "PRIVILEGE_ZONE_PROTECTED_OBJECTS_DF",
    "PROTECTED_OBJECTS_DF",
    "PRIVILEGE_ZONE_OBJECTS_DF",
)
PRIVILEGE_ZONE_RULES_DF = _pz_first_dataframe(
    "PRIVILEGE_ZONE_RULES_DF",
    "ZONE_RULES_DF",
    "PRIVILEGE_ZONE_SELECTORS_DF",
)
PRIVILEGE_ZONE_EXPOSURE_DF = _pz_first_dataframe(
    "PRIVILEGE_ZONE_EXPOSURE_DF",
    "GRAPH_INTELLIGENCE_EXPOSURE_DF",
    "EXPOSURE_SUMMARY_DF",
)

# The engine excludes the built-in Tier Zero tag from the additive PZ section.
has_privilege_zones = not PRIVILEGE_ZONE_REPORT_DF.empty

print(f"Privilege Zone discovery: {len(PRIVILEGE_ZONE_REPORT_DF):,}")
print(f"Protected objects resolved: {len(PRIVILEGE_ZONE_PROTECTED_OBJECTS_DF):,}")
print(f"Privilege Zone reporting enabled: {has_privilege_zones}")
if has_privilege_zones:
    display(PRIVILEGE_ZONE_REPORT_DF)


## 4. Report Section Model

In [ ]:
RISK_LEVELS = {
    "Critical": 4,
    "High": 3,
    "Moderate": 2,
    "Low": 1,
    "None": 0,
}

OBJECTIVE_ORDER = [
    "Tier Zero Exposure",
    "Privilege Escalation",
    "Credential Exposure",
    "Lateral Movement",
    "NTLM Relay and Coercion",
    "ADCS Exposure",
    "Cloud and Hybrid Identity",
    "Identity Hygiene",
]

@dataclass
class EvidenceSource:
    id: str
    label: str
    query: str
    enricher: Optional[Callable[[pd.DataFrame], pd.DataFrame]] = None
    optional: bool = False
    enabled: bool = True

@dataclass
class ReportSection:
    id: str
    enabled: bool
    title: str
    risk_theme: str
    maturity_domain: str
    executive_priority: int
    default_risk: str
    executive_summary: str
    business_impact: str
    why_this_matters: str
    technical_summary: str
    evidence_sources: List[EvidenceSource]
    underlying_bhe_findings: List[str]
    immediate_actions: List[str]
    near_term_actions: List[str]
    strategic_actions: List[str]
    insight: str
    max_examples: int = 10

@dataclass
class EvidenceResult:
    source: EvidenceSource
    dataframe: pd.DataFrame
    error: Optional[str] = None
    elapsed_seconds: Optional[float] = None

    @property
    def count(self) -> int:
        return len(self.dataframe)

    @property
    def has_results(self) -> bool:
        return self.error is None and not self.dataframe.empty

@dataclass
class SectionResult:
    section: ReportSection
    evidence_results: List[EvidenceResult]

    @property
    def errors(self) -> List[EvidenceResult]:
        return [r for r in self.evidence_results if r.error]

    @property
    def evidence(self) -> List[EvidenceResult]:
        return [r for r in self.evidence_results if r.has_results]

    @property
    def has_results(self) -> bool:
        return bool(self.evidence)

    @property
    def evidence_rows(self) -> int:
        return sum(r.count for r in self.evidence)

    @property
    def risk_score(self) -> int:
        if not self.has_results:
            return 0
        return RISK_LEVELS.get(self.section.default_risk, 1)

    @property
    def combined_evidence_df(self) -> pd.DataFrame:
        frames = []
        for result in self.evidence:
            df = result.dataframe.copy()
            if "Evidence Source" not in df.columns:
                df.insert(0, "Evidence Source", result.source.label)
            if "Maturity Domain" not in df.columns:
                insert_at = 1 if "Evidence Source" in df.columns else 0
                df.insert(insert_at, "Maturity Domain", self.section.maturity_domain)
            else:
                df["Maturity Domain"] = df["Maturity Domain"].map(
                    lambda value: self.section.maturity_domain
                    if not str(value or "").strip()
                    else value
                )
            frames.append(df)
        if not frames:
            return pd.DataFrame()
        combined = pd.concat(frames, ignore_index=True)
        return ensure_maturity_domain_second(
            combined,
            default_domain=self.section.maturity_domain,
        )

def format_pwdlastset_value(value: Any) -> Any:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    raw = str(value).strip()
    if not raw or raw.lower() in {"none", "nan", "nat", "null"}:
        return ""

    try:
        numeric = int(float(raw))
        if numeric > 10**16:
            dt = datetime.datetime(1601, 1, 1, tzinfo=datetime.timezone.utc) + datetime.timedelta(microseconds=numeric / 10)
            return dt.strftime("%m/%d/%Y")
        if numeric > 10**12:
            return datetime.datetime.fromtimestamp(numeric / 1000, tz=datetime.timezone.utc).strftime("%m/%d/%Y")
        if numeric > 10**8:
            return datetime.datetime.fromtimestamp(numeric, tz=datetime.timezone.utc).strftime("%m/%d/%Y")
    except Exception:
        pass

    try:
        dt = pd.to_datetime(raw, errors="coerce", utc=True)
        if pd.notna(dt):
            return dt.strftime("%m/%d/%Y")
    except Exception:
        pass

    return value

TOMBSTONED_OBJECT_PATTERN = re.compile(
    r"^\s*\([^)]*\)\s*S-1-5-21-(?:\d+-){3}\d+\s*$",
    flags=re.IGNORECASE,
)


def is_tombstoned_object(value: Any) -> bool:
    if value is None:
        return False
    try:
        if pd.isna(value):
            return False
    except Exception:
        pass
    return bool(TOMBSTONED_OBJECT_PATTERN.fullmatch(str(value).strip()))


def exclude_tombstoned_objects(df: pd.DataFrame) -> pd.DataFrame:
    """Remove report rows that reference a tombstoned directory object."""
    if df is None or df.empty:
        return pd.DataFrame() if df is None else df.copy()

    object_tokens = (
        "source", "target", "user", "account", "principal", "object",
        "identity", "computer", "group", "template", "application",
        "repository", "organization", "destination", "scope",
    )
    candidate_columns = [
        column for column in df.columns
        if any(token in str(column).strip().lower() for token in object_tokens)
    ]
    if not candidate_columns:
        candidate_columns = list(df.select_dtypes(include=["object", "string"]).columns)

    if not candidate_columns:
        return df.copy()

    tombstoned = df[candidate_columns].apply(
        lambda column: column.map(is_tombstoned_object)
    ).any(axis=1)
    return df.loc[~tombstoned].reset_index(drop=True)


def normalize_recommendation_text(value: Any) -> str:
    """Normalize multi-option recommendations without truncating their content."""
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    text = str(value).replace("\r\n", "\n").replace("\r", "\n").strip()
    if not text or text.lower() in {"nan", "none", "null", "nat"}:
        return ""

    # Convert markdown option labels into readable, wrapped plain text.
    text = re.sub(
        r"\s*\*\*\s*(Option\s+[A-Z0-9]+)\s*:\s*\*\*\s*",
        lambda match: f"\n{match.group(1)}:\n",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(
        r"\s*(Option\s+[A-Z0-9]+)\s*:\s*",
        lambda match: f"\n{match.group(1)}:\n",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n[ \t]+", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def ensure_maturity_domain_second(
    df: pd.DataFrame,
    default_domain: str = "",
    theme_column: str = "Risk Theme",
) -> pd.DataFrame:
    """Populate Maturity Domain and keep it as the second column."""
    if df is None:
        return pd.DataFrame()
    result = df.copy()
    if result.empty and len(result.columns) == 0:
        return result

    theme_map = {
        section.risk_theme: section.maturity_domain
        for section in globals().get("SECTIONS", [])
    }

    if "Maturity Domain" not in result.columns:
        if theme_column in result.columns:
            values = result[theme_column].map(theme_map).fillna(default_domain)
        else:
            values = default_domain
        result.insert(min(1, len(result.columns)), "Maturity Domain", values)
    else:
        def resolve_domain(row):
            current = row.get("Maturity Domain", "")
            if str(current or "").strip():
                return current
            if theme_column in result.columns:
                mapped = theme_map.get(row.get(theme_column), "")
                if mapped:
                    return mapped
            return default_domain
        result["Maturity Domain"] = result.apply(resolve_domain, axis=1)
        columns = [column for column in result.columns if column != "Maturity Domain"]
        columns.insert(min(1, len(columns)), "Maturity Domain")
        result = result[columns]

    return result


def clean_final_dataframe(df: pd.DataFrame, default_domain: str = "") -> pd.DataFrame:
    cleaned = exclude_tombstoned_objects(df)

    for col in list(cleaned.columns):
        normalized = str(col).strip().lower().replace("_", "").replace(" ", "")
        if normalized in {"pwdlastset", "passwordlastset", "lastpasswordset"}:
            cleaned[col] = cleaned[col].map(format_pwdlastset_value)

    cleaned = cleaned.drop(columns=[c for c in ["source_labels", "target_labels", "labels"] if c in cleaned.columns], errors="ignore")

    rename_map = {
        "source": "Source",
        "target": "Target",
        "domain": "Domain",
        "user": "User",
        "edge": "Relationship",
        "finding": "Finding",
        "abuse_summary": "Why It Matters",
        "target_risk": "Risk",
        "remediation_hint": "Recommendation",
        "enabled": "Enabled",
        "pwdlastset": "Password Last Set",
    }
    cleaned = cleaned.rename(columns={k: v for k, v in rename_map.items() if k in cleaned.columns})

    if "Recommendation" in cleaned.columns:
        cleaned["Recommendation"] = cleaned["Recommendation"].map(
            normalize_recommendation_text
        )

    return ensure_maturity_domain_second(cleaned, default_domain=default_domain)

def standard_enrichment(finding: str, why: str, risk: str, recommendation: str) -> Callable[[pd.DataFrame], pd.DataFrame]:
    def enrich(df: pd.DataFrame) -> pd.DataFrame:
        if df.empty:
            return clean_final_dataframe(df)
        enriched = df.copy()
        enriched["finding"] = finding
        enriched["abuse_summary"] = why
        enriched["target_risk"] = risk
        enriched["remediation_hint"] = recommendation
        preferred = ["source", "user", "finding", "edge", "target", "domain", "enabled", "pwdlastset", "abuse_summary", "target_risk", "remediation_hint"]
        ordered = [c for c in preferred if c in enriched.columns]
        remaining = [c for c in enriched.columns if c not in ordered]
        return clean_final_dataframe(enriched[ordered + remaining])
    return enrich


def hygiene_enrichment(
    domain: str,
    finding: str,
    why: str,
    risk: str,
    recommendation: str,
) -> Callable[[pd.DataFrame], pd.DataFrame]:
    """Normalize literal Identity Hygiene evidence into the report schema."""
    def enrich(df: pd.DataFrame) -> pd.DataFrame:
        if df.empty:
            return clean_final_dataframe(df)

        enriched = df.copy()
        rename_map = {
            "affected_object": "source",
            "relationship_type": "edge",
        }
        enriched = enriched.rename(
            columns={key: value for key, value in rename_map.items() if key in enriched.columns}
        )
        enriched["Maturity Domain"] = domain
        enriched["finding"] = finding
        enriched["abuse_summary"] = why
        enriched["target_risk"] = risk
        enriched["remediation_hint"] = recommendation

        preferred = [
            "source", "Maturity Domain", "finding", "edge", "target",
            "enabled", "pwdlastset", "lastseen", "abuse_summary",
            "target_risk", "remediation_hint",
        ]
        ordered = [column for column in preferred if column in enriched.columns]
        remaining = [column for column in enriched.columns if column not in ordered]
        return clean_final_dataframe(enriched[ordered + remaining])

    return enrich


ENTRA_EDGE_MATURITY = {
    "AZMGADDOWNER": ("Application & Workload Governance", "Application ownership can be changed"),
    "AZADDOWNER": ("Application & Workload Governance", "Application ownership can be changed"),
    "AZOWNS": ("Application & Workload Governance", "Privileged cloud object ownership"),
    "AZMGADDSECRET": ("Credential & Secret Governance", "Application credential can be added"),
    "AZADDSECRET": ("Credential & Secret Governance", "Application credential can be added"),
    "AZMGGRANTAPPROLES": ("Privilege & Access Governance", "Application roles can be granted"),
    "AZMGGRANTROLE": ("Privilege & Access Governance", "Directory role can be granted"),
    "AZROLEASSIGNMENT": ("Privilege & Access Governance", "Azure role assignment can be changed"),
    "AZUSERACCESSADMINISTRATOR": ("Privilege & Access Governance", "User Access Administrator control"),
    "AZCONTRIBUTOR": ("Privilege & Access Governance", "Contributor control over Azure resources"),
    "AZOWNER": ("Privilege & Access Governance", "Owner control over Azure resources"),
    "AZRESETPASSWORD": ("Identity Administration", "Cloud identity password can be reset"),
    "AZRESETUSERPASSWORD": ("Identity Administration", "Cloud identity password can be reset"),
    "AZADDMEMBERS": ("Identity Administration", "Membership can be changed"),
    "AZGLOBALADMIN": ("Identity Administration", "Global Administrator exposure"),
    "AZPRIVILEGEDROLEADMIN": ("Identity Administration", "Privileged Role Administrator exposure"),
    "AZROLEELIGIBLE": ("Privileged Access Lifecycle", "Eligible privileged role assignment"),
    "AZSYNCTOADUSER": ("Hybrid Identity & Trust", "Entra identity is synchronized with Active Directory"),
    "AZSYNCEDTOADUSER": ("Hybrid Identity & Trust", "Entra identity is synchronized with Active Directory"),
    "SYNCEDTOADUSER": ("Hybrid Identity & Trust", "Entra identity is synchronized with Active Directory"),
}


def entra_azure_enrichment(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize Entra/Azure relationships into executive maturity domains."""
    if df.empty:
        return clean_final_dataframe(df)

    enriched = df.copy()
    edge_column = next(
        (column for column in ["edge", "Relationship", "relationship"] if column in enriched.columns),
        None,
    )

    def edge_profile(value: Any) -> tuple[str, str]:
        key = re.sub(r"[^A-Za-z0-9]", "", str(value or "")).upper()
        return ENTRA_EDGE_MATURITY.get(
            key,
            ("Cloud Control Relationships", "Privileged Entra ID or Azure control relationship"),
        )

    if edge_column:
        profiles = enriched[edge_column].map(edge_profile)
        enriched["Maturity Domain"] = profiles.map(lambda item: item[0])
        enriched["finding"] = profiles.map(lambda item: item[1])
    else:
        enriched["Maturity Domain"] = "Cloud Control Relationships"
        enriched["finding"] = "Privileged Entra ID or Azure control relationship"

    enriched["abuse_summary"] = (
        "The source can influence a cloud identity, application, role assignment, "
        "credential, or synchronized trust relationship."
    )
    enriched["target_risk"] = (
        "Entra ID and Azure control relationships can provide direct or indirect "
        "access to the cloud identity control plane."
    )
    enriched["remediation_hint"] = (
        "Validate business need, remove unnecessary standing control, use eligible "
        "role activation where supported, and monitor changes to privileged cloud objects."
    )

    preferred = [
        "source", "Maturity Domain", "finding", "edge", "target",
        "abuse_summary", "target_risk", "remediation_hint",
    ]
    ordered = [column for column in preferred if column in enriched.columns]
    remaining = [column for column in enriched.columns if column not in ordered]
    return clean_final_dataframe(enriched[ordered + remaining])

DANGEROUS_PERMISSION_NAMES = {
    "GenericAll": "Full control over target",
    "GenericWrite": "Write access to target attributes",
    "WriteDacl": "Can modify target permissions",
    "WriteOwner": "Can take ownership of target",
    "Owns": "Owns target object",
    "AddMember": "Can add members to target group",
    "ForceChangePassword": "Can reset target password",
    "AllExtendedRights": "Has all extended rights on target",
}

def dangerous_permission_enrichment(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return clean_final_dataframe(df)
    enriched = df.copy()
    if "edge" in enriched.columns:
        enriched["finding"] = enriched["edge"].map(lambda e: DANGEROUS_PERMISSION_NAMES.get(str(e), "Dangerous delegated permission"))
    else:
        enriched["finding"] = "Dangerous delegated permission"
    enriched["abuse_summary"] = "The source has a permission that may enable control or escalation against the target."
    enriched["target_risk"] = "Delegated permissions can create hidden privilege escalation paths."
    enriched["remediation_hint"] = "Validate business need and remove unnecessary delegated rights."
    preferred = ["source", "finding", "edge", "target", "abuse_summary", "target_risk", "remediation_hint"]
    ordered = [c for c in preferred if c in enriched.columns]
    remaining = [c for c in enriched.columns if c not in ordered]
    return clean_final_dataframe(enriched[ordered + remaining])

## Branding and Output Naming


In [ ]:
def safe_filename_part(value: Any) -> str:
    """Return a filesystem-safe report filename component."""
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", str(value or "").strip())
    return text.strip("._-") or "BloodHound_Report"


def report_output_stem() -> str:
    """Build the shared stem used by the PDF and supporting workbook."""
    customer = safe_filename_part(report_cfg.get("customer_name", "Customer"))
    version = safe_filename_part(report_cfg.get("report_version", "1.0"))
    return f"{customer}_BHE_Identity_Risk_Report_{version}"


display(Markdown(f"Output stem will be: `{report_output_stem()}`"))


## Professional Writing Rules


Rules:

1. Lead with customer-facing risk.
2. Use formal BloodHound / identity terminology.
3. Do not invent labels or names.
4. `Understanding the Risk` explains impact in plain language without analogies.
5. Keep collection failures visible in a Collection Issues section.
6. Keep technical evidence in the appendix or collapsed evidence sections.

In [ ]:
def formal_terms_for_section(section: "ReportSection") -> str:
    terms = []
    for finding in section.underlying_bhe_findings:
        if finding.startswith("T0"):
            terms.append(finding)
        elif finding.startswith("LargeDefaultGroups"):
            terms.append(finding)
        elif finding.startswith("AzureT0"):
            terms.append(finding)
        elif finding in {"Kerberoasting", "ASREPRoasting", "NonT0DCSyncers"}:
            terms.append(finding)

    terms = list(dict.fromkeys(terms))
    if not terms:
        return "BloodHound attack path relationships mapped to this risk theme."
    if len(terms) > 8:
        return ", ".join(terms[:8]) + ", and related BloodHound findings."
    return ", ".join(terms)

def executive_takeaway_for_section(section: "ReportSection") -> str:
    defaults = {
        "Tier Zero Exposure": "Several identity relationships may allow access to systems or identities that control the broader environment. Reducing these paths should be treated as a top remediation priority.",
        "Privilege Escalation": "Delegated permissions may allow users or groups to gain more control than their role requires. Cleaning up these permissions can reduce hidden paths to privileged access.",
        "Credential Exposure": "Some identity relationships may expose credentials or authentication material. Reducing these exposures limits an attacker's ability to turn one compromised account into broader access.",
        "Lateral Movement": "Administrative access paths may allow an attacker to move from one compromised identity or system to others. Reducing unnecessary remote administration lowers blast radius.",
        "NTLM Relay and Coercion": "Coercion and relay paths may allow attackers to abuse authentication flows rather than directly stealing passwords. These paths should be reviewed carefully where they involve Tier Zero or sensitive services.",
        "ADCS Exposure": "Certificate Services issues can allow attackers to obtain authentication material without needing a user's password. These paths can be high impact and should be validated carefully.",
        "Cloud and Hybrid Identity": "Cloud and hybrid relationships can extend identity risk beyond Active Directory into Entra ID, Azure, and other connected platforms.",
        "Identity Hygiene": "Identity hygiene issues may not create compromise alone, but they make other attack paths easier to use and harder to govern."
    }
    return defaults.get(section.risk_theme, section.executive_summary)

def what_bloodhound_found_for_section(section: "ReportSection") -> str:
    terms = formal_terms_for_section(section)
    return (
        f"BloodHound mapped this section to formal attack path relationships such as {terms}. "
        f"These terms are technical evidence labels; the report groups them into the broader risk theme of {section.risk_theme}."
    )

def risk_context_for_section(section: "ReportSection") -> str:
    contexts = {
        "Tier Zero Exposure": (
            "Tier Zero assets are the systems, identities, and control points that can affect the security of the broader environment. "
            "A path into Tier Zero does not always mean immediate compromise, but it shows that existing relationships may give an attacker a viable route toward control of critical identity infrastructure."
        ),
        "Privilege Escalation": (
            "Privilege escalation risk is created when a principal has permissions that can change another object, group, policy, or account in a way that increases effective access. "
            "These rights are often granted for legitimate administration, but unnecessary delegation can create hidden paths around the intended access model."
        ),
        "Credential Exposure": (
            "Credential exposure paths matter because attackers often prefer to obtain reusable authentication material rather than rely on noisy exploitation. "
            "Reducing these paths limits opportunities for offline password attacks, directory replication abuse, or access to managed account secrets."
        ),
        "Lateral Movement": (
            "Lateral movement paths represent ways an attacker could expand access after compromising an initial identity or system. "
            "Remote administration and local administrator relationships may be expected in some cases, but broad or poorly tiered access increases blast radius."
        ),
        "NTLM Relay and Coercion": (
            "Coercion and relay paths abuse authentication behavior rather than a user's password directly. "
            "They are most concerning when they can be chained to privileged services, certificate infrastructure, domain controllers, or other Tier Zero assets."
        ),
        "ADCS Exposure": (
            "Certificate Services can issue authentication material that is trusted by the environment. "
            "When templates, enrollment rights, or CA administration are misconfigured, attackers may be able to obtain certificates that allow them to authenticate as another identity."
        ),
        "Cloud and Hybrid Identity": (
            "Hybrid identity relationships can allow risk to cross between on-prem Active Directory and cloud control planes. "
            "This matters because cloud roles, managed identities, app ownership, or synchronization relationships may create paths to privileged administration outside traditional AD boundaries."
        ),
        "Identity Hygiene": (
            "Identity hygiene issues increase the durability and usefulness of compromised access. "
            "Stale accounts, non-expiring passwords, and weak governance do not always create a direct attack path alone, but they make other paths easier to exploit and harder to detect."
        ),
    }
    return contexts.get(section.risk_theme, section.why_this_matters)

display(Markdown("Professional writing helpers loaded."))

## 5. Risk Theme Report Sections

In [ ]:
# ---------------------------------------------------------------------------
# Authoritative risk-theme metadata registry
# Presentation language and recommendations live here exactly once.
# Evidence query definitions remain in SECTIONS below.
# ---------------------------------------------------------------------------
THEME_METADATA: Dict[str, Dict[str, Any]] = {'tier_zero_exposure': {'title': 'Tier Zero Exposure',
                        'risk_theme': 'Tier Zero Exposure',
                        'maturity_domain': 'Privileged Access Governance',
                        'executive_priority': 1,
                        'default_risk': 'Critical',
                        'executive_summary': 'The assessment identified paths that may lead to Tier Zero '
                                             'assets, the systems and identities that control the rest of '
                                             'the environment.',
                        'business_impact': 'Compromise of Tier Zero can result in broad control over '
                                           'identity infrastructure and downstream systems.',
                        'why_this_matters': 'Attackers do not need direct Domain Admin membership if other '
                                            'paths eventually lead to equivalent control.',
                        'technical_summary': 'Representative evidence uses last-hop relationships from '
                                             'non-Tier Zero principals into objects tagged as Tier Zero.',
                        'underlying_bhe_findings': ['T0Admins',
                                                    'T0MemberOf',
                                                    'T0Logins',
                                                    'T0PSRemote',
                                                    'T0RDP',
                                                    'T0DCOM',
                                                    'T0SQLAdmin'],
                        'immediate_actions': ['Review non-Tier Zero identities with paths into Tier Zero.',
                                              'Remove direct administrative access that is not explicitly '
                                              'required.',
                                              'Validate that all Tier Zero assets are tagged correctly.'],
                        'near_term_actions': ['Reduce nested group relationships into Tier Zero.',
                                              'Separate operational administration from Tier Zero '
                                              'administration.',
                                              'Document approved Tier Zero administration paths.'],
                        'strategic_actions': ['Implement continuous monitoring for new paths into Tier Zero.',
                                              'Formalize a Tier Zero governance model.',
                                              'Review Tier Zero access as part of recurring identity '
                                              'security operations.'],
                        'insight': 'Tier Zero exposure reflects governance gaps that allow operational '
                                   'identities to retain unnecessary routes into critical control systems.'},
 'privilege_escalation': {'title': 'Privilege Escalation Opportunities',
                          'risk_theme': 'Privilege Escalation',
                          'maturity_domain': 'Identity Governance & Delegation',
                          'executive_priority': 2,
                          'default_risk': 'High',
                          'executive_summary': 'Identity analysis identified delegated permissions that may '
                                               'allow users or groups to escalate privileges beyond their '
                                               'intended roles.',
                          'business_impact': 'Delegated control can quietly bypass formal administrative '
                                             'boundaries and create shortcuts to privileged access.',
                          'why_this_matters': 'Privilege escalation often comes from ordinary-looking '
                                              'permissions that accumulate into administrative control.',
                          'technical_summary': 'Representative evidence includes dangerous ACL-style '
                                               'relationships such as GenericAll, WriteDacl, WriteOwner, '
                                               'AddMember, and ForceChangePassword.',
                          'underlying_bhe_findings': ['T0GenericAll',
                                                      'T0GenericWrite',
                                                      'T0WriteDACL',
                                                      'T0WriteOwner',
                                                      'T0Owns',
                                                      'T0AddMember',
                                                      'T0ForceChangePassword',
                                                      'T0AllExtendedRights',
                                                      'LargeDefaultGroupsGenericAll',
                                                      'LargeDefaultGroupsWriteDacl'],
                          'immediate_actions': ['Review broad delegated permissions on sensitive users, '
                                                'groups, computers, and GPOs.',
                                                'Remove GenericAll, WriteDacl, and WriteOwner where not '
                                                'explicitly required.',
                                                'Validate ownership of privileged groups and administrative '
                                                'objects.'],
                          'near_term_actions': ['Replace broad delegation with narrow, task-specific '
                                                'administration.',
                                                'Audit inherited permissions on privileged containers and '
                                                'groups.',
                                                'Document approved delegated administration paths.'],
                          'strategic_actions': ['Continuously monitor changes to dangerous permissions.',
                                                'Move privileged workflows toward just-in-time access where '
                                                'practical.',
                                                'Add delegated permission review to identity governance '
                                                'processes.'],
                          'insight': 'Delegated permissions frequently accumulate over time and can provide '
                                     'administrative control well beyond their intended operational '
                                     'purpose.'},
 'credential_exposure': {'title': 'Credential Exposure',
                         'risk_theme': 'Credential Exposure',
                         'maturity_domain': 'Credential Security',
                         'executive_priority': 3,
                         'default_risk': 'High',
                         'executive_summary': 'Review of the identity graph identified credential exposures '
                                              'that could allow attackers to obtain or crack account '
                                              'secrets.',
                         'business_impact': 'Credential exposure can turn limited access into durable access '
                                            'to privileged systems and identities.',
                         'why_this_matters': 'Attackers often prefer stealing or cracking credentials '
                                             'because it lets them blend into normal authentication flows.',
                         'technical_summary': 'Representative evidence includes DCSync and '
                                              'Kerberoasting-style exposure.',
                         'underlying_bhe_findings': ['NonT0DCSyncers',
                                                     'Kerberoasting',
                                                     'ASREPRoasting',
                                                     'T0ReadLAPS',
                                                     'T0ReadGMSA',
                                                     'T0DumpSMSA'],
                         'immediate_actions': ['Review principals with DCSync or replication-like '
                                               'permissions.',
                                               'Prioritize Kerberoastable accounts with old passwords or '
                                               'privileged access.',
                                               'Rotate high-risk service account passwords.'],
                         'near_term_actions': ['Move eligible services to managed service accounts.',
                                               'Reduce unnecessary SPNs.',
                                               'Review LAPS and gMSA read access.'],
                         'strategic_actions': ['Implement recurring service account governance.',
                                               'Continuously monitor credential exposure paths.',
                                               'Alert on unexpected directory replication behavior.'],
                         'insight': 'Credential exposure shifts attacks away from monitored systems and into '
                                    'offline techniques that are significantly harder to detect.'},
 'lateral_movement': {'title': 'Lateral Movement',
                      'risk_theme': 'Lateral Movement',
                      'maturity_domain': 'Administrative Tiering',
                      'executive_priority': 4,
                      'default_risk': 'High',
                      'executive_summary': 'The assessment found administrative access paths that may help '
                                           'attackers move across systems after initial compromise.',
                      'business_impact': 'Overly broad remote administration makes it easier for an attacker '
                                         'to expand from one compromised identity to many systems.',
                      'why_this_matters': 'Lateral movement paths often represent legitimate administration, '
                                          'but unnecessary breadth increases blast radius.',
                      'technical_summary': 'Analysis focuses on AdminTo relationships by default, with '
                                           'BloodHound Enterprise mappings for PSRemote, RDP, DCOM, and '
                                           'SQLAdmin.',
                      'underlying_bhe_findings': ['T0PSRemote',
                                                  'LargeDefaultGroupsPSRemote',
                                                  'T0RDP',
                                                  'LargeDefaultGroupsRDP',
                                                  'T0DCOM',
                                                  'LargeDefaultGroupsDCOM',
                                                  'T0SQLAdmin',
                                                  'LargeDefaultGroupsSQLAdmin'],
                      'immediate_actions': ['Review remote administration paths to sensitive systems.',
                                            'Remove broad local administrator assignments.',
                                            'Prioritize paths from non-Tier Zero identities to sensitive '
                                            'assets.'],
                      'near_term_actions': ['Segment workstation, server, and Tier Zero administration.',
                                            'Reduce standing administrative rights.',
                                            'Implement privileged access workstations where appropriate.'],
                      'strategic_actions': ['Continuously monitor local administrator drift.',
                                            'Adopt just-in-time local administration where feasible.',
                                            'Define approved remote administration patterns.'],
                      'insight': 'Administrative access should be governed as an enterprise capability '
                                 'because individually acceptable permissions can combine into high-impact '
                                 'attack paths.'},
 'ntlm_relay_coercion': {'title': 'NTLM Relay and Coercion',
                         'risk_theme': 'NTLM Relay and Coercion',
                         'maturity_domain': 'Authentication Hardening',
                         'executive_priority': 5,
                         'default_risk': 'High',
                         'executive_summary': 'Identity analysis identified coercion and relay relationships '
                                              'that may allow attackers to abuse authentication flows.',
                         'business_impact': 'Relay paths can allow attackers to turn authentication behavior '
                                            'into access, especially where privileged services or Tier Zero '
                                            'systems are involved.',
                         'why_this_matters': 'These paths may not require stealing a password directly; they '
                                             'can abuse how systems authenticate to each other.',
                         'technical_summary': 'Analysis evaluates NTLM coercion and relay relationships '
                                              'available in the graph.',
                         'underlying_bhe_findings': ['T0CoerceAndRelayNTLMToADCS',
                                                     'T0CoerceAndRelayNTLMToLDAP',
                                                     'T0CoerceAndRelayNTLMToLDAPS',
                                                     'T0CoerceAndRelayNTLMToSMB',
                                                     'T0CoerceToTGT',
                                                     'LargeDefaultGroupsCoerceAndRelayNTLMToSMB'],
                         'immediate_actions': ['Review relay/coercion paths involving Tier Zero or sensitive '
                                               'services.',
                                               'Prioritize SMB/LDAP/LDAPS relay exposure where signing or '
                                               'channel binding is not enforced.',
                                               'Validate ADCS relay exposure separately from general NTLM '
                                               'relay risk.'],
                         'near_term_actions': ['Harden LDAP signing and channel binding where applicable.',
                                               'Restrict unnecessary authentication coercion paths.',
                                               'Review service exposure that enables relay to privileged '
                                               'targets.'],
                         'strategic_actions': ['Adopt an NTLM reduction strategy where feasible.',
                                               'Monitor for authentication coercion and relay activity.',
                                               'Integrate relay-path review into identity attack path '
                                               'management.'],
                         'insight': 'Coercion and relay paths abuse authentication behavior rather than a '
                                    "user's password directly."},
 'adcs_exposure': {'title': 'Certificate Services Exposure',
                   'risk_theme': 'ADCS Exposure',
                   'maturity_domain': 'Certificate Services Governance',
                   'executive_priority': 6,
                   'default_risk': 'High',
                   'executive_summary': 'Certificate Services misconfigurations can allow attackers to '
                                        'obtain trusted authentication material without needing a password.',
                   'business_impact': 'ADCS abuse can bypass normal password controls and create durable '
                                      'privilege escalation paths into highly privileged identities.',
                   'why_this_matters': 'Certificate-based attacks are high impact because the resulting '
                                       'certificate may be accepted as the target identity by systems that '
                                       'trust the issuing PKI.',
                   'technical_summary': 'Representative evidence includes risky certificate template '
                                        'settings, certificate authority administrative control, and '
                                        'enrollment relationships using explicit ADCS object types and '
                                        'edges.',
                   'underlying_bhe_findings': ['T0ADCSESC1',
                                               'T0ADCSESC3',
                                               'T0ADCSESC4',
                                               'T0ADCSESC6a',
                                               'T0ADCSESC6b',
                                               'T0ADCSESC9a',
                                               'T0ADCSESC9b',
                                               'T0ADCSESC10a',
                                               'T0ADCSESC10b',
                                               'T0ADCSESC13',
                                               'T0ManageCA',
                                               'T0ManageCertificates',
                                               'T0GoldenCert'],
                   'immediate_actions': ['Review authentication-capable templates that allow subject '
                                         'control, omit the security extension, or lack approval controls.',
                                         'Restrict ManageCA and ManageCertificates rights to dedicated Tier '
                                         'Zero PKI administrators.',
                                         'Review template control and enrollment rights for broad or '
                                         'non-privileged groups.'],
                   'near_term_actions': ['Require manager approval or authorized signatures where '
                                         'appropriate.',
                                         'Remove unnecessary enrollment and template modification rights.',
                                         'Unpublish unused or unsafe templates and review CA publication '
                                         'scope.'],
                   'strategic_actions': ['Implement recurring ADCS security reviews.',
                                         'Monitor certificate issuance and CA administration for suspicious '
                                         'activity.',
                                         'Document approved certificate administration and enrollment '
                                         'workflows.'],
                   'insight': 'Certificate-based authentication is inherently trusted, making certificate '
                              'governance as important as password governance.'},
 'cloud_hybrid_identity': {'title': 'Cloud and Hybrid Identity',
                           'risk_theme': 'Cloud and Hybrid Identity',
                           'maturity_domain': 'Hybrid Identity Governance',
                           'executive_priority': 7,
                           'default_risk': 'High',
                           'executive_summary': 'Enterprise analysis identified Entra ID, Azure, and hybrid '
                                                'identity relationships that expose cloud control-plane '
                                                'privileges or connect on-premises identities to cloud '
                                                'administration.',
                           'business_impact': 'Compromise of synchronized identities, privileged role '
                                              'assignments, applications, service principals, or Azure '
                                              'resource permissions can provide broad control across the '
                                              'cloud environment.',
                           'why_this_matters': 'Cloud privilege is distributed across directory roles, '
                                               'application ownership, credentials, Azure RBAC, and '
                                               'synchronization. A path does not need to end at Global '
                                               'Administrator to create control-plane risk.',
                           'technical_summary': 'Representative evidence is organized across application and '
                                                'workload governance, credential governance, privilege and '
                                                'access governance, identity administration, '
                                                'privileged-access lifecycle, and hybrid trust.',
                           'underlying_bhe_findings': ['AzureT0AddOwner',
                                                       'AzureT0AddSecret',
                                                       'AzureT0MGGrantRole',
                                                       'AzureT0MGGrantAppRoles',
                                                       'AzureT0ResetPassword',
                                                       'AzureT0RoleEligible',
                                                       'AzureT0SyncedToEntraUser',
                                                       'AzureT0UserAccessAdministrator',
                                                       'AzureT0Contributor',
                                                       'AzureT0Owner',
                                                       'AzureT0GlobalAdmin',
                                                       'AzureT0PrivilegedRoleAdmin'],
                           'immediate_actions': ['Review identities with direct control over privileged '
                                                 'roles, applications, service principals, credentials, and '
                                                 'Azure resource permissions.',
                                                 'Remove stale owners, unused application secrets, '
                                                 'unnecessary role assignments, and standing access that can '
                                                 'change the cloud control plane.',
                                                 'Validate synchronized identities that connect on-prem '
                                                 'compromise paths to privileged Entra ID or Azure access.'],
                           'near_term_actions': ['Move eligible privileged assignments to just-in-time '
                                                 'activation and require strong approval and authentication '
                                                 'controls.',
                                                 'Establish recurring reviews for application ownership, '
                                                 'service-principal credentials, Microsoft Graph '
                                                 'permissions, and Azure RBAC.',
                                                 'Separate cloud administration from daily-use and on-prem '
                                                 'identities wherever practical.'],
                           'strategic_actions': ['Define Entra ID and Azure Tier Zero assets and govern them '
                                                 'with the same rigor as on-prem identity infrastructure.',
                                                 'Monitor cross-platform attack paths between Active '
                                                 'Directory, Entra ID, applications, and Azure resources.',
                                                 'Measure cloud privilege reduction, credential age, '
                                                 'eligible-versus-standing access, and ownership review '
                                                 'completion as identity-program maturity indicators.'],
                           'insight': 'Cloud identity maturity depends on governing every route to '
                                      'control—not only highly visible directory roles.'},
 'identity_hygiene': {'title': 'Identity Lifecycle & Hygiene',
                      'risk_theme': 'Identity Hygiene',
                      'maturity_domain': 'Identity Lifecycle Management',
                      'executive_priority': 8,
                      'default_risk': 'Moderate',
                      'executive_summary': 'The assessment identified weaknesses across password governance, '
                                           'authentication hardening, identity lifecycle, and administrative '
                                           'hygiene.',
                      'business_impact': 'Weak lifecycle and authentication controls allow credentials, '
                                         'accounts, and legacy trust settings to remain useful longer than '
                                         'intended.',
                      'why_this_matters': 'These conditions rarely create compromise alone, but they reveal '
                                          'inconsistent identity governance and make other attack paths '
                                          'easier to exploit and harder to contain.',
                      'technical_summary': 'Representative evidence spans four identity-program maturity '
                                           'domains rather than treating hygiene as a single password-policy '
                                           'issue.',
                      'underlying_bhe_findings': ['PasswordNotRequired',
                                                  'PasswordNeverExpires',
                                                  'Kerberoasting',
                                                  'UnconstrainedDelegation',
                                                  'DisabledIdentityLifecycleReview',
                                                  'AdminCount'],
                      'immediate_actions': ['Remove unnecessary password-policy exceptions and harden legacy '
                                            'authentication settings.',
                                            'Prioritize privileged, service, and legacy identities with '
                                            'roastable credentials or unconstrained delegation.',
                                            'Review disabled identities for stale ownership, memberships, '
                                            'delegation, and administrative access.'],
                      'near_term_actions': ['Establish a governed exception process for non-expiring '
                                            'passwords and service identities.',
                                            'Add deprovisioning checks for graph relationships, privileged '
                                            'memberships, ownership, and delegated access.',
                                            'Review AdminCount identities for stale privilege and '
                                            'protected-object drift.'],
                      'strategic_actions': ['Measure identity hygiene across password governance, '
                                            'authentication hardening, lifecycle management, and '
                                            'administrative hygiene.',
                                            'Run recurring lifecycle and exception reviews using BloodHound '
                                            'evidence.',
                                            'Track remediation trends as an identity-program maturity '
                                            'indicator.'],
                      'insight': 'Identity hygiene is not a list of isolated account settings. It shows '
                                 'whether identity controls are consistently governed across the full '
                                 'account lifecycle.'}}

def theme_metadata(section_id: str) -> Dict[str, Any]:
    """Return a defensive copy of the authoritative metadata for a report section."""
    if section_id not in THEME_METADATA:
        raise KeyError(f"Unknown report section metadata: {section_id}")
    metadata = THEME_METADATA[section_id]
    return {
        key: list(value) if isinstance(value, list) else value
        for key, value in metadata.items()
    }


SECTIONS: List[ReportSection] = [
    ReportSection(
        id='tier_zero_exposure',
        enabled=True,
        evidence_sources=[
                    EvidenceSource(
                        id="tier_zero_paths",
                        label="Paths to Tier Zero",
                        query="""
        MATCH (source)-[edge]->(target)
        WHERE target.system_tags CONTAINS 'admin_tier_0'
          AND NOT source.system_tags CONTAINS 'admin_tier_0'
        RETURN source.name AS source,
               type(edge) AS edge_type,
               target.name AS target
        LIMIT 50
        """,
                        enricher=standard_enrichment(
                            "Tier Zero Chokepoint",
                            "The source has a direct final-step relationship into a Tier Zero asset.",
                            "Tier Zero exposure can enable full identity compromise.",
                            "Break unnecessary path segments into Tier Zero and restrict delegated control."
                        )
                    )
                ],
        **theme_metadata('tier_zero_exposure'),
    ),
    ReportSection(
        id='privilege_escalation',
        enabled=True,
        evidence_sources=[
                    EvidenceSource(
                        id="dangerous_permissions",
                        label="Dangerous Delegated Permissions",
                        query="""
        MATCH (s)-[r]->(t)
        WHERE type(r) IN ['GenericAll','GenericWrite','WriteDacl','WriteOwner','Owns','AddMember','ForceChangePassword','AllExtendedRights']
        RETURN s.name AS source, type(r) AS edge, t.name AS target
        LIMIT 75
        """,
                        enricher=dangerous_permission_enrichment
                    )
                ],
        **theme_metadata('privilege_escalation'),
    ),
    ReportSection(
        id='credential_exposure',
        enabled=True,
        evidence_sources=[
                    EvidenceSource(
                        id="dcsync",
                        label="DCSync Exposure",
                        query="""
        MATCH (s)-[r]->(d)
        WHERE type(r) = 'DCSync'
        RETURN s.name AS source, type(r) AS edge, d.name AS target
        LIMIT 50
        """,
                        enricher=standard_enrichment(
                            "Can replicate directory secrets",
                            "The source may be able to perform DCSync-style replication and retrieve password material.",
                            "DCSync can lead directly to domain credential theft.",
                            "Remove replication rights from nonessential principals and monitor replication API usage."
                        )
                    ),
                    EvidenceSource(
                        id="kerberoastable",
                        label="Kerberoastable Accounts",
                        query="""
        MATCH (u)
        WHERE u.hasspn = true OR u.serviceprincipalnames IS NOT NULL
        RETURN u.name AS source, u.enabled AS enabled, u.pwdlastset AS pwdlastset
        LIMIT 100
        """,
                        enricher=standard_enrichment(
                            "Kerberoastable service account",
                            "The account has an SPN and may allow attackers to request a service ticket for offline password cracking.",
                            "Weak or reused service account passwords can lead to privilege escalation or lateral movement.",
                            "Use managed service accounts, rotate stale passwords, and review service account privileges."
                        )
                    )
                ],
        **theme_metadata('credential_exposure'),
    ),
    ReportSection(
        id='lateral_movement',
        enabled=True,
        evidence_sources=[
                    EvidenceSource(
                        id="admin_to",
                        label="Local Administrator Paths",
                        query="""
        MATCH (s)-[r]->(t)
        WHERE type(r) = 'AdminTo'
        RETURN s.name AS source, type(r) AS edge, t.name AS target
        LIMIT 75
        """,
                        enricher=standard_enrichment(
                            "Local administrative control",
                            "The source may have administrative access to the target system.",
                            "Local administrator rights can enable lateral movement and credential exposure.",
                            "Restrict local administrator rights and enforce tiered administration."
                        )
                    )
                ],
        **theme_metadata('lateral_movement'),
    ),
    ReportSection(
        id='ntlm_relay_coercion',
        enabled=True,
        evidence_sources=[
                    EvidenceSource(
                        id="ntlm_smb_relay_paths",
                        label="NTLM Relay to SMB",
                        query="""
        MATCH (s)-[r]->(t)
        WHERE type(r) = 'CoerceAndRelayNTLMToSMB'
        RETURN s.name AS source, type(r) AS edge, t.name AS target
        LIMIT 75
        """,
                        enricher=standard_enrichment(
                            "NTLM relay to SMB",
                            "The source has a relationship that may support coerced authentication and relay to SMB.",
                            "SMB relay can contribute to privilege escalation or administrative access when signing protections are not enforced.",
                            "Enforce SMB signing where required, restrict coercion paths, and review affected administrative targets."
                        ),
                        optional=True
                    ),
                    EvidenceSource(
                        id="ntlm_coerce_to_tgt_paths",
                        label="NTLM Coercion to TGT",
                        query="""
        MATCH (s)-[r]->(t)
        WHERE type(r) = 'CoerceToTGT'
        RETURN s.name AS source, type(r) AS edge, t.name AS target
        LIMIT 75
        """,
                        enricher=standard_enrichment(
                            "NTLM coercion to Kerberos TGT",
                            "The source has a relationship that may allow authentication coercion to obtain Kerberos authentication material.",
                            "Coerce-to-TGT paths can support privilege escalation without directly stealing a user's password.",
                            "Restrict coercion-capable services, reduce unnecessary privileged authentication exposure, and monitor for abnormal ticket requests."
                        ),
                        optional=True,
                        enabled=False  # Unsupported direct graph edge in BHE 9.4; native finding coverage remains enabled.
                    )
                ],
        **theme_metadata('ntlm_relay_coercion'),
    ),
    ReportSection(
        id='adcs_exposure',
        enabled=True,
        evidence_sources=[
                    EvidenceSource(
                        id="risky_certificate_templates",
                        label="Risky Certificate Templates",
                        query="""
        MATCH (t:CertTemplate)
        WHERE t.authenticationenabled = true
          AND (
            t.enrolleesuppliessubject = true
            OR t.nosecurityextension = true
            OR t.requiresmanagerapproval = false
          )
        RETURN t.name AS template,
               t.authenticationenabled AS authentication_enabled,
               t.enrolleesuppliessubject AS enrollee_supplies_subject,
               t.nosecurityextension AS no_security_extension,
               t.requiresmanagerapproval AS requires_manager_approval,
               t.authorizedsignatures AS authorized_signatures,
               t.effectiveekus AS effective_ekus,
               t.enrollmentflag AS enrollment_flags,
               t.schemaversion AS schema_version
        LIMIT 100
        """,
                        enricher=standard_enrichment(
                            "Potentially risky certificate template",
                            "The template supports authentication and has settings that may increase abuse potential.",
                            "Risky template settings can allow certificate-based impersonation or privilege escalation.",
                            "Review enrollment permissions, subject name controls, EKUs, manager approval, signatures, and the security extension."
                        ),
                        optional=True
                    ),
                    EvidenceSource(
                        id="certificate_authority_admin_control",
                        label="Certificate Authority Administrative Control",
                        query="""
        MATCH (s)-[r]->(ca:EnterpriseCA)
        WHERE type(r) IN ['ManageCA','ManageCertificates','GenericAll','GenericWrite','WriteDacl','WriteOwner','Owns']
        RETURN s.name AS source, type(r) AS edge, ca.name AS target
        LIMIT 100
        """,
                        enricher=standard_enrichment(
                            "Certificate authority administrative control",
                            "The source has administrative or ownership control over an enterprise certification authority.",
                            "Control of a CA can allow certificate issuance, configuration changes, or creation of highly privileged authentication material.",
                            "Restrict CA administration to dedicated Tier Zero identities and review all delegated control."
                        ),
                        optional=True
                    ),
                    EvidenceSource(
                        id="certificate_template_admin_control",
                        label="Certificate Template Administrative Control",
                        query="""
        MATCH (s)-[r]->(t:CertTemplate)
        WHERE type(r) IN [
            'GenericAll',
            'GenericWrite',
            'WriteDacl',
            'WriteOwner',
            'Owns'
        ]
        RETURN s.name AS source, type(r) AS edge, t.name AS target
        LIMIT 30
        """,
                        enricher=standard_enrichment(
                            "Certificate template administrative control",
                            "The source can modify, own, or otherwise administer a certificate template.",
                            "Control of an authentication-capable template can enable certificate-based privilege escalation.",
                            "Restrict template administration to approved PKI identities and remove unnecessary delegated control."
                        ),
                        optional=True
                    ),
                    EvidenceSource(
                        id="certificate_enrollment_relationships",
                        label="Certificate Enrollment Relationships",
                        query="""
        MATCH (s)-[r]->(t:CertTemplate)
        WHERE type(r) = 'Enroll'
        RETURN s.name AS source, type(r) AS edge, t.name AS target
        LIMIT 150
        """,
                        enricher=standard_enrichment(
                            "Certificate enrollment relationship",
                            "The source is permitted to enroll in the certificate template.",
                            "Enrollment becomes dangerous when combined with authentication-capable or otherwise misconfigured templates.",
                            "Review who can enroll and remove broad enrollment rights from risky templates."
                        ),
                        optional=True
                    ),
                    EvidenceSource(
                        id="certificate_template_publication",
                        label="Certificate Template Publication",
                        query="""
        MATCH (t:CertTemplate)-[r]->(ca:EnterpriseCA)
        WHERE type(r) = 'PublishedTo'
        RETURN t.name AS source, type(r) AS edge, ca.name AS target
        LIMIT 150
        """,
                        enricher=standard_enrichment(
                            "Published certificate template",
                            "The template is published to an enterprise certification authority.",
                            "A risky template must generally be published before it can be abused through that CA.",
                            "Unpublish unused or unsafe templates and review publication against approved PKI design."
                        ),
                        optional=True
                    )
                ],
        **theme_metadata('adcs_exposure'),
    ),
    ReportSection(
        id='cloud_hybrid_identity',
        enabled=True,
        evidence_sources=[
                    EvidenceSource(
                        id="entra_azure_control_edges",
                        label="Entra ID and Azure Maturity Evidence",
                        query="""
        MATCH (s)-[r]->(t)
        WHERE type(r) IN [
            'AZOwns','AZMGAddOwner','AZMGAddSecret','AZMGGrantAppRoles',
            'AZMGGrantRole','AZResetPassword','AZResetUserPassword',
            'AZUserAccessAdministrator','AZAddMembers','AZAddSecret','AZAddOwner',
            'AZRoleAssignment','AZContributor','AZOwner','AZRoleEligible',
            'AZGlobalAdmin','AZPrivilegedRoleAdmin','AZSyncedToADUser',
            'AZSyncToADUser','SyncedToADUser'
        ]
        RETURN s.name AS source,
               type(r) AS edge,
               t.name AS target
        LIMIT 250
        """,
                        enricher=entra_azure_enrichment,
                        optional=True
                    )
                ],
        **theme_metadata('cloud_hybrid_identity'),
    ),
    ReportSection(
        id='identity_hygiene',
        enabled=True,
        evidence_sources=[
                    EvidenceSource(
                        id="password_not_required",
                        label="Password Governance — Password Not Required",
                        query="""
        MATCH (u)
        WHERE u.passwordnotreqd = true
        RETURN u.name AS affected_object,
               u.enabled AS enabled,
               u.pwdlastset AS pwdlastset,
               'PasswordNotRequired' AS condition
        LIMIT 250
        """,
                        enricher=hygiene_enrichment(
                            "Password Governance",
                            "Password Not Required",
                            "The account is configured with a password-policy exception that can permit weak or absent password enforcement.",
                            "High",
                            "Remove unnecessary Password Not Required settings and formally govern any documented exception."
                        ),
                        optional=True,
                    ),
                    EvidenceSource(
                        id="password_never_expires",
                        label="Password Governance — Password Never Expires",
                        query="""
        MATCH (u)
        WHERE u.pwdneverexpires = true
        RETURN u.name AS affected_object,
               u.enabled AS enabled,
               u.pwdlastset AS pwdlastset,
               'PasswordNeverExpires' AS condition
        LIMIT 250
        """,
                        enricher=hygiene_enrichment(
                            "Password Governance",
                            "Password Never Expires",
                            "Long-lived password exceptions increase persistence risk when credentials are exposed.",
                            "Moderate",
                            "Establish password lifecycle governance, review exceptions, and move eligible service identities to managed credentials."
                        ),
                        optional=True,
                    ),
                    EvidenceSource(
                        id="kerberoastable",
                        label="Authentication Hardening — Kerberoastable",
                        query="""
        MATCH (u)
        WHERE u.hasspn = true AND u.enabled = true
        RETURN u.name AS affected_object,
               u.enabled AS enabled,
               u.pwdlastset AS pwdlastset,
               'Kerberoastable' AS condition
        LIMIT 250
        """,
                        enricher=hygiene_enrichment(
                            "Authentication Hardening",
                            "Kerberoastable Account",
                            "A service principal can be requested for offline password-cracking attempts.",
                            "Moderate",
                            "Prioritize privileged or stale service accounts, rotate credentials, and use managed service accounts where possible."
                        ),
                        optional=True,
                    ),
                    EvidenceSource(
                        id="unconstrained_delegation",
                        label="Authentication Hardening — Unconstrained Delegation",
                        query="""
        MATCH (n)
        WHERE n.unconstraineddelegation = true
        RETURN n.name AS affected_object,
               n.enabled AS enabled,
               n.lastseen AS lastseen,
               'UnconstrainedDelegation' AS condition
        LIMIT 250
        """,
                        enricher=hygiene_enrichment(
                            "Authentication Hardening",
                            "Unconstrained Delegation",
                            "Legacy delegation can expose reusable authentication material when privileged identities connect to the affected system.",
                            "High",
                            "Remove unconstrained delegation and replace it with constrained or resource-based delegation where required."
                        ),
                        optional=True,
                    ),
                    EvidenceSource(
                        id="disabled_identity_lifecycle_review",
                        label="Identity Lifecycle — Disabled Accounts Requiring Review",
                        query="""
        MATCH (u)
        WHERE u.enabled = false
        RETURN u.name AS affected_object,
               u.admincount AS admincount,
               u.pwdlastset AS pwdlastset,
               'DisabledIdentityLifecycleReview' AS condition
        LIMIT 10
        """,
                        enricher=hygiene_enrichment(
                            "Identity Lifecycle",
                            "Disabled Account Requiring Lifecycle Review",
                            "A disabled identity remains present in the identity graph and should be reviewed for stale privilege, ownership, memberships, and delegated access.",
                            "Moderate",
                            "Complete deprovisioning by reviewing and removing obsolete memberships, ownership, delegation, and administrative access associated with disabled identities."
                        ),
                        optional=True,
                    ),
                    EvidenceSource(
                        id="admincount_identities",
                        label="Administrative Hygiene — AdminCount Identities",
                        query="""
        MATCH (u)
        WHERE u.admincount = true
        RETURN u.name AS affected_object,
               u.enabled AS enabled,
               u.pwdlastset AS pwdlastset,
               'AdminCount' AS condition
        LIMIT 250
        """,
                        enricher=hygiene_enrichment(
                            "Administrative Hygiene",
                            "AdminCount Identity",
                            "Historic or current administrative identities require review for stale privilege and protected-object drift.",
                            "Low",
                            "Review AdminCount identities, validate current privileged status, and remove stale administrative access or protection flags."
                        ),
                        optional=True,
                    ),
                ],
        **theme_metadata('identity_hygiene'),
    ),
]


## Extension Registry

In [ ]:
# --- Extension registry ---
import sys
PLUGINS_DIR = PROJECT_ROOT / "plugins"
if str(PLUGINS_DIR) not in sys.path:
    sys.path.insert(0, str(PLUGINS_DIR))

from plugin_loader import ExtensionManager

extensions_config_value = cfg.get("extensions_config")
if extensions_config_value:
    extensions_config_path = Path(extensions_config_value)
    if not extensions_config_path.is_absolute():
        extensions_config_path = PROJECT_ROOT / extensions_config_path
else:
    extensions_config_path = EXTENSIONS_FILE
if not extensions_config_path.exists():
    raise FileNotFoundError(
        f"Missing extension registry: {extensions_config_path}. "
        f"Expected the file at {EXTENSIONS_FILE} or set extensions_config in bhe_report_config.json."
    )

extension_registry = json.loads(
    extensions_config_path.read_text(encoding="utf-8")
).get("extensions", [])

# Normalize all extension paths against PROJECT_ROOT.
for extension in extension_registry:
    module_value = extension.get("module", "")
    if module_value:
        module_path = Path(module_value)
        if not module_path.is_absolute():
            # Support either "plugins/foo.py" or just "foo.py".
            candidate = PROJECT_ROOT / module_path
            if not candidate.exists():
                candidate = PLUGIN_DIR / module_path.name
            extension["module"] = str(candidate.resolve())

    search_value = extension.get("saved_search_zip", "")
    if search_value:
        search_path = Path(search_value)
        if not search_path.is_absolute():
            # Support either "searches/saved_searches.zip" or just "saved_searches.zip".
            candidate = PROJECT_ROOT / search_path
            if not candidate.exists():
                candidate = SEARCH_DIR / search_path.name
            extension["saved_search_zip"] = str(candidate.resolve())

extension_manager = ExtensionManager(PROJECT_ROOT, extension_registry)

for ext in extension_registry:
    risk_theme = ext.get("risk_theme", ext.get("title", ext["id"]))
    if risk_theme not in OBJECTIVE_ORDER:
        OBJECTIVE_ORDER.append(risk_theme)

    platform_name = str(ext.get("platform", ext["id"].title())).strip()
    extension_maturity_domains = {
        "github": "DevSecOps Identity Governance",
        "okta": "Workforce Identity Governance",
        "jamf": "Endpoint Identity Governance",
        "aws": "Cloud Workload Identity Governance",
        "gcp": "Cloud Workload Identity Governance",
        "sailpoint": "Identity Governance & Administration",
        "cyberark": "Privileged Access Governance",
        "beyondtrust": "Privileged Access Governance",
    }
    maturity_domain = ext.get(
        "maturity_domain",
        extension_maturity_domains.get(platform_name.lower(), f"{platform_name} Identity Governance"),
    )

    SECTIONS.append(ReportSection(
        id=f"extension_{ext['id']}",
        enabled=bool(ext.get("enabled", True)),
        title=ext.get("title", ext["id"].title()),
        risk_theme=risk_theme,
        maturity_domain=maturity_domain,
        executive_priority=int(ext.get("priority", 50)),
        default_risk=ext.get("severity", "High"),
        executive_summary=ext.get("executive_summary", ""),
        business_impact=ext.get("business_impact", ""),
        why_this_matters=ext.get("why_this_matters", ""),
        technical_summary=ext.get("technical_summary", ""),
        evidence_sources=[EvidenceSource(
            id=f"extension_source_{ext['id']}",
            label=f"{ext.get('platform', ext['id'].title())} Extension",
            query=f"EXTENSION::{ext['id']}",
            optional=True,
            enabled=True,
        )],
        underlying_bhe_findings=[f"Extension:{ext['id']}"],
        immediate_actions=ext.get("immediate_actions", []),
        near_term_actions=ext.get("near_term_actions", []),
        strategic_actions=ext.get("strategic_actions", []),
        insight=ext.get("insight", ""),
    ))

display(Markdown(f"Loaded `{len(extension_registry)}` extension definition(s)."))

## 6. Optional Report Designer

In [ ]:
# ---------------------------------------------------------------------------
# Deterministic report options
# ---------------------------------------------------------------------------
# Interactive ipywidgets were removed because the browser/kernel widget
# communication channel could intermittently stall notebook execution.
# Edit these plain values when a different run configuration is required.

REPORT_OPTIONS = {
    "max_evidence_rows": int(report_cfg.get("max_evidence_rows", 10)),
    "include_no_result_sections": bool(report_cfg.get("include_no_result_sections", False)),
    "enabled_section_ids": None,  # None keeps every configured section enabled.
}

enabled_section_ids = REPORT_OPTIONS["enabled_section_ids"]
for section in SECTIONS:
    section.enabled = (
        True if enabled_section_ids is None
        else section.id in set(enabled_section_ids)
    )

report_cfg["max_evidence_rows"] = REPORT_OPTIONS["max_evidence_rows"]
report_cfg["include_no_result_sections"] = REPORT_OPTIONS["include_no_result_sections"]

print(
    f"Report options applied: "
    f"{sum(section.enabled for section in SECTIONS)} sections enabled, "
    f"up to {report_cfg['max_evidence_rows']} evidence rows, "
    f"include empty sections={report_cfg['include_no_result_sections']}."
)


## 7. Run Collection

In [ ]:
def _extension_not_present_error(error: Exception | str) -> bool:
    """Return True only for the expected absent-platform response from BHE."""
    message = str(error).lower()
    return (
        "404" in message
        and "resource not found" in message
    )


unavailable_extension_ids: set[str] = set()
unavailable_extension_themes: set[str] = set()


def run_evidence_source(source: EvidenceSource):
    start = time.time()
    extension_id = None

    try:
        if source.query.startswith("EXTENSION::"):
            extension_id = source.query.split("::", 1)[1]

            # Do not retry a platform already confirmed absent in this tenant.
            if extension_id in unavailable_extension_ids:
                return None

            extension_result = extension_manager.collect(extension_id, bh)
            df = extension_result.evidence_df.copy()

            if extension_result.coverage_status == "Collection Issue" and df.empty:
                collection_note = (
                    extension_result.collection_note
                    or f"{extension_id} extension collection failed"
                )

                if _extension_not_present_error(collection_note):
                    unavailable_extension_ids.add(extension_id)
                    extension_manager._results.pop(extension_id, None)
                    return None

                raise RuntimeError(collection_note)
        else:
            df = bh.run(source.query)

        if source.enricher:
            df = source.enricher(df)
        else:
            df = clean_final_dataframe(df)

        return EvidenceResult(
            source=source,
            dataframe=df,
            elapsed_seconds=round(time.time() - start, 2),
        )

    except Exception as e:
        # GitHub, Okta, and Jamf are optional. A 404/resource-not-found
        # response means the platform is not present in this tenant, not that
        # collection failed. Remove the cached result and skip the section.
        if extension_id and _extension_not_present_error(e):
            unavailable_extension_ids.add(extension_id)
            extension_manager._results.pop(extension_id, None)
            return None

        return EvidenceResult(
            source=source,
            dataframe=pd.DataFrame(),
            error=f"{type(e).__name__}: {e}",
            elapsed_seconds=round(time.time() - start, 2),
        )


section_results: Dict[str, SectionResult] = {}

for section in SECTIONS:
    if not section.enabled:
        continue

    display(Markdown(f"Running **{section.title}**..."))

    evidence_results = []
    for source in section.evidence_sources:
        if not source.enabled:
            continue

        evidence_result = run_evidence_source(source)
        if evidence_result is not None:
            evidence_results.append(evidence_result)

    # An optional extension section with no result was confirmed unavailable
    # for this tenant. Do not add it to the report, summaries, or error tables.
    if section.id.startswith("extension_") and not evidence_results:
        extension_id = section.id.removeprefix("extension_")
        if extension_id in unavailable_extension_ids:
            unavailable_extension_themes.add(section.risk_theme)
            continue

    section_results[section.id] = SectionResult(
        section=section,
        evidence_results=evidence_results,
    )

# Hide unavailable optional platforms everywhere downstream in this run.
if unavailable_extension_ids:
    SECTIONS[:] = [
        section
        for section in SECTIONS
        if not (
            section.id.startswith("extension_")
            and section.id.removeprefix("extension_") in unavailable_extension_ids
        )
    ]
    OBJECTIVE_ORDER[:] = [
        theme
        for theme in OBJECTIVE_ORDER
        if theme not in unavailable_extension_themes
    ]

run_summary_rows = []
for result in section_results.values():
    for evidence in result.evidence_results:
        run_summary_rows.append({
            "Priority": f"P{result.section.executive_priority}",
            "Risk Theme": result.section.risk_theme,
            "Report Section": result.section.title,
            "Evidence Source": evidence.source.label,
            "Default Risk": result.section.default_risk,
            "Rows": evidence.count,
            "Elapsed Seconds": evidence.elapsed_seconds,
            "Status": "Collection Error" if evidence.error else ("No Evidence Returned" if evidence.dataframe.empty else "Evidence Returned"),
            "Error": evidence.error or "",
        })

run_summary_df = pd.DataFrame(run_summary_rows)
display(run_summary_df)

### 7A. Native Finding Catalogs


In [ ]:
# ---------------------------------------------------------------------------
# API-first BloodHound finding inventory and Representative Actionable Paths
# ---------------------------------------------------------------------------

from urllib.parse import urlencode


# Full native finding catalog observed in the BHE Attack Paths UI.
# Counts and narratives are built from this catalog. Representative cards are
# selected separately so a section count never depends on the number of cards.
NATIVE_FINDING_CATALOG = {
    "tier_zero_exposure": [
        "T0Admins",
        "T0Logins",
        "T0MarkSensitive",
        "T0AllowedToDelegate",
        "T0SyncLAPSPassword",
        "T0WriteAccountRestrictions",
        "T0AllExtendedRights",
        "T0GenericAll",
        "T0GenericWrite",
        "T0WriteDACL",
        "T0Owns",
        "T0AddMember",
        "T0DumpSMSA",
        "T0ReadGMSA",
        "T0GoldenCert",
        "T0CoerceAndRelayNTLMToSMB",
        "T0ADCSESC1",
        "T0ADCSESC4",
        "T0ADCSESC9a",
        "T0SyncedToADUser",
        "NonT0DCSyncers",
    ],
    "privilege_escalation": [
        "T0GenericAll",
        "T0GenericWrite",
        "T0WriteDACL",
        "T0Owns",
        "T0AddMember",
        "T0AllExtendedRights",
        "T0WriteAccountRestrictions",
    ],
    "credential_exposure": [
        "NonT0DCSyncers",
        "T0ReadGMSA",
        "T0DumpSMSA",
        "T0SyncLAPSPassword",
        "T0GoldenCert",
    ],
    "lateral_movement": [
        "T0Logins",
        "T0AllowedToDelegate",
        "T0Admins",
    ],
    "ntlm_relay_coercion": [
        "T0CoerceAndRelayNTLMToSMB",
    ],
    "adcs_exposure": [
        "T0ADCSESC1",
        "T0ADCSESC4",
        "T0ADCSESC9a",
        "T0GoldenCert",
    ],
    "cloud_hybrid_identity": [
        "T0SyncedToADUser",
        "AzureT0AddOwner",
        "AzureT0AddSecret",
        "AzureT0MGGrantRole",
        "AzureT0MGGrantAppRoles",
        "AzureT0ResetPassword",
        "AzureT0RoleEligible",
        "AzureT0SyncedToEntraUser",
        "AzureT0UserAccessAdministrator",
        "AzureT0Contributor",
        "AzureT0Owner",
        "AzureT0GlobalAdmin",
        "AzureT0PrivilegedRoleAdmin",
    ],
}


REPRESENTATIVE_FINDING_PRIORITY = {
    "tier_zero_exposure": [
        "NonT0DCSyncers",
        "T0GenericAll",
        "T0GenericWrite",
        "T0WriteDACL",
        "T0ADCSESC1",
        "T0SyncedToADUser",
    ],
    "privilege_escalation": [
        "T0GenericAll",
        "T0GenericWrite",
        "T0WriteDACL",
        "T0Owns",
        "T0AddMember",
    ],
    "credential_exposure": [
        "NonT0DCSyncers",
        "T0ReadGMSA",
        "T0DumpSMSA",
        "T0GoldenCert",
    ],
    "lateral_movement": [
        "T0Admins",
        "T0Logins",
        "T0AllowedToDelegate",
    ],
    "ntlm_relay_coercion": [
        "T0CoerceAndRelayNTLMToSMB",
    ],
    "adcs_exposure": [
        "T0ADCSESC1",
        "T0ADCSESC4",
        "T0ADCSESC9a",
        "T0GoldenCert",
    ],
    "cloud_hybrid_identity": [
        "AzureT0AddSecret",
        "AzureT0AddOwner",
        "AzureT0MGGrantRole",
        "AzureT0RoleEligible",
        "AzureT0SyncedToEntraUser",
        "T0SyncedToADUser",
    ],
}


NATIVE_EDGE_LABELS = {
    "NonT0DCSyncers": "DCSync",
    "T0ADCSESC1": "ADCS ESC1",
    "T0ADCSESC4": "ADCS ESC4",
    "T0ADCSESC9a": "ADCS ESC9",
    "T0GenericAll": "GenericAll",
    "T0GenericWrite": "GenericWrite",
    "T0WriteDACL": "WriteDACL",
    "T0Owns": "Owns",
    "T0AddMember": "AddMember",
    "AzureT0AddOwner": "Add application owner",
    "AzureT0AddSecret": "Add application secret",
    "AzureT0MGGrantRole": "Grant directory role",
    "AzureT0MGGrantAppRoles": "Grant application roles",
    "AzureT0ResetPassword": "Reset cloud identity password",
    "AzureT0RoleEligible": "Eligible privileged role",
    "AzureT0SyncedToEntraUser": "Synchronized privileged identity",
    "AzureT0UserAccessAdministrator": "User Access Administrator",
    "AzureT0Contributor": "Azure Contributor",
    "AzureT0Owner": "Azure Owner",
    "AzureT0GlobalAdmin": "Global Administrator",
    "AzureT0PrivilegedRoleAdmin": "Privileged Role Administrator",
    "T0AllExtendedRights": "AllExtendedRights",
    "T0WriteAccountRestrictions": "WriteAccountRestrictions",
    "T0Admins": "Administrative Access",
    "T0Logins": "Login Path",
    "T0AllowedToDelegate": "AllowedToDelegate",
    "T0MarkSensitive": "MarkSensitive",
    "T0SyncLAPSPassword": "SyncLAPSPassword",
    "T0ReadGMSA": "ReadGMSAPassword",
    "T0DumpSMSA": "DumpSMSAPassword",
    "T0GoldenCert": "Golden Certificate",
    "T0SyncedToADUser": "SyncedToADUser",
    "T0CoerceAndRelayNTLMToSMB": "Coerce and Relay NTLM to SMB",
}


### 7B. Native Finding Collection Helpers


In [ ]:


def bhe_get_json(
    path: str,
    params: Optional[Dict[str, Any]] = None,
    timeout: int = 45,
) -> Dict[str, Any]:
    params = params or {}

    query = urlencode(
        [
            (key, value)
            for key, value in params.items()
            if value is not None
        ],
        doseq=True,
    )

    uri = path + (f"?{query}" if query else "")

    response = requests.get(
        bh.base_url + uri,
        headers=bh._headers("GET", uri),
        verify=bh.verify_ssl,
        timeout=min(timeout, bh.timeout),
    )

    try:
        response.raise_for_status()
    except requests.HTTPError as error:
        raise RuntimeError(
            f"HTTP {response.status_code}: {response.text[:1000]}"
        ) from error

    return response.json()


def native_finding_metadata(finding_id: str) -> Dict[str, Any]:
    payload = bhe_get_json(f"/api/v2/findings/{finding_id}")
    return payload.get("data", {}) or {}


def is_tier_zero_props(properties: Dict[str, Any]) -> bool:
    tags = str((properties or {}).get("system_tags", "")).lower()
    return "admin_tier_0" in tags


def _extract_finding_ids_from_catalog_payload(payload: Any) -> List[str]:
    """Recursively extract native finding IDs from the product finding catalog."""
    found = set()

    def walk(value: Any) -> None:
        if isinstance(value, dict):
            title = str(value.get("title") or value.get("name") or "")
            candidate = value.get("id") or value.get("finding_id") or value.get("findingId") or value.get("slug")
            if candidate and (
                "privilege zone" in title.casefold()
                or any(key in value for key in ("short_description", "short_remediation", "type"))
            ):
                found.add(str(candidate).strip())
            for child in value.values():
                walk(child)
        elif isinstance(value, list):
            for child in value:
                walk(child)

    walk(payload)
    return sorted(item for item in found if item)


def discover_native_finding_ids() -> List[str]:
    """Discover the full native finding catalog instead of only the report's curated IDs."""
    discovered = set()
    catalog_paths = [
        "/api/v2/findings",
        "/api/v2/findings?skip=0&limit=1000",
        "/api/v2/findings?limit=1000",
    ]
    for path in catalog_paths:
        try:
            payload = bhe_get_json(path)
            discovered.update(_extract_finding_ids_from_catalog_payload(payload))
        except Exception:
            continue
    return sorted(discovered)


def all_native_finding_ids() -> List[str]:
    curated = {
        finding_id
        for finding_ids in NATIVE_FINDING_CATALOG.values()
        for finding_id in finding_ids
    }
    discovered = set(discover_native_finding_ids())
    combined = sorted(curated | discovered)
    print(
        f"Native finding catalog: {len(curated):,} curated + "
        f"{len(discovered):,} discovered = {len(combined):,} unique IDs"
    )
    return combined


def collect_native_finding_inventory(
    rows_per_request: int = 3,
) -> tuple[pd.DataFrame, Dict[str, Dict[str, Any]], Dict[str, List[Dict[str, Any]]], pd.DataFrame]:
    available = bhe_get_json("/api/v2/available-domains")
    environments = available.get("data", []) or []

    usable_environments = [
        environment
        for environment in environments
        if isinstance(environment, dict)
        and bool(environment.get("collected", False))
        and str(environment.get("type", "")).lower() in {
            "active-directory",
            "azure",
        }
    ]

    usable_environments.sort(
        key=lambda environment: float(
            environment.get("impactValue") or 0
        ),
        reverse=True,
    )

    metadata_cache: Dict[str, Dict[str, Any]] = {}
    inventory_rows = []
    detail_rows_by_finding: Dict[str, List[Dict[str, Any]]] = {}
    status_rows = []

    for finding_id in all_native_finding_ids():
        try:
            metadata_cache[finding_id] = native_finding_metadata(finding_id)
        except Exception as error:
            metadata_cache[finding_id] = {}
            status_rows.append({
                "Finding ID": finding_id,
                "Environment": "",
                "Status": "Metadata Error",
                "Finding Count": 0,
                "Error": f"{type(error).__name__}: {error}",
            })
            continue

        for environment in usable_environments:
            environment_id = str(environment.get("id", ""))
            environment_name = str(environment.get("name", ""))

            if not environment_id:
                continue

            params = {
                "finding": finding_id,
                "skip": 0,
                "limit": rows_per_request,
                "sort_by": [
                    "-exposure_percentage",
                    "-exposure_count",
                    "-impact_percentage",
                    "-impact_count",
                ],
                "asset_group_tag_id": 1,
                "Accepted": "eq:false",
            }

            try:
                payload = bhe_get_json(
                    f"/api/v2/domains/{environment_id}/details",
                    params=params,
                )

                count = int(payload.get("count", 0) or 0)
                rows = payload.get("data", []) or []

                status_rows.append({
                    "Finding ID": finding_id,
                    "Environment": environment_name,
                    "Status": "Success",
                    "Finding Count": count,
                    "Error": "",
                })

                metadata = metadata_cache.get(finding_id, {})

                exposure_counts = [
                    int(row.get("ExposureCount", 0) or 0)
                    for row in rows
                    if isinstance(row, dict)
                ]
                impact_counts = [
                    int(row.get("ImpactCount", 0) or 0)
                    for row in rows
                    if isinstance(row, dict)
                ]

                inventory_rows.append({
                    "Finding ID": finding_id,
                    "Finding Title": metadata.get("title") or finding_id,
                    "Finding Type": metadata.get("type"),
                    "Environment": environment_name,
                    "Environment ID": environment_id,
                    "Finding Count": count,
                    "Exposure Count": max(exposure_counts, default=0),
                    "Impact Count": max(impact_counts, default=0),
                    "Short Description": metadata.get("short_description"),
                    "Short Remediation": metadata.get("short_remediation"),
                })

                if rows:
                    detail_rows_by_finding.setdefault(finding_id, []).extend(rows)

            except requests.Timeout:
                status_rows.append({
                    "Finding ID": finding_id,
                    "Environment": environment_name,
                    "Status": "Timeout",
                    "Finding Count": 0,
                    "Error": "Request timed out.",
                })

            except Exception as error:
                status_rows.append({
                    "Finding ID": finding_id,
                    "Environment": environment_name,
                    "Status": "Error",
                    "Finding Count": 0,
                    "Error": f"{type(error).__name__}: {error}",
                })

    return (
        pd.DataFrame(inventory_rows),
        metadata_cache,
        detail_rows_by_finding,
        pd.DataFrame(status_rows),
    )


### 7C. Representative Native Path Selection


In [ ]:


def native_path_from_row(
    row: Dict[str, Any],
    finding_id: str,
    metadata: Dict[str, Any],
) -> Optional[Dict[str, Any]]:
    if not isinstance(row, dict) or bool(row.get("Accepted", False)):
        return None

    source_props = row.get("FromPrincipalProps", {}) or {}
    target_props = row.get("ToPrincipalProps", {}) or {}

    source_name = (
        row.get("FromPrincipalName")
        or source_props.get("name")
        or row.get("FromPrincipal")
    )
    target_name = (
        row.get("ToPrincipalName")
        or target_props.get("name")
        or row.get("ToPrincipal")
    )

    if not source_name or not target_name:
        return None

    if is_tier_zero_props(source_props):
        return None

    return {
        "finding_id": finding_id,
        "finding_title": metadata.get("title") or finding_id,
        "relationship": NATIVE_EDGE_LABELS.get(finding_id) or finding_id,
        "source": source_name,
        "source_kind": row.get("FromPrincipalKind"),
        "target": target_name,
        "target_kind": row.get("ToPrincipalKind"),
        "target_is_tier_zero": (
            is_tier_zero_props(target_props)
            or finding_id.startswith("T0")
            or finding_id == "NonT0DCSyncers"
        ),
        "environment": row.get("Environment"),
        "severity": row.get("Severity"),
        "impact_count": row.get("ImpactCount"),
        "exposure_count": row.get("ExposureCount"),
        "short_description": metadata.get("short_description"),
        "short_remediation": metadata.get("short_remediation"),
    }


def select_representative_paths(
    detail_rows_by_finding: Dict[str, List[Dict[str, Any]]],
    metadata_cache: Dict[str, Dict[str, Any]],
    max_paths_per_section: int = 3,
) -> Dict[str, List[Dict[str, Any]]]:
    """Select unique card types first, then distinct examples from the same theme."""
    paths_by_section: Dict[str, List[Dict[str, Any]]] = {}

    for section_id, priority_ids in REPRESENTATIVE_FINDING_PRIORITY.items():
        candidates = []
        seen_fingerprints = set()

        # Build a section-scoped candidate pool in the configured priority order.
        for finding_id in priority_ids:
            for row in detail_rows_by_finding.get(finding_id, []):
                path = native_path_from_row(
                    row,
                    finding_id,
                    metadata_cache.get(finding_id, {}),
                )
                if path is None:
                    continue

                fingerprint = (
                    str(path["source"]).upper(),
                    str(path["relationship"]).upper(),
                    str(path["target"]).upper(),
                )
                if fingerprint in seen_fingerprints:
                    continue

                seen_fingerprints.add(fingerprint)
                candidates.append(path)

        selected = []
        selected_fingerprints = set()
        seen_relationships = set()

        # First pass: one card per unique relationship/finding type.
        for path in candidates:
            relationship = re.sub(
                r"[^A-Za-z0-9]+",
                "",
                str(path.get("relationship") or path.get("finding_id") or ""),
            ).upper()
            if not relationship or relationship in seen_relationships:
                continue

            fingerprint = (
                str(path["source"]).upper(),
                str(path["relationship"]).upper(),
                str(path["target"]).upper(),
            )
            seen_relationships.add(relationship)
            selected_fingerprints.add(fingerprint)
            selected.append(path)

            if len(selected) >= max_paths_per_section:
                break

        # Second pass: fill remaining slots with distinct source-target examples
        # from the same section when fewer unique types are available.
        if len(selected) < max_paths_per_section:
            for path in candidates:
                fingerprint = (
                    str(path["source"]).upper(),
                    str(path["relationship"]).upper(),
                    str(path["target"]).upper(),
                )
                if fingerprint in selected_fingerprints:
                    continue

                selected_fingerprints.add(fingerprint)
                selected.append(path)
                if len(selected) >= max_paths_per_section:
                    break

        paths_by_section[section_id] = selected

    return paths_by_section


### 7D. Native Finding Collection Execution


In [ ]:


try:
    (
        NATIVE_FINDING_INVENTORY_DF,
        NATIVE_FINDING_METADATA,
        NATIVE_DETAIL_ROWS,
        NATIVE_FINDING_STATUS_DF,
    ) = collect_native_finding_inventory()

    NATIVE_ACTIONABLE_PATHS = select_representative_paths(
        NATIVE_DETAIL_ROWS,
        NATIVE_FINDING_METADATA,
        max_paths_per_section=3,
    )

    NATIVE_ACTIONABLE_PATHS_DF = pd.DataFrame([
        {
            "Section ID": section_id,
            **path,
        }
        for section_id, paths in NATIVE_ACTIONABLE_PATHS.items()
        for path in paths
    ])

    display(Markdown("## Native BloodHound Finding Inventory"))
    display(
        NATIVE_FINDING_INVENTORY_DF[
            [
                "Finding ID",
                "Finding Title",
                "Environment",
                "Finding Count",
                "Exposure Count",
                "Impact Count",
            ]
        ].sort_values(
            ["Finding Count", "Exposure Count"],
            ascending=False,
        )
    )

    display(Markdown("## Native Representative Actionable Paths"))

    if NATIVE_ACTIONABLE_PATHS_DF.empty:
        display(Markdown(
            "No representative actionable paths were returned."
        ))
    else:
        display(
            NATIVE_ACTIONABLE_PATHS_DF[
                [
                    "Section ID",
                    "finding_title",
                    "source",
                    "relationship",
                    "target",
                    "environment",
                ]
            ]
        )

except Exception as error:
    NATIVE_FINDING_INVENTORY_DF = pd.DataFrame()
    NATIVE_FINDING_METADATA = {}
    NATIVE_DETAIL_ROWS = {}
    NATIVE_ACTIONABLE_PATHS = {}
    NATIVE_ACTIONABLE_PATHS_DF = pd.DataFrame()
    NATIVE_FINDING_STATUS_DF = pd.DataFrame([{
        "Finding ID": "",
        "Environment": "",
        "Status": "Collector Error",
        "Finding Count": 0,
        "Error": f"{type(error).__name__}: {error}",
    }])

    display(Markdown(
        "⚠️ Native finding inventory could not be loaded. "
        "The report will continue with extension and custom evidence only."
    ))

In [ ]:
# ---------------------------------------------------------------------------
# Privilege Zone active finding integration
# ---------------------------------------------------------------------------
# Native BloodHound findings that explicitly reference a Privilege Zone are
# normalized here so the PDF can distinguish configuration coverage from an
# actively exposed customer-defined boundary.

def _pz_find_col(frame: pd.DataFrame, candidates: list[str]):
    lookup = {str(column).strip().casefold(): column for column in frame.columns}
    for candidate in candidates:
        matched = lookup.get(str(candidate).strip().casefold())
        if matched is not None:
            return matched
    return None


def build_privilege_zone_findings_dataframe() -> pd.DataFrame:
    inventory = globals().get("NATIVE_FINDING_INVENTORY_DF", pd.DataFrame())
    output_columns = [
        "Finding ID", "Finding Title", "Finding Count", "Severity",
        "Environment", "Zone Name", "Zone Mapping", "Source",
    ]
    if not isinstance(inventory, pd.DataFrame) or inventory.empty:
        return pd.DataFrame(columns=output_columns)

    title_col = _pz_find_col(inventory, ["Finding Title", "Title", "Name"])
    count_col = _pz_find_col(inventory, ["Finding Count", "Findings", "Count", "Active Findings"])
    id_col = _pz_find_col(inventory, ["Finding ID", "ID", "FindingId"])
    severity_col = _pz_find_col(inventory, ["Severity", "Risk Level", "Risk", "Level"])
    environment_col = _pz_find_col(inventory, ["Environment", "Environment Name", "Domain", "Tenant"])
    if title_col is None or count_col is None:
        return pd.DataFrame(columns=output_columns)

    findings = inventory.copy()
    findings[count_col] = pd.to_numeric(findings[count_col], errors="coerce").fillna(0).astype(int)
    title_text = findings[title_col].astype(str)
    mask = title_text.str.contains(r"privilege\s+zone", case=False, na=False, regex=True) & findings[count_col].gt(0)
    findings = findings.loc[mask].copy()
    if findings.empty:
        return pd.DataFrame(columns=output_columns)

    zone_frame = globals().get("PRIVILEGE_ZONE_REPORT_DF", pd.DataFrame())
    zone_names = []
    if isinstance(zone_frame, pd.DataFrame) and not zone_frame.empty:
        zone_col = _pz_find_col(zone_frame, ["Zone Name", "Privilege Zone", "Zone", "Name"])
        if zone_col:
            zone_names = [str(value).strip() for value in zone_frame[zone_col].dropna() if str(value).strip()]

    rows = []
    for _, row in findings.iterrows():
        title = str(row.get(title_col, "")).strip()
        combined = " | ".join(str(value) for value in row.tolist() if value is not None).casefold()
        matched_zone = next((name for name in zone_names if name.casefold() in combined), "")
        mapping = "Matched from native finding data" if matched_zone else ""
        if not matched_zone and len(zone_names) == 1:
            matched_zone = zone_names[0]
            mapping = "Assigned to the only configured reportable Privilege Zone"

        severity = str(row.get(severity_col, "")).strip() if severity_col else ""
        if not severity:
            severity = "Critical" if "mfa not required" in title.casefold() else "High"

        rows.append({
            "Finding ID": str(row.get(id_col, "")).strip() if id_col else "",
            "Finding Title": title,
            "Finding Count": int(row.get(count_col, 0)),
            "Severity": severity,
            "Environment": str(row.get(environment_col, "")).strip() if environment_col else "",
            "Zone Name": matched_zone,
            "Zone Mapping": mapping,
            "Source": "Native BloodHound Finding Inventory",
        })

    result = pd.DataFrame(rows, columns=output_columns)
    return result.sort_values(["Finding Count", "Finding Title"], ascending=[False, True], kind="stable").reset_index(drop=True)


PRIVILEGE_ZONE_FINDINGS_DF = build_privilege_zone_findings_dataframe()
PRIVILEGE_ZONE_ACTIVE_FINDING_COUNT = int(
    pd.to_numeric(PRIVILEGE_ZONE_FINDINGS_DF.get("Finding Count", pd.Series(dtype=int)), errors="coerce").fillna(0).sum()
) if not PRIVILEGE_ZONE_FINDINGS_DF.empty else 0

print(f"Active Privilege Zone finding categories: {len(PRIVILEGE_ZONE_FINDINGS_DF):,}")
print(f"Active Privilege Zone findings: {PRIVILEGE_ZONE_ACTIVE_FINDING_COUNT:,}")
if not PRIVILEGE_ZONE_FINDINGS_DF.empty:
    display(PRIVILEGE_ZONE_FINDINGS_DF)


def build_privilege_zone_representative_examples(max_examples: int = 2) -> pd.DataFrame:
    """Return only explicit Privilege Zone source -> relationship -> destination evidence.

    Generic representative-path rows are intentionally rejected. A row is eligible only
    when the raw native detail itself contains a Privilege Zone marker, a configured zone
    name, or a dedicated zone field. If that evidence is not present, the PDF omits the
    representative cards rather than substituting examples from another report section.
    """
    columns = [
        "Finding ID", "Finding Title", "Source", "Source Kind",
        "Relationship", "Target", "Target Kind", "Environment",
        "Zone Name", "Severity", "Instances",
    ]
    findings = globals().get("PRIVILEGE_ZONE_FINDINGS_DF", pd.DataFrame())
    detail_rows = globals().get("NATIVE_DETAIL_ROWS", {})
    metadata = globals().get("NATIVE_FINDING_METADATA", {})
    zone_frame = globals().get("PRIVILEGE_ZONE_REPORT_DF", pd.DataFrame())
    if not isinstance(findings, pd.DataFrame) or findings.empty or not isinstance(detail_rows, dict):
        return pd.DataFrame(columns=columns)

    configured_zone_names = []
    if isinstance(zone_frame, pd.DataFrame) and not zone_frame.empty:
        zone_col = _pz_find_col(zone_frame, ["Zone Name", "Privilege Zone", "Zone", "Name"])
        if zone_col:
            configured_zone_names = [
                str(value).strip() for value in zone_frame[zone_col].dropna()
                if str(value).strip()
            ]

    explicit_zone_keys = {
        "zone", "zone name", "privilege zone", "privilege_zone", "privilegezonename",
        "protected zone", "protected scope", "asset group tag", "asset_group_tag",
    }

    def row_has_explicit_zone_context(raw_row, finding_title: str, expected_zone: str) -> bool:
        if isinstance(raw_row, dict):
            normalized_keys = {str(key).strip().casefold() for key in raw_row.keys()}
            for key, value in raw_row.items():
                key_text = str(key).strip().casefold()
                value_text = str(value or "").strip().casefold()
                if key_text in explicit_zone_keys and value_text:
                    return True
            row_text = " | ".join(str(value) for value in raw_row.values() if value is not None).casefold()
        else:
            normalized_keys = set()
            row_text = str(raw_row or "").casefold()

        markers = ["privilege zone", "protected scope"]
        markers.extend(name.casefold() for name in configured_zone_names if name)
        if expected_zone and expected_zone.casefold() not in {"", "privilege zone"}:
            markers.append(expected_zone.casefold())
        return any(marker and marker in row_text for marker in markers)

    candidates = []
    seen = set()
    ordered = findings.sort_values("Finding Count", ascending=False, kind="stable")
    for _, finding in ordered.iterrows():
        finding_id = str(finding.get("Finding ID", "") or "").strip()
        finding_title = str(finding.get("Finding Title", "") or finding_id).strip()
        if not finding_id or "privilege zone" not in finding_title.casefold():
            continue
        zone_name = str(finding.get("Zone Name", "") or "Privilege Zone").strip()
        severity = str(finding.get("Severity", "High") or "High").strip()
        instances = int(pd.to_numeric(finding.get("Finding Count", 0), errors="coerce") or 0)
        for raw_row in detail_rows.get(finding_id, []):
            if not row_has_explicit_zone_context(raw_row, finding_title, zone_name):
                continue
            path = native_path_from_row(raw_row, finding_id, metadata.get(finding_id, {}))
            if not path:
                continue
            source = str(path.get("source") or "").strip()
            relationship = str(path.get("relationship") or "").strip()
            target = str(path.get("target") or "").strip()
            if not source or not relationship or not target:
                continue
            fingerprint = (source.casefold(), relationship.casefold(), target.casefold())
            if fingerprint in seen:
                continue
            seen.add(fingerprint)
            candidates.append({
                "Finding ID": finding_id,
                "Finding Title": str(path.get("finding_title") or finding_title),
                "Source": source,
                "Source Kind": str(path.get("source_kind") or "Identity"),
                "Relationship": relationship,
                "Target": target,
                "Target Kind": str(path.get("target_kind") or "Protected Object"),
                "Environment": str(path.get("environment") or finding.get("Environment") or ""),
                "Zone Name": zone_name,
                "Severity": severity,
                "Instances": instances,
            })
            break
        if len(candidates) >= max_examples:
            break

    return pd.DataFrame(candidates, columns=columns)


PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF = build_privilege_zone_representative_examples(2)
print(f"Privilege Zone representative source-edge-destination examples: {len(PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF):,}")
if not PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF.empty:
    display(PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF)


## 4B. Privilege Zone Native Detail Collection

Collects real object-level Privilege Zone findings from the same native BloodHound detail records used by the report themes. This supports Active Directory and Microsoft Entra Privilege Zone findings without substituting generic examples.


In [ ]:
# ---------------------------------------------------------------------------
# Privilege Zone object-level examples — native AD + Microsoft Entra details
# ---------------------------------------------------------------------------
# Native detail rows are keyed by BloodHound finding ID. The active Privilege
# Zone inventory is therefore the authoritative eligibility boundary: only rows
# stored under an exact active PZ finding ID may enter the representative pool.
# The raw detail payload does not need to repeat the finding ID or zone title;
# the NATIVE_DETAIL_ROWS dictionary key already preserves that association.
# Generic T0 findings and general report evidence are never substituted.

PZ_DETAILS_PAGE_SIZE = 500
PZ_MAX_EXAMPLES_PER_ZONE = 2


def _clean_identifier(value: Any) -> str | None:
    if value is None:
        return None
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "null"}:
        return None
    return text


def _looks_like_ad_sid(value: Any) -> bool:
    return bool(re.fullmatch(r"S-1-5-21-(?:\d+-){2}\d+", str(value or "").strip()))


def _looks_like_uuid(value: Any) -> bool:
    return bool(re.fullmatch(
        r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}",
        str(value or "").strip(),
    ))


def _platform_from_finding(finding_id: str, finding_type: str = "", environment_id: str = "") -> str:
    combined = f"{finding_id} {finding_type}".casefold()
    if finding_id.startswith("AzureT0") or "entra" in combined or "azure" in combined:
        return "Microsoft Entra ID"
    if _looks_like_uuid(environment_id):
        return "Microsoft Entra ID"
    return "Active Directory"


def _first_detail_value(record: dict, keys: list[str]):
    for key in keys:
        value = record.get(key)
        if value not in (None, ""):
            return value
    return None


def _normalized_text(value: Any) -> str:
    return re.sub(r"[^a-z0-9]+", " ", str(value or "").casefold()).strip()


def _active_pz_finding_record(finding_id: str, environment: str = "") -> dict | None:
    """Return the exact active Privilege Zone inventory record for a finding ID.

    A record is eligible only when its BloodHound inventory title explicitly identifies
    a Privilege Zone finding family. This prevents generic T0/native findings from
    entering the representative-example pool merely because a zone exists.
    """
    frame = globals().get("PRIVILEGE_ZONE_FINDINGS_DF", pd.DataFrame())
    if not isinstance(frame, pd.DataFrame) or frame.empty:
        return None
    matches = frame[frame["Finding ID"].astype(str).eq(str(finding_id))].copy()
    if environment and "Environment" in matches.columns:
        env_matches = matches[
            matches["Environment"].astype(str).str.casefold().eq(str(environment).casefold())
        ]
        if not env_matches.empty:
            matches = env_matches
    if matches.empty:
        return None
    row = matches.iloc[0].to_dict()
    title = str(row.get("Finding Title") or "").strip()
    if "privilege zone" not in title.casefold():
        return None
    return row


def _row_declares_same_pz_finding(raw_row: dict, finding_id: str, inventory_title: str) -> bool:
    """Require the raw detail to identify the same PZ finding family.

    We accept an exact finding ID match or a normalized title match. Rows that carry
    neither identifier are rejected from the PDF representative cards.
    """
    if not isinstance(raw_row, dict):
        return False
    id_keys = [
        "FindingID", "FindingId", "finding_id", "findingId", "Finding", "finding",
        "Type", "type", "FindingType", "finding_type",
    ]
    title_keys = [
        "FindingTitle", "finding_title", "Title", "title", "Name", "name",
        "RiskTitle", "risk_title",
    ]
    expected_id = str(finding_id or "").strip().casefold()
    expected_title = _normalized_text(inventory_title)
    for key in id_keys:
        value = str(raw_row.get(key) or "").strip().casefold()
        if value and value == expected_id:
            return True
    for key in title_keys:
        value = _normalized_text(raw_row.get(key))
        if value and expected_title and value == expected_title:
            return True
    return False


def _zone_for_active_finding(finding_id: str, environment: str = "") -> tuple[str, str]:
    frame = globals().get("PRIVILEGE_ZONE_FINDINGS_DF", pd.DataFrame())
    if not isinstance(frame, pd.DataFrame) or frame.empty:
        return "", ""
    matches = frame[frame["Finding ID"].astype(str).eq(str(finding_id))]
    if environment and "Environment" in matches.columns:
        environment_matches = matches[
            matches["Environment"].astype(str).str.casefold().eq(str(environment).casefold())
        ]
        if not environment_matches.empty:
            matches = environment_matches
    if matches.empty:
        return "", ""
    row = matches.iloc[0]
    return str(row.get("Zone Name") or "Privilege Zone"), str(row.get("Severity") or "High")


def _normalize_native_pz_row(raw_row: dict, finding_id: str, metadata: dict) -> dict | None:
    path = native_path_from_row(raw_row, finding_id, metadata)
    if not path:
        return None
    source = str(path.get("source") or "").strip()
    relationship = str(path.get("relationship") or finding_id).strip()
    target = str(path.get("target") or "").strip()
    if not source or not relationship or not target:
        return None

    environment = str(path.get("environment") or raw_row.get("Environment") or "").strip()
    active_record = _active_pz_finding_record(finding_id, environment)
    if not active_record:
        return None
    inventory_title = str(active_record.get("Finding Title") or finding_id).strip()
    # The row is already scoped by NATIVE_DETAIL_ROWS[finding_id]. Requiring the
    # payload to repeat the same ID/title incorrectly rejects valid BloodHound
    # detail rows because many detail schemas contain only object/path fields.
    zone_name = str(active_record.get("Zone Name") or "Privilege Zone").strip()
    inventory_severity = str(active_record.get("Severity") or "High").strip()
    finding_type = str(metadata.get("finding_type") or metadata.get("type") or "")
    environment_id = str(
        _first_detail_value(raw_row, ["EnvironmentID", "EnvironmentId", "environment_id", "DomainSID", "TenantID"])
        or ""
    ).strip()

    return {
        "Zone ID": "",
        "Zone": zone_name or "Privilege Zone",
        "Finding": finding_id,
        "Finding Title": inventory_title,
        "Source Principal": source,
        "Source Principal Kind": str(path.get("source_kind") or "Identity"),
        "Relationship": relationship,
        "Target Principal": target,
        "Target Principal Kind": str(path.get("target_kind") or "Protected Object"),
        "Environment": environment,
        "Environment ID": environment_id,
        "Platform": _platform_from_finding(finding_id, finding_type, environment_id),
        "Severity": str(path.get("severity") or inventory_severity or "High"),
        "Exposure Percentage": _first_detail_value(raw_row, ["ExposurePercentage", "exposure_percentage"]),
        "Exposure Count": path.get("exposure_count") or _first_detail_value(raw_row, ["ExposureCount", "exposure_count"]),
        "Impact Percentage": _first_detail_value(raw_row, ["ImpactPercentage", "impact_percentage"]),
        "Impact Count": path.get("impact_count") or _first_detail_value(raw_row, ["ImpactCount", "impact_count"]),
        "Accepted": _first_detail_value(raw_row, ["Accepted", "accepted"]),
        "Collection Route": "Exact Native PZ Finding Detail",
    }


def _pz_details_records(payload: Any) -> list[dict]:
    if not isinstance(payload, dict):
        return []
    data = payload.get("data")
    if isinstance(data, list):
        return [row for row in data if isinstance(row, dict)]
    if isinstance(data, dict):
        for key in ("records", "findings", "items", "results"):
            rows = data.get(key)
            if isinstance(rows, list):
                return [row for row in rows if isinstance(row, dict)]
    return []


def _normalize_fallback_pz_row(record: dict, finding_id: str, zone_id: str, zone_name: str,
                               environment_id: str, environment_name: str, platform: str,
                               severity: str) -> dict:
    finding = _first_detail_value(record, ["Finding", "finding", "FindingID", "finding_id", "finding_type"]) or finding_id
    source = _first_detail_value(record, ["FromPrincipalName", "from_principal_name", "FromPrincipal", "from_principal", "SourceName", "source_name"])
    target = _first_detail_value(record, ["ToPrincipalName", "to_principal_name", "ToPrincipal", "to_principal", "TargetName", "target_name"])
    relationship = _first_detail_value(record, ["Relationship", "relationship", "EdgeKind", "edge_kind", "FindingType", "finding_type"])
    if not relationship:
        relationship = NATIVE_EDGE_LABELS.get(finding_id) or finding_id
    return {
        "Zone ID": zone_id,
        "Zone": zone_name,
        "Finding": finding,
        "Finding Title": str(globals().get("NATIVE_FINDING_METADATA", {}).get(finding_id, {}).get("title") or finding_id),
        "Source Principal": source,
        "Source Principal Kind": _first_detail_value(record, ["FromPrincipalKind", "from_principal_kind", "SourceKind", "source_kind"]),
        "Relationship": relationship,
        "Target Principal": target,
        "Target Principal Kind": _first_detail_value(record, ["ToPrincipalKind", "to_principal_kind", "TargetKind", "target_kind"]),
        "Environment": _first_detail_value(record, ["Environment", "environment", "Domain", "domain"]) or environment_name,
        "Environment ID": environment_id,
        "Platform": platform,
        "Severity": _first_detail_value(record, ["Severity", "severity"]) or severity or "High",
        "Exposure Percentage": _first_detail_value(record, ["ExposurePercentage", "exposure_percentage"]),
        "Exposure Count": _first_detail_value(record, ["ExposureCount", "exposure_count"]),
        "Impact Percentage": _first_detail_value(record, ["ImpactPercentage", "impact_percentage"]),
        "Impact Count": _first_detail_value(record, ["ImpactCount", "impact_count"]),
        "Accepted": _first_detail_value(record, ["Accepted", "accepted"]),
        "Collection Route": "Domain Details Fallback",
    }


# 1. Build the primary dataset directly from the tenant's native PZ detail rows.
_active_pz_ids = set(
    PRIVILEGE_ZONE_FINDINGS_DF.get("Finding ID", pd.Series(dtype=str)).dropna().astype(str)
) if isinstance(PRIVILEGE_ZONE_FINDINGS_DF, pd.DataFrame) else set()
_native_detail_rows = globals().get("NATIVE_DETAIL_ROWS", {}) or {}
_native_metadata = globals().get("NATIVE_FINDING_METADATA", {}) or {}
_pz_object_rows = []
_pz_native_validation_rows = []

for finding_id in sorted(_active_pz_ids):
    raw_rows = list(_native_detail_rows.get(finding_id, []) or [])
    normalized_count = 0
    for raw_row in raw_rows:
        normalized = _normalize_native_pz_row(raw_row, finding_id, _native_metadata.get(finding_id, {}) or {})
        if normalized:
            _pz_object_rows.append(normalized)
            normalized_count += 1
    _pz_native_validation_rows.append({
        "Finding ID": finding_id,
        "Finding Title": str((_native_metadata.get(finding_id, {}) or {}).get("title") or finding_id),
        "Native Detail Rows": len(raw_rows),
        "Normalized PZ Rows": normalized_count,
        "Platform": _platform_from_finding(finding_id, str((_native_metadata.get(finding_id, {}) or {}).get("finding_type") or "")),
        "Status": "Exact native PZ examples available" if normalized_count else "No native detail rows returned",
        "Selection Rule": "Exact active PZ finding ID key; payload repetition not required",
    })

PZ_NATIVE_DETAIL_VALIDATION_DF = pd.DataFrame(_pz_native_validation_rows)

# 2. Zone-scoped object-detail collection
# ---------------------------------------------------------------------------
# Native finding detail rows are environment-wide and do not prove that an
# object relationship belongs to a specific configured Privilege Zone. The
# representative cards therefore use only rows returned by the BloodHound
# domain-details endpoint while the exact zone tag is supplied through
# asset_group_tag_id. Native rows remain available for diagnostics, but they
# are never eligible for the PDF examples.

_inventory = globals().get("NATIVE_FINDING_INVENTORY_DF", pd.DataFrame())
_zone_frame = globals().get("REPORTABLE_PRIVILEGE_ZONES_DF", pd.DataFrame())

_pz_call_rows = []
_zone_scoped_rows = []

if isinstance(_inventory, pd.DataFrame) and not _inventory.empty and isinstance(_zone_frame, pd.DataFrame) and not _zone_frame.empty:
    active_inventory = _inventory[
        _inventory["Finding ID"].astype(str).isin(_active_pz_ids)
        & pd.to_numeric(_inventory.get("Finding Count", 0), errors="coerce").fillna(0).gt(0)
    ].copy()

    for _, zone_row in _zone_frame.iterrows():
        zone_id = str(zone_row.get("Zone ID") or zone_row.get("Asset Group Tag ID") or "").strip()
        zone_name = str(zone_row.get("Zone Name") or zone_row.get("Zone") or "Privilege Zone").strip()
        if not zone_id:
            continue

        for _, inventory_row in active_inventory.iterrows():
            finding_id = str(inventory_row.get("Finding ID") or "").strip()
            environment_id = str(inventory_row.get("Environment ID") or "").strip()
            environment_name = str(inventory_row.get("Environment") or "").strip()
            finding_type = str(inventory_row.get("Finding Type") or "")
            finding_title = str(inventory_row.get("Finding Title") or finding_id).strip()
            severity = str(inventory_row.get("Severity") or "High").strip()

            if not finding_id or not environment_id or not (_looks_like_ad_sid(environment_id) or _looks_like_uuid(environment_id)):
                continue

            platform = _platform_from_finding(finding_id, finding_type, environment_id)
            params = {
                "finding": finding_id,
                "skip": 0,
                "limit": PZ_DETAILS_PAGE_SIZE,
                "sort_by": ["-exposure_percentage", "-exposure_count", "-impact_percentage", "-impact_count"],
                "asset_group_tag_id": zone_id,
                "Accepted": "eq:false",
            }
            result = bhe_signed_get(f"/api/v2/domains/{environment_id}/details", params=params)
            payload = result.get("payload", {})
            page_rows = _pz_details_records(payload) if result.get("ok") else []

            _pz_call_rows.append({
                "Zone ID": zone_id,
                "Zone": zone_name,
                "Environment": environment_name,
                "Environment ID": environment_id,
                "Platform": platform,
                "Finding": finding_id,
                "Finding Title": finding_title,
                "HTTP Status": result.get("status_code"),
                "Succeeded": bool(result.get("ok")),
                "Returned Rows": len(page_rows),
                "Elapsed Seconds": result.get("elapsed_seconds"),
                "Error": str(result.get("error") or "")[:1000],
            })

            for raw_row in page_rows:
                normalized = _normalize_fallback_pz_row(
                    raw_row, finding_id, zone_id, zone_name,
                    environment_id, environment_name, platform, severity,
                )
                normalized["Finding"] = finding_id
                normalized["Finding Title"] = finding_title
                normalized["Collection Route"] = "Zone-Scoped Domain Details"
                _zone_scoped_rows.append(normalized)

DOMAIN_DETAILS_CALLS_DF = pd.DataFrame(_pz_call_rows)

# Exact native PZ rows are primary. The domain-details route is retained only
# as a supplemental fallback for an exact active PZ finding ID.
_native_exact_df = pd.DataFrame(_pz_object_rows)
_zone_fallback_df = pd.DataFrame(_zone_scoped_rows)
_frames = [frame for frame in (_native_exact_df, _zone_fallback_df) if isinstance(frame, pd.DataFrame) and not frame.empty]
OBJECT_LEVEL_FINDINGS_DF = pd.concat(_frames, ignore_index=True, sort=False) if _frames else pd.DataFrame()
if not OBJECT_LEVEL_FINDINGS_DF.empty:
    OBJECT_LEVEL_FINDINGS_DF = (
        OBJECT_LEVEL_FINDINGS_DF
        .dropna(subset=["Source Principal", "Target Principal"])
        .drop_duplicates(subset=["Zone", "Finding", "Source Principal", "Relationship", "Target Principal"])
        .sort_values(
            ["Zone", "Platform", "Finding", "Source Principal", "Target Principal"],
            ascending=[True, True, True, True, True], kind="stable", na_position="last",
        ).reset_index(drop=True)
    )

# 3. Select diverse, explicitly PZ-scoped examples.
# Selection is platform-aware: when both AD and Entra examples are available,
# show one from each. When only one platform is present, fill both slots from it.
def _pz_severity_rank(value: Any) -> int:
    return {"critical": 0, "high": 1, "moderate": 2, "medium": 2, "low": 3}.get(
        str(value or "").strip().casefold(), 9
    )


def _pz_candidate_key(row: pd.Series) -> tuple:
    return (
        _pz_severity_rank(row.get("Severity")),
        str(row.get("Finding") or row.get("Relationship") or "").casefold(),
        str(row.get("Source Principal") or "").casefold(),
        str(row.get("Target Principal") or "").casefold(),
    )


def _pz_pick_distinct(candidates: pd.DataFrame, selected: list, seen_fingerprints: set, seen_findings: set) -> bool:
    if candidates.empty:
        return False
    for _, row in candidates.sort_values(
        ["Severity", "Finding", "Source Principal", "Target Principal"],
        kind="stable", na_position="last",
    ).iterrows():
        fingerprint = (
            str(row.get("Source Principal") or "").casefold(),
            str(row.get("Relationship") or "").casefold(),
            str(row.get("Target Principal") or "").casefold(),
        )
        finding_key = str(row.get("Finding") or row.get("Relationship") or "").casefold()
        if not all(fingerprint) or fingerprint in seen_fingerprints:
            continue
        # Prefer a new finding category, but allow repeats during the fill pass.
        if finding_key in seen_findings and len(selected) == 0:
            continue
        selected.append(row)
        seen_fingerprints.add(fingerprint)
        seen_findings.add(finding_key)
        return True
    return False


_pz_representatives = []
_STRICT_PZ_REPRESENTATIVE_POOL_DF = OBJECT_LEVEL_FINDINGS_DF.copy() if not OBJECT_LEVEL_FINDINGS_DF.empty else pd.DataFrame()
if not _STRICT_PZ_REPRESENTATIVE_POOL_DF.empty:
    # A native row is authoritative for its finding. Suppress fallback duplicates
    # for that same finding while retaining fallback-only findings.
    native_findings = set(
        _STRICT_PZ_REPRESENTATIVE_POOL_DF.loc[
            _STRICT_PZ_REPRESENTATIVE_POOL_DF["Collection Route"].astype(str).eq("Exact Native PZ Finding Detail"),
            "Finding",
        ].astype(str)
    )
    _STRICT_PZ_REPRESENTATIVE_POOL_DF = _STRICT_PZ_REPRESENTATIVE_POOL_DF[
        _STRICT_PZ_REPRESENTATIVE_POOL_DF["Collection Route"].astype(str).eq("Exact Native PZ Finding Detail")
        | ~_STRICT_PZ_REPRESENTATIVE_POOL_DF["Finding"].astype(str).isin(native_findings)
    ].copy()

if not _STRICT_PZ_REPRESENTATIVE_POOL_DF.empty:
    for zone_name, group in _STRICT_PZ_REPRESENTATIVE_POOL_DF.groupby("Zone", sort=False):
        group = group.copy()
        group["_severity_rank"] = group["Severity"].map(_pz_severity_rank)
        group = group.sort_values(
            ["_severity_rank", "Finding", "Source Principal", "Target Principal"],
            kind="stable", na_position="last",
        )
        selected = []
        seen_findings = set()
        seen_fingerprints = set()

        ad_group = group[group["Platform"].astype(str).eq("Active Directory")]
        entra_group = group[group["Platform"].astype(str).eq("Microsoft Entra ID")]

        # One per platform when both are present.
        if not ad_group.empty and not entra_group.empty:
            _pz_pick_distinct(ad_group, selected, seen_fingerprints, seen_findings)
            _pz_pick_distinct(entra_group, selected, seen_fingerprints, seen_findings)
        else:
            primary = ad_group if not ad_group.empty else entra_group if not entra_group.empty else group
            while len(selected) < PZ_MAX_EXAMPLES_PER_ZONE:
                before = len(selected)
                _pz_pick_distinct(primary, selected, seen_fingerprints, seen_findings)
                if len(selected) == before:
                    break

        # Fill any remaining slot from any available platform with a distinct row.
        while len(selected) < PZ_MAX_EXAMPLES_PER_ZONE:
            before = len(selected)
            _pz_pick_distinct(group, selected, seen_fingerprints, seen_findings)
            if len(selected) == before:
                break

        for rank, row in enumerate(selected[:PZ_MAX_EXAMPLES_PER_ZONE], start=1):
            _pz_representatives.append({
                "Zone ID": row.get("Zone ID"),
                "Zone": zone_name,
                "Example Rank": rank,
                "Platform": row.get("Platform"),
                "Finding": row.get("Finding"),
                "Finding Title": row.get("Finding Title"),
                "Source Principal": row.get("Source Principal"),
                "Source Principal Kind": row.get("Source Principal Kind"),
                "Relationship": row.get("Relationship"),
                "Target Principal": row.get("Target Principal"),
                "Target Principal Kind": row.get("Target Principal Kind"),
                "Environment": row.get("Environment"),
                "Environment ID": row.get("Environment ID"),
                "Severity": row.get("Severity"),
                "Collection Route": row.get("Collection Route"),
            })

REPRESENTATIVE_RELATIONSHIPS_DF = pd.DataFrame(_pz_representatives)
if not REPRESENTATIVE_RELATIONSHIPS_DF.empty:
    PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF = REPRESENTATIVE_RELATIONSHIPS_DF.rename(columns={
        "Zone": "Zone Name",
        "Source Principal": "Source",
        "Source Principal Kind": "Source Kind",
        "Target Principal": "Target",
        "Target Principal Kind": "Target Kind",
    }).copy()
    PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF["Instances"] = 1
else:
    PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF = pd.DataFrame(columns=[
        "Zone Name", "Platform", "Finding", "Finding Title", "Source", "Source Kind",
        "Relationship", "Target", "Target Kind", "Environment", "Environment ID",
        "Severity", "Collection Route", "Instances",
    ])

PZ_PLATFORM_COLLECTION_VALIDATION_DF = (
    OBJECT_LEVEL_FINDINGS_DF.groupby(["Platform", "Collection Route"], dropna=False)
    .size().reset_index(name="Normalized Rows")
    if not OBJECT_LEVEL_FINDINGS_DF.empty else pd.DataFrame()
)

FINDING_COLLECTION_VALIDATION_DF = pd.DataFrame([{
    "Active PZ Finding IDs": len(_active_pz_ids),
    "Finding IDs With Native Detail Rows": int((PZ_NATIVE_DETAIL_VALIDATION_DF.get("Normalized PZ Rows", pd.Series(dtype=int)) > 0).sum()) if not PZ_NATIVE_DETAIL_VALIDATION_DF.empty else 0,
    "Eligible Exact PZ Rows": len(_STRICT_PZ_REPRESENTATIVE_POOL_DF),
    "Zone-Scoped API Calls": len(DOMAIN_DETAILS_CALLS_DF),
    "Rejected Native Rows": sum(len(rows) for rows in globals().get("NATIVE_DETAIL_ROWS", {}).values()) if isinstance(globals().get("NATIVE_DETAIL_ROWS", {}), dict) else 0,
    "Normalized Findings": len(OBJECT_LEVEL_FINDINGS_DF),
    "Exact Native PZ Rows": int((OBJECT_LEVEL_FINDINGS_DF.get("Collection Route", pd.Series(dtype=str)).astype(str) == "Exact Native PZ Finding Detail").sum()) if not OBJECT_LEVEL_FINDINGS_DF.empty else 0,
    "Representative Examples": len(REPRESENTATIVE_RELATIONSHIPS_DF),
    "Validation Status": "PASS - exact active PZ examples populated" if len(REPRESENTATIVE_RELATIONSHIPS_DF) else "NO OBJECT-LEVEL DETAILS RETURNED FOR ACTIVE PZ FINDINGS",
}])

print(f"Active Privilege Zone finding IDs: {len(_active_pz_ids):,}")
print(f"Eligible exact Privilege Zone object-level findings: {len(_STRICT_PZ_REPRESENTATIVE_POOL_DF):,}")
print(f"Privilege Zone fallback detail API calls: {len(DOMAIN_DETAILS_CALLS_DF):,}")
print(f"Privilege Zone representative examples: {len(PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF):,}")
display(FINDING_COLLECTION_VALIDATION_DF)
if not PZ_NATIVE_DETAIL_VALIDATION_DF.empty:
    display(PZ_NATIVE_DETAIL_VALIDATION_DF)
if not PZ_PLATFORM_COLLECTION_VALIDATION_DF.empty:
    display(PZ_PLATFORM_COLLECTION_VALIDATION_DF)
if not PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF.empty:
    display(PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF)


# ---------------------------------------------------------------------------
# v1.5.28 — authoritative active-finding detail queries
# ---------------------------------------------------------------------------
# The Active Privilege Zone Findings table is the source of truth. Resolve each
# displayed finding category to its exact native BloodHound finding key, query
# object details with that key and the configured zone tag, and build report
# examples only from rows returned by those exact requests. Environment-wide
# NATIVE_DETAIL_ROWS are diagnostic only and are not eligible for PDF cards.

def _pz_col(frame: pd.DataFrame, candidates: list[str]) -> str | None:
    if not isinstance(frame, pd.DataFrame) or frame.empty:
        return None
    by_norm = {re.sub(r"[^a-z0-9]", "", str(c).casefold()): c for c in frame.columns}
    for candidate in candidates:
        key = re.sub(r"[^a-z0-9]", "", candidate.casefold())
        if key in by_norm:
            return by_norm[key]
    return None


def _pz_norm(value: Any) -> str:
    return re.sub(r"[^a-z0-9]+", " ", str(value or "").casefold()).strip()


_active = globals().get("PRIVILEGE_ZONE_FINDINGS_DF", pd.DataFrame())
_inventory = globals().get("NATIVE_FINDING_INVENTORY_DF", pd.DataFrame())
_zones = globals().get("REPORTABLE_PRIVILEGE_ZONES_DF", pd.DataFrame())
if not isinstance(_zones, pd.DataFrame) or _zones.empty:
    _zones = globals().get("PRIVILEGE_ZONE_REPORT_DF", pd.DataFrame())

_direct_rows: list[dict] = []
_direct_calls: list[dict] = []
_resolution_rows: list[dict] = []

if isinstance(_active, pd.DataFrame) and not _active.empty and isinstance(_inventory, pd.DataFrame) and not _inventory.empty:
    a_id = _pz_col(_active, ["Finding ID", "finding_id", "id"])
    a_title = _pz_col(_active, ["Finding Title", "title", "name"])
    a_env = _pz_col(_active, ["Environment", "environment_name", "domain_name"])
    a_zone = _pz_col(_active, ["Zone Name", "Zone", "zone_name"])
    a_sev = _pz_col(_active, ["Severity", "severity"])

    i_id = _pz_col(_inventory, ["Finding ID", "finding_id", "id"])
    i_title = _pz_col(_inventory, ["Finding Title", "title", "name"])
    i_env = _pz_col(_inventory, ["Environment", "environment_name", "domain_name", "Domain"])
    i_env_id = _pz_col(_inventory, ["Environment ID", "environment_id", "domain_id", "Domain ID", "DomainSID", "Tenant ID", "tenant_id"])
    i_type = _pz_col(_inventory, ["Finding Type", "finding_type", "type"])

    z_id = _pz_col(_zones, ["Zone ID", "Asset Group Tag ID", "asset_group_tag_id", "id"])
    z_name = _pz_col(_zones, ["Zone Name", "Zone", "name"])

    inventory_work = _inventory.copy()
    inventory_work["__id"] = inventory_work[i_id].astype(str).str.strip() if i_id else ""
    inventory_work["__title_norm"] = inventory_work[i_title].map(_pz_norm) if i_title else ""
    inventory_work["__env_norm"] = inventory_work[i_env].map(_pz_norm) if i_env else ""

    for _, active_row in _active.iterrows():
        active_id = str(active_row.get(a_id, "") if a_id else "").strip()
        active_title = str(active_row.get(a_title, "") if a_title else "").strip()
        active_env = str(active_row.get(a_env, "") if a_env else "").strip()
        active_zone = str(active_row.get(a_zone, "") if a_zone else "").strip()
        active_severity = str(active_row.get(a_sev, "High") if a_sev else "High").strip() or "High"

        candidates = inventory_work.iloc[0:0]
        resolution = ""
        if active_id:
            candidates = inventory_work[inventory_work["__id"].eq(active_id)]
            if not candidates.empty:
                resolution = "Exact active finding ID"
        if candidates.empty and active_title:
            candidates = inventory_work[inventory_work["__title_norm"].eq(_pz_norm(active_title))]
            if not candidates.empty:
                resolution = "Exact active finding title"
        if not candidates.empty and active_env:
            env_match = candidates[candidates["__env_norm"].eq(_pz_norm(active_env))]
            if not env_match.empty:
                candidates = env_match

        resolved_ids = sorted({str(v).strip() for v in candidates["__id"].tolist() if str(v).strip()})
        _resolution_rows.append({
            "Active Finding ID": active_id,
            "Active Finding Title": active_title,
            "Active Environment": active_env,
            "Resolved Finding IDs": ", ".join(resolved_ids),
            "Resolution": resolution or "UNRESOLVED",
            "Candidate Inventory Rows": len(candidates),
        })

        for _, inv_row in candidates.drop_duplicates(subset=["__id", i_env_id] if i_env_id else ["__id"]).iterrows():
            finding_id = str(inv_row.get("__id", "")).strip()
            environment_id = str(inv_row.get(i_env_id, "") if i_env_id else "").strip()
            environment_name = str(inv_row.get(i_env, active_env) if i_env else active_env).strip()
            finding_type = str(inv_row.get(i_type, "") if i_type else "").strip()
            if not finding_id or not environment_id:
                continue

            zone_candidates = _zones
            if isinstance(_zones, pd.DataFrame) and not _zones.empty and active_zone and z_name:
                exact_zone = _zones[_zones[z_name].map(_pz_norm).eq(_pz_norm(active_zone))]
                if not exact_zone.empty:
                    zone_candidates = exact_zone

            for _, zone_row in zone_candidates.iterrows():
                zone_id = str(zone_row.get(z_id, "") if z_id else "").strip()
                zone_name = str(zone_row.get(z_name, active_zone or "Privilege Zone") if z_name else active_zone or "Privilege Zone").strip()
                if not zone_id:
                    continue
                params = {
                    "finding": finding_id,
                    "skip": 0,
                    "limit": PZ_DETAILS_PAGE_SIZE,
                    "sort_by": ["-exposure_percentage", "-exposure_count", "-impact_percentage", "-impact_count"],
                    "asset_group_tag_id": zone_id,
                    "Accepted": "eq:false",
                }
                result = bhe_signed_get(f"/api/v2/domains/{environment_id}/details", params=params)
                page_rows = _pz_details_records(result.get("payload", {})) if result.get("ok") else []
                _direct_calls.append({
                    "Zone ID": zone_id,
                    "Zone": zone_name,
                    "Environment": environment_name,
                    "Environment ID": environment_id,
                    "Finding": finding_id,
                    "Finding Title": active_title,
                    "Resolution": resolution,
                    "HTTP Status": result.get("status_code"),
                    "Succeeded": bool(result.get("ok")),
                    "Returned Rows": len(page_rows),
                    "Elapsed Seconds": result.get("elapsed_seconds"),
                    "Error": str(result.get("error") or "")[:1000],
                })
                platform = _platform_from_finding(finding_id, finding_type, environment_id)
                for raw_row in page_rows:
                    normalized = _normalize_fallback_pz_row(
                        raw_row, finding_id, zone_id, zone_name,
                        environment_id, environment_name, platform, active_severity,
                    )
                    if not normalized.get("Source Principal") or not normalized.get("Target Principal"):
                        continue
                    normalized["Finding"] = finding_id
                    normalized["Finding Title"] = active_title or finding_id
                    normalized["Collection Route"] = "Exact Active PZ Finding Query"
                    _direct_rows.append(normalized)

PZ_ACTIVE_FINDING_RESOLUTION_DF = pd.DataFrame(_resolution_rows)
DOMAIN_DETAILS_CALLS_DF = pd.DataFrame(_direct_calls)
OBJECT_LEVEL_FINDINGS_DF = pd.DataFrame(_direct_rows)
if not OBJECT_LEVEL_FINDINGS_DF.empty:
    OBJECT_LEVEL_FINDINGS_DF = (
        OBJECT_LEVEL_FINDINGS_DF
        .drop_duplicates(subset=["Zone", "Finding", "Source Principal", "Relationship", "Target Principal"])
        .sort_values(["Zone", "Platform", "Finding Title", "Source Principal"], kind="stable")
        .reset_index(drop=True)
    )

# Platform-aware two-card selection, exclusively from exact active-finding queries.
def _pick_pz_examples(frame: pd.DataFrame, limit: int = 2) -> pd.DataFrame:
    if not isinstance(frame, pd.DataFrame) or frame.empty:
        return pd.DataFrame()
    work = frame.copy()
    severity_rank = {"critical": 0, "high": 1, "moderate": 2, "medium": 2, "low": 3}
    work["__severity"] = work["Severity"].astype(str).str.casefold().map(severity_rank).fillna(9)
    work = work.sort_values(["__severity", "Finding Title", "Relationship"], kind="stable")
    selected = []
    used = set()
    for platform in ["Active Directory", "Microsoft Entra ID"]:
        subset = work[work["Platform"].astype(str).eq(platform)]
        if not subset.empty:
            idx = subset.index[0]
            selected.append(work.loc[idx])
            used.add(idx)
    if len(selected) < limit:
        for idx, row in work.iterrows():
            if idx in used:
                continue
            selected.append(row)
            used.add(idx)
            if len(selected) >= limit:
                break
    result = pd.DataFrame(selected[:limit]).drop(columns=["__severity"], errors="ignore")
    return result.reset_index(drop=True)

REPRESENTATIVE_RELATIONSHIPS_DF = _pick_pz_examples(OBJECT_LEVEL_FINDINGS_DF, PZ_MAX_EXAMPLES_PER_ZONE)
if not REPRESENTATIVE_RELATIONSHIPS_DF.empty:
    PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF = REPRESENTATIVE_RELATIONSHIPS_DF.rename(columns={
        "Zone": "Zone Name",
        "Source Principal": "Source",
        "Source Principal Kind": "Source Kind",
        "Target Principal": "Target",
        "Target Principal Kind": "Target Kind",
    }).copy()
    PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF["Instances"] = 1
else:
    PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF = pd.DataFrame(columns=[
        "Zone Name", "Platform", "Finding", "Finding Title", "Source", "Source Kind",
        "Relationship", "Target", "Target Kind", "Environment", "Environment ID",
        "Severity", "Collection Route", "Instances",
    ])

PZ_PLATFORM_COLLECTION_VALIDATION_DF = (
    OBJECT_LEVEL_FINDINGS_DF.groupby(["Platform", "Collection Route"], dropna=False)
    .size().reset_index(name="Normalized Rows")
    if not OBJECT_LEVEL_FINDINGS_DF.empty else pd.DataFrame()
)
FINDING_COLLECTION_VALIDATION_DF = pd.DataFrame([{
    "Active PZ Finding Categories": len(_active) if isinstance(_active, pd.DataFrame) else 0,
    "Resolved Active Finding Categories": int(PZ_ACTIVE_FINDING_RESOLUTION_DF["Resolution"].ne("UNRESOLVED").sum()) if not PZ_ACTIVE_FINDING_RESOLUTION_DF.empty else 0,
    "Exact Active-Finding API Calls": len(DOMAIN_DETAILS_CALLS_DF),
    "Exact Active-Finding Detail Rows": len(OBJECT_LEVEL_FINDINGS_DF),
    "Representative Examples": len(REPRESENTATIVE_RELATIONSHIPS_DF),
    "Validation Status": "PASS - examples came from active PZ finding queries" if len(REPRESENTATIVE_RELATIONSHIPS_DF) else "NO OBJECT-LEVEL DETAILS RETURNED BY ACTIVE PZ FINDING QUERIES",
}])

print("PZ example source: exact queries driven by Active Privilege Zone Findings")
print(f"Resolved active PZ categories: {FINDING_COLLECTION_VALIDATION_DF.iloc[0]['Resolved Active Finding Categories']:,}")
print(f"Exact active-finding detail rows: {len(OBJECT_LEVEL_FINDINGS_DF):,}")
print(f"Representative examples: {len(REPRESENTATIVE_RELATIONSHIPS_DF):,}")
display(FINDING_COLLECTION_VALIDATION_DF)
if not PZ_ACTIVE_FINDING_RESOLUTION_DF.empty:
    display(PZ_ACTIVE_FINDING_RESOLUTION_DF)
if not PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF.empty:
    display(PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF)

# ---------------------------------------------------------------------------
# v1.5.30 — direct detail queries from active PZ findings
# ---------------------------------------------------------------------------
# The active finding inventory already identifies the Privilege Zone finding
# family. Query that exact finding directly. Some BHE detail routes return no
# rows when asset_group_tag_id is supplied because the PZ scope is encoded in
# the finding itself, so each active finding is attempted zone-scoped first and
# then directly without the redundant zone filter. No generic/native evidence
# is eligible for the report cards.

def _pz_all_environment_rows() -> pd.DataFrame:
    frames = []
    for name in [
        "NATIVE_FINDING_INVENTORY_DF", "DOMAIN_INVENTORY_DF", "ENVIRONMENT_INVENTORY_DF",
        "DOMAINS_DF", "ENVIRONMENTS_DF", "POSTURE_ENVIRONMENTS_DF",
    ]:
        frame = globals().get(name)
        if isinstance(frame, pd.DataFrame) and not frame.empty:
            frames.append(frame.copy())
    return pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()


def _pz_environment_candidates(active_row: pd.Series, inventory: pd.DataFrame) -> list[tuple[str, str, str]]:
    """Return unique (environment_id, environment_name, finding_type) candidates."""
    results = []
    seen = set()
    active_id = str(active_row.get("Finding ID") or "").strip()
    active_env = str(active_row.get("Environment") or "").strip()

    def add(env_id, env_name="", finding_type=""):
        env_id = str(env_id or "").strip()
        env_name = str(env_name or "").strip()
        if not env_id or env_id.casefold() in {"nan", "none", "null"}:
            return
        key = env_id.casefold()
        if key not in seen:
            seen.add(key)
            results.append((env_id, env_name, str(finding_type or "").strip()))

    if isinstance(inventory, pd.DataFrame) and not inventory.empty:
        id_col = _pz_col(inventory, ["Finding ID", "finding_id", "id"])
        env_id_col = _pz_col(inventory, ["Environment ID", "environment_id", "domain_id", "Domain ID", "DomainSID", "Tenant ID", "tenant_id", "id"])
        env_col = _pz_col(inventory, ["Environment", "Environment Name", "environment_name", "Domain", "Tenant", "name"])
        type_col = _pz_col(inventory, ["Finding Type", "finding_type", "type"])
        candidates = inventory
        if id_col and active_id:
            exact = inventory[inventory[id_col].astype(str).str.strip().eq(active_id)]
            if not exact.empty:
                candidates = exact
        if env_col and active_env:
            exact_env = candidates[candidates[env_col].astype(str).str.casefold().eq(active_env.casefold())]
            if not exact_env.empty:
                candidates = exact_env
        if env_id_col:
            for _, row in candidates.iterrows():
                add(row.get(env_id_col), row.get(env_col) if env_col else active_env, row.get(type_col) if type_col else "")

    # Fall back to all known environment tables when the finding inventory does
    # not repeat an environment identifier.
    all_env = _pz_all_environment_rows()
    if isinstance(all_env, pd.DataFrame) and not all_env.empty:
        env_id_col = _pz_col(all_env, ["Environment ID", "environment_id", "domain_id", "Domain ID", "DomainSID", "Tenant ID", "tenant_id", "id"])
        env_col = _pz_col(all_env, ["Environment", "Environment Name", "environment_name", "Domain", "Tenant", "name"])
        if env_id_col:
            candidates = all_env
            if env_col and active_env:
                exact_env = candidates[candidates[env_col].astype(str).str.casefold().eq(active_env.casefold())]
                if not exact_env.empty:
                    candidates = exact_env
            for _, row in candidates.iterrows():
                add(row.get(env_id_col), row.get(env_col) if env_col else active_env, "")
    return results


def _pz_zone_candidates(active_row: pd.Series) -> list[tuple[str, str]]:
    zones = globals().get("REPORTABLE_PRIVILEGE_ZONES_DF", pd.DataFrame())
    if not isinstance(zones, pd.DataFrame) or zones.empty:
        zones = globals().get("PRIVILEGE_ZONE_REPORT_DF", pd.DataFrame())
    active_zone = str(active_row.get("Zone Name") or "Privilege Zone").strip()
    results = []
    if isinstance(zones, pd.DataFrame) and not zones.empty:
        zid = _pz_col(zones, ["Zone ID", "Asset Group Tag ID", "asset_group_tag_id", "id"])
        zname = _pz_col(zones, ["Zone Name", "Privilege Zone", "Zone", "name"])
        work = zones
        if zname and active_zone:
            exact = work[work[zname].astype(str).str.casefold().eq(active_zone.casefold())]
            if not exact.empty:
                work = exact
        for _, row in work.iterrows():
            zone_id = str(row.get(zid) if zid else "").strip()
            zone_name = str(row.get(zname) if zname else active_zone).strip() or active_zone
            if zone_id:
                results.append((zone_id, zone_name))
    return results or [("", active_zone)]


_active = globals().get("PRIVILEGE_ZONE_FINDINGS_DF", pd.DataFrame())
_inventory = globals().get("NATIVE_FINDING_INVENTORY_DF", pd.DataFrame())
_direct_rows = []
_direct_calls = []

if isinstance(_active, pd.DataFrame) and not _active.empty:
    for _, active_row in _active.iterrows():
        finding_id = str(active_row.get("Finding ID") or "").strip()
        finding_title = str(active_row.get("Finding Title") or finding_id).strip()
        severity = str(active_row.get("Severity") or "High").strip() or "High"
        if not finding_id:
            continue

        environments = _pz_environment_candidates(active_row, _inventory)
        zones = _pz_zone_candidates(active_row)
        for environment_id, environment_name, finding_type in environments:
            returned_for_environment = False
            # First attempt the exact active finding with the exact zone tag.
            attempts = [(zone_id, zone_name, "Zone-scoped") for zone_id, zone_name in zones if zone_id]
            # Then query the same exact PZ finding directly. This is still PZ
            # evidence because the finding family itself is the scope boundary.
            attempts.append(("", str(active_row.get("Zone Name") or "Privilege Zone"), "Direct active-finding"))

            for zone_id, zone_name, mode in attempts:
                params = {
                    "finding": finding_id,
                    "skip": 0,
                    "limit": PZ_DETAILS_PAGE_SIZE,
                    "sort_by": ["-exposure_percentage", "-exposure_count", "-impact_percentage", "-impact_count"],
                    "Accepted": "eq:false",
                }
                if zone_id:
                    params["asset_group_tag_id"] = zone_id
                result = bhe_signed_get(f"/api/v2/domains/{environment_id}/details", params=params)
                page_rows = _pz_details_records(result.get("payload", {})) if result.get("ok") else []
                _direct_calls.append({
                    "Finding": finding_id,
                    "Finding Title": finding_title,
                    "Zone ID": zone_id,
                    "Zone": zone_name,
                    "Environment": environment_name,
                    "Environment ID": environment_id,
                    "Query Mode": mode,
                    "HTTP Status": result.get("status_code"),
                    "Succeeded": bool(result.get("ok")),
                    "Returned Rows": len(page_rows),
                    "Error": str(result.get("error") or "")[:1000],
                })
                if not page_rows:
                    continue

                platform = _platform_from_finding(finding_id, finding_type, environment_id)
                for raw_row in page_rows:
                    normalized = _normalize_fallback_pz_row(
                        raw_row, finding_id, zone_id, zone_name,
                        environment_id, environment_name, platform, severity,
                    )
                    if not normalized.get("Source Principal") or not normalized.get("Target Principal"):
                        continue
                    normalized["Finding"] = finding_id
                    normalized["Finding Title"] = finding_title
                    normalized["Zone"] = zone_name
                    normalized["Collection Route"] = mode + " exact PZ detail"
                    _direct_rows.append(normalized)
                returned_for_environment = True
                break

DOMAIN_DETAILS_CALLS_DF = pd.DataFrame(_direct_calls)
OBJECT_LEVEL_FINDINGS_DF = pd.DataFrame(_direct_rows)
if not OBJECT_LEVEL_FINDINGS_DF.empty:
    OBJECT_LEVEL_FINDINGS_DF = (
        OBJECT_LEVEL_FINDINGS_DF
        .drop_duplicates(subset=["Zone", "Finding", "Source Principal", "Relationship", "Target Principal"])
        .sort_values(["Zone", "Platform", "Finding Title", "Source Principal"], kind="stable")
        .reset_index(drop=True)
    )

REPRESENTATIVE_RELATIONSHIPS_DF = _pick_pz_examples(OBJECT_LEVEL_FINDINGS_DF, PZ_MAX_EXAMPLES_PER_ZONE)
if not REPRESENTATIVE_RELATIONSHIPS_DF.empty:
    PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF = REPRESENTATIVE_RELATIONSHIPS_DF.rename(columns={
        "Zone": "Zone Name", "Source Principal": "Source", "Source Principal Kind": "Source Kind",
        "Target Principal": "Target", "Target Principal Kind": "Target Kind",
    }).copy()
    PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF["Instances"] = 1
else:
    PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF = pd.DataFrame(columns=[
        "Zone Name", "Platform", "Finding", "Finding Title", "Source", "Source Kind",
        "Relationship", "Target", "Target Kind", "Environment", "Environment ID",
        "Severity", "Collection Route", "Instances",
    ])

FINDING_COLLECTION_VALIDATION_DF = pd.DataFrame([{
    "Active PZ Finding Categories": len(_active) if isinstance(_active, pd.DataFrame) else 0,
    "Exact PZ Detail API Calls": len(DOMAIN_DETAILS_CALLS_DF),
    "Zone-Scoped Calls With Rows": int(((DOMAIN_DETAILS_CALLS_DF.get("Query Mode", pd.Series(dtype=str)) == "Zone-scoped") & (pd.to_numeric(DOMAIN_DETAILS_CALLS_DF.get("Returned Rows", 0), errors="coerce").fillna(0) > 0)).sum()) if not DOMAIN_DETAILS_CALLS_DF.empty else 0,
    "Direct Active-Finding Calls With Rows": int(((DOMAIN_DETAILS_CALLS_DF.get("Query Mode", pd.Series(dtype=str)) == "Direct active-finding") & (pd.to_numeric(DOMAIN_DETAILS_CALLS_DF.get("Returned Rows", 0), errors="coerce").fillna(0) > 0)).sum()) if not DOMAIN_DETAILS_CALLS_DF.empty else 0,
    "Exact PZ Detail Rows": len(OBJECT_LEVEL_FINDINGS_DF),
    "Representative Examples": len(REPRESENTATIVE_RELATIONSHIPS_DF),
    "Validation Status": "PASS - exact active PZ details populated" if len(REPRESENTATIVE_RELATIONSHIPS_DF) else "NO DETAIL ROWS RETURNED FOR EXACT ACTIVE PZ FINDINGS",
}])

print("PZ example source: exact active finding IDs; zone-scoped then direct detail query")
display(FINDING_COLLECTION_VALIDATION_DF)
if not DOMAIN_DETAILS_CALLS_DF.empty:
    display(DOMAIN_DETAILS_CALLS_DF)
if not PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF.empty:
    display(PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF)

# ---------------------------------------------------------------------------
# v1.5.30 — Management Plane top-finding examples
# ---------------------------------------------------------------------------
# The Active Privilege Zone Findings table is the authoritative source for
# representative examples. For each configured zone, query the exact finding
# IDs from the displayed top-five rows with that exact zone's asset_group_tag_id.
# The first two displayed categories that return object-level detail become the
# two report cards. No direct, Tier Zero-wide, native-detail, or generic fallback
# is permitted.

PZ_DISPLAYED_FINDING_LIMIT = 5
PZ_REPRESENTATIVE_EXAMPLE_LIMIT = 2


def _pz_exact_zone_lookup(zone_name: str) -> tuple[str, str]:
    """Return the exact configured zone ID and canonical name."""
    zones = globals().get("REPORTABLE_PRIVILEGE_ZONES_DF", pd.DataFrame())
    if not isinstance(zones, pd.DataFrame) or zones.empty:
        zones = globals().get("PRIVILEGE_ZONE_REPORT_DF", pd.DataFrame())
    if not isinstance(zones, pd.DataFrame) or zones.empty:
        return "", str(zone_name or "Privilege Zone").strip()

    zid = _pz_col(zones, ["Zone ID", "Asset Group Tag ID", "asset_group_tag_id", "id"])
    zname = _pz_col(zones, ["Zone Name", "Privilege Zone", "Zone", "name"])
    if not zid:
        return "", str(zone_name or "Privilege Zone").strip()

    requested = str(zone_name or "").strip()
    work = zones
    if zname and requested:
        exact = work[work[zname].astype(str).str.strip().str.casefold().eq(requested.casefold())]
        if not exact.empty:
            work = exact
    row = work.iloc[0]
    return str(row.get(zid) or "").strip(), str(row.get(zname) if zname else requested or "Privilege Zone").strip()


def _pz_displayed_active_findings(active: pd.DataFrame) -> pd.DataFrame:
    """Mirror the top-five finding rows displayed in the PDF, per zone."""
    if not isinstance(active, pd.DataFrame) or active.empty:
        return pd.DataFrame()
    work = active.copy()
    if "Finding Count" not in work.columns:
        work["Finding Count"] = 0
    work["Finding Count"] = pd.to_numeric(work["Finding Count"], errors="coerce").fillna(0)
    if "Zone Name" not in work.columns:
        work["Zone Name"] = "Privilege Zone"
    work = work.sort_values(["Zone Name", "Finding Count"], ascending=[True, False], kind="stable")
    return work.groupby("Zone Name", sort=False, group_keys=False).head(PZ_DISPLAYED_FINDING_LIMIT).reset_index(drop=True)


_active = globals().get("PRIVILEGE_ZONE_FINDINGS_DF", pd.DataFrame())
_inventory = globals().get("NATIVE_FINDING_INVENTORY_DF", pd.DataFrame())
_displayed = _pz_displayed_active_findings(_active)
_query_rows = []
_example_rows = []

if not _displayed.empty:
    for zone_name, zone_findings in _displayed.groupby("Zone Name", sort=False):
        zone_id, canonical_zone_name = _pz_exact_zone_lookup(zone_name)
        examples_for_zone = 0
        if not zone_id:
            for _, finding_row in zone_findings.iterrows():
                _query_rows.append({
                    "Finding": str(finding_row.get("Finding ID") or "").strip(),
                    "Finding Title": str(finding_row.get("Finding Title") or "").strip(),
                    "Zone ID": "",
                    "Zone": canonical_zone_name,
                    "Query Mode": "Exact displayed finding + exact zone",
                    "Succeeded": False,
                    "Returned Rows": 0,
                    "Selected Example": False,
                    "Error": "Configured zone ID was not resolved",
                })
            continue

        for display_rank, (_, finding_row) in enumerate(zone_findings.iterrows(), start=1):
            finding_id = str(finding_row.get("Finding ID") or "").strip()
            finding_title = str(finding_row.get("Finding Title") or finding_id).strip()
            severity = str(finding_row.get("Severity") or "High").strip() or "High"
            if not finding_id:
                continue

            environments = _pz_environment_candidates(finding_row, _inventory)
            selected_for_finding = False
            for environment_id, environment_name, finding_type in environments:
                params = {
                    "finding": finding_id,
                    "asset_group_tag_id": zone_id,
                    "skip": 0,
                    "limit": PZ_DETAILS_PAGE_SIZE,
                    "sort_by": ["-exposure_percentage", "-exposure_count", "-impact_percentage", "-impact_count"],
                    "Accepted": "eq:false",
                }
                result = bhe_signed_get(f"/api/v2/domains/{environment_id}/details", params=params)
                page_rows = _pz_details_records(result.get("payload", {})) if result.get("ok") else []
                valid_rows = []
                platform = _platform_from_finding(finding_id, finding_type, environment_id)
                for raw_row in page_rows:
                    normalized = _normalize_fallback_pz_row(
                        raw_row, finding_id, zone_id, canonical_zone_name,
                        environment_id, environment_name, platform, severity,
                    )
                    if not normalized.get("Source Principal") or not normalized.get("Target Principal"):
                        continue
                    normalized["Finding"] = finding_id
                    normalized["Finding Title"] = finding_title
                    normalized["Zone ID"] = zone_id
                    normalized["Zone"] = canonical_zone_name
                    normalized["Display Rank"] = display_rank
                    normalized["Finding Count"] = int(pd.to_numeric(finding_row.get("Finding Count", 0), errors="coerce") or 0)
                    normalized["Collection Route"] = "Displayed active finding + exact configured zone"
                    valid_rows.append(normalized)

                selected = bool(valid_rows) and examples_for_zone < PZ_REPRESENTATIVE_EXAMPLE_LIMIT
                _query_rows.append({
                    "Display Rank": display_rank,
                    "Finding": finding_id,
                    "Finding Title": finding_title,
                    "Zone ID": zone_id,
                    "Zone": canonical_zone_name,
                    "Environment": environment_name,
                    "Environment ID": environment_id,
                    "Platform": platform,
                    "Query Mode": "Exact displayed finding + exact zone",
                    "HTTP Status": result.get("status_code"),
                    "Succeeded": bool(result.get("ok")),
                    "Returned Rows": len(page_rows),
                    "Valid Object Rows": len(valid_rows),
                    "Selected Example": selected,
                    "Error": str(result.get("error") or "")[:1000],
                })

                if selected:
                    _example_rows.append(valid_rows[0])
                    examples_for_zone += 1
                    selected_for_finding = True
                    break

            if examples_for_zone >= PZ_REPRESENTATIVE_EXAMPLE_LIMIT:
                break

DOMAIN_DETAILS_CALLS_DF = pd.DataFrame(_query_rows)
OBJECT_LEVEL_FINDINGS_DF = pd.DataFrame(_example_rows)
if not OBJECT_LEVEL_FINDINGS_DF.empty:
    OBJECT_LEVEL_FINDINGS_DF = (
        OBJECT_LEVEL_FINDINGS_DF
        .drop_duplicates(subset=["Zone ID", "Finding", "Source Principal", "Relationship", "Target Principal"])
        .sort_values(["Zone", "Display Rank"], kind="stable")
        .reset_index(drop=True)
    )

REPRESENTATIVE_RELATIONSHIPS_DF = OBJECT_LEVEL_FINDINGS_DF.copy()
if not REPRESENTATIVE_RELATIONSHIPS_DF.empty:
    PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF = REPRESENTATIVE_RELATIONSHIPS_DF.rename(columns={
        "Zone": "Zone Name", "Source Principal": "Source", "Source Principal Kind": "Source Kind",
        "Target Principal": "Target", "Target Principal Kind": "Target Kind",
    }).copy()
    PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF["Instances"] = 1
else:
    PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF = pd.DataFrame(columns=[
        "Zone Name", "Zone ID", "Platform", "Finding", "Finding Title", "Display Rank",
        "Source", "Source Kind", "Relationship", "Target", "Target Kind", "Environment",
        "Environment ID", "Severity", "Collection Route", "Instances",
    ])

FINDING_COLLECTION_VALIDATION_DF = pd.DataFrame([{
    "Displayed PZ Finding Categories Queried": len(_displayed),
    "Exact Zone-Scoped API Calls": len(DOMAIN_DETAILS_CALLS_DF),
    "Calls With Object Rows": int(pd.to_numeric(DOMAIN_DETAILS_CALLS_DF.get("Valid Object Rows", 0), errors="coerce").fillna(0).gt(0).sum()) if not DOMAIN_DETAILS_CALLS_DF.empty else 0,
    "Representative Examples": len(REPRESENTATIVE_RELATIONSHIPS_DF),
    "Selection Rule": "Top-five displayed finding order; exact finding ID + exact configured zone ID; first two with details",
    "Fallback Used": "No",
    "Validation Status": "PASS - displayed Management Plane findings produced examples" if len(REPRESENTATIVE_RELATIONSHIPS_DF) else "NO OBJECT DETAILS RETURNED FOR DISPLAYED FINDINGS IN EXACT CONFIGURED ZONE",
}])

print("PZ example source: top displayed findings queried with exact configured zone ID; no fallback")
display(FINDING_COLLECTION_VALIDATION_DF)
if not DOMAIN_DETAILS_CALLS_DF.empty:
    display(DOMAIN_DETAILS_CALLS_DF)
if not PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF.empty:
    display(PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF)


## 8. Executive Metrics and Theme Scoring

In [ ]:
active_sections = [r for r in section_results.values()]
sections_with_results = [r for r in active_sections if r.has_results]
collection_errors = [e for r in active_sections for e in r.errors]

def overall_posture(score: int) -> str:
    if score >= 14:
        return "Critical"
    if score >= 9:
        return "High"
    if score >= 4:
        return "Moderate"
    return "Limited"

total_risk_score = sum(r.risk_score for r in sections_with_results)
overall_risk = overall_posture(total_risk_score)

def native_theme_metrics(section_id: str) -> Dict[str, int]:
    inventory = globals().get(
        "NATIVE_FINDING_INVENTORY_DF",
        pd.DataFrame(),
    )

    finding_ids = set(
        NATIVE_FINDING_CATALOG.get(section_id, [])
    )

    if inventory.empty or not finding_ids:
        return {
            "findings": 0,
            "relationship_types": 0,
            "exposed_principals": 0,
            "impact": 0,
        }

    matching = inventory[
        inventory["Finding ID"].isin(finding_ids)
    ].copy()

    if matching.empty:
        return {
            "findings": 0,
            "relationship_types": 0,
            "exposed_principals": 0,
            "impact": 0,
        }

    for column in [
        "Finding Count",
        "Exposure Count",
        "Impact Count",
    ]:
        matching[column] = pd.to_numeric(
            matching[column],
            errors="coerce",
        ).fillna(0)

    active = matching[matching["Finding Count"] > 0]

    return {
        "findings": int(matching["Finding Count"].sum()),
        "relationship_types": int(active["Finding ID"].nunique()),
        "exposed_principals": int(matching["Exposure Count"].max()),
        "impact": int(matching["Impact Count"].max()),
    }


def extension_theme_count(theme: str) -> int:
    matching = [
        result
        for result in active_sections
        if result.section.risk_theme == theme
    ]

    if not matching:
        return 0

    return int(sum(result.evidence_rows for result in matching))


risk_theme_rows = []

for theme in OBJECTIVE_ORDER:
    section = next(
        (
            result.section
            for result in sorted(
                active_sections,
                key=lambda item: item.section.executive_priority,
            )
            if result.section.risk_theme == theme
        ),
        None,
    )

    section_id = section.id if section else ""
    native_metrics = native_theme_metrics(section_id)

    if theme in {
        "GitHub Identity & Supply Chain Risk",
        "Okta Workforce Identity Risk",
        "Jamf Device Administration Risk",
        "Identity Hygiene",
    }:
        findings = extension_theme_count(theme)
        relationship_types = 0
        exposed_principals = 0
    else:
        findings = native_metrics["findings"]
        relationship_types = native_metrics["relationship_types"]
        exposed_principals = native_metrics["exposed_principals"]

    risk_theme_rows.append({
        "Risk Theme": theme,
        "Maturity Domain": (
            section.maturity_domain
            if section is not None
            else "Identity Governance"
        ),
        "Risk Level": (
            section.default_risk
            if section is not None
            else "None"
        ),
        "Findings": findings,
        "Relationship Types": relationship_types,
        "Exposed Principals": exposed_principals,
    })


risk_theme_df = pd.DataFrame(risk_theme_rows)


top_priorities_df = pd.DataFrame([
    {
        "Priority": f"P{r.section.executive_priority}",
        "Maturity Domain": r.section.maturity_domain,
        "Report Section": r.section.title,
        "Risk Theme": r.section.risk_theme,
        "Risk Level": r.section.default_risk,
        "Immediate Focus": r.section.immediate_actions[0] if r.section.immediate_actions else ""
    }
    for r in sorted(sections_with_results, key=lambda x: x.section.executive_priority)
]).head(5)

underlying_bhe_findings_df = pd.DataFrame([
    {
        "Report Section": section.title,
        "Maturity Domain": section.maturity_domain,
        "Risk Theme": section.risk_theme,
        "Underlying BHE Finding": finding
    }
    for section in SECTIONS
    for finding in section.underlying_bhe_findings
])

display(Markdown(f"## Overall Identity Risk: {overall_risk}"))
display(Markdown(f"Risk theme score: **{total_risk_score}**"))
display(Markdown("### Risk Theme Summary"))
display(risk_theme_df)
display(Markdown("### Top Priorities"))
display(top_priorities_df)
display(Markdown("### Underlying BHE Finding Mapping"))
display(underlying_bhe_findings_df.head(100))

platform_coverage_df = extension_manager.coverage_dataframe()
display(Markdown("### Platform Coverage"))
display(platform_coverage_df)

## 9. Identity Risk Assessment Narrative

In [ ]:
def plural(n: int, word: str) -> str:
    return f"{n} {word}" + ("" if n == 1 else "s")

def build_identity_risk_assessment() -> str:
    """
    Build the executive narrative from the current Risk Themes schema.

    The Risk Themes table now uses native BHE finding counts and no longer
    includes the deprecated Report Sections or Evidence Rows columns.
    """
    active_themes = risk_theme_df.copy()

    if "Findings" in active_themes.columns:
        active_themes["Findings"] = pd.to_numeric(
            active_themes["Findings"],
            errors="coerce",
        ).fillna(0)

        active_themes = active_themes[
            active_themes["Findings"] > 0
        ].sort_values(
            ["Findings", "Risk Theme"],
            ascending=[False, True],
        )
    else:
        active_themes = active_themes.iloc[0:0]

    top_themes = (
        ", ".join(active_themes["Risk Theme"].head(3).tolist())
        if not active_themes.empty
        else "the assessed identity risk themes"
    )

    total_native_findings = (
        int(active_themes["Findings"].sum())
        if not active_themes.empty
        else 0
    )

    if overall_risk in {"Critical", "High"}:
        opening = (
            f"The assessment identified {overall_risk.lower()} identity risk "
            f"across {top_themes}."
        )
    else:
        opening = (
            f"The assessment identified notable identity risk across "
            f"{top_themes}."
        )

    if total_native_findings:
        finding_sentence = (
            f"BloodHound Enterprise reported {total_native_findings:,} "
            f"native finding instance(s) across the active risk themes."
        )
    else:
        finding_sentence = (
            "The report did not receive native finding counts for the active "
            "risk themes, so technical evidence remains available in the appendix."
        )

    narrative = f"""
{opening}

{finding_sentence}

The highest-value remediation work should focus on reducing exposure to Tier Zero assets, removing unnecessary delegated control, limiting credential exposure, and reducing broad administrative movement paths. Full technical evidence is preserved in the appendix for engineering review.
"""

    return narrative.strip()


identity_risk_assessment = build_identity_risk_assessment()
display(Markdown(identity_risk_assessment))

## 10. Preview Report Sections

In [ ]:
show_empty = bool(report_cfg.get("include_no_result_sections", False))
max_rows = int(report_cfg.get("max_evidence_rows", 10))

for result in sorted(section_results.values(), key=lambda x: x.section.executive_priority):
    if not result.has_results and not show_empty:
        continue

    section = result.section
    display(Markdown(f"""
## P{section.executive_priority}: {section.title}

**Risk Theme:** {section.risk_theme}  
**Maturity Domain:** {section.maturity_domain}  
**Risk Level:** {section.default_risk}  
**Evidence Rows:** {result.evidence_rows}

### Risk Summary
{executive_takeaway_for_section(section)}

### Business Impact
{section.business_impact}

### What BloodHound Found
{what_bloodhound_found_for_section(section)}

### Why It Matters
{risk_context_for_section(section)}

### Underlying BHE Findings
{", ".join(section.underlying_bhe_findings)}
"""))

    evidence_df = result.combined_evidence_df
    if not evidence_df.empty:
        display(evidence_df.head(max_rows))
    else:
        display(Markdown("No evidence returned for this section."))

## 11A. Supporting Evidence Model


### 11A.1 Evidence Normalization and Matching


In [ ]:
def customer_facing_dataframe(df: pd.DataFrame, default_domain: str = "") -> pd.DataFrame:
    clean = clean_final_dataframe(df, default_domain=default_domain).copy()

    drop_columns = []
    for column in clean.columns:
        name = str(column).strip()

        if name.lower() in {
            "result_type",
            "resulttype",
            "row_type",
            "raw",
        }:
            drop_columns.append(column)
            continue

        if re.fullmatch(r"col_?\d+", name, flags=re.IGNORECASE):
            drop_columns.append(column)

    clean = clean.drop(columns=drop_columns, errors="ignore")

    if clean.empty:
        return pd.DataFrame()

    for column in clean.columns:
        clean[column] = clean[column].map(
            lambda value: "" if not meaningful_value(value) else value
        )

    keep_columns = []
    for column in clean.columns:
        if not clean[column].astype(str).str.strip().eq("").all():
            keep_columns.append(column)

    if not keep_columns:
        return pd.DataFrame()

    clean = clean[keep_columns]

    rename_map = {}
    for column in clean.columns:
        text = str(column).strip()
        text = re.sub(r"[_\-]+", " ", text)
        text = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        rename_map[column] = text.title()

    clean = clean.rename(columns=rename_map)
    return ensure_maturity_domain_second(clean, default_domain=default_domain)


def evidence_value(row: pd.Series, names: List[str]) -> Any:
    return first_present(row, names)


def concise_detail(value: Any, limit: int = 180) -> str:
    if not meaningful_value(value):
        return ""
    text = re.sub(r"\s+", " ", str(value)).strip()
    return text if len(text) <= limit else text[: limit - 3].rstrip() + "..."


GENERIC_OBJECT_LABELS = {
    "GH_ORGANIZATION", "GH_TEAMROLE", "GH_USER", "GH_REPOSITORY",
    "GH_WORKFLOW", "GH_SECRET", "GH_TEAM", "GH_APPLICATION",
    "OKTA_GROUP", "OKTA_USER", "OKTA_APPLICATION", "OKTA_ORGANIZATION",
    "OKTA_CONTAINS", "JAMF_USER", "JAMF_GROUP", "JAMF_COMPUTER",
    "JAMF_POLICY", "JAMF_SMARTGROUP", "JAMF_CONFIGURATIONPROFILE",
    "JAMF_PACKAGE", "JAMF_SITE",
}


def generic_object_label(value: Any) -> bool:
    if not meaningful_value(value):
        return True
    text = re.sub(r"[\\s\\-]+", "_", str(value).strip()).upper()
    if text in GENERIC_OBJECT_LABELS:
        return True
    return bool(re.fullmatch(r"(?:GH|OKTA|JAMF)_[A-Z0-9_]+", text))


def relationship_bucket(value: Any) -> str:
    """Normalize display labels so related rows count as one evidence type."""
    if not meaningful_value(value):
        return ""
    text = re.sub(r"[^A-Za-z0-9]+", "", str(value)).upper()
    aliases = {
        "CANREPLICATEDIRECTORYSECRETS": "DCSYNC",
        "NONTO DCSYNCERS": "DCSYNC",
        "NON T0 DCSYNCERS": "DCSYNC",
        "KERBEROASTABLESERVICEACCOUNT": "KERBEROASTING",
        "KERBEROASTABLE": "KERBEROASTING",
        "ASREPROASTABLE": "ASREPROASTING",
        "PASSWORDNEVEREXPIRES": "PASSWORDNEVEREXPIRES",
    }
    return aliases.get(text, text)



# Supporting Technical Evidence must stay inside the current risk theme before
# representative rows are selected. These buckets intentionally assign shared
# findings (for example Golden Certificate) to their primary report section.
SECTION_EVIDENCE_BUCKETS = {
    "tier_zero_exposure": {
        "ADMINISTRATIVEACCESS", "LOGINPATH", "ALLOWEDTODELEGATE",
        "GENERICALL", "GENERICWRITE", "WRITEDACL", "WRITEOWNER", "OWNS",
        "ADDMEMBER", "ALLEXTENDEDRIGHTS", "WRITEACCOUNTRESTRICTIONS",
        "MARKSENSITIVE", "DCSYNC", "READGMSAPASSWORD", "DUMPSMSAPASSWORD",
        "SYNCLAPSPASSWORD", "READLAPSPASSWORD", "COERCEANDRELAYNTLMTOSMB",
        "ADCSESC1", "ADCSESC4", "ADCSESC9", "SYNCEDTOADUSER",
    },
    "privilege_escalation": {
        "GENERICALL", "GENERICWRITE", "WRITEDACL", "WRITEOWNER", "OWNS",
        "ADDMEMBER", "FORCECHANGEPASSWORD", "ALLEXTENDEDRIGHTS",
        "WRITEACCOUNTRESTRICTIONS",
    },
    "credential_exposure": {
        "DCSYNC", "READGMSAPASSWORD", "DUMPSMSAPASSWORD",
        "SYNCLAPSPASSWORD", "READLAPSPASSWORD", "KERBEROASTING",
        "ASREPROASTING",
    },
    "lateral_movement": {
        "ADMINISTRATIVEACCESS", "LOGINPATH", "ADMINTO", "CANPSREMOTE",
        "CANRDP", "EXECUTEDCOM", "ALLOWEDTODELEGATE", "HASSESSION",
    },
    "ntlm_relay_coercion": {
        "COERCEANDRELAYNTLMTOSMB", "NTLMRELAY", "NTLMCOERCION",
        "PRINTERBUG", "PETITPOTAM", "WEBDAVRELAY", "SMBSIGNINGDISABLED",
    },
    "adcs_exposure": {
        "ADCSESC1", "ADCSESC2", "ADCSESC3", "ADCSESC4", "ADCSESC5",
        "ADCSESC6", "ADCSESC7", "ADCSESC8", "ADCSESC9", "ADCSESC10",
        "ADCSESC11", "ADCSESC12", "ADCSESC13", "ADCSESC14", "ADCSESC15",
        "GOLDENCERTIFICATE", "MANAGECA", "MANAGECERTIFICATES", "ENROLL",
        "PUBLISHEDTO", "CERTIFICATETEMPLATECONTROL",
    },
    "cloud_hybrid_identity": {
        "SYNCEDTOADUSER", "SYNCEDIDENTITY",
        "AZOWNS", "AZADDOWNER", "AZADDSECRET", "AZADDMEMBERS",
        "AZMGADDOWNER", "AZMGADDSECRET", "AZMGGRANTAPPROLES",
        "AZMGGRANTROLE", "AZRESETPASSWORD", "AZRESETUSERPASSWORD",
        "AZUSERACCESSADMINISTRATOR", "AZGLOBALADMIN",
        "AZPRIVILEGEDROLEADMIN", "AZROLEASSIGNMENT",
        "AZAPPADMIN", "AZCLOUDAPPADMIN",
    },
    "identity_hygiene": {
        "PASSWORDNOTREQUIRED", "PASSWORDNEVEREXPIRES",
        "ASREPROASTABLE", "ASREPROASTING",
        "KERBEROASTABLE", "KERBEROASTING",
        "UNCONSTRAINEDDELEGATION",
        "DISABLEDIDENTITYRELATIONSHIP", "DISABLEDIDENTITYLIFECYCLEREVIEW",
        "ADMINCOUNT", "STALEPASSWORD", "INACTIVEACCOUNT",
        "INACTIVEPRIVILEGEDACCOUNT", "DISABLEDACCOUNTSTILLPRIVILEGED",
        "SERVICEACCOUNTHYGIENE", "WEAKSERVICEACCOUNTHYGIENE",
    },
}

PLATFORM_THEME_TOKENS = {
    "GitHub Identity & Supply Chain Risk": ("GITHUB", "GH_", "GH "),
    "Okta Workforce Identity Risk": ("OKTA",),
    "Jamf Device Administration Risk": ("JAMF",),
}


def evidence_type_value(row: pd.Series) -> Any:
    return evidence_value(row, [
        "Relationship / Finding", "Finding / Abuse Condition",
        "Finding / Condition", "Relationship", "Finding",
    ])


def evidence_row_text(row: pd.Series) -> str:
    return " | ".join(
        str(value).upper()
        for value in row.tolist()
        if meaningful_value(value)
    )


def evidence_matches_section(section: ReportSection, row: pd.Series) -> bool:
    """Keep evidence in its owning report theme before unique-type selection."""
    platform_tokens = PLATFORM_THEME_TOKENS.get(section.risk_theme)
    if platform_tokens:
        text = evidence_row_text(row)
        return any(token in text for token in platform_tokens)

    allowed = SECTION_EVIDENCE_BUCKETS.get(section.id)
    if not allowed:
        return True

    bucket = relationship_bucket(evidence_type_value(row))
    if bucket in allowed:
        return True

    # Native finding titles sometimes include a descriptive prefix/suffix.
    text = re.sub(r"[^A-Z0-9]+", "", evidence_row_text(row))
    return any(token and token in text for token in allowed)


def filter_evidence_to_section(section: ReportSection, df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()
    keep = df.apply(lambda row: evidence_matches_section(section, row), axis=1)
    return df.loc[keep].reset_index(drop=True)


### 11A.2 Representative Evidence Selection


In [ ]:


def unique_representative_rows(df: pd.DataFrame, max_rows: int = 5) -> pd.DataFrame:
    """Select distinct, actionable evidence without padding repeated examples."""
    if df is None or df.empty:
        return pd.DataFrame()

    limit = min(int(max_rows), 5)
    working = df.copy()

    type_columns = [
        "Relationship / Finding", "Finding / Abuse Condition",
        "Finding / Condition", "Relationship", "Edge Type", "Edge", "Finding",
    ]
    source_columns = [
        "Evidence Source", "Source", "Account / Principal",
        "Principal / Group", "Certificate Template",
    ]
    target_columns = [
        "Target", "Tier Zero Target", "Affected Object", "Destination",
    ]

    type_column = next((column for column in type_columns if column in working.columns), None)
    source_column = next((column for column in source_columns if column in working.columns), None)
    target_column = next((column for column in target_columns if column in working.columns), None)

    # Remove exact duplicate examples before representative selection.
    identity_columns = [
        column for column in (source_column, type_column, target_column)
        if column is not None
    ]
    if identity_columns:
        working = working.drop_duplicates(subset=identity_columns, keep="first")

    if type_column is None and source_column is None:
        return working.head(limit).reset_index(drop=True)

    structural_edges = {
        "CONTAINS", "MEMBEROF", "HASSESSION", "AZCONTAINS",
    }

    def normalized_source(row: pd.Series) -> str:
        if source_column is None:
            return ""
        value = row.get(source_column)
        return str(value).strip().upper() if meaningful_value(value) else ""

    def normalized_type(row: pd.Series) -> str:
        if type_column is None:
            return ""
        return relationship_bucket(row.get(type_column))

    # Prefer actionable control relationships before structural graph edges.
    working = working.assign(
        _evidence_priority=working.apply(
            lambda row: 1 if normalized_type(row) in structural_edges else 0,
            axis=1,
        )
    ).sort_values("_evidence_priority", kind="stable")

    selected = []
    selected_indices = set()
    seen_types = set()
    seen_sources = set()
    seen_pairs = set()

    # First, prefer one representative row per unique finding/relationship type.
    for index, row in working.iterrows():
        finding_type = normalized_type(row)
        evidence_source = normalized_source(row)
        pair = (evidence_source, finding_type)
        if not finding_type or finding_type in seen_types or pair in seen_pairs:
            continue
        seen_types.add(finding_type)
        if evidence_source:
            seen_sources.add(evidence_source)
        seen_pairs.add(pair)
        selected.append(row.drop(labels=["_evidence_priority"], errors="ignore"))
        selected_indices.add(index)
        if len(selected) >= limit:
            return pd.DataFrame(selected).reset_index(drop=True)

    # Then allow another row only when it introduces a new evidence source.
    for index, row in working.iterrows():
        if index in selected_indices:
            continue
        finding_type = normalized_type(row)
        evidence_source = normalized_source(row)
        pair = (evidence_source, finding_type)
        if not evidence_source or evidence_source in seen_sources or pair in seen_pairs:
            continue
        seen_sources.add(evidence_source)
        if finding_type:
            seen_types.add(finding_type)
        seen_pairs.add(pair)
        selected.append(row.drop(labels=["_evidence_priority"], errors="ignore"))
        if len(selected) >= limit:
            break

    return pd.DataFrame(selected).reset_index(drop=True)


def extension_named_objects(row: pd.Series) -> List[Any]:
    """Find likely named customer objects in extension output without using prose fields."""
    preferred_columns = [
        "Source", "Target", "Organization", "Repository", "Repository Name",
        "Workflow", "Workflow Name", "Secret", "Secret Name", "Team", "Team Name",
        "User", "User Name", "Group", "Group Name", "Application", "Application Name",
        "Policy", "Policy Name", "Computer", "Computer Name", "Device", "Device Name",
        "Smart Group", "Configuration Profile", "Package", "Site", "Tenant", "Name",
    ]
    values = []
    lowered = {str(column).lower(): column for column in row.index}
    for requested in preferred_columns:
        actual = lowered.get(requested.lower())
        if actual is None:
            continue
        value = row.get(actual)
        if not meaningful_value(value) or generic_object_label(value):
            continue
        text = str(value).strip()
        if len(text) > 160 or text in values:
            continue
        values.append(text)
    return values


# Core evidence normalization helpers preserved from frozen v2.8.
def meaningful_value(value: Any) -> bool:
    if value is None:
        return False
    try:
        if pd.isna(value):
            return False
    except (TypeError, ValueError):
        pass
    return str(value).strip().lower() not in {"", "nan", "none", "null", "nat", "[]", "{}"}

def first_present(row: pd.Series, names: List[str]) -> Any:
    lowered = {str(k).lower(): k for k in row.index}
    for name in names:
        key = lowered.get(name.lower())
        if key is not None and meaningful_value(row.get(key)):
            return row.get(key)
    return ""


### 11A.3 Native and Collected Evidence Frames


In [ ]:

def native_technical_evidence_dataframe(
    section: ReportSection,
    max_rows: int = 5,
) -> pd.DataFrame:
    """Return concrete customer examples from native BHE finding details."""
    if section.id not in {
        "tier_zero_exposure",
        "privilege_escalation",
        "credential_exposure",
        "lateral_movement",
        "ntlm_relay_coercion",
        "adcs_exposure",
        "cloud_hybrid_identity",
        "identity_hygiene",
    }:
        return pd.DataFrame()

    detail_rows = globals().get("NATIVE_DETAIL_ROWS", {}) or {}
    metadata_cache = globals().get("NATIVE_FINDING_METADATA", {}) or {}
    finding_ids = NATIVE_FINDING_CATALOG.get(section.id, [])

    examples = []
    seen = set()

    for finding_id in finding_ids:
        metadata = metadata_cache.get(finding_id, {}) or {}
        for raw_row in detail_rows.get(finding_id, []) or []:
            path = native_path_from_row(raw_row, finding_id, metadata)
            if not path:
                continue

            source = path.get("source")
            relationship = path.get("relationship") or path.get("finding_title")
            target = path.get("target")

            # Technical evidence is proof. Path-oriented themes require two named
            # customer objects; account-oriented themes require a named principal.
            if section.id in {"credential_exposure", "identity_hygiene"}:
                if not all(meaningful_value(value) for value in [source, relationship]):
                    continue
            elif not all(meaningful_value(value) for value in [source, relationship, target]):
                continue

            fingerprint = (
                str(source).strip().upper(),
                str(relationship).strip().upper(),
                str(target or "").strip().upper(),
            )
            if fingerprint in seen:
                continue
            seen.add(fingerprint)

            evidence_source = concise_detail(
                path.get("finding_title")
                or NATIVE_EDGE_LABELS.get(finding_id)
                or "Native BloodHound Finding"
            )

            if section.id == "identity_hygiene":
                example = {
                    "Evidence Source": evidence_source,
                    "Maturity Domain": section.maturity_domain,
                    "Account / Principal": source,
                    "Finding / Condition": relationship,
                    "Risk": concise_detail(path.get("severity") or section.default_risk),
                    "Recommendation": concise_detail(path.get("short_remediation")),
                }
            else:
                example = {
                    "Evidence Source": evidence_source,
                    "Maturity Domain": section.maturity_domain,
                    "Source": source,
                    "Relationship / Finding": relationship,
                    "Target": target,
                    "Relevant Detail": concise_detail(
                        path.get("short_description") or path.get("finding_title")
                    ),
                    "Risk": concise_detail(path.get("severity") or section.default_risk),
                    "Recommendation": concise_detail(path.get("short_remediation")),
                }
                example = {key: value for key, value in example.items() if meaningful_value(value)}

            examples.append(example)

    native_examples = filter_evidence_to_section(section, pd.DataFrame(examples))
    native_examples = ensure_maturity_domain_second(native_examples, default_domain=section.maturity_domain)
    return ensure_maturity_domain_second(
        unique_representative_rows(native_examples, max_rows=max_rows),
        default_domain=section.maturity_domain,
    )


def concrete_collected_evidence_dataframe(
    section: ReportSection,
    df: pd.DataFrame,
    max_rows: int = 5,
) -> pd.DataFrame:
    """Normalize collected evidence and reject summary-only rows."""
    normalized = technical_evidence_dataframe(section, df, max_rows=max_rows)
    if normalized.empty:
        return normalized

    normalized = filter_evidence_to_section(section, normalized)
    if normalized.empty:
        return normalized

    path_sections = {
        "tier_zero_exposure",
        "privilege_escalation",
        "lateral_movement",
        "ntlm_relay_coercion",
        "adcs_exposure",
        "cloud_hybrid_identity",
    }

    if section.id in path_sections:
        required = ["Source", "Relationship / Finding", "Target"]
        if not all(column in normalized.columns for column in required):
            return pd.DataFrame()
        keep = normalized.apply(
            lambda row: all(meaningful_value(row.get(column)) for column in required),
            axis=1,
        )
        normalized = normalized.loc[keep].reset_index(drop=True)

    elif section.id == "credential_exposure":
        source_column = "Source" if "Source" in normalized.columns else "Account / Principal"
        if source_column not in normalized.columns or "Relationship / Finding" not in normalized.columns:
            return pd.DataFrame()
        keep = normalized.apply(
            lambda row: all(meaningful_value(row.get(column)) for column in [source_column, "Relationship / Finding"]),
            axis=1,
        )
        normalized = normalized.loc[keep].reset_index(drop=True)

    elif section.id == "identity_hygiene":
        if "Account / Principal" not in normalized.columns or "Finding / Condition" not in normalized.columns:
            return pd.DataFrame()
        keep = normalized.apply(
            lambda row: all(meaningful_value(row.get(column)) for column in ["Account / Principal", "Finding / Condition"]),
            axis=1,
        )
        normalized = normalized.loc[keep].reset_index(drop=True)

    return unique_representative_rows(normalized, max_rows=max_rows)



def select_maturity_domain_examples(df: pd.DataFrame, max_rows: int = 5) -> pd.DataFrame:
    """Prefer breadth across maturity domains, then distinct findings and objects."""
    if df is None or df.empty:
        return pd.DataFrame()
    limit = min(int(max_rows), 5)
    working = df.drop_duplicates().copy()
    domain_col = "Maturity Domain" if "Maturity Domain" in working.columns else None
    finding_col = next((c for c in ["Finding / Condition", "Relationship / Finding", "Relationship"] if c in working.columns), None)
    source_col = next((c for c in ["Source", "Account / Principal"] if c in working.columns), None)
    selected, selected_idx = [], set()
    seen_domains, seen_findings, seen_sources = set(), set(), set()

    if domain_col:
        for idx, row in working.iterrows():
            domain = str(row.get(domain_col, "") or "").strip().upper()
            if not domain or domain in seen_domains:
                continue
            seen_domains.add(domain)
            if finding_col:
                seen_findings.add(str(row.get(finding_col, "") or "").strip().upper())
            if source_col:
                seen_sources.add(str(row.get(source_col, "") or "").strip().upper())
            selected.append(row); selected_idx.add(idx)
            if len(selected) >= limit:
                return pd.DataFrame(selected).reset_index(drop=True)

    if finding_col:
        for idx, row in working.iterrows():
            if idx in selected_idx:
                continue
            finding = str(row.get(finding_col, "") or "").strip().upper()
            if not finding or finding in seen_findings:
                continue
            seen_findings.add(finding)
            selected.append(row); selected_idx.add(idx)
            if len(selected) >= limit:
                return pd.DataFrame(selected).reset_index(drop=True)

    for idx, row in working.iterrows():
        if idx in selected_idx:
            continue
        source = str(row.get(source_col, "") or "").strip().upper() if source_col else str(idx)
        if source_col and (not source or source in seen_sources):
            continue
        seen_sources.add(source)
        selected.append(row)
        if len(selected) >= limit:
            break
    return pd.DataFrame(selected).reset_index(drop=True)


### 11A.4 Cloud and Hybrid Evidence


In [ ]:


ENTRA_FINDING_MATURITY = {
    "AzureT0AddSecret": ("Application & Service Principal Governance", "Add Secret to Privileged Application"),
    "AzureT0AddOwner": ("Application & Service Principal Governance", "Add Owner to Privileged Application or Service Principal"),
    "AzureT0MGGrantRole": ("Privileged Role Governance", "Grant Privileged Entra ID Role"),
    "AzureT0MGGrantAppRoles": ("Application & Service Principal Governance", "Grant Privileged Application Role"),
    "AzureT0ResetPassword": ("Privileged Role Governance", "Reset Password for Privileged Cloud Identity"),
    "AzureT0RoleEligible": ("Privileged Role Governance", "Eligible Assignment to Privileged Entra ID Role"),
    "AzureT0SyncedToEntraUser": ("Hybrid Identity & Synchronization", "Synced Identity with Privileged Entra ID Access"),
    "T0SyncedToADUser": ("Hybrid Identity & Synchronization", "Entra User Synced to AD User in Privilege Zone"),
    "AzureT0UserAccessAdministrator": ("Azure Resource Administration", "User Access Administrator over Privileged Azure Scope"),
    "AzureT0Contributor": ("Azure Resource Administration", "Contributor over Privileged Azure Scope"),
    "AzureT0Owner": ("Azure Resource Administration", "Owner over Privileged Azure Scope"),
    "AzureT0GlobalAdmin": ("Privileged Role Governance", "Global Administrator Exposure"),
    "AzureT0PrivilegedRoleAdmin": ("Privileged Role Governance", "Privileged Role Administrator Exposure"),
}


def cloud_hybrid_actionable_path_dataframe(
    section: ReportSection,
    collected_df: pd.DataFrame,
    max_rows: int = 5,
) -> pd.DataFrame:
    """Build Cloud & Hybrid evidence from the same native paths used by the cards."""
    if section.id != "cloud_hybrid_identity":
        return pd.DataFrame()

    rows = []
    paths = (globals().get("NATIVE_ACTIONABLE_PATHS", {}) or {}).get(section.id, []) or []

    for path in paths:
        finding_id = str(path.get("finding_id") or "").strip()
        relationship = path.get("relationship") or path.get("finding_title")
        domain, finding = ENTRA_FINDING_MATURITY.get(
            finding_id,
            ENTRA_EDGE_MATURITY.get(
                re.sub(r"[^A-Za-z0-9]", "", str(relationship or "")).upper(),
                ("Cloud Control Relationships", path.get("finding_title") or relationship),
            ),
        )
        source = path.get("source")
        target = path.get("target")
        if not all(meaningful_value(value) for value in [source, relationship, target]):
            continue

        detail = path.get("short_description") or path.get("finding_title")
        recommendation = path.get("short_remediation")
        rows.append({
            "Evidence Source": concise_detail(path.get("finding_title") or "Native BloodHound Finding"),
            "Maturity Domain": domain,
            "Source": source,
            "Finding / Condition": finding or relationship,
            "Relationship": relationship,
            "Target": target,
            "Relevant Detail": concise_detail(detail),
            "Risk": concise_detail(path.get("severity") or section.default_risk),
            "Recommendation": normalize_recommendation_text(recommendation),
        })

    # Keep the stable Cypher evidence as a fallback/fill source, especially for
    # synchronized identities not returned by the native actionable-path API.
    if collected_df is not None and not collected_df.empty:
        collected = technical_evidence_dataframe(section, collected_df, max_rows=5)
        if not collected.empty:
            rows.extend(collected.to_dict("records"))

    if not rows:
        return pd.DataFrame()

    result = pd.DataFrame(rows)
    result = result.drop_duplicates(
        subset=[c for c in ["Source", "Finding / Condition", "Target"] if c in result.columns],
        keep="first",
    )

    domain_order = {
        "Hybrid Identity & Synchronization": 0,
        "Application & Service Principal Governance": 1,
        "Privileged Role Governance": 2,
        "Azure Resource Administration": 3,
        "Cloud Control Relationships": 4,
    }
    result["__domain_order"] = result.get("Maturity Domain", "").map(
        lambda value: domain_order.get(str(value), 9)
    )
    result = result.sort_values("__domain_order", kind="stable").drop(columns="__domain_order")
    return select_maturity_domain_examples(result, max_rows=max_rows)


### 11A.5 Section Evidence Assembly


In [ ]:

def section_technical_evidence_dataframe(
    section: ReportSection,
    collected_df: pd.DataFrame,
    max_rows: int = 5,
) -> pd.DataFrame:
    """Prefer native named examples, then fill remaining slots from Cypher evidence."""

    # ADCS needs deliberate family-level selection. Native ESC findings often
    # occupy the first slots, while fast Cypher evidence such as ManageCA and
    # ManageCertificates appears later in the combined dataset. Select those
    # relationship families before applying the five-row report limit.
    if section.id == "adcs_exposure":
        limit = min(int(max_rows), 5)
        native_df = native_technical_evidence_dataframe(section, max_rows=limit)

        collected_examples = pd.DataFrame()
        if collected_df is not None and not collected_df.empty:
            clean = customer_facing_dataframe(collected_df)

            relationship_column = next(
                (
                    column for column in [
                        "Edge", "Relationship", "Edge Type", "Type", "Kind"
                    ]
                    if column in clean.columns
                ),
                None,
            )

            if relationship_column is not None:
                priority_families = [
                    "MANAGECA",
                    "MANAGECERTIFICATES",
                    "GENERICALL",
                    "GENERICWRITE",
                    "WRITEDACL",
                    "WRITEOWNER",
                    "OWNS",
                    "ENROLL",
                    "PUBLISHEDTO",
                ]

                selected_rows = []
                selected_indices = set()

                for family in priority_families:
                    for index, row in clean.iterrows():
                        if index in selected_indices:
                            continue
                        relationship = relationship_bucket(
                            row.get(relationship_column)
                        )
                        if relationship != family:
                            continue
                        selected_rows.append(row)
                        selected_indices.add(index)
                        break

                if selected_rows:
                    # Build these rows directly in the same generic path schema
                    # used by native ADCS findings. The normal ADCS formatter
                    # emits Certificate Template / Enterprise CA columns, which
                    # are then rejected by the path-evidence validator that
                    # expects Source / Relationship / Target.
                    generic_rows = []
                    for row in selected_rows:
                        source_value = evidence_value(
                            row,
                            ["Source", "Principal", "Principal / Group", "User", "Group"],
                        )
                        relationship_value = evidence_value(
                            row,
                            ["Edge", "Relationship", "Finding", "Type", "Kind"],
                        )
                        target_value = evidence_value(
                            row,
                            [
                                "Target", "Enterprise Ca", "Enterprise CA",
                                "Certificate Authority", "Template",
                                "Certificate Template",
                            ],
                        )
                        evidence_source_value = evidence_value(
                            row,
                            ["Evidence Source"],
                        )
                        detail_value = evidence_value(
                            row,
                            ["Relevant Detail", "Why It Matters", "Abuse Summary"],
                        )
                        risk_value = evidence_value(
                            row,
                            ["Risk", "Target Risk"],
                        )
                        recommendation_value = evidence_value(
                            row,
                            ["Recommendation", "Remediation Hint"],
                        )

                        if not (
                            meaningful_value(source_value)
                            and meaningful_value(relationship_value)
                            and meaningful_value(target_value)
                        ):
                            continue

                        generic_rows.append({
                            "Evidence Source": (
                                evidence_source_value
                                or "ADCS Administrative and Enrollment Control"
                            ),
                            "Source": source_value,
                            "Relationship / Finding": relationship_value,
                            "Target": target_value,
                            "Relevant Detail": concise_detail(
                                detail_value
                                or "The source has delegated administrative, enrollment, or publication control over an ADCS object."
                            ),
                            "Risk": concise_detail(risk_value or section.default_risk),
                            "Recommendation": concise_detail(
                                recommendation_value
                                or "Review the delegated ADCS relationship and remove unnecessary control or enrollment rights."
                            ),
                        })

                    if generic_rows:
                        collected_examples = pd.DataFrame(generic_rows)
                        collected_examples = filter_evidence_to_section(
                            section,
                            collected_examples,
                        )
                        collected_examples = unique_representative_rows(
                            collected_examples,
                            max_rows=limit,
                        )

        frames = [
            frame for frame in [native_df, collected_examples]
            if frame is not None and not frame.empty
        ]
        if not frames:
            return pd.DataFrame()

        combined = pd.concat(frames, ignore_index=True, sort=False)
        combined = filter_evidence_to_section(section, combined)

        dedupe_columns = [
            column for column in [
                "Source", "Principal / Group",
                "Relationship / Finding", "Finding / Abuse Condition",
                "Relationship", "Target", "Enterprise CA",
            ]
            if column in combined.columns
        ]
        if dedupe_columns:
            combined = combined.drop_duplicates(
                subset=dedupe_columns,
                keep="first",
            )
        else:
            combined = combined.drop_duplicates(keep="first")

        return unique_representative_rows(combined, max_rows=limit)

    if section.id == "cloud_hybrid_identity":
        cloud_df = cloud_hybrid_actionable_path_dataframe(
            section,
            collected_df,
            max_rows=max_rows,
        )
        if not cloud_df.empty:
            return cloud_df

    native_df = native_technical_evidence_dataframe(section, max_rows=max_rows)
    remaining = max(0, min(int(max_rows), 5) - len(native_df))

    collected = pd.DataFrame()
    if remaining and collected_df is not None and not collected_df.empty:
        collected = concrete_collected_evidence_dataframe(
            section,
            collected_df,
            max_rows=remaining,
        )

    frames = [frame for frame in [native_df, collected] if not frame.empty]
    if not frames:
        return pd.DataFrame()

    combined = pd.concat(frames, ignore_index=True, sort=False)
    combined = filter_evidence_to_section(section, combined)
    if combined.empty:
        return pd.DataFrame()

    dedupe = [
        column for column in ["Source", "Relationship / Finding", "Target"]
        if column in combined.columns
    ]
    if dedupe:
        combined = combined.drop_duplicates(subset=dedupe, keep="first")
    else:
        combined = combined.drop_duplicates(keep="first")

    # Extension-backed sections need evidence selected from the full collected
    # extension dataset. Native summary rows alone can collapse a platform to
    # one row per saved search, hiding the breadth of affected objects.
    platform = platform_section_kind(section)
    if platform:
        platform_rows = technical_evidence_dataframe(
            section,
            collected_df,
            max_rows=max_rows,
        ) if collected_df is not None and not collected_df.empty else pd.DataFrame()

        frames = [frame for frame in [native_df, platform_rows] if not frame.empty]
        if not frames:
            return pd.DataFrame()

        combined = pd.concat(frames, ignore_index=True, sort=False)
        source_column = next(
            (column for column in [
                "Source", "Principal / Object", "Account / Principal"
            ] if column in combined.columns),
            None,
        )
        finding_column = next(
            (column for column in [
                "Finding / Condition", "Relationship / Finding", "Finding"
            ] if column in combined.columns),
            None,
        )
        domain_column = "Maturity Domain" if "Maturity Domain" in combined.columns else None

        dedupe_columns = [column for column in [finding_column, source_column] if column]
        if dedupe_columns:
            combined = combined.drop_duplicates(subset=dedupe_columns, keep="first")
        else:
            combined = combined.drop_duplicates(keep="first")

        if domain_column:
            combined = combined.copy()
            combined["__maturity_priority"] = combined.apply(
                lambda row: platform_maturity_priority(
                    platform,
                    row.get(domain_column, ""),
                    row.get(finding_column, "") if finding_column else "",
                ),
                axis=1,
            )
            combined = combined.sort_values(
                by=["__maturity_priority"],
                kind="stable",
            ).drop(columns=["__maturity_priority"])

        limit = min(int(max_rows), 5)
        selected = []
        selected_indices = set()
        seen_domains = set()
        seen_findings = set()
        seen_sources = set()

        # First: one example from each maturity domain.
        if domain_column:
            for index, row in combined.iterrows():
                domain = str(row.get(domain_column, "") or "").strip().upper()
                source = str(row.get(source_column, "") or "").strip().upper() if source_column else ""
                if not domain or domain in seen_domains or not source:
                    continue
                seen_domains.add(domain)
                seen_sources.add(source)
                if finding_column:
                    seen_findings.add(str(row.get(finding_column, "") or "").strip().upper())
                selected.append(row)
                selected_indices.add(index)
                if len(selected) >= limit:
                    return pd.DataFrame(selected).reset_index(drop=True)

        # Second: distinct finding families.
        if finding_column:
            for index, row in combined.iterrows():
                if index in selected_indices:
                    continue
                finding = str(row.get(finding_column, "") or "").strip().upper()
                source = str(row.get(source_column, "") or "").strip().upper() if source_column else ""
                if not finding or finding in seen_findings or not source:
                    continue
                seen_findings.add(finding)
                seen_sources.add(source)
                selected.append(row)
                selected_indices.add(index)
                if len(selected) >= limit:
                    return pd.DataFrame(selected).reset_index(drop=True)

        # Finally: fill remaining slots with different affected objects.
        for index, row in combined.iterrows():
            if index in selected_indices:
                continue
            source = str(row.get(source_column, "") or "").strip().upper() if source_column else ""
            if not source or source in seen_sources:
                continue
            seen_sources.add(source)
            selected.append(row)
            if len(selected) >= limit:
                break

        return pd.DataFrame(selected).reset_index(drop=True)

    if section.id == "identity_hygiene":
        account_column = next(
            (column for column in [
                "Account / Principal", "Source", "Account"
            ] if column in combined.columns),
            None,
        )
        condition_column = next(
            (column for column in [
                "Finding / Condition", "Relationship / Finding", "Finding"
            ] if column in combined.columns),
            None,
        )
        dedupe_columns = [
            column for column in [condition_column, account_column]
            if column is not None
        ]
        if dedupe_columns:
            combined = combined.drop_duplicates(
                subset=dedupe_columns,
                keep="first",
            )

        limit = min(int(max_rows), 5)
        selected = []
        selected_indices = set()
        seen_conditions = set()

        # Prefer different hygiene conditions first.
        if condition_column is not None:
            for index, row in combined.iterrows():
                condition = str(row.get(condition_column, "") or "").strip().upper()
                if not condition or condition in seen_conditions:
                    continue
                seen_conditions.add(condition)
                selected.append(row)
                selected_indices.add(index)
                if len(selected) >= limit:
                    return pd.DataFrame(selected).reset_index(drop=True)

        # Fill remaining slots with distinct affected accounts.
        seen_accounts = {
            str(row.get(account_column, "") or "").strip().upper()
            for row in selected
        } if account_column is not None else set()

        for index, row in combined.iterrows():
            if index in selected_indices:
                continue
            account = (
                str(row.get(account_column, "") or "").strip().upper()
                if account_column is not None else ""
            )
            if account_column is not None and (not account or account in seen_accounts):
                continue
            if account:
                seen_accounts.add(account)
            selected.append(row)
            if len(selected) >= limit:
                break

        return pd.DataFrame(selected).reset_index(drop=True)

    return unique_representative_rows(combined, max_rows=max_rows)


## 11B. Platform and Theme Evidence Selection


### 11B.1 Platform Evidence Helpers


In [ ]:
def platform_section_kind(section: ReportSection) -> str:
    text = f"{getattr(section, 'risk_theme', '')} {getattr(section, 'title', '')}".lower()
    for platform in ("github", "okta", "jamf"):
        if platform in text:
            return platform
    return ""

def readable_platform_object_type(value: Any) -> str:
    if not meaningful_value(value):
        return ""
    text = re.sub(r"^(?:GH|Okta|jamf)_", "", str(value).strip(), flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", re.sub(r"[_\-]+", " ", text)).strip().title()

def opaque_platform_scope(value: Any) -> bool:
    if not meaningful_value(value):
        return True
    text = str(value).strip()
    if generic_object_label(text):
        return True
    compact = re.sub(r"[^A-Za-z0-9]", "", text)
    if re.fullmatch(r"[0-9A-Fa-f]{24,}", compact):
        return True
    if re.fullmatch(r"[A-Za-z0-9]{18,24}", compact) and any(ch.isdigit() for ch in compact):
        return True
    if re.fullmatch(r"[0-9A-Fa-f]{8}(?:-[0-9A-Fa-f]{4}){3}-[0-9A-Fa-f]{12}", text):
        return True
    return False

def identity_hygiene_literal_dataframe(section: ReportSection, df: pd.DataFrame, max_rows: int = 5) -> pd.DataFrame:
    if section.id != "identity_hygiene" or df is None or df.empty:
        return pd.DataFrame()
    examples = []
    limit = min(int(max_rows), 5)
    for _, raw in df.iterrows():
        if str(raw.get("result_type", "") or "").strip().lower() != "literal":
            continue
        numbered = []
        for column in df.columns:
            match = re.fullmatch(r"col_?(\d+)", str(column), flags=re.IGNORECASE)
            if match:
                numbered.append((int(match.group(1)), column))
        numbered.sort()
        values = [raw.get(column) for _, column in numbered]
        for offset in range(0, len(values) - 2, 3):
            account, enabled, password_last_set = values[offset:offset + 3]
            if not meaningful_value(account):
                continue
            examples.append({
                "Evidence Source": raw.get("Evidence Source", "Identity Hygiene"),
                "Account / Principal": account,
                "Password Last Set": format_pwdlastset_value(password_last_set),
                "Enabled": enabled,
                "Finding / Condition": raw.get("Finding") or raw.get("Evidence Source"),
                "Risk": concise_detail(raw.get("Risk") or raw.get("Why It Matters")),
                "Recommendation": normalize_recommendation_text(raw.get("Recommendation")),
            })
            if len(examples) >= limit:
                return pd.DataFrame(examples)
    return pd.DataFrame(examples)

def opengraph_maturity_domain(platform: str, finding: Any, risk: Any = "") -> str:
    """Map extension-specific findings into a stable executive maturity model."""
    text = f"{finding} {risk}".lower()

    if platform == "github":
        if any(term in text for term in ["secret", "credential", "token", "pat", "key"]):
            return "Secrets & Credential Governance"
        if any(term in text for term in ["workflow", "actions", "branch", "pull request", "code owner", "repository"]):
            return "Software Delivery Governance"
        if any(term in text for term in ["owner", "admin", "role", "permission", "visibility", "member"]):
            return "Privilege & Access Governance"
        return "Platform Governance"

    if platform == "okta":
        if any(term in text for term in ["mfa", "factor", "password", "authentication"]):
            return "Authentication Assurance"
        if any(term in text for term in ["secret", "access key", "client", "credential"]):
            return "Secrets & Application Governance"
        if any(term in text for term in ["synced", "hybrid", "scim", "lifecycle", "inactive", "staged"]):
            return "Identity Lifecycle & Federation"
        if any(term in text for term in ["admin", "privileged", "role", "console", "tier zero"]):
            return "Privilege & Access Governance"
        return "Workforce Identity Governance"

    if platform == "jamf":
        if any(term in text for term in ["tenant", "account", "admin", "privilege", "role", "path"]):
            return "Privilege & Access Governance"
        if any(term in text for term in ["computer", "device", "endpoint", "inventory"]):
            return "Device Administration Governance"
        if any(term in text for term in ["policy", "profile", "package", "script"]):
            return "Configuration & Deployment Governance"
        return "Endpoint Identity Governance"

    return "Identity Governance"


def platform_maturity_priority(platform: str, domain: Any, finding: Any = "") -> int:
    domain_text = str(domain or "").strip().lower()
    finding_text = str(finding or "").strip().lower()

    if platform == "okta":
        ordered_domains = [
            "authentication assurance",
            "secrets & application governance",
            "identity lifecycle & federation",
            "privilege & access governance",
            "workforce identity governance",
        ]
        for position, expected in enumerate(ordered_domains):
            if expected in domain_text:
                return position
        # Prefer direct privileged and MFA findings over generic rows.
        if "mfa" in finding_text or "authentication" in finding_text:
            return 0
        if any(term in finding_text for term in ["secret", "access key", "client"]):
            return 1
        if any(term in finding_text for term in ["synced", "hybrid"]):
            return 2
        if any(term in finding_text for term in ["admin", "privileged", "console"]):
            return 3
        return 9

    if platform == "github":
        ordered_domains = [
            "software delivery governance",
            "secrets & credential governance",
            "privilege & access governance",
            "platform governance",
        ]
        for position, expected in enumerate(ordered_domains):
            if expected in domain_text:
                return position

    if platform == "jamf":
        ordered_domains = [
            "privilege & access governance",
            "device administration governance",
            "configuration & deployment governance",
            "endpoint identity governance",
        ]
        for position, expected in enumerate(ordered_domains):
            if expected in domain_text:
                return position

    return 9


def select_diverse_platform_examples(df: pd.DataFrame, max_rows: int = 5) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()

    limit = min(int(max_rows), 5)
    selected = []
    selected_indices = set()
    seen_domains = set()
    seen_findings = set()
    seen_objects = set()

    # First show breadth across executive maturity domains.
    for index, row in df.iterrows():
        domain = str(row.get("Maturity Domain", "") or "").strip().upper()
        source = str(row.get("Source", "") or "").strip().upper()
        if not source or not domain or domain in seen_domains:
            continue
        seen_domains.add(domain)
        seen_findings.add(str(row.get("Finding / Condition", "") or "").strip().upper())
        seen_objects.add(source)
        selected.append(row)
        selected_indices.add(index)
        if len(selected) >= limit:
            return pd.DataFrame(selected).reset_index(drop=True)

    # Then add distinct finding types.
    for index, row in df.iterrows():
        if index in selected_indices:
            continue
        source = str(row.get("Source", "") or "").strip().upper()
        finding = str(row.get("Finding / Condition", "") or "").strip().upper()
        if not source or not finding or finding in seen_findings:
            continue
        seen_findings.add(finding)
        seen_objects.add(source)
        selected.append(row)
        selected_indices.add(index)
        if len(selected) >= limit:
            return pd.DataFrame(selected).reset_index(drop=True)

    # Finally fill remaining slots with distinct affected objects.
    for index, row in df.iterrows():
        if index in selected_indices:
            continue
        source = str(row.get("Source", "") or "").strip().upper()
        if not source or source in seen_objects:
            continue
        seen_objects.add(source)
        selected.append(row)
        if len(selected) >= limit:
            break

    return pd.DataFrame(selected).reset_index(drop=True)


### 11B.2 Technical Evidence Assembly


In [ ]:

def technical_evidence_dataframe(
    section: ReportSection,
    df: pd.DataFrame,
    max_rows: int = 5,
) -> pd.DataFrame:
    """Build a compact, example-driven table from collected evidence."""
    identity_literals = identity_hygiene_literal_dataframe(section, df, max_rows=max_rows)
    if not identity_literals.empty:
        return identity_literals

    clean = customer_facing_dataframe(df, default_domain=section.maturity_domain)
    if clean.empty:
        return pd.DataFrame()

    platform = platform_section_kind(section)

    rows = []
    for _, row in clean.iterrows():
        evidence_source = evidence_value(row, ["Evidence Source"])
        source = evidence_value(row, [
            "Source", "User", "Account", "Account Name", "Principal",
            "Principal Name", "Member", "Start", "Source Name",
        ])
        relationship = evidence_value(row, [
            "Relationship", "Edge", "Finding", "Type", "Kind",
            "Finding Type", "Condition",
        ])
        target = evidence_value(row, [
            "Target", "Domain", "Computer", "Group", "End", "Target Name",
            "Enterprise Ca", "Ca", "Certificate Authority",
        ])

        if section.risk_theme in {
            "GitHub Identity & Supply Chain Risk",
            "Okta Workforce Identity Risk",
            "Jamf Device Administration Risk",
        }:
            named_objects = extension_named_objects(row)
            if generic_object_label(source):
                source = ""
            if generic_object_label(target):
                target = ""
            if not meaningful_value(source) and named_objects:
                source = named_objects[0]
            if not meaningful_value(target):
                target = next(
                    (value for value in named_objects if str(value) != str(source)),
                    "",
                )

        finding = evidence_value(row, ["Finding", "Finding Name", "Condition"])
        why = evidence_value(row, ["Why It Matters", "Abuse Summary"])
        risk = evidence_value(row, ["Risk", "Target Risk"])
        recommendation = evidence_value(row, ["Recommendation", "Remediation Hint"])

        if section.id == "adcs_exposure":
            template = evidence_value(row, [
                "Template", "Certificate Template", "Template Name", "Name",
            ])
            ca = evidence_value(row, [
                "Enterprise Ca", "Ca", "Certificate Authority", "Target",
            ])
            principal = evidence_value(row, [
                "Source", "Principal", "Principal Group", "User", "Group",
            ])
            abuse_condition = evidence_value(row, [
                "Finding", "Finding Abuse Condition", "Condition",
            ])

            setting_parts = []
            setting_map = [
                ("Authentication Enabled", "Authentication enabled"),
                ("Enrollee Supplies Subject", "Enrollee supplies subject"),
                ("No Security Extension", "No security extension"),
                ("Requires Manager Approval", "Manager approval"),
                ("Authorized Signatures", "Authorized signatures"),
                ("Effective Ekus", "EKUs"),
                ("Enrollment Flags", "Enrollment flags"),
                ("Schema Version", "Schema version"),
            ]
            for column, label in setting_map:
                value = evidence_value(row, [column])
                if meaningful_value(value):
                    setting_parts.append(f"{label}: {value}")

            relevant_detail = "; ".join(setting_parts)
            if not relevant_detail:
                relevant_detail = evidence_value(row, [
                    "Relevant Detail", "Why It Matters", "Abuse Summary",
                ])

            output = {
                "Evidence Source": evidence_source,
                "Certificate Template": template,
                "Enterprise CA": ca,
                "Principal / Group": principal,
                "Finding / Abuse Condition": abuse_condition or finding,
                "Relationship": relationship,
                "Relevant Detail": concise_detail(relevant_detail),
                "Risk": concise_detail(risk),
                "Recommendation": normalize_recommendation_text(recommendation),
            }

        elif platform:
            platform_source = evidence_value(row, [
                "Principal / Object", "Source", "Label", "User", "Account",
                "Group", "Organization", "Repository", "Workflow", "Team",
            ])
            platform_finding = evidence_value(
                row,
                ["Finding", "Search", "Finding Name", "Condition"],
            )
            platform_relationship = evidence_value(
                row,
                ["Relationship", "Edge", "Kind"],
            )
            object_type = readable_platform_object_type(
                evidence_value(row, ["Object Type", "Kind", "Target"])
                or platform_relationship
            )

            properties = {}
            raw_properties = evidence_value(row, ["Properties"])
            if meaningful_value(raw_properties):
                try:
                    properties = (
                        json.loads(raw_properties)
                        if isinstance(raw_properties, str)
                        else dict(raw_properties)
                    )
                except Exception:
                    properties = {}

            detail_parts = []

            if platform == "github":
                for key, label in [
                    ("org_name", "Organization"),
                    ("repository_name", "Repository"),
                    ("workflow_name", "Workflow"),
                    ("secret_name", "Secret"),
                    ("html_url", "URL"),
                ]:
                    value = properties.get(key)
                    if meaningful_value(value):
                        detail_parts.append(f"{label}: {value}")

            elif platform == "okta":
                for key, label in [
                    ("displayName", "Display name"),
                    ("status", "Status"),
                    ("roleType", "Role"),
                    ("appType", "Application type"),
                    ("authenticationFactors", "Authentication factors"),
                ]:
                    value = properties.get(key)
                    if meaningful_value(value):
                        detail_parts.append(f"{label}: {value}")
                if bool(evidence_value(row, ["Is Tier Zero"])):
                    detail_parts.append("Tier Zero object")

            elif platform == "jamf":
                for key, label in [
                    ("accessLevel", "Access level"),
                    ("Tier", "Tier"),
                    ("computer", "Computer"),
                    ("site", "Site"),
                    ("displayname", "Display name"),
                    ("appType", "Application type"),
                ]:
                    value = properties.get(key)
                    if meaningful_value(value):
                        detail_parts.append(f"{label}: {value}")
                if bool(evidence_value(row, ["Is Tier Zero"])):
                    detail_parts.append("Tier Zero object")

            platform_target = evidence_value(row, [
                "Target / Scope", "Target", "Repository", "Application",
                "Organization", "Workflow", "Team", "Computer", "Policy",
                "Configuration Profile", "Package", "Site",
            ])
            if opaque_platform_scope(platform_target):
                platform_target = ""

            if not meaningful_value(platform_target):
                platform_target = next(
                    (
                        properties.get(key)
                        for key in [
                            "org_name", "repository_name", "workflow_name",
                            "computer", "site", "displayName", "environment_name",
                        ]
                        if meaningful_value(properties.get(key))
                    ),
                    "",
                )

            relevant_detail = "; ".join(detail_parts)
            if not relevant_detail:
                relevant_detail = why or evidence_value(
                    row,
                    ["Relevant Detail", "Description"],
                )

            maturity_domain = opengraph_maturity_domain(
                platform,
                platform_finding or platform_relationship,
                risk,
            )

            output = {
                "Evidence Source": evidence_source,
                "Maturity Domain": maturity_domain,
                "Source": platform_source,
                "Finding / Condition": platform_finding or platform_relationship,
                "Object Type": object_type,
                "Target / Scope": platform_target,
                "Relevant Detail": concise_detail(relevant_detail),
                "Risk": concise_detail(risk),
                "Recommendation": normalize_recommendation_text(recommendation),
            }

        elif section.id == "cloud_hybrid_identity":
            cloud_source = evidence_value(row, [
                "Source", "User", "Account", "Principal", "Source Name",
            ])
            cloud_relationship = evidence_value(row, [
                "Relationship", "Edge", "Type", "Kind",
            ])
            cloud_target = evidence_value(row, [
                "Target", "Application", "Service Principal", "Group",
                "Role", "Resource", "Target Name",
            ])
            cloud_finding = finding or evidence_value(row, ["Finding Name", "Condition"])
            maturity_domain = evidence_value(row, ["Maturity Domain", "Domain"])
            if not meaningful_value(maturity_domain):
                normalized_edge = re.sub(
                    r"[^A-Za-z0-9]", "", str(cloud_relationship or "")
                ).upper()
                maturity_domain, inferred_finding = ENTRA_EDGE_MATURITY.get(
                    normalized_edge,
                    ("Cloud Control Relationships", "Privileged Entra ID or Azure control relationship"),
                )
                if not meaningful_value(cloud_finding):
                    cloud_finding = inferred_finding

            output = {
                "Evidence Source": evidence_source,
                "Maturity Domain": maturity_domain,
                "Source": cloud_source,
                "Finding / Condition": cloud_finding or cloud_relationship,
                "Relationship": cloud_relationship,
                "Target": cloud_target,
                "Relevant Detail": concise_detail(why),
                "Risk": concise_detail(risk),
                "Recommendation": normalize_recommendation_text(recommendation),
            }

        elif section.id == "identity_hygiene":
            account = evidence_value(row, [
                "User", "Source", "Account", "Account Name", "Principal",
                "Principal Name", "Name",
            ])
            account_type = evidence_value(row, [
                "Account Type", "Type", "Kind", "Label",
            ])
            maturity_domain = evidence_value(row, [
                "Maturity Domain", "Domain",
            ]) or section.maturity_domain
            password_last_set = evidence_value(row, [
                "Password Last Set", "Pwdlastset", "Last Password Set",
            ])
            enabled = evidence_value(row, ["Enabled", "Is Enabled"])
            condition = finding or relationship or evidence_source

            condition_text = str(condition or "Identity hygiene condition")
            condition_key = condition_text.lower()
            if "password not required" in condition_key:
                hygiene_impact = "The account can authenticate without a required password, increasing the chance of unauthorized access."
            elif "password never expires" in condition_key:
                hygiene_impact = "A long-lived password can remain useful to an attacker and is less likely to benefit from regular credential rotation."
            elif "kerberoast" in condition_key:
                hygiene_impact = "The service account can be targeted for offline password cracking through Kerberos service tickets."
            elif "unconstrained delegation" in condition_key:
                hygiene_impact = "The account or computer may capture reusable privileged Kerberos credentials when users authenticate to it."
            elif "disabled" in condition_key:
                hygiene_impact = "Disabled identities can retain graph relationships, ownership, or delegated access that should be reviewed during deprovisioning."
            elif "admincount" in condition_key:
                hygiene_impact = "AdminCount indicates a privileged-history condition that can leave protected permissions or unexpected administrative exposure."
            else:
                hygiene_impact = concise_detail(why or risk) or "The condition weakens identity lifecycle or authentication controls and can increase the usefulness of other attack paths."

            output = {
                "Evidence Source": evidence_source,
                "Maturity Domain": maturity_domain,
                "Account / Principal": account,
                "Account Type": account_type,
                "Password Last Set": password_last_set,
                "Enabled": enabled,
                "Finding / Condition": condition,
                "Target": account or "Affected identity",
                "Why It Matters": hygiene_impact,
                "Risk": concise_detail(risk or why),
                "Recommendation": normalize_recommendation_text(recommendation),
            }

        else:
            relevant_detail = evidence_value(row, [
                "Relevant Detail", "Why It Matters", "Abuse Summary",
                "Permission", "Detail", "Description",
            ])
            output = {
                "Evidence Source": evidence_source,
                "Maturity Domain": section.maturity_domain,
                "Source": source,
                "Relationship / Finding": relationship or finding,
                "Target": target,
                "Relevant Detail": concise_detail(relevant_detail),
                "Risk": concise_detail(risk),
                "Recommendation": normalize_recommendation_text(recommendation),
            }

        output = {
            key: value
            for key, value in output.items()
            if meaningful_value(value)
        }

        if output:
            rows.append(output)

    if not rows:
        return pd.DataFrame()

    result = pd.DataFrame(rows)

    # Remove columns that contain no useful data after normalization.
    result = result.dropna(axis=1, how="all")
    result = result.loc[:, [
        column for column in result.columns
        if not result[column].map(meaningful_value).eq(False).all()
    ]]

    # Prefer distinct customer examples instead of repeated generic blurbs.
    dedupe_columns = [
        column for column in [
            "Evidence Source", "Source", "Account / Principal",
            "Certificate Template", "Principal / Group",
            "Relationship / Finding", "Finding / Abuse Condition", "Target",
        ]
        if column in result.columns
    ]
    if dedupe_columns:
        result = result.drop_duplicates(subset=dedupe_columns, keep="first")
    else:
        result = result.drop_duplicates(keep="first")

    result = ensure_maturity_domain_second(
        result,
        default_domain=section.maturity_domain,
    )

    if platform:
        selected = select_diverse_platform_examples(result, max_rows=max_rows)
    elif section.id == "cloud_hybrid_identity":
        selected = select_maturity_domain_examples(result, max_rows=max_rows)
    else:
        selected = unique_representative_rows(result, max_rows=max_rows)

    return ensure_maturity_domain_second(
        selected,
        default_domain=section.maturity_domain,
    )


### 11B.3 ReportLab Imports


In [ ]:



# ---------------------------------------------------------------------------
# Plain PDF proof-of-concept renderer
# ---------------------------------------------------------------------------
# This renderer intentionally avoids the HTML presentation layer. It prioritizes
# complete content, readable pagination, and deterministic PDF output.

from xml.sax.saxutils import escape as xml_escape
from reportlab.lib import colors
from reportlab.graphics.shapes import Circle, Drawing, Line, Path as GraphicsPath, Polygon, Rect, String
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.platypus import (
    BaseDocTemplate,
    Frame,
    PageTemplate,
    PageBreak,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    KeepTogether,
    NextPageTemplate,
    HRFlowable,
    Image,
)


## 11C. PDF Visual System and Rendering Helpers


### 11C.1 Embedded Brand Assets


In [ ]:
import base64
# Embedded v1.0.19 brand assets.
SPECTEROPS_LOGO_V116_B64 = "..."
ENTRA_ID_LOGO_V116_B64 = "iVBORw0KGgoAAAANSUhEUgAAAlgAAAJYCAMAAACJuGjuAAAC9FBMVEX////6/v/t+v7b9//G8v+y7v+c6f+Q5/+C5P964v934f9o1/srpukhoec1qelJsuxqwO6IzfKm2fXM6fno9v36/P39/v7w+//W9v+37/6W6P+B4/9y3/9p3f9m3f9b0/sQnecBk+MBlOQClOQMmOUWneZXuO3A5fjO9P+i6/9t3v9l3P+u3fat7f9/yfFeu+73/f+Z6f9v3//l+f6m6/7X7vvB8f9+4/+Z1vT0/P9q3v/R9f+24Pe88P+S0fN1xfBn3f8FleSf6f/8/f2g1/T+/v/L8/7i+P/p+v+H5f/f9v6U6f+o7v904P+K5v+N5v+E5P8+rutl3f/J8/8BleUCkuICj98DitkChdQDfs0EdsQFbbsFZ7UFYrAFXaoGVaIGT5sGS5cGR5MGRpIFRZIHR5MISJNOrORokL7r8PZvlMDy9fkGRpMMS5V/oMf3+fsSUJePrc8YVJqnvtkfWZ22yN4KSZQtY6PL2Og9b6rV4O1MerHk6/NahbeCosn5+vyXstIkXJ+6zeLa5O9ljbzv8/h1mcP09/qJp8sUUZiatNMdV52twtswaKESiNBCc604fK5SfrNk3P+MtL8pYqC48//E8PaexMp8prkkXpyt8P/M+P+dw8mXvMJ0oLcaV5kQTpbL+P+Vu8FsmbTK+P+WvMKWu8Jfj7GvxNyUusFVh63C0uXH9/+RuMBFeqg2aqfR3evE9v+PtsA8cqS/9f+Gr73m7PSXvcJ7ncZxnraVu8JmlLKhudae7P+xxdxQgqtHdq/f5/F/qbux8f+l7v9Ziq7F1OZAdaY1b6NKfqmVvML//v6exMll3P5VueA1d6dezfFCkr4nWo8iUIYNSpJk2PpOqtMwbJ4hT4UhUYlaw+k8hLIkVYshToQiUIVh0/dJn8oqYZWH0+9SsdlcyO1AjbqzxNfk6vCJor/G0uBggagtZZiesspAaJZn3//X3+lyj7IrV4uzw9VPc58jU4k1X5AmU4h0kbNjg6qtvtJWeqN8mLhrDNjoAAAZzUlEQVR4AezBAQEAEAAAIADg/11DVAEAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA4Asx5VJbH3Ptcx87dpXoNgxEYXguMyYxe1pLDjMzlWn/66lKYY7eer41/IKZ+8en55fXSDRGhwMwTMt2XM/n3968DYQQUobxx8RLMmoQwEFVpdJuJssTKiz5TxA+PueitCeAk7O867OyMixFqLYKkRgB7O7EuvRYWRPWpK14MVkigF2zKleYt4elCJVWjAB2UK15zNvDmqT1XCeAbUqNJvPuYSnivtUmgI2qlx3eLywlLNYJYD2j6zLvH5YMHnsGAaxx0vf4oLCkiA9KBLBSO9/hA8OSIhyaBLBCOz3iQ8NSxs+nBLCk/W7Ex4QlUBbse19tCWtSlkkAc0q1ER8Zlipr2CaAGScXHT42LCV8eU8AE0YjwzrCEvEPBgH89fGTxzrCUuI5AvjLumVdYYn7JAH89tllbWFJ8VgnBeDLV9YYlgyevhEBmN9Za1gySESJAAt31hyWxKKUoPRjpD0sOS6UCP5rJ3aH9Yclw8FP9s4CWY4YhoJhZkZVjTVhZobPjHv/u6TmBkn+PK9X6j5DF3XZeoRSwqhArPLgSuJQCkuPt0wiVted/34I0nLrtGnEGkLp1UOQNow+k4nlXdZQCqvbZjqxvOQMpXDzjinEIpQSRkcXi1AK11/8Hl0sQincGMLo2GIRSvHq2FurIZYTSnPxeMvqiFUIpZk4etpGF4tQCiunbHSxeFEKqztWUaw5XpQSRgViDbxK8PUeXj/5XVusjg/SGcLofastlvcvOYAbnBs/nlp9sUr/OXQohaUhjNYXa/ggHTmUwrctm45Y5TwfpAmj/yQWoRTe7drUxPKyFzSUwpGd/SmK5d1F3v2F5Msbs/piEUq5BSkXi0uSIW9B3rdpi+VzlwmlhFGBWN5/jpWz8Gpyz1oQa45QGotvZ2x0sQilcPSTNSKWd4TSOLxbs9HFIpTCkZPWkFg+RyiNwZcT1pRYPkcoJYxKxCKUEkYVYrn3hNJZZ2kIo82JNdczuTPb3Fi+Zw2K5YVQGnAkpwGxfI5QOsusf7JGxfJygVCaPYxKxHLfS/5BmjAqEsuTX5IkjKrE6oJekiSMCsTiRSnr4QKxwl+ShKWPT619sTzzNjlhVChWeRD/kiRhVC9W/G1y1sNlYnFJkvVwuVg5t8lZDxeKxeQOIzkCsXJP7rAeLhMr9+QO6+F6sQilhFGFWFG2yeHG4y2TiqUi0uQO6+HtiFVifZAmjArEIpSyHq4XK/42OSM5KrHivygljP6ebbHmLjX47g8Ov/htDYhFKGU9vD2xCKUx1sOlYhFKGcmRiEUoZSRHLxaTO/HXw9ViEUpZD//D3l1lvQ3FQABeSUHyWrIDM99rh5MyM/OiG1KZW4/qczSPedX3QyYwcFj8WcZflFoxqgyLOQzDgKI4Tj4mjiMKdo9/KYzsmyTHvh6OgMU7UVGcZnlRVnXTNK2kaeq6KvIsTeKIQ2b7gPRY18PxsHhnKsnysm5a57u+777M7hHv2qYqsjSmMDzhIvsmyZGM5OBhCaqi3pE6iZp+I+LL73TlO1wnWyPYJrdiFAxLVMV7VDMR9ZMcdbmmzJMoZLZt8pEVowBYoqps3K+Y+kJX79sqT3e2AvsmyVEVowBYzHFWNK7/TVUfce1tJRRaUTqe9XAALKYkr91HVH9oqymzWHNyx9bDz/5HsIg5SsvW/xUrseXqfG7v+xvBejgAVhhnlft7VWLLN4ul3Vj3uyD1YVHAcVa7Xlj9E1qz1dqurDiSow+LmKMdq3+lSrLZXrhod9YrRvVhMaWV67vpv46/dBlcwVsmsh6uDouYk6LtuukA2Vy5qlGU2nq4PiwKOM4bYfXvZV27brfWXw/HwyLmtLoxgCuJv3nLrq0+koOHxXHeDsRKcvuO3VtvJAcPS35d+W46bPzde3ZxxZEcPCziSP67Gjb3gUWpFaP6sDguHMLV9AaqKLViVB8WBZzWvptCsn3w0O6OKEb1YRFzhvgzKEXpo4HfUWqZyEiOKiziSJ4NQuKxRamN5OBhiavCdVNkHj+x24PXw/GwiOPS/6UrK0qtGP0SFnEs7RUw/vZTuz++GMXBElf44ItSK0aBsPaupt1UI8+emwF8MQqCpehq6l+8NAX4kRwILHGlk+0rfFFqIzkIWPJ8UCuv3+CLUitGAbA4KjRdTf3bd/Ci1EZyALAod5qurCh9z95ZbjmRPHD0r2ufcclX3N3dHV5gT+80mll3d8NdhpFupnC3sMHdXmopHMLWgZ7K9E3nV89wzyS5U/W7yEiOB7BS5sqe1RKlfuvhBLDWvJ06V2GwFiRKVQ/3cHotnLOuxoKV9lnPEaWqh3s4G4Z13LipJg8gS6LU1xbkeAJXb23esnXb9hoAWJ9mQ5RKjNrTq8uOui076xsaCWQ1ZUKUSoxarnrOjqItO+P6XX8AvmYFf/wJ0Fmqh3vgqmuP5vtgmd17mgBkhSv3iqzKF6O5XuMHNEfNFqzY7Nv/KeFv1qYDYqPixWivXq8fjB7+xYrNocOLAWTVHDkqOlpSD+9F4GruMcvVfbBMbI6feAdAVsgXpYrkuLna8MZJy9VDsExcf2o1gazF60+LkIRitAdCYL25ua45egKWiQunNhGkw2KJ0qT1cARXczbXRdFTYNnDEKW1Z5IsSUo0MMTowB1RVAJW4a8PAWQFTR/wRClfjNpIDkOMNj8PVtHUn30fIUrPkXWWIjkOMXreCqySv1jFe6K0lkDWxxdECjiS4xCjVmCVgGXJ2ndRolT18JaJ0ReAdV+UhgSyqKJUkRwHVzOsGH0hWJasjQxReumyeKkwMdrZitF/AsvqrNU1ALISLEmqHs4Qoy8GyxQLVxiidClsSVL1cLcYvWrFqAMsYwrbPpYo1RZkQjHqAMsYhCgNm5TcqRwxeu2+GHWDVTQGIUrDDyVKK06MOsCyZMXXawlkkZI7iuQ4xGgnK7AcYOFE6Wc3xI67Ho4QWPMsVy8Dlr33BxGlSu5UmBh1gyVRqnr4S56FG4ZZMeoGS6JUW5AtF6NusKwoZTyQdiR3FMkBilE3WPYUGA+k1Sani9FXBcsUEA+kw/e/LxWl4mpRFwJXI69ZgfWKYFlR2gQgS8kdTCSnRGC9VsKVAyycKA20JMkUo7lZVmC9OlgmvidKFwPIypckdxTJQYhRK7CSgGVF6U2EKPXQJlckxzdX863ASgaWies33gKI0pLkjiI5BDFqBVYysB6K0rxEqcRo6RakFVgJwIKJ0lqUKJUY7XI1ihKAxROlgdrk90+7oQwxOjtKABZSlKpNTonkWDHawwqsFoJVNGYPQ5QquWPFKGYLsqVgmdjEjCVJidKZQDGaHCz7QPo2SZRKjAK2IH2AdV+UIvb+AG1yidH7W5AewCKJ0iB1USox+qblyhdYVpQqueM4EqMJwTJ3CoglySDd5I4iOVaMegCLIkrVJm+HieR4B8spStUmlxh1g8UXpWG6bXJFcnyDZe/9VbMoVSQnd38L0gNYVFEaHPm82iI5dDHqBospStUmbz+LJUb9g2WTO4wbpetPqx4OePKcACyKKFVypy0oklM+sEyxUN1tckVy/IOVQnJHbfIpQymRnHKDdcfU74Ikd6pBlHaHbkH6B8uKUkRyJ79y712J0RQiOeUCy1Rlm1z18PKDhWmT57Oe3JmO3oL0DxaoTd56S5Kqh5cfLP9tcolSfiSnVcAyMSS586kVpRKjqWxB+gVLyR2JUQ9g8UVpvrWTO4rklBmsO5DkTr6V2+SK5HgDi74k6T+5o3q4K5JTbrBMDEnuhJtuZE6MzudvQfoHC5fcCbMmSv8zrxdGjKYAltVZjOROUMbkjiI5KYCl5I7EqH+wUG3yT7PTJv83rh6eAlj2eBWlSu7wIzk+weKL0sCnKJUYHWnFaLpgcZI7mRClo95iilEPYEmUqh7u3oL0CxZflOYrv00+CCRGUwALK0r9tckVyQGAxVmSXOxflEqM+gNLolRbkBSw1CbPUCSnLgKAxVuS/PY9iVEPkRwOWJzkTkWK0imgejgALKAozXtok6seDgLLxBBRmvewJKlIDgAs3JJk6CG5o0gOCKz7S5J5Alke2uSK5HDAUps8A/VwJlic5I4HUSoxCgALKEp/kxj1EMkBgAUTpU2V0iafgonkwMEy9ZDkTmW0yduBIjlssNzJHYlScD2cDRY6uSMx6ojkcMGSKJUY9QAWX5QG9DZ5e5AYJYElUZqBSM5Cx5NnAFhqkycQo7NpYpQPljGQ5M5ibnKHUg/fEUU4sPiiNPDQJpcYBYCFE6UBtE0+pXsX5JNnPlhFU0SI0sBDm1yRHBBYpggRpeEPHpI7iuRgwHKIUiV3ps/lRnL4YGGSOzUe2uSqh4PAUnIHXg93cMUGyyZ3CKI0cLTJFcnhgsUXpYGjTa5IDhks/pIkRZS2G4qJ5FQ2WEruZKAeTgSraHYj2uTBygseyFIkBwOWKWKSOwcAYvQNxhak5arSwbKiFLEkGXgQpRKjHLAsWYzkTj7tNnn7Tr0I9fDODjFKBIu/JLk4XVE6gh/J4YPFTO6kKkrb8iM5fLCgbfLA0SaXGOWCxRelYWrJnXbgejgfLP6SZGiTO6qHo8BScgcgRgGRHBhYnOTOphuK5PDB4id3CDdK/zOPH8nhg6U2OTaSYwUWDywldwD1cMCTZwBY1CXJYLFtk1edGI0iBlhZfiBt2+Sqh2cKrHvJHaQoBdTDAZEcPlh8UfpRO2Y9HBDJ4YMlUfpfUCQn22DZe38XFzPa5Bwxyt+C5IN1f0kyxCZ3VA/ng8VfkrysejgLLA9LkkhRqkgOHyy1yTGRnLoo+2Dx2uTfgcQofwuSD5app7TJJ/Hr4ZxIDh8sm9zhilJFcvhgOdrk+z+FtskVyeGDxV+SDBzJHUVyuGDxRWlgRanq4XywlNzp3yOX4SfPfLBMEZPc8aqzpnTsihaj2QfLFCHJndoPfEqHDnMokRw6WNm/UbryR49fsGbgxWj2wbpDaZN/8bU3495nPEGM2ifPfLCyL0oXn2nr64OwC0FgdbJcVTNYJjZFwr2//EpPQ5L9OyGM+9zNUXX/xYqNuXJiMaJescLPKnLPHMJhzTl/rC5qrlawYlPYeL0R4bLCplVe/mC9nkOchb3Gv3G1uWr/YtXHDasX1yC4CvNf/QL4g+Vza7TnrJN11fkvncKp2+9DsLLxigseXj0PyHFOr15vzj7oHy04WEVTOHR2u/0UpJz8Ny1/HT2zS45E1oau/r/E8+9jXblZa/9cccBq/Ju9u9qV5AiCMJzLcL9tzlszMy8znzcos92H0czMzMx8aWamZzKEamTGqOrxhPJ7hpDOv6nqnTkjrVvk5XHT+l3Ea795J6K9ossa45y4yQlKEY9haUQ7L114nnEW8OkuE/EYlkK089KNL7PPGpynEvEYlka0825ojLFii9dHRLzysJ6qHu2E9s7rjbFxmfdExGNY3UZ73VkR0j1XG+PknxMrIh7Dkop2wo3Txthrpf9/HXzXUYh4vWG9XTvaeTM3GGO190TEY1jdRftZ/KyqusYYO7wnIh7DErm089oPjcG3u8xzGgxLJdp57T3G2Nd7IuIxLJVo500Y4yTnyVziMSyRSztv2Bj+s4h4DEvl0s4rNayI+MrDwpt2RHsMS/8Szw+Lf9OuOKz9PIuIx7Ai2gs11lGeRcRjWDKXdt5EoTtWRDyGFdGepSuNscmziPhKw8Kl/QFE+0BpLzTGYs8i4isN6yni0t5Xlxpj0UrPIuIxrIj27OzLjLH+IM8i4jEszUt795/ZH73cwSPiMayI9qy98iJjHLHDs4j40sN6m7m09//VzLXGWHeoZxHxGJbG8xheerMxyskneRYRj2GVjfb2rAH1xLRx5h/vWUQ8hqUQ7bz21kuMs+4YzyLiMSyJaOedv8ZIex7kEBGPYSlc2nnt2FXGOnG7Q0Q8MSzi0i75b0I4bqVDRDyGxUf7W7i0D7I0c5Px5u/jBKEPWzEsgWjnta9fYrx1p6500I94fljy0Q4z440VMH+ng37E88PSjvYslfrRk4UnOUsg4jEsgUs774l3rYwj9j7YQSniiWER0d6eJeDSC6yQzcsdFCIefw+5YRHRruDKOWP93O8nOehHPD8s4WiH4SsaK+aI1Q76Ec8PSzfaa/x278ZNLoGIeAyLuLSLuPAjK2rBMgf9iOeHpRntkO753Ao7eZsLIJ7TYFhEtCtIU9NW2rohLEs/4vlhKUY7pJsnG4tl8RHPDUsq2qH8rmDd+uUO+hHPD0vp0p6lK29prIpmz30c9COeHdbPz2NktK9f1VgtR29Z6a4f8fSw3paKdpi58yKr6MRDj3LXj3huWFrRDmnizfOsquaAne76Ec8M6ymtaIcLX7jcapt/6L4u5Of/J54flmK048rw5kfWgWbPJSvFI54Y1qMvCl3aYfjOl9dYN048dZnrRjwxrKfkoj2dNXvFtdaZZsPqg0QjnhiWYrTfeM1F1qkVQ5oRTwxL79KezrrwllHjRMTnS/x/HdZTitE+conVFxGPYek/j8kSH+36Ec9/2PrzsPSjHRIT7Tz5iP+bYUW08yLiMSz5aIcuoz0iHsMSj3boMtoj4jGsiHZSXOLx9/DvhhWXdl5c4jGsiPbiIuIxrLi0lxYR//Ow8vOYl+LSTomI/92wvuafx0S0x3Oan4ele2lvmWivb8Ni4YjHsPSiHchor2+ecMRjWIpv2tu+RntEPIZVOtrj0h4Rj2HJRXvb52iP5zQYlly049I+MBrJiL/vkUfj0l5eXOLv+1Yo2uFsPtpp8WHr8hOej2gvLiL+ux/Yu2ttJ7owjOOf9R/upMf1CrCaa0iDOxUdDt25DCpuAx3LZpId2TOZCHbcW87G4cWfCevM5nmPS7r/mvVb7xrxg1PctC8axG9x43BVKoVRRT1xp6uTZRzt3MTjWcXVmtZ1ERY37UQ81FWj2TI6EWFx007EI1ml7cxopbCwiHYiXqBd2a6wsIh2buIF2m1WaFjctPPCVoF2pZwJq3z7aK5oJ+IBtCtnwirnvmkn4gG05xYWz2kn4re8RzseFtFOxEu0y7CIdiIeR7srYZ3MddPOc+JxtIuweCEqEY+jXYRFtBPx8KYdD4toJ+Il2vGweHoMES/RjofFc9q5iZdox8PiLR+JeIl2PCyeHkPEy007HhbRTsRLtONhcdNOxEu0Fy4sol3O34cOL2K0i7CIdiIeRzsQFjftRLxEOxgW0U7ES7QDYRHtRLxAOx4Wz2nnJl6gHQ+Lp8cQ8QLteFhEOxEv0O5OWADaifjc0Q6Exfu0E/EA2kVYRHshZ+d/673Fg3YRVnE37ZwVezd7vwXt7oRFtP/IPNq/vJMOMKu4K9AOhFWkTTvn+pUl6YCy8hpNiXY8rGI8EZWz6tK5bmpLyDurXj8TWQFhEe2Fm5tPj1ZDNC2RVZRpnSggrOJv2jnPnt9t93JkvNewWWmlXAiLm3ZgVr54Odz3Y/ywZV8e+00gKxEW0V7keXR9ZLQSVHvemI0DqMoLq0HL2KwcCotoR2Z8YlINN7uhbetXJ/X7w3LBUPCwiHZwzkxNazNT6/9aW2MLVUW1loGzEmHxPu0OIH5WGzMz3Kw24tLYj4trrBT3un1ZlQNhXSTac0L8XKKNUZUg6vZSb+yHogr99lA2YzBZibD4RFSnZgHxiVLaHriy+/22v1BXaexLfdnfemnoV/tB1tJG41XJsIh2xxBvF5vaxqVaWa0Ztbt+L0zj2HszcZymYcPvtqPm/aylDBAVEBY37UVEvLJpvavLqJlKNly7Hww1X08Q3K9lldaMNrIpV8Ky92m/MwC0E/GJej/ajnkz77/akU05E9bJwaGdiFf44GFx0+7eJj5J/sywyuWBop2In598xc6dU2UQBFEYdYEhnHTcGVuGAIQhY7RA1gbq1b/UHQ1fcE+dN30NCQva+xE/Lixob0L8rLBWH9ohfkBYB+2/bWh3iZ8S1trQ3o/4/rDMYyA+FBa0Q3wgrAHzGN9rG+JPWGPQDvFXe1h+RIX4QFgu7RAfCMs8BuIfNqw1YR4D8f1h3RXabeKvJwlrf/0MQDvE94a1BqId4iNhubRDfD6s9+8BaIf4QFjeaXeJj4flR1SIz4e1oX0A4gNhubRDfCAsTz5CfCQsP6LaxOfDgnaIz4fl0g7xkbCgHeIDYUE7xFeEBe3mNPmwFrRDfEVY0A7x+bBc2iE+ENYyj4H4grCg3ZwmEpYfUSE+Hxa0Q3xBWOYxEB8IC9ohPhIWtJvT5MOCdogPhOXSDvGBsKAd4gNhQTvEB8Ly5KNLfCYsr8dAfElY0A7x6bCWTTvEJ8KCdogvDgvabeIzYW2bdogPhAXt54P4krBc2s8H8aVhubSfD+Lrwtr/aH/5a9euDZuNATAMagl7wJ+ZmRmDG7jMGO7ThVFtmJmZSV/U5m6Gp3gFAc65ic8L65qbdhIjPiOsa+k/7Rjx6bCMdvJHfDosoz3NiM8Ny017mhGfEZbRnmbEp8Ny057NiE+HdTjaP4cUjPhEWFmjHSN+cSoWD+ta5WOhm3a4v7Q8H2OhsK49rNbcD1DM3aXlqRjTYT15//xugOJ+vpyca43x/LCuXbv19cfLn+FSoLk0Mbo41xq3nArr2oNrr742fnlXChng8cTSyvLcVGtXjPthPXhw81bla/29598fh2xQ/jkxPTi5PDY339rd9vDv/9fVH3+a3v0sBwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAK2AT0M1/vqYVGEMAAAAASUVORK5CYII="
GITHUB_LOGO_V13_B64 = "iVBORw0KGgoAAAANSUhEUgAAADIAAAAxCAYAAACYq/ofAAANzUlEQVR4nMWaeZRV1ZWHv33Ove/Vq6pXA1UFzSQyK1JMCiKICoqipJ26ddlxiCQLNWrUaK9W2qE7hu6O3Ymrl5rEFsdolDaGOCDgBBIjHYiCBgEHQItJKIqh5vfevffs/uO+V8ASZFDg1jr/VN17z/nu3mfvfX67RFU5XFdzS5uCggrpdLEctokA79t6Ud2GjfqX91fywfJPqavbyJb6erZv20k2CDAYyirS2qWmmu7dOjOgd09Gj6zlxJOGfGtw3wjkszXr9bU3FjFn3tu8/+FfaWzYgroIEDA+SrxOF0XgQlyUBRyIkEhXMKD/AL1g0gQuvnASw4cN/EZQciiu9eHytfrQjJm8+NI8dmzeDGLw0sUYTxBjQQEUVQcuQjUCdTEEggLqBM3m0EjxS8sZN240t0z9B84999RDAjookM+/2KT3/WIGTz7zImFbK6a0HPwkoFgNEIlAJQZQBxrtNkJAEREQgxofJIGTIlwEkmvHEDD+9FP46d03c9Kog3O7AwaZ8dizese//jdNWzZh09WYRBGRM0ROEBxWQkTivY26vBXCPUAEBREQD8THiY8jCSJ4VjDqcK07sQmfH15zFXdOu4GK8tIDAtovyI7GFr3x1unMfOppvKIkydIUqh6hJojUx2EAMIQFp9kLRDw0fydiwfggPmp8BMEzER5ZPOOIAqWtuZn+Q0fy/OP3UTu4335hvhZk08Z6vfR7t/HuW/NIlFXg++AZQY1H4FKEmkCxxK+IEBxxuI1BdgE5ds1jYosYDzEeiIcR8EyARw5LFkXJRsVkmjOUdOrE3OceZNwZI78WxuzrD1s2N+iFl93Au2/NwyurQY3BOY2XqsIe+KKIEEMQb2pVze8TjUfeWpIfIIDEz+VdUhUUi8PDuRBbnKK1cSdnXTyF+W8t+lrX2adFJk6eom/OeYlEeRWqDofFWIvn+YjxcOoROYPDxl9EIkRDIELdrrErWsWwRgSMR0QSxEeMwYhgjcOIQ4Mcqo7AeahGeNbhMgHFpSW8MedZRp44aK+W2atFbrx1ur4552W8dGcsGSw5XGQJspBpbiezo4koF2CM4nkGa8CKw+TdS8QhBsRIxzAGjCgmv2AjIMYgRnDOEbZnyDbuhESKREkl6hQRxZDBT0FL43YunfJPNDRs3+uX/wrIc8/P04cfegRTWgPEX9SIQbMtTD5nIk889gBTrrmKHt274tqaCJoacGEWsQbxLGIMzkGYCwgzOcJsjjAbEIYOpxKHXeujxsO1t+FaGrAEjBhZy1333MlHS+Yw7cfX4NpbQE3sxlGAV1JC3YplXH3Lfft3rfXr67V27AU0N2xFkj4SZfAlABLk2tt4fe6LTDhzdIdp35y/RGfOnscrc96icWMdLsyhYvCKiqmorKSkuARrLUEY0tzaQuP27USZJlCH9ZIMGnEKU6Z8l3NOG8Vx/Xt2vPezT9Zq7ajzCTFYk0WIUA1RU0zQuJ0nn3iY7111oewTZOr19+jjMx7HpCvQKIcQxiBqCXMh7y9ZSG1t36/46MZNDfr8S6+yub6J4bWDOH5gb7p2riLhexhriCJHS0sbGzdvYcmHy1mx4lNOHTWKKy87d5+R6PgRk/XjFWvxSjxwcTQT8chmHemaHqz94FWqKtO7nldVVJW/vPeRJquGqC09Xm3FYLVlA9VL99VUeQ9NlffS4op+Wre+Xgv3H+4xYdKViu2jttNQtWUDNVHeW5Nlx6hXMVCRKr3r3l/tsZaOPfKzh54gaNyBJLw47mPjxCWx3yPkI9CRuYpLS4nDdRymI7U4DJYsfqqIBx97hi31jR3uZABWr16ns2fPR5MloPl6CEM+9CDGQyOhqbnliIHsbGrBeImOtTgskVoMIX7So3HDZ7w8Z2HH/QZg1mt/IrdjCzZhUY3i8lsExcSWMUnCzA4+WL76iEA0bN2mq1Z9jvipOEuK5JOw5JOmghp++8LcPUFemTsfK2BNiKjmS478l5AEQTYiXdWNIYP6HxGQ6poqGTN6OEZzsWMZ8pUDRGoJIyCRYvHSpWzYtE0BzIZ1m3XFsmWYpMW4AC2UF/kyQsUnbN/GP99+O0Nq+xzW4+ru18+n30ayJIWGEVIoZfJ7JXIWPJ/M1k28v/QjAMzKT9fQvGMbxvfytRHE5YRirBBls1T07M9VV154pBgAGDCgl5x++slotiU2CTGI5veuMYCGvLNoaQzywYcriXIZVCyuAIFicBgjSNDChDFjqOl0eMWDvV2TzjoVdSGKTyH4yG4gRoSVqz4DwPt0TV1HTYoK8dknLvBiq1jGjBp+pBkAGDtqKFKUxjniNKDkN4vDGAeeZd2mjQCYLfXboCMaxDG7UF4LivETdO/e5aiA9OjamaJ0GTiHSCEdWMTYuJL2PbY37ohB2lrbAJMPtaYjRBR+rJ+korz0qIDUdKmW4nRpPu7YPYbBYIxHLpejsbldvSAMd4tvFiX2Qy1ErSgkCMKjAgIg4sXrM4Ca/O8kbxlAwTnFM0bygoCJT2oUoAQVg3OOHU3NRwViW/02zWSyiPUo6GExiUPFA1V8P4FnDSZVVJTf6rtFhYKrqUGDgI8/XnNUQFavXkf7zp2IZ3e5fUdMjU+oZWXlpEuLxNRUd8obxOyxoVzhdK2Gz7+oOyogi5YtJwpyqLFAwXPidTk8XBDRvWs3AEyPnt0QdsuesgvAOUWNz5JlK44KyJx5C0E8pHBmUqGQzBRBXUif3scCYGoH9cdaL3YmiYMvhQINRVIlrF+7mrf/uOzwyfZ7uZa+v1zfXbQUkyoilpoK6kuH6AqinHB8vxhk5PDBpNJlaBjmS+Zdwk2ciASNlJ/858NHkoOf/tevCdpbwZi8TqYdioyogyjEFBUz6sTBMUi//r3kuNpaXLYNFUPBAws4aISkyln8x/k8+Ktnj4hVHn1spr768mtISRka5UBjQVxxGA0RidBsG1179GTICX1jEICzJ56GiwKcxvSGENEoLlHyOSayFdxx9/3M/N0bhxXml488p9ffdDckS2JL4HC6S/ATclgJ0WwTZ542hnRJQjpALj5vPKakAhflY5VGcUrJBESNDbjAQVExgfpMvelfuP+h5751mC82fKlTrrtTb/zRNJyfBEusF7tYrYwTdIQhgjCLSaS4/NLzO57vUFHGnTtFFy14B5tKgQpRpo2e/Y/jvPHjeH726zTXt0KqBIkykG1h2Mkj+MGVFzDp7LH07F5zyJXx2rr1+r+z3uT+Xz9Bw+o1mPJKREys7OdPqPGJMBYAPeOImhs4afx5LJr3ZMe8HSAvz1mgF19yDeInEYGotZ3qY3qwZfVCWbFqvU69aTrv/enP2JJKxOVwuRY0ytGpS2dGnjSEIYP7csro4UyedMZ+oVZ+tEr/56nf89Enn7PkwxW0bKjDpoqwyQSRy5dJ+ZwWl0wOjxxiIqIQXKaJl1+cxXnnnPJVOUhVGTvxuyp+b7VlJ6gtG6FiqvRHP763Q3b5/vXTNZEaosXV47S8+iQtqx6pxRUj1E8N06JUL73vF48dkFy0ePF7SqKPYo9VKRukftUgLerUV4sqeqtfMUBNxWA1lUPVVA5TUzlCvU61WlzdX4tr+qvxO+uZF133lXn2EOgWvL1Yzzr3KvATiCgagKWdhfNnccrooQLwt5fcpnNfmkO6sgJ1AYhHkFVGDBvAOwufPmAXu+jym/XFP7yGTXkYl8VICGrinov4cVEohSIxxDc5NIjAJlm0YDbDa4/dY649tN/xZ5ws1173fVx7CyoexjdEYcTV19/Vcc8/3nQFiZISstlmsplWMu1tZFq/ZMzoEQfKAMBP7vghyaSHCbJY43BqiNSLG0dxrkOIBW8Rg9MEUXszt99281cgYC9thcbGFh07+VpWLP4/vNI0FkOuaRPT7pzGv917iwCs+uQLbWxsJoyieJKc0m9gT7p26XTAFtn85VYdMGISudZtWF/IRR6R+nHhKnGrweSLcoxP2LSFieddwNzfP7TXOfbaH1mzZpOOnnQF29ZvwitOQRBBNsvceU9x5oTRhxyh9pxjvdaO+g5EzVjPkY08IufHhzoRRCwigvEsUVsTg2qH8NbsZ6jeh3aw1/5I377dZMGsGVR160bY1oZNJFCr/OD6e/j447pvJYfkgmCXmJnvU1kJ8STESARo3hI76d6nH7N/9+g+IWA/PcQ1azbqhPOvZd3qVSSSacJMK+nqzlx2yWROOKEfZSXFOKesXbuB8WeMYMLpJx+wtVasXK0jx/0dhlaMCQlDYhVHBDUJIk0RtezguGHDmfPCo/Q+pvrr372/ULl16w6deNG1inRXUzxIpaRWkV6K7a0m0U/91AD1E130/geePiilfuXK1VpcNVSLqwZqcVUfLarorYmKYzXZqa96Jb1V/Bo9+8KpWt/QfEDv3WcztHBVV1fI67Meln//+XRSpWkIcyQrayit/htSFdUkSisRr5qE7x+oMXZ9RAeR84lIEZkiotAj29hMKl3Kf/zsPl77wyNSU3Vgffb9ghSuabdeLQvm/obTzxxPmGmnvWk7UZQjyv+PQDKRPCiIwvHbqUeQEYKdTYhaLr70Cv684FVuv3XKwQWVQ2nCzHzhbZ1w/g1a2vVktcmBahM9dcYTLx2Ua33y6VpNpo9Tm+yvnXqN1cun3q3vLPrrITeSvlFXafnKOr3zJ7/U2lEX6ZO/feWgFrFx42b9zt9fp/c/8ButW/flIQMUxv8DtGBieIxb2AQAAAAASUVORK5CYII="
OKTA_LOGO_V13_B64 = "iVBORw0KGgoAAAANSUhEUgAAAFoAAAAhCAYAAABZTbwAAAATxElEQVR4nM2aeZTcRbXHP7fq9+ttprunJ9skGTOBsAVIWCSYx2JQwIUHioqKcERENjEYBVkiEpT9EGVHdsSIATUYQNk3FwR8KLggoiQK2TNZmK2nl9+v6r4/fj2dDCQhPjz47jl9uk+fqrpV37p17/feKlFV3glZumSZ/vnFVwjCAEHAgjplu0kT2HbbLnk7Y7/22lL9+6KlpFMZPB7nI0SVPXfbhbZS8W2N/W8TVX1HPjffdqdCQTEdihmtBB0KLXrOOZfp2x37zLMv1nSmQ1Mt22uQm6Q2M0HDzFhdeO9Db3vsf9cneKc21DsPpAgLRURjFIj6lVQu97bHVgGRFGG+Fe/q+LrgYwH5/2HMAOadUhR5BQRLRCB1jElAUOzbHrvuHGiE0SpWKwSBQ12NOHIA9PT0vTP+cQvyjgFtRUAMFk9gPFYUjCDev+2xk0V4ROsYagTUMDjUJUA75+jrH/iPgv2OuQ5jTHKUxQIeEYv5Nx1t5xVVj6oHVVBQ9XifYGusoZBv/Y/6kXcMaDEGEcFLCAhIgIjh38F6nPd473EO1BtQITkoydiltv888/iXgF61YoVG9ZhCW4FiW9u/NHljLGIMKiFODYYQweK3Auj+/rLm8y1v0tff06f5toJ4VbwHr4L3Ali8CgO16F+Z4mZl1epujaoR2WyGkaNH/J82bYtAv7LoH3rfA7/g2ed+z6LF/2RVdzdx7MgXikzapkv/6917ctghM5i21+5vqVyMIMmP5HSLBTFbBPrFF/+qp511IZVKBQmtVmoRqhBXI0ptrVx+6Rz22GMXYufwqiiKV0FVMKkWLr70Km763nwVlCAU0qFF1PK1r57Mhw6esdk59/f36cNPPMt9Dz7Bi399me7ubqJqnVQ6zcSJnfrevfbkqCM/wi67ThaAr82+QJ/7/ctksiEintjFjOsYz7yb527QsSnO989Xl+jMr8zR9rFT1YajNJ0Zpa358Vps79LiiG20UOzSXEuXpjLjNDdiJ/3gx07QXz393BY56y3z7labGq+Zkbtrun2qZke/W4P0RP3GN6/eZL9ly5brpF0PUCMlTaXGqLWjFBmlUFLI6Wlfu7DZ7/iZX1cxJc2UujTIT1Dbuo0G+e01zE7QTK5TW/MTtFQaryPaRmu+pV2vvfnOzc51wT0P69Tph6pJjdbAtmgmXdLWwjgttHVpoThRW1o7NZsdpdnCJD155hxdsXKdjt9hPzV2jIbZCZpKj9VU2KbtY6doX1+5qedNFr3gxw/oCbPOpmfVEiQsksq3EVolsIpFQUBTKZQU3gXEtYhH73+EXz75G2bOPE7nXnD6Ji1FSHy0iEFFSQzZIxsFxL6+AS0UkqB1zElnsvjFF8iNHElAhVjTVKsGP9jPccd/me/MPafZ0RoDaHJiEMAgeIJMmiAwhIEnFUQYY/F9SrFQaOjr10Ih3xzn1NMv1msvvwIwhPkcqSCNtYo1YIygEiLkcGqJBiNuuekWnnvxFSqVOkFrkSAliGTQaopsNkPcYD3wBtdx/XXf01NmngHpLGHbaLxTcIrHoQgaJGCpCuoVvEdDSxCWcFGdKy69hKVLV+pdt3/7TWArCo2NSv7wyb+u2mwzBPIZZ3xLn3zgQVLt45B4AIfHecWXV/PRTx/DlVfMGTZ2EBhAUGhsZDJPRJAGX/cIeFARbGPVG2/yJ476ov70znnY1g7ECqIe74XACMYYxFgwFlEDDkwmjaedF555liCbQxFc7DEWVIW69yAb2HPz190/WainzJxN2JIlmwGJq4h6FPCkiDWkXoN6xeMiANsYyKJeCUJIF8dw9/zbmXXmJW9yvIkFD1mdBzyq2khkNsjV19ykl195I+liGyGDeIGILPWBfqbP+DC3X38Z+ZbssI00YpP0kCEKaRCxeOeJazFRDeqDQnXA4+IIo0n3fIPynXr6ebrwznlkiyMJTYy4JBZ4k6YWBZT7Ywb7Yqr9daLBOuoS6CQMCFpam1B6DfDeoBhcg2Y2jQFg6T+X6FFfnAOpNDYA4wfxkgYV4rrHR31kWtvpGNdJJpulr3+A19esxTvFZLIgiigYE2FbR3PdNTez37Td9ZOf/HATkCiKG1ROE2sWQVVxugGzn//sAf3a2XMh04oxEUZinMlQ76kwfvup3HHrtbSVCm92TUaaACMGIxZVx4j2dor5FhRBBGyjSWsu0+x67wOP6rVX30qmOAorEU4VMSHeG2r9g4wa38m0d+/JmDGj8N6xsnsNf/jL3+nt7oYglawdRTVIGI+65IPBb5SMBQAXXX4j9XUrsIUS6sp4MXgxuCgmlclw/EmzOProw9lu2/GkU2leX9fHs8/9katumc/zzz6DpDKIGLy3iE1o3Ncvu4GDDtpfS6XEapQkKOAdqMf7JKmoVcsAvPDCC/rZk8/DeTChxbs6JgxxVU+2rcSC27/LpEljt8BukmRIBBCLr/Yz+7Rz+dznj6BaqWLEEoYhxUJ62BgXXHglRhRjDRo7MCHeWVTrzDr1VL4861i27RozrM/zL/xNr7zphyz40UK8kOyeF1QM4JLToCRWPQR098punb/wIWymBUuMYog0RVyNaB/byc/m38w+++42TFGhNUNX18F8+oiDOW32d/T662/GhVkST+QJszmWvvxnFt77CMcd+/EGDgb1ivcO1ZjE5US0tiRFpc+eOJu+NauxLa2oj1CbJqoLVir86NbrmD59ly1TyIa7EFFEkljQOXYUhZa0FFrSm+zy4KO/1Bf/8CdSrSnUR3gCYp9CXY0br72WL3z+8E3q3HOPHWXe9eczsetdOvfSK/A2lXiuhnsExXtFN7Jo89CTTzGwehU2FWCI8RIQR4JpKXLfJkB+o1x+yenysY8dhqvUcCTuJrCCes89D/yi2c77JEVW71Dv8N4BwvLl3Rxx5En6l+efx2RbUR8jYnCxJapVuO6KKzjsI5vnvE2cm2xDMKJJ0G44yXXr1mtfX3/TvPobdY8HH3sa4gqBcah6nMkQl/s47vPHbxbkjeX8s0+Qgw45CF8dRCRhOkkc8qgfcpUNoB/9ze8SMmBMEqhMALUyJ37hc+z7FiAPycXnzSJVLDZKoQaPR1JZnvnDX1i0eIUOAY16VBPXoS6CsJX5d93NwgX3EOSK4GMECxqCq3He7LM47gsff8s5KNrw0RsIHoBp0Asxpsk+YEMQfPqZ34EJEoDE4CNHdsRYzjz9xK1ZNgAXnH0KJpNBncOIR3BAYlBsDPRLLy0GCRL6JgE4RVoKfPaTh261sokTO2XvPXaFaj8Yi/OC2jR93d0s+seSBhoK2iB5jQKQMQ5DnXQuxEqE4BFCXGWAGTP2Y855p2zVRgsNqtY4us140JAgCMi3Di8qLVm2WhcvWozaVFIXkRBfq/KePaey3TbjtjrN3n3qTjJ55+2Renk40Dp8DmZgoAzGJNxYLHhPYUQ7EyZ0bK0uALomdCC+khSOPGAMPo5Yu/71RFGDSEqDP4Mi4rHGEZg6gXENXushyPLyq0v5w/N/2uqK0xAlHuLrXn3TijdVuevt72ewvw81gtekPqLqmDChc6vX3NPTqwDjxo9G3GAD6EYFsUll2Wj9KgxlVIol9rqx1W+VxI3SpPqGIp8cn8FqPVFkE6SHGJ4MK2kakADEouoJ0hlWv7aMDx5xMq+9tmyrZ6JDChpjb6kyWK3WcVEtWbeaBiwWF7vN9nmjNBMeI6ivA66ZiCWbvZFF5/MtDUqUACE2pNLTR/fa17daIcDSZSvxWNS7xAWIIkbIZpKIb21AM6GAhiMVHCGRT1EtVxK+awSRCJsrsm7lao455dy31L0BUG0EpOToxi6J+r0bBcIhCWyAscEGt4NBTIYlK1dv9ZqLxYTTr+9eh1fb8Ms+4dWN4D8kpqtrPPh6ErVVwVr8YA+PPvZUs9Hmbif6+5L/V63s1t8+93sIMglrILFsk04zfuwoYKgeAag0axJqLPVanTCXZZ8Z70d8vWFXgI+wuRE88/hDzD7321u06qFAm1hT49s7XBwDyQ3LG2VUqUhLIY/G8QYjy7Tw++dfZPnKNQob2MmWZOmSZfqnP7+EBmm8cw1wtZErbAT0tN0nN6wrCVZCjAlbufq2O+npLWsCyeYkmcf5l11Lbc1ybCrhxoqg9Rojx4xm5x22ARpAi2xUg7Bo7BnXOZEnfraAXz06X96773RcpQ9jAwweQ0RQ6OCqK6/iwQef2Oyi1ftGIG0wGq9476nVas02/QPlYf073zVGdth5R3xUQbGJjYUhUe8arvru94EN7GRLcs2N36e6ZnkjyXKN0yVJwuI2AvqD752OyeVRF2OMx/oaQSak+5U/c9yp525RYb6Ql1vv+LHecN2t2GwGSwXwOLVoVOYDM/aho6N9aBuT4GSSNFlMiNbLfPzQDzNt7ykCcNXl3yTbUiCOooaH0eQKzOQ49tRvsGLl2mFg9fb2Nt2y4LASgTq8grqY7jVrABpZq9LfPxzs9x+wD14dTgMSJh4juXauuvp73HPvI29pzffe/4Ree81NhNkMQqLXa1K4cnFMXI83AL3bHlNkn+l7oNWeJG8Xj6GKyZa490cL+OBHT9YX/vT3Nyldu369fv2C6/T4z5+BGoOxYLQOorgoQlKtnHTcUc32zisiBtOorIlJrpwINlS4dt55e7n4/LOIB9eDCsbEqK9jsq30rHiVE079xrA5DN0JoonbM01q5UFS/PTnj7C+p09LbQUpFlrljbc0nzn8EGyuhHMkpwwFY3AqHHXCGXz3hh9sFuwrr/+hHn3Ml4miCBsKaJycZE2MyVUG6R0oN9sntY45s3jfLx5LynxGGywgwmTz/OqR+znw2d8ydd9putP2k8hmQlavXMWvn3qGlS//Ecm0JVHXObwVlACtruUzJ5zM9GmTmwtTbQBtQJQkIoshekOUnznzGPn5Q0/o4489QljoQHwd1TphfjRP3n83l31nDz3z9JOS3Krh90ePGYlXTe4MVVEfYzJZnv3Nr3nfh49kyi6TNZMNCa1h+bJujj7iUD595OEydcpkOfQjh+q9P76LoDgOIcnmJJXGu5gzzjqXG+f9VA88eAbbdo3Dx47Fry7m8V88zd9+90ecWmw2jXeVxrOJIdoK1gpfOOU09th5F/3qaSchQxF79py5OveSi5Bse2LVEmCMYK3BeIerVXBxvZHZOWwQYtNZnHfU4xhMiJoUfrCXbabsxf88vpARpQ0WdPO8hXrKCaeSLRbBDxJrQKWnn1O+MovrvjN7mKUtW7pKp+3/36xf04PN5TDqCEOPxhEuinn04fuY/p4NWetTTz+nB8w4lCCbB6qJZaoHBVcfBF8jDCxiLJVqmXPPm8s3zztNAF5Z9Krue9An6F3bg83lk6OvQipwBOKJBwdxUT3x4ygBdUxgiW0LtdgjGiNGUZ/QSWvBGk9oDUQ1vIu47Y67NtSjLzn/DDns08cQl3vxPo1Yi1XFapzcULS0kMqXSOfbyORbyeQsoSljpYoNDJ4Q17eOMdtO5pEF84aBDImfREyyeUYSxSLIJp6WdL6rQ269aS74Gr7uk0TARaRCg8YRx5z0dfoHas1jvd8+02Tv/fanVh4Ak0EkuREKbUw2F5LNt9LSlidfKhKmCqTTKQBef71Xt99uosy76QrSmRS+WkUkxBjBqMdonSCXJSyMIMiXCFpLhIU8YUYwWkXE4SJPXK7gfZhcQCMYDIFE5FosYSpFOgyHr3LhHVfL5048GeIqvlIDK0hgEufuE14axy75dh6PwYvFVer4gdVM2fsAnnrobiZt27GJmnHjqFubpPo2QEyAyqYThEM+8H45/ksnEg+uwasFEZyLMdki/3jpdxz/pdnD2l8591uUOjqIK4OISSc1GxPgMTinRJEnihPaGTVo19BrqQ994L1y949vpTRmJL68FlGP2MTXelXiRmCLo4g48jifuD5X7iXfXuDozx2HVvtQLGJDVAK8BHgCPEnd/U3mdPv1F8kPfnADk6ZMwVUj6oN16pEkhFwCPCkizVCNQsr9SqWvTLo4krPOuZCnHruLSdtsAmSS6yZjkncXrlEkF+LkumwzctG5X2H39+xDbaCXWmSpVqBerUIqz4L585jzrauanffeaxe5/54fsMOUXakPVqjXhMiHRJom0iyVKKQ8oERxnXot2dyhhAPg4PfvI08//hMOO+KjWI2Jy4NEdZ9cgRmLBBZFiOpKZSCiVi4zduIOzL/lau647VL5zLHH4sr91KueWAPqPkU1SuN84yZqyy9AF+qBHzlBOybtr+nCTmrCThUzRjEdGhYm6Y67HahnnjNX//bK0rd8tbngvge1tbSDto3ZTQtjdtNCx26aKe6gp8+eu8W+f315sb57/49pcfw0HdE5TUd07qXFzr20dcxUzY3cSR9+7Olh/de93qvnXXSNTpp6oKbbdlRJdTZer47TsLC9jt3mPXrDjfO3qPOxJ57VTx0zSzsnH6DZUbtqqmU7DXOTNMhtp/lxe+r0Az6l13/3h7p+fc+wcb73/ft0p70P18zo3TRs20mzI3bRkZ2768OP/lqbwXBLsnpNr776z2W8vOhV+gfKjCi1sfPO2zHhXR2UCrmtrnS99Ld/6FDNWIxgRGgrFRg9YssviXr6ylouJ5e4Q9XO2MXUaxEjR7ZRLLyZ5697vV//vmgJK9asoVKuMrJUoqtrHKNGFBnZvonrsE3IilVrdfGrS1m2fDX1KKKQa2Hyjtuy044TN/8mpFzRJa8tJ3KObDpDS0uWzvGj5X8BrFljh+Hh8mMAAAAASUVORK5CYII="
JAMF_LOGO_V13_B64 = "iVBORw0KGgoAAAANSUhEUgAAACwAAAAsCAYAAAAehFoBAAAIvUlEQVR4nLWZ3Y9eVRXGf2vvfeaj0+mUAgVpjYSm1Fgbglqq4UKFKCGRC24Ixjsv/Be80X/AGxMvvNKEKzXR1BtRrATSgB+ApYWCtJYWSltb2qml0+nHzHv2Xl7sz/N2agy+nHTmTPe7zz7PWutZaz17v6KqtNfSlRUNCgoIICIszE8Ja1xXrq5qCAFN8wFu2zCz5txJXQ7g/OKy/vw3r/LuyQv0Cn3fR7BGMEawxqg1ghiDoCCgqgQfCEEJKCEoIoKzVjtnefpbu/na7s9MHLyoKj/4yXP6t4MnWTdtCcldxggiBjEGI4IxBhEBAUFRDYQQ0KARvEYjRAwI2M7y/e8+yp4Htk4UtDn2/nl96+i/mJ+bwhpD5+KPcwbnBGcFa00EbAzGSDRABGvqT2eFzjmmnGV2usN45S9/Pz5JrAC41VVPCAErBpHIx+RI4m8FFEWTZwFCHNEAaJwlYAiICILBCATfTxyw8QFCH2JYQx4WMuRy5czS+Gf6RfsEIok2Unj+iQCuCw9fIEp0XTusEXF+RqnAc1Qk/eU/GcAhJlpFlOEWCJrv2cnNdIl1Y3BJCo6vIZsc4BC0ATAELinEkrCLDL0pheP12TxXIycmD1iJJQkiBSS5MDM4B1gSpuzzQgBpfVwJogrOmokDdqqSaNmCbC5pR7IXo2cFiVyWaoahcvvuOzcOXnbm3JK+/PpxTp+9xMgHfPKUNQZnDZ2zOGdxzmBNfWfA8/AX7mfnts3i8uItoELC5jNpfd4ygVIDB7Z1neXBnVvKOr989pD+4cV3uPjRdVb7VRQIIWAkxtAaE+u9tRhjEYGgSlBlxfc8/8p7/PB7j6mLHKDyrXGvoohKMyaJxxK5k5mhyRwTvb+yMmLXjq3s+uwWAfjxM/t138vHmJuZZXa2YzoYVGNtj/SLXdUYh1iLMZFKset6plVYXrrB3j8dilqiembtJKl4m8SKbYRaJ1JbBqY7w1OP7QLgt/ve0uf//C4bFxYYjVYJuSqlphP/RVqpBgg0eRCbUzRMuLp8AzNIquLaBCCDKyGXZiBVkPy8RHF0Y2XE1x+6n89tv0uWlq7p7148wtzcPL33QCCb2nqjVKLcVTUkg+JdQ0DxqCqmZrkOm4IOx3J7ri+h0lfAmMjJe+6Y58lvfB6Avx54j4uXryPZe5pAFSfI0AkN6PjukJ5TCPF5Z3Ld1Gq5tA8NaCIljHF9KXdB8CPPVx/awR23rxeAA++cpbMG1R40IFL1SJMWRQVStMpAAyQjDYrgSnMYJ8ZNeiHPkMEUaT7bsM7x8IPbADi/eEX/+f4FnLX06hMYbWp57PpSVmherdmwYalVBSOEwsGIra25DSXaH+qLRQTrHKurIx7YsZUt98wLwJETF7i8vELMw3CLiI0hJWPIWkVTNcmIFLd50zwzM47RyDPb2RwjTFJdsUmYcifdjOROB6OVEQsbZnji0QfLq984ehaI3K050XhLGt8WfTWuacaNFNzWTy3IU098WZ/59Uv4PmDEoBpq6AvobEAci1snmJ7quPfTd/KdJ77Etns3FX+9fewcXTdVdjCF+5n32XODTtrkyDj+BMEBfPvxXXLXpjl99dBxxDpsag5RoKdKK7GFGmswxjI1bdmwbh3b793M7p13DwL7ysGTeuHSVbpuGu/7ug9oO+RNFGykVJnfJl6c4fJLHtlznzyy5z4mcb3+9qmYUbmhDHBWqVoVQLW3jmlDhPh/BCYupy4vreqb75ym6yyqvgHShr2h2cCcCvamKw1NHPCbx86yuHQdaxuWFHCmds/yWRq4qd2Oo42I3a2mfNxr/ytHIx1Ke89dsmqQgjV3tszXxtvaUClHJxWoyV1HT5zXt4+cYmp6uqkO/+OlLTWahlGKVaTRRAHv/eNBem/KhiC+aRy1NHAKWv6rdU1PmBjgvfve0AOHP2B6dobQCJ285crvjZALhqr0Ghpn5uSaX7xr5P/n8EfLK7r3ucM8+8JhZtato/ceDTW0A98VYdNcaxQJLR80VVri6dPHAnz5ynVdvHSNA/84w/7XjnPqzCUW1s/S+z4exqimnQnk3YiSxQ6oDikhoqUul9Rrmkr0cKgePre4rL/6/SEWL10DUXzv8T7gQz0RUiAExStcvz7i8vIKN1ZGzM041s/N0Pehyev26CC+WKiIB2UtU31MBLVNpVBCEuAf/ewFjpxYZG52umhV7z3e+6j2NZ6moVGoO2fpOsfC+pkk6DQledoMSE66WxbWinaMEu0GVxPwSIe4o3HvfbCoJ0//m00Lc4QUKg0Ba4RgG7VFbqeR/IKgXhvxnePYqrKqwaIpjRGNgaVqSExSbabUpAMj4BTBda4unjd+6gnqoexuMwxTLI8p3eiFvJMeP/oSKR6XcXfqOB2aTUIpJbWKGBGDhHGxHppO0yimrDJbMM2tkSnDGc1WZ3DuWNtfGpIxkGZ4XIbETSjZUU0RkmZnUTXfWj9DhxXYA80eo1R3EWN2rrGuNMBbA12dn/jTnhkkLVBOf3JoSsHP4xWEJBkxoNjA3w2oYmjsflLYzhCoRAyBgFM0ZeZ4KaIKmEIFGQvR2nUg5VPlfnuqlDBXNpgY/ptWyHgV0XRSpIoDRb0v65ZwtVvskrY3t6S2B2RfZrFWdVfDgQQ63pqjrwFQqc9p9K0Gj6K4/D1FUI8GECPp26BW8SuYTIfUxTIdMlgh7dO05pE23a21sqkwec3q3DR7EJnAqB/RWcHdvnGehdvmuXDxCjNdlyYFFINo5RUaO42oYY3DLcoXNirVo6UcNqEreFItT/cIfMgzxaAoq6ueUa985Yvb4/d0rx0+oz/9xUssXrrGlDWpu0k9YaRpGGLioZ+hjjeNY9A6NAU2gdY2M4Xy9ZkR0xQErWuFQB8CxjqefPwhnv7mDpG8yIcXr+rpD5ewxuJszIr4LWeKnsmA85lFXNe0nkkKpgKWknilHjc0lZzIjcgphimMvGfU99yxcY6td8UDmv8A+KLZJTZSiOoAAAAASUVORK5CYII="
SPECTEROPS_LOGO_V116_PATH = PROJECT_ROOT / "assets" / "specterops_logo_v116.png"
ENTRA_ID_LOGO_V116_PATH = PROJECT_ROOT / "assets" / "microsoft_entra_id_v116.png"
GITHUB_LOGO_V13_PATH = PROJECT_ROOT / "assets" / "github_logo_v13.png"
OKTA_LOGO_V13_PATH = PROJECT_ROOT / "assets" / "okta_logo_v13.png"
JAMF_LOGO_V13_PATH = PROJECT_ROOT / "assets" / "jamf_logo_v13.png"
for _asset_path_v116, _asset_b64_v116 in [
    (SPECTEROPS_LOGO_V116_PATH, SPECTEROPS_LOGO_V116_B64),
    (ENTRA_ID_LOGO_V116_PATH, ENTRA_ID_LOGO_V116_B64),
    (GITHUB_LOGO_V13_PATH, GITHUB_LOGO_V13_B64),
    (OKTA_LOGO_V13_PATH, OKTA_LOGO_V13_B64),
    (JAMF_LOGO_V13_PATH, JAMF_LOGO_V13_B64),
]:
    _asset_path_v116.parent.mkdir(parents=True, exist_ok=True)
    if not _asset_path_v116.exists():
        _asset_path_v116.write_bytes(base64.b64decode(_asset_b64_v116))


# Embedded v1.0.3 BloodHound cover artwork supplied for this report build.
BHE_COVER_GRAPH_V103_B64 = "..."

def ensure_bhe_cover_graph_asset() -> Path:
    import base64 as _base64
    asset_dir = PROJECT_ROOT / "assets"
    asset_dir.mkdir(parents=True, exist_ok=True)
    asset_path = asset_dir / "bhe_cover_graph_v1018.png"
    try:
        if (not asset_path.exists()) or asset_path.stat().st_size < 10000:
            asset_path.write_bytes(_base64.b64decode(BHE_COVER_GRAPH_V103_B64))
    except Exception:
        fallback = OUTPUT_DIR / "bhe_cover_graph_v1018.png"
        fallback.parent.mkdir(parents=True, exist_ok=True)
        fallback.write_bytes(_base64.b64decode(BHE_COVER_GRAPH_V103_B64))
        return fallback
    return asset_path

BHE_COVER_GRAPH_V103_PATH = ensure_bhe_cover_graph_asset()


### 11C.2 PDF Colors, Icons, and Brand Badges


In [ ]:

# ---------------------------------------------------------------------------
# PDF visual system - SpecterOps report design language
# ---------------------------------------------------------------------------
PDF_COLORS = {
    # 2026 SpecterOps Brand Standards - canonical report palette.
    "ink": colors.HexColor("#0D0A30"),
    "muted": colors.HexColor("#5C6F8A"),
    "line": colors.HexColor("#E3EAF0"),
    "panel": colors.HexColor("#F3F8FC"),
    "white": colors.white,
    "accent": colors.HexColor("#2C2677"),
    "accent_dark": colors.HexColor("#0D0A30"),
    "blue": colors.HexColor("#4A3BD7"),
    "neon_blue": colors.HexColor("#5465FF"),
    "light_blue": colors.HexColor("#66A3FF"),
    "green": colors.HexColor("#05CE8B"),
    "green_dark": colors.HexColor("#008E62"),
    "orange": colors.HexColor("#F46036"),
    "critical": colors.HexColor("#6536C8"),
    "high": colors.HexColor("#E4251B"),
    "moderate": colors.HexColor("#F36C21"),
    "medium": colors.HexColor("#F36C21"),
    "low": colors.HexColor("#F2C94C"),
    "informational": colors.HexColor("#8EAADB"),
    "github": colors.HexColor("#0D0A30"),
    "okta": colors.HexColor("#2C2677"),
    "jamf": colors.HexColor("#66A3FF"),
    "entra": colors.HexColor("#5465FF"),
    "ad": colors.HexColor("#4A3BD7"),
}


def pdf_text(value: Any) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    text = str(value).strip()
    if text.lower() in {"nan", "none", "null", "nat"}:
        return ""
    return xml_escape(text).replace("\n", "<br/>")


def severity_color(level: Any):
    return PDF_COLORS.get(str(level or "").strip().lower(), PDF_COLORS["accent"])


THEME_GLYPHS = {
    "Tier Zero Exposure": "T0",
    "Privilege Escalation": "UP",
    "Credential Exposure": "KEY",
    "Lateral Movement": "MOVE",
    "NTLM Relay and Coercion": "NTLM",
    "ADCS Exposure": "ADCS",
    "Cloud and Hybrid Identity": "CLOUD",
    "Identity Hygiene": "ID",
    "GitHub Identity & Supply Chain Risk": "GH",
    "Okta Workforce Identity Risk": "OKTA",
    "Jamf Device Administration Risk": "JAMF",
}

PLATFORM_BRANDS = {
    "Active Directory": ("AD", PDF_COLORS["ad"]),
    "Azure / Entra ID": ("Microsoft Entra ID", PDF_COLORS["entra"]),
    "GitHub": ("GitHub", PDF_COLORS["github"]),
    "Okta": ("okta", PDF_COLORS["okta"]),
    "Jamf": ("jamf", PDF_COLORS["jamf"]),
}


def platform_brand_badge(platform: str, styles: Dict[str, ParagraphStyle], width: float = 1.28 * inch) -> Table:
    label, brand_color = PLATFORM_BRANDS.get(platform, (platform, PDF_COLORS["accent"]))
    logo_style = ParagraphStyle(
        f"PlatformBrand{re.sub(r'[^A-Za-z0-9]', '', platform)}",
        parent=styles["inverse"], fontName="Helvetica-Bold",
        fontSize=8.0 if len(label) < 10 else 6.8, leading=9.0,
        alignment=TA_CENTER,
    )
    table = Table([[Paragraph(pdf_text(label), logo_style)]], colWidths=[width], rowHeights=[0.34 * inch])
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), brand_color),
        ("BOX", (0, 0), (-1, -1), 0.45, brand_color),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("ALIGN", (0, 0), (-1, -1), "CENTER"),
        ("LEFTPADDING", (0, 0), (-1, -1), 6),
        ("RIGHTPADDING", (0, 0), (-1, -1), 6),
        ("TOPPADDING", (0, 0), (-1, -1), 4),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
    ]))
    return table


def _theme_symbol(theme: str, accent, size: float = 34) -> Drawing:
    """Solid shield with the approved category glyphs used by the HTML report."""
    d = Drawing(size, size)
    cx = size / 2
    shield = [
        cx, size * 0.93,
        size * 0.82, size * 0.78,
        size * 0.77, size * 0.34,
        cx, size * 0.10,
        size * 0.23, size * 0.34,
        size * 0.18, size * 0.78,
    ]
    d.add(Polygon(points=shield, fillColor=accent, strokeColor=accent, strokeWidth=0.8))
    white = colors.white
    low = theme.lower()

    if "tier zero" in low:
        # Crown: privileged control / Tier Zero.
        d.add(Polygon(points=[
            size*.31,size*.48, size*.36,size*.66, size*.47,size*.55,
            size*.55,size*.68, size*.64,size*.55, size*.71,size*.66,
            size*.68,size*.48,
        ], fillColor=None, strokeColor=white, strokeWidth=1.7))
        d.add(Line(size*.32,size*.44,size*.68,size*.44,strokeColor=white,strokeWidth=1.7))
        d.add(Line(size*.36,size*.39,size*.64,size*.39,strokeColor=white,strokeWidth=1.7))
    elif "privilege" in low or "permission" in low:
        # Key: delegated rights / privilege escalation.
        d.add(Circle(size*.39,size*.59,size*.09,fillColor=None,strokeColor=white,strokeWidth=1.8))
        d.add(Line(size*.46,size*.52,size*.67,size*.31,strokeColor=white,strokeWidth=2.2))
        d.add(Line(size*.58,size*.40,size*.65,size*.47,strokeColor=white,strokeWidth=1.6))
        d.add(Line(size*.63,size*.35,size*.69,size*.41,strokeColor=white,strokeWidth=1.6))
    elif "credential" in low:
        # Identity card: exposed passwords, secrets, and account material.
        d.add(Rect(size*.29,size*.36,size*.42,size*.34,rx=1.5,ry=1.5,fillColor=None,strokeColor=white,strokeWidth=1.5))
        d.add(Circle(size*.40,size*.57,size*.055,fillColor=None,strokeColor=white,strokeWidth=1.3))
        d.add(Line(size*.35,size*.46,size*.45,size*.46,strokeColor=white,strokeWidth=1.3))
        d.add(Line(size*.51,size*.59,size*.65,size*.59,strokeColor=white,strokeWidth=1.2))
        d.add(Line(size*.51,size*.50,size*.65,size*.50,strokeColor=white,strokeWidth=1.2))
    elif "lateral" in low:
        # Hierarchy/network: movement between systems and administrative tiers.
        d.add(Rect(size*.43,size*.62,size*.14,size*.13,fillColor=None,strokeColor=white,strokeWidth=1.4))
        d.add(Rect(size*.24,size*.31,size*.14,size*.13,fillColor=None,strokeColor=white,strokeWidth=1.4))
        d.add(Rect(size*.43,size*.31,size*.14,size*.13,fillColor=None,strokeColor=white,strokeWidth=1.4))
        d.add(Rect(size*.62,size*.31,size*.14,size*.13,fillColor=None,strokeColor=white,strokeWidth=1.4))
        d.add(Line(cx,size*.62,cx,size*.51,strokeColor=white,strokeWidth=1.4))
        d.add(Line(size*.31,size*.51,size*.69,size*.51,strokeColor=white,strokeWidth=1.4))
        for x in (size*.31,size*.50,size*.69): d.add(Line(x,size*.51,x,size*.44,strokeColor=white,strokeWidth=1.4))
    elif "ntlm" in low:
        # Interlocking links: coercion and relay.
        d.add(Circle(size*.42,size*.57,size*.11,fillColor=None,strokeColor=white,strokeWidth=1.7))
        d.add(Circle(size*.58,size*.43,size*.11,fillColor=None,strokeColor=white,strokeWidth=1.7))
        d.add(Line(size*.45,size*.52,size*.55,size*.48,strokeColor=white,strokeWidth=1.8))
    elif "adcs" in low or "certificate" in low:
        # Certificate/document.
        d.add(Rect(size*.31,size*.34,size*.38,size*.40,fillColor=None,strokeColor=white,strokeWidth=1.5))
        d.add(Line(size*.38,size*.62,size*.62,size*.62,strokeColor=white,strokeWidth=1.2))
        d.add(Line(size*.38,size*.54,size*.62,size*.54,strokeColor=white,strokeWidth=1.2))
        d.add(Circle(cx,size*.40,size*.055,fillColor=None,strokeColor=white,strokeWidth=1.3))
        d.add(Line(size*.47,size*.35,size*.44,size*.28,strokeColor=white,strokeWidth=1.2))
        d.add(Line(size*.53,size*.35,size*.56,size*.28,strokeColor=white,strokeWidth=1.2))
    elif "cloud" in low:
        # Cloud control plane.
        d.add(Circle(size*.40,size*.52,size*.12,fillColor=None,strokeColor=white,strokeWidth=1.5))
        d.add(Circle(size*.56,size*.56,size*.15,fillColor=None,strokeColor=white,strokeWidth=1.5))
        d.add(Line(size*.29,size*.42,size*.71,size*.42,strokeColor=white,strokeWidth=1.7))
        d.add(Circle(cx,size*.46,size*.025,fillColor=white,strokeColor=white))
    elif "hygiene" in low:
        # Identity/user with control indicator.
        d.add(Circle(cx,size*.61,size*.08,fillColor=None,strokeColor=white,strokeWidth=1.5))
        hygiene_arc = GraphicsPath()
        hygiene_arc.moveTo(size*.35, size*.31)
        hygiene_arc.curveTo(size*.39, size*.50, size*.61, size*.50, size*.65, size*.31)
        hygiene_arc.strokeColor = white
        hygiene_arc.strokeWidth = 1.5
        hygiene_arc.fillColor = None
        d.add(hygiene_arc)
        d.add(Circle(size*.67,size*.39,size*.045,fillColor=None,strokeColor=white,strokeWidth=1.2))
        d.add(Line(size*.67,size*.31,size*.67,size*.35,strokeColor=white,strokeWidth=1.2))
    elif "github" in low:
        # Code brackets: source and software supply chain.
        d.add(Line(size*.38,size*.62,size*.28,size*.52,strokeColor=white,strokeWidth=1.8))
        d.add(Line(size*.28,size*.52,size*.38,size*.42,strokeColor=white,strokeWidth=1.8))
        d.add(Line(size*.62,size*.62,size*.72,size*.52,strokeColor=white,strokeWidth=1.8))
        d.add(Line(size*.72,size*.52,size*.62,size*.42,strokeColor=white,strokeWidth=1.8))
        d.add(Line(size*.55,size*.68,size*.45,size*.36,strokeColor=white,strokeWidth=1.6))
    elif "okta" in low:
        # Compact Okta mark centered inside the shield without stretching the wordmark.
        d.add(Circle(cx, size*.53, size*.145, fillColor=None, strokeColor=white, strokeWidth=1.8))
        d.add(Circle(cx, size*.53, size*.045, fillColor=white, strokeColor=white))
    elif "jamf" in low:
        # Managed device.
        d.add(Rect(size*.35,size*.31,size*.30,size*.43,rx=2,ry=2,fillColor=None,strokeColor=white,strokeWidth=1.6))
        d.add(Line(size*.42,size*.65,size*.58,size*.65,strokeColor=white,strokeWidth=1.2))
        d.add(Circle(cx,size*.37,size*.018,fillColor=white,strokeColor=white))
    else:
        d.add(Circle(cx,size*.54,size*.075,fillColor=white,strokeColor=white))
    return d

def platform_logo_for_theme(risk_theme: str, max_w: float = 42, max_h: float = 42):
    low = str(risk_theme or "").lower()
    if "github" in low:
        return Image(str(GITHUB_LOGO_V13_PATH), width=min(max_w, 38), height=min(max_h, 38), kind="proportional")
    if "okta" in low:
        return Image(str(OKTA_LOGO_V13_PATH), width=min(max_w, 42), height=min(max_h, 18), kind="proportional")
    if "jamf" in low:
        return Image(str(JAMF_LOGO_V13_PATH), width=min(max_w, 36), height=min(max_h, 36), kind="proportional")
    return None


### 11C.3 Theme Components and Styles


In [ ]:


def theme_header(title: str, risk_theme: str, priority: int, styles: Dict[str, ParagraphStyle], risk_color=None) -> Table:
    title_stack = [
        Paragraph(f"PRIORITY {priority} &nbsp;|&nbsp; {pdf_text(risk_theme).upper()}", styles["eyebrow"]),
        Paragraph(pdf_text(title), styles["h1"]),
    ]
    icon_color = risk_color or PDF_COLORS["blue"]
    platform_logo = platform_logo_for_theme(risk_theme, 42, 42)
    header_icon = platform_logo if platform_logo is not None else _theme_symbol(risk_theme, icon_color, 48)
    table = Table([[header_icon, title_stack]], colWidths=[0.78 * inch, 6.22 * inch], hAlign="LEFT")
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), colors.HexColor("#F3F8FC")),
        ("BOX", (0, 0), (-1, -1), 0.8, PDF_COLORS["line"]),
        ("LINEABOVE", (0, 0), (-1, 0), 3.4, icon_color),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("ALIGN", (0, 0), (0, 0), "CENTER"),
        ("LEFTPADDING", (0, 0), (0, 0), 7),
        ("RIGHTPADDING", (0, 0), (0, 0), 4),
        ("LEFTPADDING", (1, 0), (1, 0), 10),
        ("RIGHTPADDING", (1, 0), (1, 0), 10),
        ("TOPPADDING", (0, 0), (-1, -1), 7),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
    ]))
    return table


def risk_theme_card(row: pd.Series, styles: Dict[str, ParagraphStyle]) -> Table:
    theme = str(row.get("Risk Theme", "Risk Theme"))
    risk = str(row.get("Risk Level", ""))
    findings = row.get("Findings", row.get("Finding Count", ""))
    accent = severity_color(risk)
    title_style = ParagraphStyle("ThemeCardTitle109", parent=styles["h3"], fontSize=8.5, leading=10.4, spaceBefore=0, spaceAfter=3)
    meta_style = ParagraphStyle("ThemeCardMeta109", parent=styles["small"], fontSize=6.8, leading=8.3, spaceAfter=0)
    body = [Paragraph(pdf_text(theme), title_style), Paragraph(f"<b>{pdf_text(risk)}</b> &nbsp;|&nbsp; {pdf_text(findings)} findings", meta_style)]
    table = Table([[_theme_symbol(theme, accent, 49), body]], colWidths=[0.72 * inch, 2.68 * inch], rowHeights=[0.68 * inch])
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), colors.white),
        ("BOX", (0, 0), (-1, -1), 0.85, PDF_COLORS["line"]),
        ("LINEABOVE", (0, 0), (-1, 0), 3.5, accent),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("ALIGN", (0, 0), (0, 0), "CENTER"),
        ("LEFTPADDING", (0, 0), (0, 0), 7),
        ("RIGHTPADDING", (0, 0), (0, 0), 4),
        ("LEFTPADDING", (1, 0), (1, 0), 7),
        ("RIGHTPADDING", (1, 0), (1, 0), 7),
        ("TOPPADDING", (0, 0), (-1, -1), 6),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
    ]))
    return table


def risk_theme_card_grid(df: pd.DataFrame, styles: Dict[str, ParagraphStyle]) -> Table:
    cards = [risk_theme_card(row, styles) for _, row in df.iterrows()]
    rows = []
    for i in range(0, len(cards), 2):
        rows.append([cards[i], cards[i + 1] if i + 1 < len(cards) else ""])
    grid = Table(rows, colWidths=[3.48 * inch, 3.48 * inch], hAlign="LEFT")
    grid.setStyle(TableStyle([
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 0),
        ("RIGHTPADDING", (0, 0), (-1, -1), 6),
        ("TOPPADDING", (0, 0), (-1, -1), 2.5),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 2.5),
    ]))
    return grid


def pdf_styles() -> Dict[str, ParagraphStyle]:
    sample = getSampleStyleSheet()
    return {
        "title": ParagraphStyle(
            "PDFTitle", parent=sample["Title"], fontName="Helvetica-Bold",
            fontSize=27, leading=31, textColor=PDF_COLORS["accent_dark"],
            alignment=TA_LEFT, spaceAfter=14,
        ),
        "cover_customer": ParagraphStyle(
            "CoverCustomer", parent=sample["Normal"], fontName="Helvetica-Bold",
            fontSize=20, leading=24, textColor=PDF_COLORS["accent_dark"], spaceAfter=4,
        ),
        "cover_meta": ParagraphStyle(
            "CoverMeta", parent=sample["Normal"], fontName="Helvetica",
            fontSize=10.5, leading=15, textColor=colors.HexColor("#3C4F7C"), spaceAfter=3,
        ),
        "eyebrow": ParagraphStyle(
            "Eyebrow", parent=sample["Normal"], fontName="Helvetica-Bold",
            fontSize=8.5, leading=11, textColor=PDF_COLORS["green_dark"],
            spaceBefore=2, spaceAfter=5,
        ),
        "h1": ParagraphStyle(
            "PDFH1", parent=sample["Heading1"], fontName="Helvetica-Bold",
            fontSize=23.5, leading=28, textColor=PDF_COLORS["accent_dark"],
            spaceBefore=5, spaceAfter=13,
        ),
        "h2": ParagraphStyle(
            "PDFH2", parent=sample["Heading2"], fontName="Helvetica-Bold",
            fontSize=14, leading=17.5, textColor=PDF_COLORS["accent_dark"],
            spaceBefore=12, spaceAfter=7,
        ),
        "h3": ParagraphStyle(
            "PDFH3", parent=sample["Heading3"], fontName="Helvetica-Bold",
            fontSize=10.5, leading=14, textColor=PDF_COLORS["accent_dark"],
            spaceBefore=7, spaceAfter=4,
        ),
        "body": ParagraphStyle(
            "PDFBody", parent=sample["BodyText"], fontName="Helvetica",
            fontSize=9.6, leading=13.8, textColor=PDF_COLORS["ink"], spaceAfter=7,
        ),
        "lead": ParagraphStyle(
            "PDFLead", parent=sample["BodyText"], fontName="Helvetica",
            fontSize=11.4, leading=16.4, textColor=PDF_COLORS["muted"], spaceAfter=9,
        ),
        "small": ParagraphStyle(
            "PDFSmall", parent=sample["BodyText"], fontName="Helvetica",
            fontSize=8.25, leading=11.1, textColor=PDF_COLORS["muted"], spaceAfter=4,
        ),
        "meta": ParagraphStyle(
            "PDFMeta", parent=sample["BodyText"], fontName="Helvetica",
            fontSize=8.4, leading=11, textColor=PDF_COLORS["muted"], spaceAfter=3,
        ),
        "label": ParagraphStyle(
            "PDFLabel", parent=sample["BodyText"], fontName="Helvetica-Bold",
            fontSize=8, leading=10, textColor=PDF_COLORS["accent_dark"],
        ),
        "metric_value": ParagraphStyle(
            "MetricValue", parent=sample["Normal"], fontName="Helvetica-Bold",
            fontSize=22, leading=24, textColor=PDF_COLORS["accent"], spaceAfter=3,
        ),
        "metric_label": ParagraphStyle(
            "MetricLabel", parent=sample["Normal"], fontName="Helvetica-Bold",
            fontSize=7.3, leading=9, textColor=PDF_COLORS["muted"],
        ),
        "table": ParagraphStyle(
            "PDFTable", parent=sample["BodyText"], fontName="Helvetica",
            fontSize=7.35, leading=9.5, textColor=PDF_COLORS["ink"],
        ),
        "table_header": ParagraphStyle(
            "PDFTableHeader", parent=sample["BodyText"], fontName="Helvetica-Bold",
            fontSize=7.0, leading=8.8, textColor=colors.HexColor("#475467"),
        ),
        "inverse": ParagraphStyle(
            "Inverse", parent=sample["BodyText"], fontName="Helvetica-Bold",
            fontSize=9, leading=12, textColor=colors.white,
        ),
        "evidence_label": ParagraphStyle(
            "EvidenceLabel", parent=sample["Normal"], fontName="Helvetica-Bold",
            fontSize=7.5, leading=9, textColor=PDF_COLORS["green_dark"], spaceAfter=2,
        ),
    }


### 11C.4 Document Template and PDF Helpers


In [ ]:


class StyledIdentityRiskDocTemplate(BaseDocTemplate):
    def __init__(self, filename: str, **kwargs):
        super().__init__(filename, **kwargs)
        self.cover_data = kwargs.pop("cover_data", {}) if "cover_data" in kwargs else {}
        cover_frame = Frame(0, 0, letter[0], letter[1], id="cover", leftPadding=0, rightPadding=0, topPadding=0, bottomPadding=0)
        body_frame = Frame(self.leftMargin, self.bottomMargin, self.width, self.height, id="body")
        self.addPageTemplates([
            PageTemplate(id="cover", frames=cover_frame, onPage=self._draw_cover),
            PageTemplate(id="whitepaper", frames=body_frame, onPage=self._draw_page_chrome),
        ])

    @staticmethod
    def _asset_path(value):
        if not value:
            return None
        path = Path(str(value)).expanduser()
        if not path.is_absolute():
            path = PROJECT_ROOT / path
        return path if path.exists() else None

    @staticmethod
    def _draw_fitted_image(canvas, path, x, y, box_w, box_h, anchor="center"):
        if not path:
            return False
        try:
            from reportlab.lib.utils import ImageReader
            image = ImageReader(str(path))
            iw, ih = image.getSize()
            scale = min(box_w / iw, box_h / ih)
            dw, dh = iw * scale, ih * scale
            dx = x + (box_w - dw) / 2
            dy = y + (box_h - dh) / 2
            if anchor == "left":
                dx = x
            canvas.drawImage(image, dx, dy, width=dw, height=dh, mask="auto", preserveAspectRatio=True)
            return True
        except Exception:
            return False

    def _draw_cover(self, canvas, doc):
        """Cover closely aligned to the 2026 SpecterOps Brand Standards Guide."""
        w, h = letter
        canvas.saveState()
        data = getattr(self, "cover_data", {}) or {}
        title = str(data.get("assessment_name", "BloodHound Enterprise Identity Risk Report"))
        customer = str(data.get("customer_name", "Customer"))
        generated = str(data.get("generated_date", ""))
        classification = str(data.get("classification", "Confidential"))
        version = str(data.get("report_version", ""))

        # Use one solid vector rectangle for the cover background. The previous
        # multi-band gradient could display and print as visible horizontal stripes.
        cover_background = colors.HexColor("#211B5A")
        green = colors.HexColor("#05CE8B")
        violet = colors.HexColor("#4A3BD7")
        orange = colors.HexColor("#F46036")
        white = colors.white
        muted = colors.HexColor("#D7D6E6")

        canvas.setFillColor(cover_background)
        canvas.rect(0, 0, w, h, fill=1, stroke=0)

        # Official full-color SpecterOps logo. The clear-space box follows the
        # brand guide cap-height rule and preserves the native aspect ratio.
        self._draw_fitted_image(canvas, SPECTEROPS_LOGO_V116_PATH, 40, h - 91, 226, 49, anchor="left")

        # v1.0.19 cover illustration: dense, thin-line hexagon network inspired
        # by page 11 of the 2026 Brand Standards Guide. Lines remain at or below
        # 1 pt, several cells intentionally continue off the right edge, and a
        # single orange route travels naturally through the topology.
        import math
        radius = 34
        x_step = radius * 1.50
        y_step = radius * math.sqrt(3)
        origin_x = 414
        origin_y = 590
        hex_centers = []
        for col in range(6):
            for row in range(9):
                cx = origin_x + col * x_step
                cy = origin_y - row * y_step + (y_step / 2 if col % 2 else 0)
                if 105 <= cy <= 665:
                    hex_centers.append((cx, cy))
        for idx, (cx, cy) in enumerate(hex_centers):
            # Slight variation adds depth without departing from the thin-line style.
            stroke = colors.HexColor("#4A3BD7") if (idx % 5 == 0) else colors.HexColor("#332C82")
            canvas.setStrokeColor(stroke)
            canvas.setLineWidth(0.62 if idx % 5 else 0.82)
            pts = []
            for j in range(6):
                angle = math.radians(30 + 60 * j)
                pts.extend([cx + radius * math.cos(angle), cy + radius * math.sin(angle)])
            path = canvas.beginPath(); path.moveTo(pts[0], pts[1])
            for j in range(2, len(pts), 2):
                path.lineTo(pts[j], pts[j + 1])
            path.close(); canvas.drawPath(path, fill=0, stroke=1)

        # One continuous orange attack path, aligned to hexagon edges.
        orange_path = [(390, 604), (414, 590), (414, 531), (465, 502),
                       (465, 443), (516, 414), (516, 355), (567, 326),
                       (567, 267), (618, 238)]
        canvas.setStrokeColor(orange); canvas.setLineWidth(0.95)
        route = canvas.beginPath(); route.moveTo(*orange_path[0])
        for point in orange_path[1:]: route.lineTo(*point)
        canvas.drawPath(route, fill=0, stroke=1)
        canvas.setFillColor(orange)
        canvas.circle(orange_path[0][0], orange_path[0][1], 2.1, fill=1, stroke=0)
        canvas.circle(orange_path[-1][0], orange_path[-1][1], 2.1, fill=1, stroke=0)

        # Lower-left hierarchy mirrors the Brand Standards Guide cover.
        canvas.setFillColor(green); canvas.setFont("Helvetica", 14)
        canvas.drawString(40, 225, generated[-4:] if len(generated) >= 4 else "2026")
        canvas.setFillColor(white); canvas.setFont("Helvetica-Bold", 31)
        words=title.replace("\n"," ").split(); lines=[]; current=[]
        for word in words:
            candidate=" ".join(current+[word])
            if canvas.stringWidth(candidate,"Helvetica-Bold",31) <= 305: current.append(word)
            else:
                if current: lines.append(" ".join(current))
                current=[word]
        if current: lines.append(" ".join(current))
        text=canvas.beginText(40,184); text.setLeading(35)
        for line in lines[:4]: text.textLine(line)
        canvas.drawText(text)

        canvas.setFillColor(green); canvas.rect(40, 84, 220, 3, fill=1, stroke=0)
        canvas.setFillColor(muted); canvas.setFont("Helvetica-Bold", 7.2); canvas.drawString(40, 63, "PREPARED FOR")
        canvas.setFillColor(white); canvas.setFont("Helvetica-Bold", 13); canvas.drawString(40, 44, customer)
        if generated:
            canvas.setFillColor(muted); canvas.setFont("Helvetica", 7.2); canvas.drawString(40, 29, f"REPORT DATE  {generated}")
        canvas.setFillColor(muted); canvas.setFont("Helvetica", 7); canvas.drawRightString(w-36, 24, classification + (f" | {version}" if version else ""))
        canvas.restoreState()

    def _draw_page_chrome(self, canvas, doc):
        w, h = letter
        canvas.saveState()
        canvas.setFillColor(PDF_COLORS["green"])
        canvas.rect(0, h - 4, w * 0.34, 4, fill=1, stroke=0)
        canvas.setFillColor(PDF_COLORS["neon_blue"])
        canvas.rect(w * 0.34, h - 4, w * 0.32, 4, fill=1, stroke=0)
        canvas.setFillColor(PDF_COLORS["accent"])
        canvas.rect(w * 0.66, h - 4, w * 0.34, 4, fill=1, stroke=0)
        canvas.setStrokeColor(colors.HexColor("#D5DBE7"))
        canvas.line(self.leftMargin, h - 35, w - self.rightMargin, h - 35)
        canvas.setFillColor(PDF_COLORS["muted"])
        canvas.setFont("Helvetica", 7.4)
        canvas.drawString(self.leftMargin, h - 26, "BLOODHOUND ENTERPRISE IDENTITY RISK REPORT")
        canvas.setFillColor(PDF_COLORS["accent_dark"])
        canvas.rect(0, 0, w, 30, fill=1, stroke=0)
        canvas.setFillColor(colors.white)
        canvas.setFont("Helvetica", 7.4)
        canvas.drawString(self.leftMargin, 11.5, "SpecterOps | Confidential")
        canvas.drawRightString(w - self.rightMargin, 11.5, f"Page {doc.page - 1}")
        canvas.restoreState()


def paragraph(value: Any, style: ParagraphStyle) -> Paragraph:
    return Paragraph(pdf_text(value), style)


def section_title(title: str, styles: Dict[str, ParagraphStyle], eyebrow: str = "") -> List[Any]:
    flows: List[Any] = []
    if eyebrow:
        flows.append(Paragraph(pdf_text(eyebrow.upper()), styles["eyebrow"]))
    flows.append(Paragraph(pdf_text(title), styles["h1"]))
    flows.append(HRFlowable(width="100%", thickness=0.8, color=colors.HexColor("#BFD8C9"), spaceAfter=10))
    return flows


def callout_panel(title: str, body: Any, styles: Dict[str, ParagraphStyle], accent=None) -> Table:
    accent = accent or PDF_COLORS["green_dark"]
    content = [
        Paragraph(pdf_text(title), styles["h3"]),
        Paragraph(pdf_text(body), styles["body"]),
    ]
    table = Table([["", content]], colWidths=[0.08 * inch, 6.82 * inch], hAlign="LEFT")
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), colors.HexColor("#FBFCFE")),
        ("BACKGROUND", (0, 0), (0, 0), accent),
        ("BOX", (0, 0), (-1, -1), 0.6, colors.HexColor("#D7DCE7")),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (1, 0), (1, 0), 13),
        ("RIGHTPADDING", (1, 0), (1, 0), 13),
        ("TOPPADDING", (1, 0), (1, 0), 10),
        ("BOTTOMPADDING", (1, 0), (1, 0), 9),
        ("LEFTPADDING", (0, 0), (0, 0), 0),
        ("RIGHTPADDING", (0, 0), (0, 0), 0),
    ]))
    return table


def metric_card(value: Any, label: str, styles: Dict[str, ParagraphStyle], background=None) -> Table:
    background = background or colors.white
    value_text = str(value)
    value_style = ParagraphStyle(
        f"MetricValue{len(value_text)}",
        parent=styles["metric_value"],
        fontSize=16 if len(value_text) > 8 else (18 if len(value_text) > 6 else 22),
        leading=18 if len(value_text) > 8 else (20 if len(value_text) > 6 else 24),
    )
    content = [
        Paragraph(pdf_text(value), value_style),
        Paragraph(pdf_text(label.upper()), styles["metric_label"]),
    ]
    table = Table([[content]], colWidths=[1.62 * inch], rowHeights=[0.78 * inch])
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), background),
        ("BOX", (0, 0), (-1, -1), 0.65, colors.HexColor("#CFD5E3")),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("LEFTPADDING", (0, 0), (-1, -1), 12),
        ("RIGHTPADDING", (0, 0), (-1, -1), 12),
        ("TOPPADDING", (0, 0), (-1, -1), 8),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 8),
    ]))
    return table


def simple_table(
    df: pd.DataFrame,
    styles: Dict[str, ParagraphStyle],
    max_rows: int = 30,
    max_columns: int = 6,
    column_widths: Optional[List[float]] = None,
) -> Table:
    if df is None or df.empty:
        table = Table([[Paragraph("No rows returned.", styles["small"])]], colWidths=[7.0 * inch])
        table.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, -1), PDF_COLORS["panel"]), ("BOX", (0, 0), (-1, -1), 0.5, PDF_COLORS["line"]), ("PADDING", (0, 0), (-1, -1), 8)]))
        return table

    view = df.copy().head(max_rows)
    if len(view.columns) > max_columns:
        view = view.iloc[:, :max_columns]

    headers = [Paragraph(pdf_text(col), styles["table_header"]) for col in view.columns]
    rows = [headers]
    for _, row in view.iterrows():
        rows.append([Paragraph(pdf_text(row.get(col, "")), styles["table"]) for col in view.columns])

    if column_widths is None:
        width = 7.0 * inch / max(1, len(view.columns))
        column_widths = [width] * len(view.columns)

    table = Table(rows, colWidths=column_widths, repeatRows=1, hAlign="LEFT")
    commands = [
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#F0F2F7")),
        ("TEXTCOLOR", (0, 0), (-1, -1), PDF_COLORS["ink"]),
        ("BOX", (0, 0), (-1, -1), 0.55, colors.HexColor("#CFD5E3")),
        ("INNERGRID", (0, 0), (-1, -1), 0.25, PDF_COLORS["line"]),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 5),
        ("RIGHTPADDING", (0, 0), (-1, -1), 5),
        ("TOPPADDING", (0, 0), (-1, -1), 4),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
    ]
    for row_index in range(2, len(rows), 2):
        commands.append(("BACKGROUND", (0, row_index), (-1, row_index), colors.HexColor("#FBFCFE")))
    table.setStyle(TableStyle(commands))
    return table


def add_bullets(story: List[Any], items: List[str], styles: Dict[str, ParagraphStyle]) -> None:
    for item in items or []:
        if str(item or "").strip():
            story.append(Paragraph(f'<font color="#229B59">&#9679;</font>&nbsp;&nbsp;{pdf_text(item)}', styles["body"]))


def action_columns(section: "ReportSection", styles: Dict[str, ParagraphStyle]) -> Table:
    def action_block(label: str, items: List[str]):
        body = [Paragraph(label.upper(), styles["evidence_label"])]
        valid = [item for item in (items or []) if str(item or "").strip()]
        if not valid:
            body.append(Paragraph("No actions specified.", styles["small"]))
        for item in valid:
            body.append(Paragraph(f"&#8226; {pdf_text(item)}", styles["small"]))
        return body
    data = [[
        action_block("Immediate", section.immediate_actions),
        action_block("Near-Term", section.near_term_actions),
        action_block("Strategic", section.strategic_actions),
    ]]
    table = Table(data, colWidths=[2.29 * inch] * 3, hAlign="LEFT")
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), colors.HexColor("#FBFCFE")),
        ("BOX", (0, 0), (-1, -1), 0.6, colors.HexColor("#CFD5E3")),
        ("INNERGRID", (0, 0), (-1, -1), 0.35, PDF_COLORS["line"]),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 10),
        ("RIGHTPADDING", (0, 0), (-1, -1), 10),
        ("TOPPADDING", (0, 0), (-1, -1), 9),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 8),
    ]))
    return table


def evidence_row_story(row: pd.Series, styles: Dict[str, ParagraphStyle], row_number: int) -> List[Any]:
    visible = []
    for column, value in row.items():
        text = str(value or "").strip()
        if not text or text.lower() in {"nan", "none", "null", "nat"}:
            continue
        visible.append((str(column), value))

    left = []
    right = []
    for index, (column, value) in enumerate(visible):
        block = [
            Paragraph(pdf_text(column).upper(), styles["evidence_label"]),
            Paragraph(pdf_text(value), styles["small"]),
        ]
        (left if index % 2 == 0 else right).extend(block)

    heading = Table([[
        Paragraph(f"{row_number:02d}", styles["inverse"]),
        Paragraph("REPRESENTATIVE SUPPORTING EVIDENCE", styles["label"]),
    ]], colWidths=[0.38 * inch, 6.52 * inch])
    heading.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (0, 0), PDF_COLORS["accent"]),
        ("BACKGROUND", (1, 0), (1, 0), colors.HexColor("#F4F7FC")),
        ("BOX", (0, 0), (-1, -1), 0.5, colors.HexColor("#D5DDEB")),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("ALIGN", (0, 0), (0, 0), "CENTER"),
        ("LEFTPADDING", (1, 0), (1, 0), 9),
    ]))
    body = Table([[left, right]], colWidths=[3.45 * inch, 3.45 * inch], hAlign="LEFT")
    body.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), colors.white),
        ("BOX", (0, 0), (-1, -1), 0.5, colors.HexColor("#D5DDEB")),
        ("INNERGRID", (0, 0), (-1, -1), 0.3, PDF_COLORS["line"]),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 10),
        ("RIGHTPADDING", (0, 0), (-1, -1), 10),
        ("TOPPADDING", (0, 0), (-1, -1), 8),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 7),
    ]))
    return [KeepTogether([heading, body, Spacer(1, 8)])]


## 11D. Styled PDF Document Assembly


In [ ]:
def remediated_during_assessment_items() -> List[Dict[str, str]]:
    """Return explicitly validated remediated items for the dashboard.

    The widget is intentionally conditional. Items may come from an upstream
    API-derived DataFrame named REMEDIATED_DURING_ASSESSMENT_DF or from the
    optional report.remediated_during_assessment configuration list. Generic
    status rows are accepted only when their status explicitly says resolved,
    remediated, or closed; zero current findings alone is not treated as proof.
    """
    rows: List[Dict[str, str]] = []

    def add_item(item: Dict[str, Any]) -> None:
        title = str(
            item.get("Finding") or item.get("Finding Title") or item.get("Title")
            or item.get("Exposure") or item.get("Name") or ""
        ).strip()
        platform = str(item.get("Platform") or item.get("Source") or "").strip()
        detail = str(
            item.get("Outcome") or item.get("Detail") or item.get("Summary")
            or item.get("Validation") or item.get("Description") or ""
        ).strip()
        status = str(item.get("Status") or item.get("State") or "Resolved").strip()
        status_low = status.casefold()
        if status_low and not any(token in status_low for token in ("resolved", "remediated", "closed", "fixed", "validated")):
            return
        if not title:
            return
        dedupe = (title.casefold(), platform.casefold())
        if any((r["title"].casefold(), r["platform"].casefold()) == dedupe for r in rows):
            return
        rows.append({"title": title, "platform": platform, "detail": detail, "status": status or "Resolved"})

    configured = report_cfg.get("remediated_during_assessment", []) or []
    if isinstance(configured, dict):
        configured = [configured]
    for item in configured:
        if isinstance(item, str):
            add_item({"Title": item, "Status": "Resolved"})
        elif isinstance(item, dict):
            add_item(item)

    for name in ("REMEDIATED_DURING_ASSESSMENT_DF", "RESOLVED_FINDINGS_DF"):
        frame = globals().get(name)
        if isinstance(frame, pd.DataFrame) and not frame.empty:
            for item in frame.to_dict("records"):
                add_item(item)

    status_frame = globals().get("NATIVE_FINDING_STATUS_DF")
    if isinstance(status_frame, pd.DataFrame) and not status_frame.empty and "Status" in status_frame.columns:
        explicit = status_frame[status_frame["Status"].astype(str).str.contains(
            r"resolved|remediated|closed|fixed|validated", case=False, regex=True, na=False
        )]
        for item in explicit.to_dict("records"):
            add_item(item)

    return rows


def build_plain_pdf_report(pdf_path: Path) -> Path:
    """Build v1.3 Executive Summary redesign with compact single-page density."""
    styles = pdf_styles()

    # Compact variants used only by the executive PDF layout.
    compact_body = ParagraphStyle(
        "CompactBody", parent=styles["body"], fontSize=8.45, leading=11.0,
        spaceAfter=3.5, textColor=PDF_COLORS["ink"],
    )
    compact_small = ParagraphStyle(
        "CompactSmall", parent=styles["small"], fontSize=7.45, leading=9.5,
        spaceAfter=2.5, textColor=PDF_COLORS["muted"],
    )
    compact_h2 = ParagraphStyle(
        "CompactH2", parent=styles["h2"], fontSize=10.8, leading=13.0,
        spaceBefore=7, spaceAfter=5,
    )
    compact_label = ParagraphStyle(
        "CompactLabel", parent=styles["evidence_label"], fontSize=6.8,
        leading=8.0, spaceAfter=1,
    )

    customer_name = report_cfg.get("customer_name", "Customer")
    assessment_name = report_cfg.get("assessment_name", "BloodHound Enterprise Identity Risk Report")
    prepared_by = report_cfg.get("prepared_by", "SpecterOps")
    prepared_for = report_cfg.get("prepared_for", "Executive Leadership and Identity Security Stakeholders")
    classification = report_cfg.get("classification", "Confidential")
    report_version = report_cfg.get("version", "")
    generated_date = datetime.datetime.now().strftime("%B %d, %Y")

    doc = StyledIdentityRiskDocTemplate(
        str(pdf_path),
        pagesize=letter,
        rightMargin=0.55 * inch,
        leftMargin=0.55 * inch,
        topMargin=0.52 * inch,
        bottomMargin=0.48 * inch,
        title=str(assessment_name),
        author=str(prepared_by),
    )
    doc.cover_data = {
        "assessment_name": assessment_name,
        "customer_name": customer_name,
        "prepared_by": prepared_by,
        "prepared_for": prepared_for,
        "classification": classification,
        "report_version": report_version,
        "generated_date": generated_date,
        "specterops_logo_path": (cfg.get("branding", {}) or {}).get("specterops_logo_path", "assets/specterops_logo_white_tagline.png"),
        "cover_hero_image_path": (cfg.get("branding", {}) or {}).get("cover_hero_image_path") or str(BHE_COVER_GRAPH_V103_PATH),
    }

    story: List[Any] = [NextPageTemplate("whitepaper"), PageBreak()]

    # ------------------------------------------------------------------
    # Page 2 - Executive Summary (single-page, print-first layout)
    # ------------------------------------------------------------------
    story.extend(section_title("Executive Summary", styles, "Identity risk at a glance"))

    attack_paths = (
        int(globals().get("TOTAL_ENTERPRISE_TIER_ZERO_ATTACK_PATHS", globals().get("TOTAL_ENTERPRISE_ATTACK_PATHS", 0)))
        if globals().get("ATTACK_PATH_METRICS_AVAILABLE", False) else 0
    )
    platform_df = globals().get("ATTACK_PATH_PLATFORM_DF", pd.DataFrame()).copy()
    environment_df = globals().get("ATTACK_PATH_ENVIRONMENT_DF", pd.DataFrame()).copy()

    def _metric_count(row):
        for col in row.index:
            key = str(col).lower()
            if any(token in key for token in ["attack path", "count", "total", "value", "findings"]):
                try: return int(float(row[col]))
                except Exception: pass
        return 0

    def _row_value(row, tokens, fallback=""):
        for col in row.index:
            if any(token in str(col).lower() for token in tokens):
                value = str(row[col]).strip()
                if value and value.lower() not in {"nan", "none", "null"}: return value
        return fallback

    def _canonical_platform(value):
        low = str(value or "").strip().lower()
        if "entra" in low or "azure" in low: return "Microsoft Entra ID"
        if "active directory" in low or low in {"ad", "windows ad"}: return "Active Directory"
        if "github" in low: return "GitHub"
        if "okta" in low: return "Okta"
        if "jamf" in low: return "Jamf"
        return str(value or "Platform")

    platform_counts = {"Microsoft Entra ID": 0, "Active Directory": 0, "GitHub": 0, "Okta": 0, "Jamf": 0}
    if not platform_df.empty:
        for _, row in platform_df.iterrows():
            name = _canonical_platform(_row_value(row, ["platform", "name", "label"], "Platform"))
            if name in platform_counts: platform_counts[name] += _metric_count(row)
    if not risk_theme_df.empty:
        for _, row in risk_theme_df.iterrows():
            theme_name = str(row.get("Risk Theme", row.get("Report Section", ""))).lower()
            count = 0
            for key in ["Findings", "Finding Count", "Evidence Rows", "Count"]:
                if key in row:
                    try: count = int(float(row[key])); break
                    except Exception: pass
            for platform in ["GitHub", "Okta", "Jamf"]:
                if platform.lower() in theme_name and count: platform_counts[platform] = count

    def hero_icon(kind, accent, inverted=False):
        d=Drawing(42,42); ring=colors.white if inverted else accent
        d.add(Circle(21,21,18,fillColor=None,strokeColor=ring,strokeWidth=1.25))
        if kind=="risk":
            d.add(Polygon(points=[21,32,31,27,29,15,21,9,13,15,11,27],fillColor=accent,strokeColor=accent))
            d.add(String(19.3,16.4,"!",fontName="Helvetica-Bold",fontSize=12,fillColor=colors.white))
        else:
            d.add(Circle(21,21,8,fillColor=None,strokeColor=colors.white,strokeWidth=1.5)); d.add(Circle(21,21,2.5,fillColor=colors.white,strokeColor=colors.white))
            for x1,y1,x2,y2 in [(21,8,21,15),(21,27,21,34),(8,21,15,21),(27,21,34,21)]: d.add(Line(x1,y1,x2,y2,strokeColor=colors.white,strokeWidth=1.5))
        return d


    summary_title = ParagraphStyle("ExecSummaryCardTitleV13", parent=styles["h2"], fontSize=11.2, leading=12.8, textColor=PDF_COLORS["green_dark"], spaceAfter=4)
    summary_body = ParagraphStyle("ExecSummaryCardBodyV13", parent=styles["body"], fontSize=8.8, leading=10.7, textColor=PDF_COLORS["ink"], spaceAfter=0)
    connected_count = sum(1 for p in ["GitHub", "Okta", "Jamf"] if platform_counts[p])
    summary_text = (
        "During this assessment, BloodHound analyzed identity attack paths across <b>Active Directory</b> and "
        "<b>Microsoft Entra ID</b>, extending visibility into connected GitHub, Okta, and Jamf environments where applicable. "
        f"The assessment identified <b>{pdf_text(overall_risk)}</b> identity risk and <b>{attack_paths:,}</b> validated paths to Tier Zero or other critical control planes."
    )
    summary_card = Table([[
        hero_icon("risk", PDF_COLORS["green_dark"]),
        [Paragraph("Executive Assessment Summary", summary_title), Paragraph(summary_text, summary_body)]
    ]], colWidths=[0.62*inch, 6.38*inch], hAlign="LEFT")
    summary_card.setStyle(TableStyle([
        ("BACKGROUND",(0,0),(-1,-1),colors.HexColor("#F7FBF9")),("BOX",(0,0),(-1,-1),0.7,colors.HexColor("#A9D6B8")),
        ("VALIGN",(0,0),(-1,-1),"MIDDLE"),("LEFTPADDING",(0,0),(-1,-1),7),("RIGHTPADDING",(0,0),(-1,-1),7),
        ("TOPPADDING",(0,0),(-1,-1),1.5),("BOTTOMPADDING",(0,0),(-1,-1),1.5)
    ]))
    story.append(summary_card); story.append(Spacer(1,16))

    # Hero cards use the exact same 7.0-inch content width as the full-width sections.
    # The small center gutter is accounted for inside that width so neither card is inset.
    HERO_GUTTER = 0.08 * inch
    HERO_CARD_W = (7.0 * inch - HERO_GUTTER) / 2.0

    def dark_hero(title,value,desc,kind):
        tstyle=ParagraphStyle("DHt"+kind,parent=styles["h3"],fontSize=7.8,leading=8.7,textColor=colors.white,spaceAfter=1.2)
        vstyle=ParagraphStyle("DHv"+kind,parent=styles["metric_value"],fontSize=17.2,leading=17.8,textColor=PDF_COLORS["orange"] if kind=="risk" else colors.white,spaceAfter=1.0)
        dstyle=ParagraphStyle("DHd"+kind,parent=styles["small"],fontSize=6.2,leading=7.0,textColor=colors.white)
        tab=Table(
            [[hero_icon(kind,PDF_COLORS["orange"] if kind=="risk" else colors.white,True),
              [Paragraph(title.upper(),tstyle),Paragraph(value,vstyle),Paragraph(desc,dstyle)]]],
            colWidths=[0.78*inch, HERO_CARD_W-(0.78*inch)],
            rowHeights=[0.70*inch],
            hAlign="LEFT",
        )
        tab.setStyle(TableStyle([
            ("BACKGROUND",(0,0),(-1,-1),PDF_COLORS["accent_dark"]),
            ("LINEABOVE",(0,0),(-1,0),3.0,PDF_COLORS["orange"]),
            ("BOX",(0,0),(-1,-1),0.7,PDF_COLORS["accent_dark"]),
            ("VALIGN",(0,0),(-1,-1),"MIDDLE"),
            ("LEFTPADDING",(0,0),(0,0),9),
            ("LEFTPADDING",(1,0),(1,0),7),
            ("RIGHTPADDING",(0,0),(-1,-1),5),
            ("TOPPADDING",(0,0),(-1,-1),3),
            ("BOTTOMPADDING",(0,0),(-1,-1),3),
        ]))
        return tab

    hero=Table([[
        dark_hero("Overall Identity Risk",overall_risk.upper(),"Material identity exposure requires prioritized remediation across the highest-impact themes.","risk"),
        "",
        dark_hero("Enterprise Tier Zero Attack Paths",f"{attack_paths:,}","Validated paths to Tier Zero or control-plane impact.","paths")
    ]], colWidths=[HERO_CARD_W,HERO_GUTTER,HERO_CARD_W], hAlign="LEFT")
    hero.setStyle(TableStyle([
        ("VALIGN",(0,0),(-1,-1),"TOP"),
        ("LEFTPADDING",(0,0),(-1,-1),0),
        ("RIGHTPADDING",(0,0),(-1,-1),0),
        ("TOPPADDING",(0,0),(-1,-1),0),
        ("BOTTOMPADDING",(0,0),(-1,-1),0),
    ]))
    story.append(hero); story.append(Spacer(1,8))

    section_head=ParagraphStyle("ExecSectionHeadV13",parent=styles["h3"],fontSize=9.2,leading=10.2,textColor=PDF_COLORS["accent_dark"],spaceAfter=0)
    row_name=ParagraphStyle("ExecRowNameV13",parent=styles["h3"],fontSize=8.2,leading=9.2,textColor=PDF_COLORS["accent_dark"])
    platform_name=ParagraphStyle("ExecPlatformNameV13",parent=styles["h3"],fontName="Helvetica-Bold",fontSize=11.2,leading=12.2,textColor=PDF_COLORS["accent_dark"],spaceAfter=0)
    row_meta=ParagraphStyle("ExecRowMetaV13",parent=styles["small"],fontSize=6.5,leading=7.3,textColor=PDF_COLORS["muted"])
    row_num=ParagraphStyle("ExecRowNumV13",parent=styles["metric_value"],fontSize=11.6,leading=12.4,textColor=PDF_COLORS["accent_dark"],alignment=TA_RIGHT)

    def ad_mark(name):
        if name == "Microsoft Entra ID":
            return Image(str(ENTRA_ID_LOGO_V116_PATH), width=19, height=19, kind="proportional")
        # Microsoft Active Directory / Windows identity mark. Use a four-pane
        # Microsoft window instead of the generic directory-tree glyph.
        d = Drawing(22, 22)
        blue = colors.HexColor("#0078D4")
        d.add(Polygon(points=[2,3,10,2,10,10,2,10], fillColor=blue, strokeColor=blue))
        d.add(Polygon(points=[12,2,20,1,20,10,12,10], fillColor=blue, strokeColor=blue))
        d.add(Polygon(points=[2,12,10,12,10,20,2,19], fillColor=blue, strokeColor=blue))
        d.add(Polygon(points=[12,12,20,12,20,21,12,20], fillColor=blue, strokeColor=blue))
        return d

    def horizontal_section(title, rows, accent=PDF_COLORS["blue"], footer=None):
        data=[[Paragraph(title.upper(),section_head),"",""]]+rows
        colw=[0.48*inch,5.18*inch,1.34*inch]
        t=Table(data,colWidths=colw,hAlign="LEFT")
        st=[("SPAN",(0,0),(-1,0)),("BACKGROUND",(0,0),(-1,0),colors.white),("LINEBEFORE",(0,0),(0,-1),2.6,accent),("LINEBELOW",(0,0),(-1,0),0.55,colors.HexColor("#C8D2E1")),("BOX",(0,0),(-1,-1),0.55,colors.HexColor("#C8D2E1")),("VALIGN",(0,0),(-1,-1),"MIDDLE"),("LEFTPADDING",(0,0),(0,0),9),("LEFTPADDING",(1,0),(1,0),8),("RIGHTPADDING",(0,0),(-1,-1),8),("TOPPADDING",(0,0),(-1,-1),3),("BOTTOMPADDING",(0,0),(-1,-1),3)]
        for r in range(2,len(data)): st.append(("LINEABOVE",(0,r),(-1,r),0.3,colors.HexColor("#E3E8F0")))
        t.setStyle(TableStyle(st)); return t

    plat_rows=[]
    for n in ["Active Directory","Microsoft Entra ID"]:
        plat_rows.append([ad_mark(n),[Paragraph(n,platform_name),Paragraph("Tier Zero attack paths",row_meta)],Paragraph(f"{platform_counts[n]:,}",row_num)])
    story.append(horizontal_section("Enterprise Tier Zero Attack Paths by Platform",plat_rows)); story.append(Spacer(1,5))

    env_rows=[]
    if not environment_df.empty:
        for _,r in environment_df.head(5).iterrows():
            p=_canonical_platform(_row_value(r,["platform"],""))
            e=_row_value(r,["environment","name","label"],"Environment")
            c=_metric_count(r)
            platform_label = "Microsoft Entra ID" if p=="Microsoft Entra ID" else "Active Directory"
            env_rows.append([
                ad_mark(platform_label),
                [Paragraph(pdf_text(e),row_name), Paragraph(platform_label,row_meta)],
                Paragraph(f"{c:,}",row_num)
            ])
    if env_rows:
        story.append(horizontal_section("Top Environments by Attack Paths",env_rows))
        story.append(Spacer(1,5))

    logo_paths={"GitHub":GITHUB_LOGO_V13_PATH,"Okta":OKTA_LOGO_V13_PATH,"Jamf":JAMF_LOGO_V13_PATH}
    widths={"GitHub":20,"Okta":32,"Jamf":20}; heights={"GitHub":20,"Okta":12,"Jamf":20}
    conn=[]
    for n in ["GitHub","Okta","Jamf"]:
        if not platform_counts.get(n, 0):
            continue
        img=Image(str(logo_paths[n]),width=widths[n],height=heights[n],kind="proportional")
        metric="Devices assessed" if n=="Jamf" else "Identities assessed"
        conn.append([img,[Paragraph(n,row_name),Paragraph(metric,row_meta)],Paragraph(f"{platform_counts[n]:,}",row_num)])
    if conn:
        story.append(horizontal_section("Additional Connected Platform Coverage",conn,PDF_COLORS["blue"])); story.append(Spacer(1,5))
    else:
        no_og_style=ParagraphStyle("NoConnectedPlatformsV1522",parent=styles["small"],fontSize=7.2,leading=8.8,textColor=PDF_COLORS["muted"])
        no_og=Table([[Paragraph("No additional connected-platform data was collected for this assessment.",no_og_style)]],colWidths=[7.0*inch],hAlign="LEFT")
        no_og.setStyle(TableStyle([("BACKGROUND",(0,0),(-1,-1),colors.HexColor("#F7F8FB")),("BOX",(0,0),(-1,-1),0.6,colors.HexColor("#CDD3E1")),("LEFTPADDING",(0,0),(-1,-1),9),("RIGHTPADDING",(0,0),(-1,-1),9),("TOPPADDING",(0,0),(-1,-1),6),("BOTTOMPADDING",(0,0),(-1,-1),6)]))
        story.append(no_og); story.append(Spacer(1,5))

    takeaway_title=ParagraphStyle("TakeawayTitleV13",parent=styles["h3"],fontSize=8.0,leading=9.2,textColor=PDF_COLORS["green_dark"],spaceAfter=2)
    takeaway_body=ParagraphStyle("TakeawayBodyV13",parent=styles["body"],fontSize=7.7,leading=9.1,textColor=PDF_COLORS["ink"])
    takeaway=Table([[hero_icon("risk",PDF_COLORS["green_dark"]),[Paragraph("EXECUTIVE TAKEAWAY",takeaway_title),Paragraph("Identity risk is the visible outcome of weaknesses in identity program maturity. Prioritize the highest-impact paths to privileged control, then address the recurring governance and operational conditions that allow those paths to persist.",takeaway_body)]]],colWidths=[0.64*inch,6.36*inch],hAlign="LEFT")
    takeaway.setStyle(TableStyle([("BACKGROUND",(0,0),(-1,-1),colors.HexColor("#F5FBF7")),("BOX",(0,0),(-1,-1),0.65,colors.HexColor("#A9D6B8")),("VALIGN",(0,0),(-1,-1),"MIDDLE"),("LEFTPADDING",(0,0),(-1,-1),6),("RIGHTPADDING",(0,0),(-1,-1),6),("TOPPADDING",(0,0),(-1,-1),2.4),("BOTTOMPADDING",(0,0),(-1,-1),2.4)])); story.append(takeaway)
    story.append(PageBreak())

    # ------------------------------------------------------------------
    # Page 3 - Risk themes and priorities
    # ------------------------------------------------------------------
    story.extend(section_title("Identity Risk Themes", styles, "Executive risk summary"))
    story.append(Paragraph(
        "The themes below summarize the most important identity conditions identified during the assessment. Detailed scope, platform coverage, and technical mappings are retained in the Evidence Workbook.",
        styles["lead"],
    ))

    matrix_title = ParagraphStyle("ThemeMatrixTitleV121", parent=styles["h2"], fontSize=13.2, leading=14.3, textColor=PDF_COLORS["accent_dark"], spaceAfter=0)
    matrix_subtitle = ParagraphStyle("ThemeMatrixSubtitleV121", parent=styles["small"], fontSize=7.8, leading=8.6, textColor=PDF_COLORS["muted"], spaceAfter=0)
    matrix_header = ParagraphStyle("ThemeMatrixHeaderV121", parent=styles["small"], fontName="Helvetica-Bold", fontSize=7.0, leading=7.6, textColor=colors.white)
    matrix_theme = ParagraphStyle("ThemeMatrixThemeV121", parent=styles["small"], fontName="Helvetica-Bold", fontSize=7.85, leading=8.55, textColor=PDF_COLORS["accent_dark"])
    matrix_cell = ParagraphStyle("ThemeMatrixCellV121", parent=styles["small"], fontSize=7.25, leading=7.95, textColor=PDF_COLORS["ink"])
    matrix_rank = ParagraphStyle("ThemeMatrixRankV121", parent=matrix_theme, fontSize=9.3, leading=9.9, alignment=TA_CENTER)
    matrix_count = ParagraphStyle("ThemeMatrixCountV121", parent=matrix_theme, fontSize=8.9, leading=9.5, alignment=TA_RIGHT)
    badge_text = ParagraphStyle("ThemeRiskBadgeV121", parent=styles["small"], fontName="Helvetica-Bold", fontSize=7.2, leading=7.8, textColor=colors.white, alignment=TA_CENTER)

    def _context_icon(kind: str, accent) -> Drawing:
        d = Drawing(21, 21)
        d.add(Circle(10.5, 10.5, 8.1, fillColor=colors.white, strokeColor=accent, strokeWidth=1.3))
        if kind == "impact":
            d.add(Circle(11.5,11.5,3.4,fillColor=None,strokeColor=accent,strokeWidth=1.3))
            for x1,y1,x2,y2 in [(11.5,2,11.5,6),(11.5,17,11.5,21),(2,11.5,6,11.5),(17,11.5,21,11.5)]: d.add(Line(x1,y1,x2,y2,strokeColor=accent,strokeWidth=1.3))
        elif kind == "why":
            d.add(Polygon(points=[10.5,17.6,17.1,5.5,3.9,5.5],fillColor=None,strokeColor=accent,strokeWidth=1.25))
            d.add(String(9.45,7.0,"!",fontName="Helvetica-Bold",fontSize=8.1,fillColor=accent))
        elif kind == "technical":
            d.add(Circle(11.5,11.5,4.1,fillColor=None,strokeColor=accent,strokeWidth=1.4))
            for x1,y1,x2,y2 in [(11.5,1.5,11.5,5),(11.5,18,11.5,22),(1.5,11.5,5,11.5),(18,11.5,22,11.5),(4,4,7,7),(16,16,19,19),(4,19,7,16),(16,7,19,4)]: d.add(Line(x1,y1,x2,y2,strokeColor=accent,strokeWidth=1.2))
        else:
            d.add(Circle(11.5,13,5,fillColor=None,strokeColor=accent,strokeWidth=1.4)); d.add(Line(9,5.5,14,5.5,strokeColor=accent,strokeWidth=1.4)); d.add(Line(10,3,13,3,strokeColor=accent,strokeWidth=1.4))
        return d


    active_heading = Table([[
        _context_icon("technical", PDF_COLORS["accent"]),
        [Paragraph("Active Risk Themes", matrix_title), Paragraph("Ranked by business impact", matrix_subtitle)],
    ]], colWidths=[0.52*inch, 6.48*inch])
    active_heading.setStyle(TableStyle([
        ("VALIGN",(0,0),(-1,-1),"MIDDLE"),("LEFTPADDING",(0,0),(-1,-1),0),("RIGHTPADDING",(0,0),(-1,-1),5),
        ("TOPPADDING",(0,0),(-1,-1),0),("BOTTOMPADDING",(0,0),(-1,-1),4),
    ]))
    story.append(active_heading)

    matrix_rows = [[
        Paragraph("PRIORITY", matrix_header), Paragraph("", matrix_header), Paragraph("THEME", matrix_header),
        Paragraph("MATURITY DOMAIN", matrix_header), Paragraph("RISK LEVEL", matrix_header), Paragraph("FINDINGS", matrix_header),
    ]]
    for rank, (_, row) in enumerate(risk_theme_df.iterrows(), start=1):
        theme = str(row.get("Risk Theme", "Risk Theme"))
        domain = str(row.get("Maturity Domain", "Identity Governance"))
        risk = str(row.get("Risk Level", "—"))
        findings = row.get("Findings", 0)
        try: findings_text = f"{int(findings):,}"
        except Exception: findings_text = str(findings)
        accent = severity_color(risk)
        risk_badge = Table([[Paragraph(pdf_text(risk).upper(), badge_text)]], colWidths=[0.72*inch], rowHeights=[0.20*inch])
        risk_badge.setStyle(TableStyle([
            ("BACKGROUND",(0,0),(-1,-1),accent),("BOX",(0,0),(-1,-1),0.4,accent),("VALIGN",(0,0),(-1,-1),"MIDDLE"),
            ("LEFTPADDING",(0,0),(-1,-1),3),("RIGHTPADDING",(0,0),(-1,-1),3),("TOPPADDING",(0,0),(-1,-1),6),("BOTTOMPADDING",(0,0),(-1,-1),6),
        ]))
        matrix_rows.append([
            Paragraph(str(rank), matrix_rank), _theme_symbol(theme, accent, 25), Paragraph(pdf_text(theme), matrix_theme),
            Paragraph(pdf_text(domain), matrix_cell), risk_badge, Paragraph(findings_text, matrix_count),
        ])

    matrix = Table(matrix_rows, colWidths=[0.62*inch,0.42*inch,2.01*inch,2.25*inch,1.02*inch,0.68*inch], repeatRows=1)
    matrix_style = [
        ("BACKGROUND",(0,0),(-1,0),PDF_COLORS["accent_dark"]),("BOX",(0,0),(-1,-1),0.5,PDF_COLORS["line"]),
        ("VALIGN",(0,0),(-1,-1),"MIDDLE"),("ALIGN",(0,0),(1,-1),"CENTER"),("ALIGN",(5,1),(5,-1),"RIGHT"),
        ("LEFTPADDING",(0,0),(-1,-1),5),("RIGHTPADDING",(0,0),(-1,-1),5),("TOPPADDING",(0,0),(-1,-1),1.0),("BOTTOMPADDING",(0,0),(-1,-1),1.0),
    ]
    for ridx, (_, row) in enumerate(risk_theme_df.iterrows(), start=1):
        accent=severity_color(str(row.get("Risk Level","")))
        matrix_style.extend([
            ("LINEBELOW",(0,ridx),(-1,ridx),0.3,colors.HexColor("#DDE4EE")),
            ("TEXTCOLOR",(0,ridx),(0,ridx),accent),
            ("BACKGROUND",(0,ridx),(-1,ridx),colors.white if ridx%2 else colors.HexColor("#FBFCFE")),
        ])
    matrix.setStyle(TableStyle(matrix_style))
    story.append(matrix)
    story.append(Spacer(1, 5))

    top_priorities_heading = Table([[Paragraph("Top Priorities", ParagraphStyle("TopPrioritiesV121", parent=compact_h2, fontSize=11.8, leading=12.6, spaceBefore=0, spaceAfter=0))]], colWidths=[7.0*inch], hAlign="LEFT")
    top_priorities_heading.setStyle(TableStyle([("LEFTPADDING",(0,0),(-1,-1),11),("RIGHTPADDING",(0,0),(-1,-1),0),("TOPPADDING",(0,0),(-1,-1),0),("BOTTOMPADDING",(0,0),(-1,-1),0)]))
    story.append(top_priorities_heading)
    story.append(Spacer(1, 2))
    priority_header = ParagraphStyle("PriorityHeaderV121", parent=matrix_header, fontSize=6.8, leading=7.4)
    priority_cell = ParagraphStyle("PriorityCellV121", parent=matrix_cell, fontSize=6.75, leading=7.35)
    priority_rank = ParagraphStyle("PriorityRankV121", parent=matrix_theme, fontSize=8.0, leading=8.5, alignment=TA_CENTER)
    priority_rows = [[Paragraph("PRIORITY",priority_header),Paragraph("MATURITY DOMAIN",priority_header),Paragraph("RISK THEME",priority_header),Paragraph("RISK",priority_header),Paragraph("WHY IT MATTERS",priority_header)]]
    for _, prow in top_priorities_df.head(5).iterrows():
        risk=str(prow.get("Risk Level","—")); accent=severity_color(risk)
        rb=Table([[Paragraph(pdf_text(risk).upper(),badge_text)]],colWidths=[0.60*inch],rowHeights=[0.18*inch])
        rb.setStyle(TableStyle([("BACKGROUND",(0,0),(-1,-1),accent),("VALIGN",(0,0),(-1,-1),"MIDDLE"),("LEFTPADDING",(0,0),(-1,-1),2),("RIGHTPADDING",(0,0),(-1,-1),2),("TOPPADDING",(0,0),(-1,-1),0),("BOTTOMPADDING",(0,0),(-1,-1),0)]))
        priority_rows.append([
            Paragraph(pdf_text(prow.get("Priority","—")),priority_rank),
            Paragraph(pdf_text(prow.get("Maturity Domain","—")),priority_cell),
            Paragraph(pdf_text(prow.get("Risk Theme","—")),priority_cell), rb,
            Paragraph(pdf_text(prow.get("Immediate Focus","Review and reduce the highest-impact exposure.")),priority_cell),
        ])
    pt=Table(priority_rows,colWidths=[0.62*inch,1.65*inch,1.55*inch,0.72*inch,2.46*inch],repeatRows=1)
    ps=[("BACKGROUND",(0,0),(-1,0),PDF_COLORS["accent_dark"]),("BOX",(0,0),(-1,-1),0.5,PDF_COLORS["line"]),("INNERGRID",(0,0),(-1,-1),0.25,colors.HexColor("#DDE4EE")),("VALIGN",(0,0),(-1,-1),"MIDDLE"),("ALIGN",(0,0),(0,-1),"CENTER"),("LEFTPADDING",(0,0),(-1,-1),5),("RIGHTPADDING",(0,0),(-1,-1),5),("TOPPADDING",(0,0),(-1,-1),1.4),("BOTTOMPADDING",(0,0),(-1,-1),1.4)]
    for ridx in range(1,len(priority_rows)): ps.append(("BACKGROUND",(0,ridx),(-1,ridx),colors.white if ridx%2 else colors.HexColor("#FBFCFE")))
    pt.setStyle(TableStyle(ps)); story.append(pt)
    story.append(Spacer(1,6))
    theme_takeaway=Table([[_context_icon("insight",PDF_COLORS["green_dark"]),[Paragraph("EXECUTIVE TAKEAWAY",takeaway_title),Paragraph("Identity risk is the visible outcome of weaknesses in identity program maturity. Prioritize the highest-impact paths to privileged control, then address the recurring governance and operational conditions that allow those paths to persist.",takeaway_body)]]],colWidths=[0.48*inch,6.52*inch])
    theme_takeaway.setStyle(TableStyle([("BACKGROUND",(0,0),(-1,-1),colors.HexColor("#F4FBF8")),("BOX",(0,0),(-1,-1),0.65,colors.HexColor("#74C7A4")),("VALIGN",(0,0),(-1,-1),"MIDDLE"),("LEFTPADDING",(0,0),(-1,-1),6),("RIGHTPADDING",(0,0),(-1,-1),6),("TOPPADDING",(0,0),(-1,-1),3),("BOTTOMPADDING",(0,0),(-1,-1),3)]))
    story.append(theme_takeaway)

    # ------------------------------------------------------------------
    # One print-native page per risk theme.
    # Executive summary spans the page, followed by a horizontal context row,
    # then a combined representative attack path and concise evidence exhibit.
    # ------------------------------------------------------------------
    show_empty = bool(report_cfg.get("include_no_result_sections", False))
    configured_max = int(report_cfg.get("max_evidence_rows", 10))
    pdf_evidence_rows = max(1, min(configured_max, 3))

    theme_summary_style = ParagraphStyle(
        "ThemeSummaryV121", parent=styles["body"], fontSize=9.0, leading=11.1,
        textColor=PDF_COLORS["ink"], spaceAfter=0,
    )
    theme_box_heading = ParagraphStyle(
        "ThemeBoxHeadingV121", parent=styles["small"], fontName="Helvetica-Bold",
        fontSize=7.0, leading=7.7, textColor=PDF_COLORS["green_dark"], spaceAfter=2.5,
    )
    theme_box_body = ParagraphStyle(
        "ThemeBoxBodyV121", parent=styles["small"], fontSize=8.3, leading=9.65,
        textColor=PDF_COLORS["ink"], spaceAfter=0,
    )
    exhibit_heading = ParagraphStyle(
        "ThemeExhibitHeadingV121", parent=styles["h2"], fontSize=10.7, leading=12.2,
        textColor=PDF_COLORS["accent_dark"], spaceAfter=3,
    )
    exhibit_intro = ParagraphStyle(
        "ThemeExhibitIntroV121", parent=styles["small"], fontSize=7.6, leading=8.7,
        textColor=PDF_COLORS["muted"], spaceAfter=4,
    )
    path_entity = ParagraphStyle(
        "ThemePathEntityV121", parent=styles["small"], fontName="Helvetica-Bold",
        fontSize=6.8, leading=7.8, alignment=TA_CENTER, textColor=PDF_COLORS["ink"],
        splitLongWords=False,
    )
    path_meta = ParagraphStyle(
        "ThemePathMetaV108", parent=styles["small"], fontSize=5.35, leading=6.15,
        alignment=TA_CENTER, textColor=PDF_COLORS["muted"],
    )
    path_relationship = ParagraphStyle(
        "ThemePathRelationshipV108", parent=styles["small"], fontName="Helvetica-Bold",
        fontSize=6.4, leading=7.5, alignment=TA_CENTER, textColor=PDF_COLORS["blue"],
    )
    evidence_header = ParagraphStyle(
        "ThemeEvidenceHeaderV121", parent=styles["small"], fontName="Helvetica-Bold",
        fontSize=6.6, leading=7.3, textColor=colors.white, alignment=TA_LEFT,
    )
    evidence_cell = ParagraphStyle(
        "ThemeEvidenceCellV121", parent=styles["small"], fontSize=7.0, leading=7.75,
        textColor=PDF_COLORS["ink"], splitLongWords=False,
    )
    evidence_cell_center = ParagraphStyle(
        "ThemeEvidenceCellCenterV108", parent=evidence_cell, alignment=TA_CENTER,
    )
    evidence_number = ParagraphStyle(
        "ThemeEvidenceNumberV108", parent=styles["inverse"], fontSize=7.5, leading=8.2,
        alignment=TA_CENTER,
    )
    action_heading = ParagraphStyle(
        "ThemeActionHeadingV121", parent=styles["small"], fontName="Helvetica-Bold",
        fontSize=7.4, leading=8.25, textColor=PDF_COLORS["accent_dark"], spaceAfter=3,
    )
    action_body = ParagraphStyle(
        "ThemeActionBodyV121", parent=styles["small"], fontSize=6.75, leading=7.7,
        textColor=PDF_COLORS["ink"],
    )

    def _clean_value(value) -> str:
        """Normalize report cell values without leaking pandas null markers."""
        if value is None:
            return ""
        try:
            if pd.isna(value):
                return ""
        except (TypeError, ValueError):
            pass
        text = str(value).strip()
        if text.casefold() in {"", "nan", "none", "null", "nat", "<na>"}:
            return ""
        return text

    def _short_name(text: str, limit: int = 34) -> str:
        text = _clean_value(text)
        if len(text) <= limit:
            return text
        return text[: max(1, limit - 1)].rstrip() + "…"

    def _row_value(row, candidates, default="") -> str:
        normalized = {str(column).strip().lower(): column for column in row.index}
        for candidate in candidates:
            key = candidate.strip().lower()
            if key in normalized:
                value = _clean_value(row[normalized[key]])
                if value:
                    return value
        return default

    def _first_sentence(text: str, limit: int = 95) -> str:
        text = re.sub(r"\s+", " ", _clean_value(text))
        if not text:
            return "Representative evidence of the broader identity risk condition."
        sentence = re.split(r"(?<=[.!?])\s+", text, maxsplit=1)[0]
        if len(sentence) > limit:
            sentence = sentence[: max(1, limit - 1)].rstrip() + "…"
        return sentence

    relationship_impact = {
        "dcsync": "Can replicate Active Directory credential data.",
        "genericall": "Provides full control over the target object.",
        "genericwrite": "Can modify security-sensitive target attributes.",
        "writedacl": "Can grant additional permissions on the target.",
        "writeowner": "Can take ownership and change target permissions.",
        "adminto": "Provides administrative control of the target system.",
        "administrative access": "Provides administrative control of the target system.",
        "login path": "May expose privileged credentials through an interactive logon.",
        "allowedtodelegate": "Can impersonate users to the delegated target service.",
        "readgmsapassword": "Can retrieve the managed service account password.",
        "dumpmsapassword": "Can retrieve managed service account credential material.",
        "kerberoastable": "Allows offline password cracking against the service account.",
        "add application secret": "Can add credentials and assume control of the application.",
        "grant directory role": "Can grant privileged Entra ID directory roles.",
    }

    def _why_it_matters(row, relationship: str) -> str:
        relationship_key = relationship.strip().lower()
        if relationship_key in relationship_impact:
            return relationship_impact[relationship_key]
        detail = _row_value(row, [
            "Why it Matters", "Relevant Detail", "Risk", "Description",
            "Finding / Condition", "Technical Detail",
        ])
        return _first_sentence(detail)

    def compact_action_block(label: str, items: List[str], accent) -> List:
        blocks = [Paragraph(label.upper(), ParagraphStyle(
            f"ThemeAction{label}V108", parent=action_heading, textColor=accent
        ))]
        valid = [item for item in (items or []) if str(item or "").strip()]
        for item in valid[:2]:
            blocks.append(Paragraph(f"&#8226; {pdf_text(item)}", action_body))
        if not valid:
            blocks.append(Paragraph("No action specified.", action_body))
        return blocks

    def _context_icon(kind: str, accent) -> Drawing:
        d = Drawing(21, 21)
        d.add(Circle(10.5, 10.5, 8.1, fillColor=colors.white, strokeColor=accent, strokeWidth=1.3))
        if kind == "impact":
            d.add(Circle(11.5,11.5,3.4,fillColor=None,strokeColor=accent,strokeWidth=1.3))
            for x1,y1,x2,y2 in [(11.5,2,11.5,6),(11.5,17,11.5,21),(2,11.5,6,11.5),(17,11.5,21,11.5)]: d.add(Line(x1,y1,x2,y2,strokeColor=accent,strokeWidth=1.3))
        elif kind == "why":
            d.add(Polygon(points=[10.5,17.6,17.1,5.5,3.9,5.5],fillColor=None,strokeColor=accent,strokeWidth=1.25))
            d.add(String(9.45,7.0,"!",fontName="Helvetica-Bold",fontSize=8.1,fillColor=accent))
        elif kind == "technical":
            d.add(Circle(11.5,11.5,4.1,fillColor=None,strokeColor=accent,strokeWidth=1.4))
            for x1,y1,x2,y2 in [(11.5,1.5,11.5,5),(11.5,18,11.5,22),(1.5,11.5,5,11.5),(18,11.5,22,11.5),(4,4,7,7),(16,16,19,19),(4,19,7,16),(16,7,19,4)]: d.add(Line(x1,y1,x2,y2,strokeColor=accent,strokeWidth=1.2))
        else:
            d.add(Circle(11.5,13,5,fillColor=None,strokeColor=accent,strokeWidth=1.4)); d.add(Line(9,5.5,14,5.5,strokeColor=accent,strokeWidth=1.4)); d.add(Line(10,3,13,3,strokeColor=accent,strokeWidth=1.4))
        return d

    def horizontal_context_boxes(section) -> Table:
        definitions = [
            ("impact", "Business Impact", section.business_impact, PDF_COLORS["accent"]),
            ("why", "Why This Matters", section.why_this_matters, PDF_COLORS["orange"]),
            ("technical", "Technical Summary", section.technical_summary, PDF_COLORS["blue"]),
            ("insight", "Identity Program Insight", section.insight, PDF_COLORS["green_dark"]),
        ]
        cells = []
        for kind, heading, content, accent in definitions:
            heading_style = ParagraphStyle("ContextHeading109" + kind, parent=theme_box_heading, textColor=accent)
            cells.append([[ _context_icon(kind, accent), Paragraph(pdf_text(heading).upper(), heading_style)], Paragraph(pdf_text(content or "Not available."), theme_box_body)])
        table = Table([cells], colWidths=[1.765 * inch] * 4, hAlign="LEFT")
        table.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (-1, -1), colors.HexColor("#F3F8FC")),
            ("BOX", (0, 0), (-1, -1), 0.45, PDF_COLORS["line"]),
            ("INNERGRID", (0, 0), (-1, -1), 0.25, PDF_COLORS["line"]),
            ("VALIGN", (0, 0), (-1, -1), "TOP"),
            ("LEFTPADDING", (0, 0), (-1, -1), 3.5),
            ("RIGHTPADDING", (0, 0), (-1, -1), 3.5),
            ("TOPPADDING", (0, 0), (-1, -1), 3.5),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 3.5),
        ]))
        return table

    def _object_icon(object_type: str, accent, size=24, label: str = "") -> Drawing:
        """Render a distinct object icon, inferring type when evidence lacks metadata."""
        d = Drawing(size,size); cx=cy=size/2
        d.add(Circle(cx,cy,size*.40,fillColor=colors.white,strokeColor=accent,strokeWidth=1.35))
        low=((object_type or "")+" "+(label or "")).lower()
        name=(label or "").upper()
        is_computer = any(x in low for x in ["computer","workstation","device","server","controller","machine","endpoint"]) or name.endswith("$") or any(x in name for x in ["DC01","SERVER","WIN10","MAC-"])
        is_group = any(x in low for x in ["group","team","admins","users","owners","subsystem"]) or any(x in name for x in [" ADMINS"," USERS","-OWNERS","SUBSYSTEM"])
        is_domain = any(x in low for x in ["domain","tenant","directory"]) or ("@" not in name and name.count(".")>=1 and not is_computer)
        is_app = any(x in low for x in ["application","service principal","repository","organization","app "]) or any(x in name for x in ["APPLICATION","APP","GITHUB","PHANTOM-CORP"])
        if is_computer:
            d.add(Rect(size*.27,size*.36,size*.46,size*.30,fillColor=None,strokeColor=accent,strokeWidth=1.35)); d.add(Line(size*.39,size*.28,size*.61,size*.28,strokeColor=accent,strokeWidth=1.35)); d.add(Line(cx,size*.28,cx,size*.36,strokeColor=accent,strokeWidth=1.2))
        elif is_group:
            d.add(Circle(size*.39,size*.57,size*.075,fillColor=accent,strokeColor=accent)); d.add(Circle(size*.61,size*.57,size*.075,fillColor=accent,strokeColor=accent)); d.add(Circle(cx,size*.40,size*.085,fillColor=accent,strokeColor=accent)); d.add(Line(size*.32,size*.31,size*.68,size*.31,strokeColor=accent,strokeWidth=1.1))
        elif is_domain:
            d.add(Circle(cx,cy,size*.21,fillColor=None,strokeColor=accent,strokeWidth=1.25)); d.add(Line(size*.30,cy,size*.70,cy,strokeColor=accent,strokeWidth=1.0)); d.add(Line(cx,size*.30,cx,size*.70,strokeColor=accent,strokeWidth=1.0)); d.add(Circle(cx,cy,size*.05,fillColor=accent,strokeColor=accent))
        elif is_app:
            d.add(Rect(size*.31,size*.32,size*.38,size*.36,rx=2,ry=2,fillColor=None,strokeColor=accent,strokeWidth=1.25)); d.add(Circle(size*.41,size*.55,size*.035,fillColor=accent,strokeColor=accent)); d.add(Circle(size*.59,size*.55,size*.035,fillColor=accent,strokeColor=accent)); d.add(Line(size*.40,size*.43,size*.60,size*.43,strokeColor=accent,strokeWidth=1.1))
        else:
            d.add(Circle(cx,size*.59,size*.105,fillColor=accent,strokeColor=accent)); d.add(Polygon(points=[size*.33,size*.27,size*.67,size*.27,size*.60,size*.47,size*.40,size*.47],fillColor=accent,strokeColor=accent))
        return d

    def _relationship_arrow(text: str, accent, width=112) -> Table:
        """Centered relationship label above a stronger directional arrow."""
        label_style = ParagraphStyle(
            "RelationshipArrowLabelV119", parent=styles["small"], fontName="Helvetica-Bold",
            fontSize=7.4, leading=8.5, textColor=accent, alignment=TA_CENTER,
        )
        # Raise the arrow within its drawing so the relationship line aligns
        # visually with the source and target labels on either side.
        arrow = Drawing(width, 11)
        arrow_y = 8.0
        arrow.add(Line(5, arrow_y, width - 10, arrow_y, strokeColor=accent, strokeWidth=2.6))
        arrow.add(Polygon(points=[width-17,arrow_y-4.2,width-4,arrow_y,width-17,arrow_y+4.2], fillColor=accent, strokeColor=accent))
        t = Table([[Paragraph(pdf_text(_short_name(text, 42)), label_style)], [arrow]], colWidths=[width], rowHeights=[0.18*inch, 0.14*inch])
        t.setStyle(TableStyle([
            ("VALIGN",(0,0),(-1,-1),"MIDDLE"),("ALIGN",(0,0),(-1,-1),"CENTER"),
            ("LEFTPADDING",(0,0),(-1,-1),0),("RIGHTPADDING",(0,0),(-1,-1),0),
            ("TOPPADDING",(0,0),(-1,-1),0),("BOTTOMPADDING",(0,0),(-1,-1),0),
        ]))
        return t

    def representative_path_table(df: pd.DataFrame, risk_color) -> Table:
        if df.empty:
            source, relationship, target = "No source available", "Relationship", "No target available"
            source_type, target_type = "Principal", "Target"
        else:
            row = df.iloc[0]
            source = _row_value(row, ["Source", "Account / Principal", "Principal", "Identity", "Object"])
            relationship = _row_value(row, ["Relationship / Finding", "Relationship", "Finding / Condition", "Finding", "Condition"], "Observed relationship")
            target = _row_value(row, ["Target", "Target / Scope", "Destination", "Resource", "Tenant", "Object Type", "Maturity Domain"], "Affected target")
            source_type = _row_value(row, ["Source Type", "Object Type", "Type"], "Principal")
            target_type = _row_value(row, ["Target Type", "Target Object Type", "Object Type", "Target Kind"], "Target")
        def hero_name_style(name, style_name):
            length = len(str(name or ""))
            size = 6.4 if length <= 24 else (5.8 if length <= 34 else 5.2)
            return ParagraphStyle(style_name, parent=path_entity, alignment=TA_CENTER, fontSize=size, leading=size + 0.9, splitLongWords=True)
        source_text = Paragraph(pdf_text(source), hero_name_style(source, "PathSourceV122"))
        target_text = Paragraph(pdf_text(target), hero_name_style(target, "PathTargetV122"))
        rel = _relationship_arrow(relationship, risk_color, 188)
        table = Table(
            [[_object_icon(source_type, PDF_COLORS["accent_dark"], 22, source), source_text, rel, _object_icon(target_type, risk_color, 22, target), target_text]],
            colWidths=[0.38*inch,1.76*inch,2.72*inch,0.38*inch,1.76*inch],
            rowHeights=[0.62*inch], hAlign="LEFT",
        )
        table.setStyle(TableStyle([
            ("BACKGROUND", (0,0), (-1,-1), colors.white),
            ("BOX", (0,0), (-1,-1), 1.15, colors.HexColor("#8E9BB0")),
            ("VALIGN", (0,0), (-1,-1), "MIDDLE"),
            ("ALIGN", (0,0), (0,0), "CENTER"), ("ALIGN", (3,0), (3,0), "CENTER"),
            ("LEFTPADDING", (0,0), (-1,-1), 1.5), ("RIGHTPADDING", (0,0), (-1,-1), 1.5),
            ("TOPPADDING", (0,0), (-1,-1), 4), ("BOTTOMPADDING", (0,0), (-1,-1), 4),
        ]))
        return table


    def concise_evidence_table(df: pd.DataFrame) -> Table:
        rows = [[
            Paragraph("#", evidence_header),
            Paragraph("SOURCE", evidence_header),
            Paragraph("RELATIONSHIP", evidence_header),
            Paragraph("TARGET", evidence_header),
            Paragraph("WHY IT MATTERS", evidence_header),
        ]]
        for row_number, (_, row) in enumerate(df.iterrows(), start=1):
            source = _row_value(row, ["Source", "Account / Principal", "Principal", "Identity", "Object"], "—")
            relationship = _row_value(row, ["Relationship / Finding", "Relationship", "Finding / Condition", "Finding", "Condition"], "Observed")
            target = _row_value(row, ["Target", "Target / Scope", "Destination", "Resource", "Tenant", "Object Type", "Maturity Domain"], "—")
            source_type = _row_value(row, ["Source Type", "Object Type", "Type"], "Principal")
            target_type = _row_value(row, ["Target Type", "Target Object Type", "Object Type", "Target Kind"], "Target")
            rel_color = [PDF_COLORS["blue"], PDF_COLORS["orange"], PDF_COLORS["green_dark"]][(row_number - 1) % 3]
            def evidence_name_style(name, style_name):
                length = len(str(name or ""))
                size = 6.35 if length <= 24 else (5.85 if length <= 36 else 5.35)
                return ParagraphStyle(style_name, parent=evidence_cell, fontSize=size, leading=size + 0.75, splitLongWords=True)
            rows.append([
                Paragraph(str(row_number), evidence_number),
                Table([[_object_icon(source_type, PDF_COLORS["accent_dark"], 15, source), Paragraph(pdf_text(source), evidence_name_style(source, f"EvidenceSourceV122_{row_number}"))]], colWidths=[0.22*inch,1.55*inch], style=[("VALIGN",(0,0),(-1,-1),"MIDDLE"),("LEFTPADDING",(0,0),(-1,-1),0),("RIGHTPADDING",(0,0),(-1,-1),2),("TOPPADDING",(0,0),(-1,-1),0),("BOTTOMPADDING",(0,0),(-1,-1),0)]),
                Paragraph(pdf_text(relationship), ParagraphStyle("EvidenceRelationshipV122", parent=evidence_cell, fontName="Helvetica-Bold", textColor=rel_color, alignment=TA_LEFT)),
                Table([[_object_icon(target_type, rel_color, 15, target), Paragraph(pdf_text(target), evidence_name_style(target, f"EvidenceTargetV122_{row_number}"))]], colWidths=[0.22*inch,1.55*inch], style=[("VALIGN",(0,0),(-1,-1),"MIDDLE"),("LEFTPADDING",(0,0),(-1,-1),0),("RIGHTPADDING",(0,0),(-1,-1),2),("TOPPADDING",(0,0),(-1,-1),0),("BOTTOMPADDING",(0,0),(-1,-1),0)]),
                Paragraph(pdf_text(_why_it_matters(row, relationship)), evidence_cell),
            ])
        table = Table(
            rows,
            colWidths=[0.34*inch, 1.79*inch, 1.24*inch, 1.79*inch, 1.84*inch],
            repeatRows=1, hAlign="LEFT",
        )
        table.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (-1, 0), PDF_COLORS["accent_dark"]),
            ("BACKGROUND", (0, 1), (0, -1), PDF_COLORS["accent"]),
            ("TEXTCOLOR", (0, 1), (0, -1), colors.white),
            ("BOX", (0, 0), (-1, -1), 1.05, colors.HexColor("#8E9BB0")),
            ("INNERGRID", (0, 0), (-1, -1), 0.35, PDF_COLORS["line"]),
            ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
            ("ALIGN", (0, 0), (0, -1), "CENTER"),
            ("LEFTPADDING", (0, 0), (-1, -1), 3),
            ("RIGHTPADDING", (0, 0), (-1, -1), 3),
            ("TOPPADDING", (0, 0), (-1, 0), 3),
            ("BOTTOMPADDING", (0, 0), (-1, 0), 3),
            ("TOPPADDING", (0, 1), (-1, -1), 1.2),
            ("BOTTOMPADDING", (0, 1), (-1, -1), 1.2),
        ]))
        return table

    def remediation_focus_for_section(section: "ReportSection") -> str:
        if getattr(section, "immediate_actions", None):
            return section.immediate_actions[0]
        return "Prioritize reducing administrative paths into Tier Zero while validating operational exceptions."

    for result in sorted(section_results.values(), key=lambda item: item.section.executive_priority):
        if not result.has_results and not show_empty:
            continue

        section = result.section
        risk_color = severity_color(section.default_risk)
        story.append(PageBreak())
        story.append(theme_header(section.title, section.risk_theme, section.executive_priority, styles, risk_color))
        story.append(Spacer(1, 3))

        strip = Table([[
            Paragraph(pdf_text(section.default_risk).upper(), styles["inverse"]),
            Paragraph(f"<b>Maturity Domain:</b> {pdf_text(section.maturity_domain)}", compact_small),
        ]], colWidths=[1.12 * inch, 5.88 * inch], hAlign="LEFT")
        strip.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (0, 0), risk_color),
            ("BACKGROUND", (1, 0), (-1, 0), colors.HexColor("#F3F8FC")),
            ("BOX", (0, 0), (-1, -1), 0.5, PDF_COLORS["line"]),
            ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
            ("ALIGN", (0, 0), (0, 0), "CENTER"),
            ("LEFTPADDING", (1, 0), (-1, -1), 8),
            ("RIGHTPADDING", (1, 0), (-1, -1), 8),
            ("TOPPADDING", (0, 0), (-1, -1), 5.5),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 5.5),
        ]))
        story.append(strip)
        story.append(Spacer(1, 3))

        # Full-width executive narrative: calm, report-like, and engagement focused.
        story.append(Paragraph("Executive Summary", exhibit_heading))
        executive_text = pdf_text(section.executive_summary or "")
        if section.why_this_matters:
            executive_text += " " + pdf_text(section.why_this_matters)
        story.append(Paragraph(executive_text, theme_summary_style))
        story.append(Spacer(1, 4))

        # Four horizontal context boxes directly beneath the narrative.
        story.append(horizontal_context_boxes(section))
        story.append(Spacer(1, 5))

        remediation_text = remediation_focus_for_section(section)
        remediation_line = Table([[
            Paragraph("REMEDIATION FOCUS", ParagraphStyle(
                "RemediationFocusLabelV14", parent=styles["small"], fontName="Helvetica-Bold",
                fontSize=6.9, leading=8.0, textColor=PDF_COLORS["green_dark"],
            )),
            Paragraph(pdf_text(remediation_text), ParagraphStyle(
                "RemediationFocusBodyV14", parent=styles["small"], fontSize=7.65, leading=8.9,
                textColor=PDF_COLORS["ink"],
            )),
        ]], colWidths=[1.22*inch, 5.78*inch], rowHeights=[0.37*inch], hAlign="LEFT")
        remediation_line.setStyle(TableStyle([
            ("BACKGROUND",(0,0),(-1,-1),colors.HexColor("#F4FBF8")),
            ("LINEBEFORE",(0,0),(0,0),2.6,PDF_COLORS["green_dark"]),
            ("VALIGN",(0,0),(-1,-1),"MIDDLE"),
            ("LEFTPADDING",(0,0),(-1,-1),6),("RIGHTPADDING",(0,0),(-1,-1),6),
            ("TOPPADDING",(0,0),(-1,-1),3),("BOTTOMPADDING",(0,0),(-1,-1),3),
        ]))
        story.append(remediation_line)
        story.append(Spacer(1, 5))

        evidence_df = section_technical_evidence_dataframe(
            section, result.combined_evidence_df, max_rows=pdf_evidence_rows
        )
        if section.id == "identity_hygiene" and not evidence_df.empty:
            if "Target" not in evidence_df.columns:
                evidence_df["Target"] = evidence_df.get("Account / Principal", "Affected identity")
            evidence_df["Target"] = evidence_df["Target"].replace({"": None, "—": None}).fillna(evidence_df.get("Account / Principal", "Affected identity"))
            if "Why It Matters" not in evidence_df.columns:
                evidence_df["Why It Matters"] = evidence_df.get("Risk", "Identity hygiene weakness increases the usefulness of other attack paths.")
        platform_kind = platform_section_kind(section)
        if platform_kind and not evidence_df.empty:
            if "Target" not in evidence_df.columns:
                evidence_df["Target"] = ""
            for idx, erow in evidence_df.iterrows():
                existing = str(erow.get("Target", "") or "").strip()
                if existing and existing.lower() not in {"—", "nan", "none", "null", "affected target"}:
                    continue
                # Match the HTML report: prefer the actual normalized target/scope,
                # then the extension object type (Organization, Teamrole, User,
                # Clientsecret, Computer, Account, Privilegezone, etc.).
                candidate = ""
                for key in ["Target / Scope", "Object Type", "Maturity Domain"]:
                    value = str(erow.get(key, "") or "").strip()
                    if value and value.lower() not in {"—", "nan", "none", "null"}:
                        candidate = value
                        break
                if not candidate:
                    candidate = {"github": "GitHub object", "okta": "Okta object", "jamf": "Jamf object"}.get(platform_kind, "Affected object")
                evidence_df.at[idx, "Target"] = candidate


        # One combined technical exhibit: representative path followed by concise evidence.
        story.append(Spacer(1, 3))
        if platform_kind in {"github", "okta", "jamf"}:
            exhibit_title = "Representative Exposure & Evidence"
            exhibit_description = (
                "Representative control relationship or exposure and supporting examples; "
                "complete details remain in the Evidence Workbook."
            )
        else:
            exhibit_title = "Representative Attack Path & Evidence"
            exhibit_description = (
                "Representative path and supporting examples; complete details remain in the Evidence Workbook."
            )
        story.append(Paragraph(exhibit_title, exhibit_heading))
        story.append(Paragraph(exhibit_description, exhibit_intro))
        story.append(representative_path_table(evidence_df, risk_color))
        story.append(Spacer(1, 3))
        if evidence_df.empty:
            story.append(Paragraph("No representative evidence rows were available for this section.", compact_body))
        else:
            story.append(concise_evidence_table(evidence_df))
            evidence_note = (
                f"{len(evidence_df)} examples shown. Complete exposure and relationship details are in the XLSX workbook."
                if platform_kind in {"github", "okta", "jamf"}
                else f"{len(evidence_df)} examples shown. Complete evidence and relationship details are in the XLSX workbook."
            )
            story.append(Paragraph(
                evidence_note,
                ParagraphStyle("ThemeEvidenceNoteV108", parent=compact_small, textColor=PDF_COLORS["blue"], alignment=TA_CENTER, spaceBefore=3),
            ))
        story.append(Spacer(1, 4))

        # Distinct action cards create a fast executive scan path without competing with severity.
        story.append(Spacer(1, 3))
        recommended_heading = Table([[Paragraph("Recommended Actions", exhibit_heading)]], colWidths=[7.0*inch], hAlign="LEFT")
        recommended_heading.setStyle(TableStyle([("LEFTPADDING",(0,0),(-1,-1),0),("RIGHTPADDING",(0,0),(-1,-1),0),("TOPPADDING",(0,0),(-1,-1),0),("BOTTOMPADDING",(0,0),(-1,-1),0)]))
        story.append(recommended_heading)
        story.append(Spacer(1, 7))
        actions = Table([[
            compact_action_block("Immediate", section.immediate_actions, PDF_COLORS["high"]),
            compact_action_block("Near-Term", section.near_term_actions, PDF_COLORS["orange"]),
            compact_action_block("Strategic", section.strategic_actions, PDF_COLORS["blue"]),
        ]], colWidths=[(7.00 / 3) * inch] * 3, hAlign="LEFT", spaceBefore=0)
        actions.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (0, 0), colors.HexColor("#FFF4F3")),
            ("BACKGROUND", (1, 0), (1, 0), colors.HexColor("#FFF8EF")),
            ("BACKGROUND", (2, 0), (2, 0), colors.HexColor("#F3F5FF")),
            ("BOX", (0, 0), (0, 0), 0.95, PDF_COLORS["high"]),
            ("BOX", (1, 0), (1, 0), 0.95, PDF_COLORS["orange"]),
            ("BOX", (2, 0), (2, 0), 0.95, PDF_COLORS["blue"]),
            ("VALIGN", (0, 0), (-1, -1), "TOP"),
            ("LEFTPADDING", (0, 0), (-1, -1), 7),
            ("RIGHTPADDING", (0, 0), (-1, -1), 7),
            ("TOPPADDING", (0, 0), (-1, -1), 7),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
        ]))
        story.append(actions)

        # Collection notes are centralized in the final Collection Status section.

    # ------------------------------------------------------------------
    # Conditional remediation appendix.
    # ------------------------------------------------------------------
    remediated_items = remediated_during_assessment_items()
    if remediated_items:
        story.append(PageBreak())
        story.extend(section_title("Remediation Completed During Assessment", styles, "Appendix"))
        story.append(Paragraph(
            "The items below were explicitly validated as resolved, remediated, closed, fixed, or otherwise completed during the assessment window. Detailed supporting evidence remains in the Evidence Workbook.",
            styles["lead"],
        ))
        story.append(Spacer(1, 8))

        appendix_head = ParagraphStyle(
            "RemediationAppendixHeadV13", parent=styles["table_head"],
            fontSize=7.2, leading=8.2, textColor=colors.white,
        )
        appendix_body = ParagraphStyle(
            "RemediationAppendixBodyV13", parent=styles["body"],
            fontSize=7.4, leading=9.1, spaceAfter=0,
        )
        appendix_status = ParagraphStyle(
            "RemediationAppendixStatusV13", parent=styles["small"],
            fontSize=7.0, leading=8.2, textColor=PDF_COLORS["green_dark"],
            alignment=TA_CENTER,
        )
        appendix_rows = [[
            Paragraph("ITEM", appendix_head),
            Paragraph("PLATFORM", appendix_head),
            Paragraph("VALIDATED OUTCOME", appendix_head),
            Paragraph("STATUS", appendix_head),
        ]]
        for item in remediated_items:
            appendix_rows.append([
                Paragraph(pdf_text(item.get("title") or "Remediated exposure"), appendix_body),
                Paragraph(pdf_text(item.get("platform") or "Enterprise"), appendix_body),
                Paragraph(pdf_text(item.get("detail") or "Remediation was validated during the assessment."), appendix_body),
                Paragraph(pdf_text(item.get("status") or "Resolved"), appendix_status),
            ])

        remediation_table = Table(
            appendix_rows,
            colWidths=[2.05*inch, 1.05*inch, 3.10*inch, 0.80*inch],
            repeatRows=1,
            hAlign="LEFT",
        )
        remediation_style = [
            ("BACKGROUND", (0,0), (-1,0), PDF_COLORS["accent_dark"]),
            ("BOX", (0,0), (-1,-1), 0.65, colors.HexColor("#C8D2E1")),
            ("GRID", (0,0), (-1,-1), 0.35, colors.HexColor("#DDE4ED")),
            ("VALIGN", (0,0), (-1,-1), "TOP"),
            ("LEFTPADDING", (0,0), (-1,-1), 6),
            ("RIGHTPADDING", (0,0), (-1,-1), 6),
            ("TOPPADDING", (0,0), (-1,-1), 6),
            ("BOTTOMPADDING", (0,0), (-1,-1), 6),
        ]
        for row_index in range(1, len(appendix_rows)):
            if row_index % 2 == 0:
                remediation_style.append(("BACKGROUND", (0,row_index), (-1,row_index), colors.HexColor("#F7F9FC")))
        remediation_table.setStyle(TableStyle(remediation_style))
        story.append(remediation_table)

    # ------------------------------------------------------------------
    # Privilege Zone Intelligence morning renderer restored (v1.5.44)
    # ------------------------------------------------------------------
    # Privilege Zones are treated as their own protected scope, exactly as
    # Tier Zero is treated elsewhere in the report. The page always provides
    # configuration intelligence. Evidence sections render only when actual
    # source -> relationship -> protected-object attack-path records exist.
    pz_df = globals().get("PRIVILEGE_ZONE_REPORT_DF", pd.DataFrame())
    pz_findings_df = globals().get("PRIVILEGE_ZONE_FINDINGS_DF", pd.DataFrame())
    if globals().get("has_privilege_zones", False) and isinstance(pz_df, pd.DataFrame) and not pz_df.empty:
        story.append(PageBreak())
        story.extend(section_title("Privilege Zone Intelligence", styles, "Customer-defined protected scope"))
        story.append(Paragraph(
            "Privilege Zones extend BloodHound's protected-asset model beyond the built-in Tier Zero scope. "
            "This page summarizes each configured zone, the objects and protection rules that define it, and—when object-level attack-path evidence is available—representative paths into that protected scope.",
            styles["lead"],
        ))
        story.append(Spacer(1, 6))

        def _pz_col(frame, candidates):
            if not isinstance(frame, pd.DataFrame):
                return None
            lookup = {str(c).strip().casefold(): c for c in frame.columns}
            for candidate in candidates:
                matched = lookup.get(str(candidate).strip().casefold())
                if matched is not None:
                    return matched
            return None

        def _pz_num(value):
            try:
                if pd.isna(value):
                    return 0
                return int(float(value))
            except Exception:
                return 0

        def _pz_text(value, default=""):
            text = str(value if value is not None else "").strip()
            return default if not text or text.casefold() in {"nan", "none", "null"} else text

        def _pz_business_purpose(zone_name):
            name = _pz_text(zone_name, "Protected Operations")
            low = name.casefold()
            if "code" in low and ("sign" in low or "certificate" in low): return "Software Trust"
            if "tier one" in low or "tier 1" in low or low == "t1": return "Server Administration"
            if "identity" in low or "directory" in low or "entra" in low: return "Identity Administration"
            if "cloud" in low or "azure" in low or "aws" in low or "gcp" in low: return "Cloud Administration"
            if "database" in low or "data" in low: return "Data Platform Administration"
            if "network" in low: return "Network Administration"
            if "security" in low or "soc" in low: return "Security Operations"
            if "backup" in low or "recovery" in low: return "Recovery Infrastructure"
            if "production" in low or "prod" in low: return "Production Administration"
            return "Customer-Defined Control Plane"

        zone_col = _pz_col(pz_df, ["Zone", "Privilege Zone", "Zone Name", "Name"])
        protected_col = _pz_col(pz_df, ["Protected Objects", "Protected Object Count", "Objects", "Member Count"])
        rules_col = _pz_col(pz_df, ["Rule Count", "Rules", "Protection Rules", "Boundary Rules", "Reported Rule Count"])

        total_zones = len(pz_df)
        total_objects = sum(_pz_num(v) for v in pz_df[protected_col]) if protected_col else len(globals().get("PRIVILEGE_ZONE_PROTECTED_OBJECTS_DF", pd.DataFrame()))
        if rules_col:
            total_rules = sum(_pz_num(v) for v in pz_df[rules_col])
        else:
            _rules_df = globals().get("PRIVILEGE_ZONE_RULES_DF", globals().get("ZONE_RULES_DF", pd.DataFrame()))
            total_rules = len(_rules_df) if isinstance(_rules_df, pd.DataFrame) else 0

        finding_count_col = _pz_col(pz_findings_df, ["Finding Count", "Findings", "Count"])
        finding_title_col = _pz_col(pz_findings_df, ["Finding Title", "Title", "Name"])
        finding_zone_col = _pz_col(pz_findings_df, ["Zone Name", "Privilege Zone", "Zone"])
        finding_severity_col = _pz_col(pz_findings_df, ["Severity", "Risk Level", "Risk"])

        findings_for_pdf = pd.DataFrame()
        active_finding_total = 0
        finding_category_count = 0
        zone_finding_counts = {}
        zone_category_counts = {}
        if isinstance(pz_findings_df, pd.DataFrame) and not pz_findings_df.empty and finding_count_col:
            findings_for_pdf = pz_findings_df.copy()
            findings_for_pdf[finding_count_col] = pd.to_numeric(findings_for_pdf[finding_count_col], errors="coerce").fillna(0).astype(int)
            findings_for_pdf = findings_for_pdf[findings_for_pdf[finding_count_col] > 0].sort_values(finding_count_col, ascending=False, kind="stable")
            active_finding_total = int(findings_for_pdf[finding_count_col].sum())
            finding_category_count = int(len(findings_for_pdf))
            if finding_zone_col:
                for zone_name, group in findings_for_pdf.groupby(finding_zone_col, dropna=False):
                    normalized = _pz_text(zone_name)
                    if normalized:
                        zone_finding_counts[normalized.casefold()] = int(group[finding_count_col].sum())
                        zone_category_counts[normalized.casefold()] = int(len(group))

        # Build a normalized set of true object-level attack paths. Only
        # dedicated graph/path datasets are eligible. Aggregate finding rows
        # remain summary data and can never become representative examples.
        path_frames = []
        for candidate_name in [
            "PRIVILEGE_ZONE_REPRESENTATIVE_PATHS_DF",
            "PRIVILEGE_ZONE_ATTACK_PATHS_DF",
            "PRIVILEGE_ZONE_GRAPH_PATHS_DF",
            "PRIVILEGE_ZONE_PATHS_DF",
            "PZ_REPRESENTATIVE_PATHS_DF",
        ]:
            candidate = globals().get(candidate_name)
            if isinstance(candidate, pd.DataFrame) and not candidate.empty:
                path_frames.append(candidate.copy())
        # Aggregate finding rows summarize exposure categories; they are not
        # graph paths and must never be promoted into representative examples.

        normalized_paths = []
        seen_paths = set()
        placeholder_values = {
            "", "unknown", "none", "nan", "principal outside zone", "source principal",
            "protected object", "protected zone object", "target object", "privilege zone",
        }
        for frame in path_frames:
            source_col = _pz_col(frame, ["Source", "Source Principal", "Principal", "Start Node", "Start", "Source Name", "FromPrincipalName"])
            relationship_col = _pz_col(frame, ["Relationship", "Edge", "Attack Path", "Finding Title", "Title", "Name", "Finding"])
            target_col = _pz_col(frame, ["Target", "Protected Object", "Object", "End Node", "End", "Target Name", "ToPrincipalName"])
            zone_path_col = _pz_col(frame, ["Zone Name", "Privilege Zone", "Zone", "Target Zone"])
            source_kind_col = _pz_col(frame, ["Source Kind", "Source Type", "FromPrincipalKind", "Start Kind"])
            target_kind_col = _pz_col(frame, ["Target Kind", "Target Type", "ToPrincipalKind", "End Kind", "Object Type"])
            if not source_col or not target_col:
                continue
            for _, path_row in frame.iterrows():
                source = _pz_text(path_row.get(source_col, ""))
                target = _pz_text(path_row.get(target_col, ""))
                relationship = _pz_text(path_row.get(relationship_col, "Privilege Relationship") if relationship_col else "Privilege Relationship", "Privilege Relationship")
                zone_name = _pz_text(path_row.get(zone_path_col, "") if zone_path_col else "", _pz_text(pz_df.iloc[0].get(zone_col, "Privilege Zone") if zone_col else "Privilege Zone", "Privilege Zone"))
                if source.casefold() in placeholder_values or target.casefold() in placeholder_values:
                    continue
                key = (source.casefold(), relationship.casefold(), target.casefold(), zone_name.casefold())
                if key in seen_paths:
                    continue
                seen_paths.add(key)
                normalized_paths.append({
                    "source": source,
                    "source_kind": _pz_text(path_row.get(source_kind_col, "Identity") if source_kind_col else "Identity", "Identity"),
                    "relationship": relationship,
                    "target": target,
                    "target_kind": _pz_text(path_row.get(target_kind_col, "Protected Object") if target_kind_col else "Protected Object", "Protected Object"),
                    "zone": zone_name,
                })

        has_attack_paths = len(normalized_paths) > 0
        privilege_zone_render_mode = "identity_risk_theme" if has_attack_paths else "executive_summary"
        has_active_findings = active_finding_total > 0
        exposed_zones = 0
        if has_active_findings:
            if zone_finding_counts:
                exposed_zones = sum(1 for value in zone_finding_counts.values() if value > 0)
            elif total_zones == 1:
                exposed_zones = 1

        pz_overline = ParagraphStyle(
            "PZThemeOverlineV1518", parent=styles["small"], fontName="Helvetica-Bold",
            fontSize=6.1, leading=6.8, textColor=PDF_COLORS["muted"], spaceAfter=1,
        )
        pz_hero_num = ParagraphStyle(
            "PZThemeHeroNumV1518", parent=styles["h1"], fontSize=13.2, leading=13.8,
            textColor=PDF_COLORS["accent_dark"], spaceAfter=0,
        )
        pz_hero_status = ParagraphStyle(
            "PZThemeHeroStatusV1518", parent=pz_hero_num, fontSize=14.5, leading=15.0,
            textColor=PDF_COLORS["accent_dark"],
        )
        pz_hero_body = ParagraphStyle(
            "PZThemeHeroBodyV1518", parent=styles["small"], fontSize=6.2, leading=7.0,
            textColor=PDF_COLORS["ink"],
        )
        pz_metric_label = ParagraphStyle(
            "PZThemeMetricLabelV1518", parent=styles["small"], fontName="Helvetica-Bold",
            fontSize=5.5, leading=6.1, textColor=PDF_COLORS["muted"],
        )
        pz_metric_value = ParagraphStyle(
            "PZThemeMetricValueV1518", parent=styles["h3"], fontSize=9.6, leading=10.0,
            textColor=PDF_COLORS["accent_dark"],
        )

        coverage_metrics = Table([[
            [Paragraph(f"{total_objects:,}", pz_metric_value), Paragraph("Protected Objects", pz_metric_label)],
            [Paragraph(f"{total_rules:,}", pz_metric_value), Paragraph("Protection Rules", pz_metric_label)],
        ]], colWidths=[1.40*inch, 1.40*inch])
        coverage_metrics.setStyle(TableStyle([
            ("LINEBEFORE",(1,0),(1,0),0.4,colors.HexColor("#D8DEE8")),
            ("VALIGN",(0,0),(-1,-1),"MIDDLE"),
            ("LEFTPADDING",(0,0),(-1,-1),3),("RIGHTPADDING",(0,0),(-1,-1),3),
            ("TOPPADDING",(0,0),(-1,-1),1),("BOTTOMPADDING",(0,0),(-1,-1),0),
        ]))

        if privilege_zone_render_mode == "identity_risk_theme":
            analysis_word = "ATTACK PATHS FOUND"
            analysis_detail = f"{len(normalized_paths):,} object-level example{'s' if len(normalized_paths) != 1 else ''} available"
            analysis_color = PDF_COLORS["high"]
            analysis_bg = colors.HexColor("#FFF4F3")
            analysis_metrics = Table([[
                [Paragraph(f"{exposed_zones:,} of {total_zones:,}", pz_metric_value), Paragraph("Affected Zones", pz_metric_label)],
                [Paragraph(f"{finding_category_count:,}", pz_metric_value), Paragraph("Finding Categories", pz_metric_label)],
            ]], colWidths=[1.40*inch,1.40*inch])
            analysis_metrics.setStyle(TableStyle([
                ("LINEBEFORE",(1,0),(1,0),0.4,colors.HexColor("#D8DEE8")),
                ("VALIGN",(0,0),(-1,-1),"MIDDLE"),
                ("LEFTPADDING",(0,0),(-1,-1),3),("RIGHTPADDING",(0,0),(-1,-1),3),
                ("TOPPADDING",(0,0),(-1,-1),1),("BOTTOMPADDING",(0,0),(-1,-1),0),
            ]))
            coverage_content = [
                Paragraph("ZONE COVERAGE", pz_overline),
                Paragraph(f"{total_zones:,}", pz_hero_num),
                Paragraph("Configured zone" if total_zones == 1 else "Configured zones", pz_hero_body),
                Spacer(1,1), coverage_metrics,
            ]
            analysis_content = [
                Paragraph("ANALYSIS STATUS", pz_overline),
                Paragraph(analysis_word, pz_hero_status),
                Paragraph(analysis_detail, pz_hero_body),
                Spacer(1,1), analysis_metrics,
            ]
            hero = Table([[coverage_content, analysis_content]], colWidths=[3.50*inch,3.50*inch], hAlign="LEFT")
            hero.setStyle(TableStyle([
                ("BACKGROUND",(0,0),(0,0),colors.HexColor("#F3F5FF")),
                ("BACKGROUND",(1,0),(1,0),analysis_bg),
                ("BOX",(0,0),(0,0),0.7,colors.HexColor("#C7CFE8")),
                ("BOX",(1,0),(1,0),0.7,analysis_color),
                ("LINEABOVE",(0,0),(0,0),3,PDF_COLORS["blue"]),
                ("LINEABOVE",(1,0),(1,0),3,analysis_color),
                ("VALIGN",(0,0),(-1,-1),"TOP"),
                ("LEFTPADDING",(0,0),(-1,-1),10),("RIGHTPADDING",(0,0),(-1,-1),10),
                ("TOPPADDING",(0,0),(-1,-1),3),("BOTTOMPADDING",(0,0),(-1,-1),3),
            ]))
            story.append(hero)
        else:
            # Executive Summary mode is intentionally a different page treatment,
            # not a disabled version of the Identity Risk Theme page.
            executive_card_label = ParagraphStyle(
                "PZExecutiveCardLabelV1518", parent=styles["small"], fontName="Helvetica-Bold",
                fontSize=5.8, leading=6.4, textColor=PDF_COLORS["muted"], alignment=1,
            )
            executive_card_value = ParagraphStyle(
                "PZExecutiveCardValueV1518", parent=styles["h1"], fontSize=16.5, leading=17.0,
                textColor=PDF_COLORS["accent_dark"], alignment=1, spaceAfter=1,
            )
            executive_card_detail = ParagraphStyle(
                "PZExecutiveCardDetailV1518", parent=styles["small"], fontSize=5.8, leading=6.5,
                textColor=PDF_COLORS["ink"], alignment=1,
            )
            executive_cards = Table([[
                [Paragraph("CONFIGURED ZONES", executive_card_label), Paragraph(f"{total_zones:,}", executive_card_value), Paragraph("Customer-defined protected scopes", executive_card_detail)],
                [Paragraph("PROTECTED OBJECTS", executive_card_label), Paragraph(f"{total_objects:,}", executive_card_value), Paragraph("Assets governed by zone boundaries", executive_card_detail)],
                [Paragraph("PROTECTION RULES", executive_card_label), Paragraph(f"{total_rules:,}", executive_card_value), Paragraph("Rules defining protected membership", executive_card_detail)],
                [Paragraph("ACTIVE FINDINGS", executive_card_label), Paragraph(f"{active_finding_total:,}", executive_card_value), Paragraph(f"Across {finding_category_count:,} finding categories", executive_card_detail)],
            ]], colWidths=[1.75*inch]*4, hAlign="LEFT")
            executive_cards.setStyle(TableStyle([
                ("BACKGROUND",(0,0),(-1,-1),colors.HexColor("#F7F8FC")),
                ("BOX",(0,0),(-1,-1),0.75,colors.HexColor("#C7CFE0")),
                ("GRID",(0,0),(-1,-1),0.45,colors.HexColor("#D9DFE9")),
                ("LINEABOVE",(0,0),(2,0),3,PDF_COLORS["blue"]),
                ("LINEABOVE",(3,0),(3,0),3,PDF_COLORS.get("moderate", colors.HexColor("#F26B2A")) if has_active_findings else PDF_COLORS["green_dark"]),
                ("VALIGN",(0,0),(-1,-1),"MIDDLE"),
                ("LEFTPADDING",(0,0),(-1,-1),7),("RIGHTPADDING",(0,0),(-1,-1),7),
                ("TOPPADDING",(0,0),(-1,-1),7),("BOTTOMPADDING",(0,0),(-1,-1),7),
            ]))
            story.append(executive_cards)
            story.append(Spacer(1,5))
            mode_title = ParagraphStyle(
                "PZExecutiveModeTitleV1518", parent=styles["h3"], fontSize=9.5, leading=10.5,
                textColor=PDF_COLORS["accent_dark"], spaceAfter=2,
            )
            mode_body = ParagraphStyle(
                "PZExecutiveModeBodyV1518", parent=styles["body"], fontSize=7.1, leading=8.4,
                textColor=PDF_COLORS["ink"],
            )
            if has_active_findings:
                executive_assessment = (
                    f"BloodHound identified active Privilege Zone findings across {finding_category_count:,} categories. "
                    "The collected dataset did not include object-level source-to-target graph paths, so this page reports verified boundary coverage and finding prevalence without inventing representative attack paths."
                )
                mode_accent = PDF_COLORS.get("moderate", colors.HexColor("#F26B2A"))
                mode_bg = colors.HexColor("#FFF8EF")
                mode_heading = "EXECUTIVE ZONE ASSESSMENT"
            else:
                executive_assessment = (
                    f"BloodHound confirmed {total_zones:,} configured Privilege Zone{'s' if total_zones != 1 else ''} protecting {total_objects:,} objects through {total_rules:,} rules. "
                    "No active zone-specific findings or object-level source-to-target attack paths were present in the collected data."
                )
                mode_accent = PDF_COLORS["green_dark"]
                mode_bg = colors.HexColor("#F1FAF6")
                mode_heading = "EXECUTIVE ZONE ASSESSMENT"
            executive_assessment_box = Table([["", [Paragraph(mode_heading, mode_title), Paragraph(executive_assessment, mode_body)]]], colWidths=[0.08*inch,6.92*inch], hAlign="LEFT")
            executive_assessment_box.setStyle(TableStyle([
                ("BACKGROUND",(0,0),(-1,-1),mode_bg),("BACKGROUND",(0,0),(0,0),mode_accent),
                ("BOX",(0,0),(-1,-1),0.7,colors.HexColor("#C8D2E1")),("VALIGN",(0,0),(-1,-1),"TOP"),
                ("LEFTPADDING",(1,0),(1,0),10),("RIGHTPADDING",(1,0),(1,0),10),
                ("TOPPADDING",(1,0),(1,0),7),("BOTTOMPADDING",(1,0),(1,0),7),
                ("LEFTPADDING",(0,0),(0,0),0),("RIGHTPADDING",(0,0),(0,0),0),
            ]))
            story.append(executive_assessment_box)
        story.append(Spacer(1,3))

        story.append(Paragraph("Privilege Zone Summary", compact_h2))
        zone_name_style = ParagraphStyle(
            "PZThemeZoneNameV1512", parent=styles["h3"], fontSize=9.0, leading=10.0,
            textColor=PDF_COLORS["accent_dark"], spaceAfter=0,
        )
        zone_purpose_style = ParagraphStyle(
            "PZThemeZonePurposeV1512", parent=styles["small"], fontSize=6.2, leading=7.0,
            textColor=PDF_COLORS["muted"],
        )
        zone_label_style = ParagraphStyle(
            "PZThemeZoneLabelV1512", parent=styles["small"], fontName="Helvetica-Bold",
            fontSize=5.6, leading=6.2, textColor=PDF_COLORS["muted"], alignment=1,
        )
        zone_value_style = ParagraphStyle(
            "PZThemeZoneValueV1512", parent=styles["h3"], fontSize=9.0, leading=9.6,
            textColor=PDF_COLORS["ink"], alignment=1,
        )
        for card_index, (_, row) in enumerate(pz_df.head(3).iterrows()):
            name = _pz_text(row.get(zone_col,"Privilege Zone") if zone_col else "Privilege Zone","Privilege Zone")
            purpose = _pz_business_purpose(name)
            protected = _pz_num(row.get(protected_col,0)) if protected_col else 0
            if rules_col:
                rules = _pz_num(row.get(rules_col,0))
            else:
                rules = 0
                _rules_df = globals().get("PRIVILEGE_ZONE_RULES_DF", globals().get("ZONE_RULES_DF", pd.DataFrame()))
                if isinstance(_rules_df, pd.DataFrame) and not _rules_df.empty:
                    _rule_zone_col = _pz_col(_rules_df,["Zone","Privilege Zone","Zone Name","Name"])
                    if _rule_zone_col:
                        rules = int((_rules_df[_rule_zone_col].astype(str).str.strip().str.casefold() == name.strip().casefold()).sum())
            zone_findings = zone_finding_counts.get(name.casefold(), active_finding_total if total_zones == 1 else 0)
            zone_categories = zone_category_counts.get(name.casefold(), finding_category_count if total_zones == 1 else 0)
            status_text = "ATTACK PATHS" if any(p["zone"].casefold() == name.casefold() for p in normalized_paths) else ("FINDINGS" if zone_findings else "CONFIGURED")
            status_color = PDF_COLORS["high"] if status_text == "ATTACK PATHS" else (PDF_COLORS.get("moderate", colors.HexColor("#F26B2A")) if status_text == "FINDINGS" else PDF_COLORS["green_dark"])
            status_style = ParagraphStyle(
                f"PZThemeStatusV1512_{card_index}", parent=styles["small"], fontName="Helvetica-Bold",
                fontSize=6.1, leading=6.7, textColor=status_color, alignment=1,
            )
            zone_bar = Table([[
                [Paragraph(pdf_text(name),zone_name_style),Paragraph(pdf_text(purpose),zone_purpose_style)],
                [Paragraph("OBJECTS",zone_label_style),Paragraph(f"{protected:,}",zone_value_style)],
                [Paragraph("RULES",zone_label_style),Paragraph(f"{rules:,}",zone_value_style)],
                [Paragraph("CATEGORIES",zone_label_style),Paragraph(f"{zone_categories:,}",zone_value_style)],
                [Paragraph("STATUS",zone_label_style),Paragraph(status_text,status_style)],
            ]], colWidths=[2.80*inch,0.95*inch,0.85*inch,1.05*inch,1.35*inch], hAlign="LEFT")
            zone_bar.setStyle(TableStyle([
                ("BACKGROUND",(0,0),(-1,-1),colors.HexColor("#FBFCFE")),
                ("BOX",(0,0),(-1,-1),0.7,colors.HexColor("#C8D2E1")),
                ("LINEBEFORE",(0,0),(0,0),4,status_color),
                ("GRID",(1,0),(-1,-1),0.3,colors.HexColor("#DDE4ED")),
                ("VALIGN",(0,0),(-1,-1),"MIDDLE"),
                ("LEFTPADDING",(0,0),(0,0),9),("RIGHTPADDING",(0,0),(0,0),7),
                ("LEFTPADDING",(1,0),(-1,-1),3),("RIGHTPADDING",(1,0),(-1,-1),3),
                ("TOPPADDING",(0,0),(-1,-1),5),("BOTTOMPADDING",(0,0),(-1,-1),5),
            ]))
            story.append(zone_bar)
            story.append(Spacer(1,3))
        if total_zones > 3:
            story.append(Paragraph(f"{total_zones-3:,} additional configured zones are retained in the Evidence Workbook.", compact_small))
            story.append(Spacer(1,2))

        default_zone = _pz_text(
            pz_df.iloc[0].get(zone_col, "Privilege Zone") if zone_col else "Privilege Zone",
            "Privilege Zone",
        )

        if has_attack_paths:
            # The summary table is evidence-dependent. It is intentionally
            # omitted when no object-level attack paths are available.
            if not findings_for_pdf.empty:
                story.append(Paragraph("Active Privilege Zone Findings", compact_h2))
                finding_note = ParagraphStyle("PZThemeFindingNoteV1512",parent=styles["small"],fontSize=6.1,leading=7.0,textColor=PDF_COLORS["muted"],spaceAfter=2)
                shown_count = min(5, finding_category_count)
                story.append(Paragraph(
                    f"{finding_category_count:,} active finding categories were identified in the protected scope. The top {shown_count:,} are summarized below; complete finding data remains in the Evidence Workbook.",
                    finding_note,
                ))
                fh = ParagraphStyle("PZThemeFindingHeadV1512",parent=styles["small"],fontName="Helvetica-Bold",fontSize=5.7,leading=6.4,textColor=colors.white)
                ft = ParagraphStyle("PZThemeFindingTextV1512",parent=styles["small"],fontSize=5.8,leading=6.6,textColor=PDF_COLORS["ink"])
                fs = ParagraphStyle("PZThemeFindingSeverityV1512",parent=ft,fontName="Helvetica-Bold",textColor=PDF_COLORS["high"],alignment=1)
                fc = ParagraphStyle("PZThemeFindingCountV1512",parent=ft,fontName="Helvetica-Bold",alignment=2)
                finding_rows=[[Paragraph("ACTIVE FINDING",fh),Paragraph("ZONE",fh),Paragraph("SEVERITY",fh),Paragraph("COUNT",fh)]]
                for _, finding in findings_for_pdf.head(5).iterrows():
                    severity=_pz_text(finding.get(finding_severity_col,"High") if finding_severity_col else "High","High")
                    finding_rows.append([
                        Paragraph(pdf_text(_pz_text(finding.get(finding_title_col,"Privilege Zone Finding") if finding_title_col else "Privilege Zone Finding")),ft),
                        Paragraph(pdf_text(_pz_text(finding.get(finding_zone_col,"") if finding_zone_col else "",default_zone)),ft),
                        Paragraph(pdf_text(severity.upper()),fs),
                        Paragraph(f"{_pz_num(finding.get(finding_count_col,0)):,}",fc),
                    ])
                findings_table=Table(finding_rows,colWidths=[3.80*inch,1.50*inch,0.90*inch,0.80*inch],repeatRows=1,hAlign="LEFT")
                findings_table.setStyle(TableStyle([
                    ("BACKGROUND",(0,0),(-1,0),PDF_COLORS["ink"]),("GRID",(0,0),(-1,-1),0.3,colors.HexColor("#D8DEE8")),
                    ("BACKGROUND",(0,1),(-1,-1),colors.HexColor("#FFF8F7")),("VALIGN",(0,0),(-1,-1),"MIDDLE"),
                    ("LEFTPADDING",(0,0),(-1,-1),4),("RIGHTPADDING",(0,0),(-1,-1),4),
                    ("TOPPADDING",(0,0),(-1,-1),2),("BOTTOMPADDING",(0,0),(-1,-1),2),
                ]))
                story.append(findings_table)
        else:
            # Executive Summary mode includes the category summary table only.
            # Representative examples are intentionally omitted.
            if not findings_for_pdf.empty:
                story.append(Paragraph("Active Privilege Zone Findings", compact_h2))
                finding_note = ParagraphStyle("PZExecutiveFindingNoteV1518",parent=styles["small"],fontSize=6.0,leading=6.8,textColor=PDF_COLORS["muted"],spaceAfter=2)
                shown_count = min(5, finding_category_count)
                story.append(Paragraph(
                    f"{finding_category_count:,} active finding categories were identified in the protected scope. The top {shown_count:,} are summarized below; complete finding data remains in the Evidence Workbook.",
                    finding_note,
                ))
                fh = ParagraphStyle("PZExecutiveFindingHeadV1518",parent=styles["small"],fontName="Helvetica-Bold",fontSize=5.7,leading=6.4,textColor=colors.white)
                ft = ParagraphStyle("PZExecutiveFindingTextV1518",parent=styles["small"],fontSize=5.8,leading=6.6,textColor=PDF_COLORS["ink"])
                fs = ParagraphStyle("PZExecutiveFindingSeverityV1518",parent=ft,fontName="Helvetica-Bold",textColor=PDF_COLORS["high"],alignment=1)
                fc = ParagraphStyle("PZExecutiveFindingCountV1518",parent=ft,fontName="Helvetica-Bold",alignment=2)
                finding_rows=[[Paragraph("ACTIVE FINDING",fh),Paragraph("ZONE",fh),Paragraph("SEVERITY",fh),Paragraph("COUNT",fh)]]
                for _, finding in findings_for_pdf.head(5).iterrows():
                    severity=_pz_text(finding.get(finding_severity_col,"High") if finding_severity_col else "High","High")
                    finding_rows.append([
                        Paragraph(pdf_text(_pz_text(finding.get(finding_title_col,"Privilege Zone Finding") if finding_title_col else "Privilege Zone Finding")),ft),
                        Paragraph(pdf_text(_pz_text(finding.get(finding_zone_col,"") if finding_zone_col else "",default_zone)),ft),
                        Paragraph(pdf_text(severity.upper()),fs),
                        Paragraph(f"{_pz_num(finding.get(finding_count_col,0)):,}",fc),
                    ])
                findings_table=Table(finding_rows,colWidths=[3.80*inch,1.50*inch,0.90*inch,0.80*inch],repeatRows=1,hAlign="LEFT")
                findings_table.setStyle(TableStyle([
                    ("BACKGROUND",(0,0),(-1,0),PDF_COLORS["ink"]),("GRID",(0,0),(-1,-1),0.3,colors.HexColor("#D8DEE8")),
                    ("BACKGROUND",(0,1),(-1,-1),colors.HexColor("#FFF8F7")),("VALIGN",(0,0),(-1,-1),"MIDDLE"),
                    ("LEFTPADDING",(0,0),(-1,-1),4),("RIGHTPADDING",(0,0),(-1,-1),4),
                    ("TOPPADDING",(0,0),(-1,-1),2),("BOTTOMPADDING",(0,0),(-1,-1),2),
                ]))
                story.append(findings_table)
                story.append(Spacer(1,4))

            # Executive Summary mode closes with a decision-oriented priority.
            priority_title = ParagraphStyle("PZExecutivePriorityTitleV1518",parent=styles["h3"],fontSize=9.2,leading=10.2,textColor=PDF_COLORS["accent_dark"],spaceAfter=2)
            priority_body = ParagraphStyle("PZExecutivePriorityBodyV1518",parent=styles["body"],fontSize=6.9,leading=8.1,textColor=PDF_COLORS["ink"])
            if has_active_findings:
                priority_text = (
                    "Prioritize review of the highest-volume zone-specific finding categories and validate whether each protection rule still reflects the intended administrative boundary. "
                    "Object-level path evidence should be collected before presenting individual identities or systems as representative attack paths."
                )
                priority_accent = PDF_COLORS.get("moderate", colors.HexColor("#F26B2A"))
                priority_bg = colors.HexColor("#FFF8EF")
            else:
                priority_text = (
                    "Maintain the current boundary by reviewing zone membership and protection rules on a recurring basis, especially after administrative model, platform, or ownership changes."
                )
                priority_accent = PDF_COLORS["green_dark"]
                priority_bg = colors.HexColor("#F1FAF6")
            priority=Table([["",[Paragraph("RECOMMENDED EXECUTIVE ACTION",priority_title),Paragraph(priority_text,priority_body)]]],colWidths=[0.08*inch,6.92*inch],hAlign="LEFT")
            priority.setStyle(TableStyle([
                ("BACKGROUND",(0,0),(-1,-1),priority_bg),("BACKGROUND",(0,0),(0,0),priority_accent),
                ("BOX",(0,0),(-1,-1),0.7,colors.HexColor("#C8D2E1")),("VALIGN",(0,0),(-1,-1),"TOP"),
                ("LEFTPADDING",(1,0),(1,0),10),("RIGHTPADDING",(1,0),(1,0),10),("TOPPADDING",(1,0),(1,0),7),("BOTTOMPADDING",(1,0),(1,0),7),
                ("LEFTPADDING",(0,0),(0,0),0),("RIGHTPADDING",(0,0),(0,0),0),
            ]))
            story.append(priority)

        story.append(Spacer(1,2))
        story.append(Paragraph(
            "Complete Privilege Zone rules, protected-object mappings, validation details, and finding datasets are retained in the Evidence Workbook.",
            ParagraphStyle("PZThemeFootnoteV1512",parent=compact_small,fontSize=5.8,leading=6.6),
        ))

    # ------------------------------------------------------------------
    story.append(PageBreak())
    story.extend(section_title("Conclusion", styles, "Assessment closeout"))
    story.append(Paragraph(
        "The assessment identified material identity exposure across privileged access, delegated control, credential security, administrative movement, and connected identity platforms. The highest-value remediation work is to reduce paths into Tier Zero, remove unnecessary standing privilege, and address the recurring governance conditions that allow new paths to reappear.",
        styles["lead"],
    ))
    story.append(Spacer(1, 6))

    closeout_title = ParagraphStyle("CloseoutTitleV116", parent=styles["h3"], fontSize=9.2, leading=11, spaceAfter=5)
    closeout_body = ParagraphStyle("CloseoutBodyV116", parent=styles["body"], fontSize=8.2, leading=10.4, spaceAfter=0)
    focus = Table([[
        [Paragraph("Immediate Focus", closeout_title), Paragraph("Break the highest-impact paths to privileged control and validate every direct or indirect route into Tier Zero.", closeout_body)],
        [Paragraph("Program Focus", closeout_title), Paragraph("Strengthen privileged access governance, delegation reviews, credential lifecycle controls, and administrative tiering.", closeout_body)],
    ]], colWidths=[3.48*inch,3.48*inch], hAlign="LEFT")
    focus.setStyle(TableStyle([
        ("BACKGROUND",(0,0),(0,0),colors.HexColor("#FFF4F3")), ("BACKGROUND",(1,0),(1,0),colors.HexColor("#F3F5FF")),
        ("BOX",(0,0),(0,0),0.9,PDF_COLORS["high"]), ("BOX",(1,0),(1,0),0.9,PDF_COLORS["blue"]),
        ("LINEBEFORE",(0,0),(0,0),5,PDF_COLORS["high"]), ("LINEBEFORE",(1,0),(1,0),5,PDF_COLORS["blue"]),
        ("VALIGN",(0,0),(-1,-1),"TOP"), ("LEFTPADDING",(0,0),(-1,-1),12), ("RIGHTPADDING",(0,0),(-1,-1),10),
        ("TOPPADDING",(0,0),(-1,-1),10), ("BOTTOMPADDING",(0,0),(-1,-1),10),
    ]))
    story.append(focus)
    story.append(Spacer(1, 14))

    story.append(Paragraph("Collection & Data Quality", compact_h2))
    error_count = len(collection_errors) if collection_errors else 0
    collected_platforms = []
    if "platform_coverage_df" in globals() and not platform_coverage_df.empty:
        for _, prow in platform_coverage_df.iterrows():
            status = str(prow.get("Status", "") or "").lower()
            if "collect" in status or "available" in status:
                collected_platforms.append(str(prow.get("Platform", "Platform")))
    platform_text = ", ".join(collected_platforms) if collected_platforms else "the available BloodHound data sources"
    if error_count:
        collection_summary = (
            f"Evidence was collected from {pdf_text(platform_text)}. {error_count} isolated collection or query issue(s) were recorded. "
            "These issues may reduce coverage for the affected query or platform, but they do not invalidate the evidence that was successfully collected. "
            "Complete diagnostics are preserved in the Evidence Workbook."
        )
        collection_accent = PDF_COLORS["orange"]
    else:
        collection_summary = (
            f"No collection errors were detected during data collection. "
            "The report reflects the graph data and connected-platform evidence available at generation time. "
            "Detailed scope, mappings, metrics, and validation data are preserved in the Evidence Workbook."
        )
        collection_accent = PDF_COLORS["green_dark"]
    collection_box = Table([["", [Paragraph("Collection Status", closeout_title), Paragraph(collection_summary, closeout_body)]]], colWidths=[0.08*inch,6.92*inch], hAlign="LEFT")
    collection_box.setStyle(TableStyle([
        ("BACKGROUND",(0,0),(-1,-1),colors.HexColor("#FBFCFE")), ("BACKGROUND",(0,0),(0,0),collection_accent),
        ("BOX",(0,0),(-1,-1),0.75,colors.HexColor("#D7DCE7")), ("VALIGN",(0,0),(-1,-1),"TOP"),
        ("LEFTPADDING",(1,0),(1,0),12), ("RIGHTPADDING",(1,0),(1,0),12),
        ("TOPPADDING",(1,0),(1,0),10), ("BOTTOMPADDING",(1,0),(1,0),10),
        ("LEFTPADDING",(0,0),(0,0),0), ("RIGHTPADDING",(0,0),(0,0),0),
    ]))
    story.append(collection_box)

    doc.build(story)
    return pdf_path


## 12. Export Primary PDF and Supporting Evidence Workbook


In [ ]:
def safe_sheet_name(name: str) -> str:
    cleaned = re.sub(r"[^A-Za-z0-9_ -]", "", name)
    return cleaned[:31] or "Sheet"


def format_evidence_workbook(writer) -> None:
    """Apply readable wrapping and bounded widths to the exported workbook."""
    from openpyxl.styles import Alignment, Font, PatternFill
    from openpyxl.utils import get_column_letter

    header_fill = PatternFill("solid", fgColor="E9ECF3")
    for worksheet in writer.book.worksheets:
        worksheet.freeze_panes = "A2"
        if worksheet.max_row >= 1 and worksheet.max_column >= 1:
            worksheet.auto_filter.ref = worksheet.dimensions

        headers = {
            str(worksheet.cell(row=1, column=column).value or "").strip(): column
            for column in range(1, worksheet.max_column + 1)
        }

        for cell in worksheet[1]:
            cell.font = Font(bold=True, color="21184F")
            cell.fill = header_fill
            cell.alignment = Alignment(wrap_text=True, vertical="top")

        for column in range(1, worksheet.max_column + 1):
            header = str(worksheet.cell(row=1, column=column).value or "").strip()
            values = [
                str(worksheet.cell(row=row, column=column).value or "")
                for row in range(1, min(worksheet.max_row, 200) + 1)
            ]
            longest = max((len(line) for value in values for line in value.splitlines()), default=0)
            if header == "Recommendation":
                width = 55
            elif header in {"Maturity Domain", "Understanding the Risk", "Why It Matters", "Error", "Detail"}:
                width = min(max(longest + 2, 24), 42)
            else:
                width = min(max(longest + 2, 12), 32)
            worksheet.column_dimensions[get_column_letter(column)].width = width

        for row in worksheet.iter_rows(min_row=2):
            line_count = 1
            for cell in row:
                if cell.value is not None:
                    cell.value = normalize_recommendation_text(cell.value) if cell.column == headers.get("Recommendation") else cell.value
                    line_count = max(line_count, str(cell.value).count("\n") + 1)
                cell.alignment = Alignment(wrap_text=True, vertical="top")
            worksheet.row_dimensions[row[0].row].height = min(max(18, line_count * 15), 105)


output_basename = report_output_stem()
xlsx_path = OUTPUT_DIR / f"{output_basename}_evidence.xlsx"

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    run_summary_df.to_excel(writer, sheet_name="Run Summary", index=False)
    risk_theme_df.to_excel(writer, sheet_name="Risk Themes", index=False)

    if globals().get("ATTACK_PATH_METRICS_AVAILABLE", False):
        pd.DataFrame([{
            "Metric": "Total Enterprise Tier Zero Attack Paths",
            "Value": int(globals().get("TOTAL_ENTERPRISE_TIER_ZERO_ATTACK_PATHS", 0)),
            "Snapshot Date": globals().get("ATTACK_PATH_SNAPSHOT_DATE", ""),
        }]).to_excel(writer, sheet_name="Enterprise Attack Paths", index=False)
        globals().get("ATTACK_PATH_PLATFORM_DF", pd.DataFrame()).to_excel(
            writer, sheet_name="Attack Paths by Platform", index=False
        )
        globals().get("ATTACK_PATH_ENVIRONMENT_DF", pd.DataFrame()).to_excel(
            writer, sheet_name="Attack Paths by Environment", index=False
        )

    if "NATIVE_FINDING_INVENTORY_DF" in globals():
        NATIVE_FINDING_INVENTORY_DF.to_excel(
            writer,
            sheet_name="Native Finding Inventory",
            index=False,
        )

    if "NATIVE_FINDING_STATUS_DF" in globals():
        NATIVE_FINDING_STATUS_DF.to_excel(
            writer,
            sheet_name="Native Finding Status",
            index=False,
        )
    underlying_bhe_findings_df.to_excel(writer, sheet_name="BHE Finding Mapping", index=False)
    platform_coverage_df.to_excel(writer, sheet_name="Platform Coverage", index=False)

    collection_diagnostics_df = pd.DataFrame([
        {
            "Report Section": result.section.title,
            "Evidence Source": error.source.label,
            "Detail": error.error,
        }
        for result in section_results.values()
        for error in result.errors
    ])
    if collection_diagnostics_df.empty:
        collection_diagnostics_df = pd.DataFrame([{
            "Report Section": "All",
            "Evidence Source": "Collection",
            "Detail": "No collection errors were recorded.",
        }])
    collection_diagnostics_df.to_excel(writer, sheet_name="Collection Diagnostics", index=False)

    if "NATIVE_ACTIONABLE_PATHS_DF" in globals():
        NATIVE_ACTIONABLE_PATHS_DF.to_excel(
            writer,
            sheet_name="Actionable Paths",
            index=False,
        )

    if "NATIVE_PATH_STATUS_DF" in globals():
        NATIVE_PATH_STATUS_DF.to_excel(
            writer,
            sheet_name="Actionable Path Status",
            index=False,
        )
    risk_context_df = pd.DataFrame([
        {
            "Report Section": section.title,
            "Maturity Domain": section.maturity_domain,
            "Risk Theme": section.risk_theme,
            "Understanding the Risk": risk_context_for_section(section)
        }
        for section in SECTIONS
    ])
    risk_context_df.to_excel(writer, sheet_name="Risk Context", index=False)


    if globals().get("has_privilege_zones", False):
        pz_exports = [
            ("Privilege Zone Summary", globals().get("PRIVILEGE_ZONE_REPORT_DF")),
            ("PZ Protected Objects", globals().get("PRIVILEGE_ZONE_PROTECTED_OBJECTS_DF")),
            ("PZ Rules", globals().get("PRIVILEGE_ZONE_RULES_DF")),
            ("PZ Exposure", globals().get("PRIVILEGE_ZONE_EXPOSURE_DF")),
            ("PZ Validation", globals().get("PRIVILEGE_ZONE_VALIDATION_DF")),
        ]
        for sheet, frame in pz_exports:
            if isinstance(frame, pd.DataFrame) and not frame.empty:
                frame.to_excel(writer, sheet_name=safe_sheet_name(sheet), index=False)

    for ext in extension_registry:
        ext_id = ext["id"]
        ext_result = extension_manager._results.get(ext_id)
        if ext_result is None:
            continue
        platform = ext.get("platform", ext_id.title())
        status_sheet = safe_sheet_name(f"{platform} Status")
        technical_sheet = safe_sheet_name(f"{platform} Technical")
        ext_result.status_df.to_excel(writer, sheet_name=status_sheet, index=False)
        if not ext_result.technical_df.empty:
            ext_result.technical_df.to_excel(writer, sheet_name=technical_sheet, index=False)

    for result in section_results.values():
        sheet_name = safe_sheet_name(result.section.title)
        df = ensure_maturity_domain_second(
            clean_final_dataframe(result.combined_evidence_df),
            default_domain=result.section.maturity_domain,
        )
        if df.empty:
            if result.errors:
                df = pd.DataFrame([
                    {"status": "collection note", "evidence_source": e.source.label, "detail": e.error}
                    for e in result.errors
                ])
            else:
                df = pd.DataFrame([{"status": "No rows returned"}])
        df.to_excel(writer, sheet_name=sheet_name, index=False)

display(Markdown(f"Evidence workbook written to `{xlsx_path}`"))


In [ ]:
# Maturity-domain and evidence-table validation
validation_rows = []
for section in SECTIONS:
    result = section_results.get(section.id)
    if result is None:
        validation_rows.append({
            "Report Section": section.title,
            "Maturity Domain": section.maturity_domain,
            "Status": "No section result",
        })
        continue
    rendered_df = section_technical_evidence_dataframe(
        section,
        result.combined_evidence_df,
        max_rows=5,
    )
    rendered_df = ensure_maturity_domain_second(
        rendered_df,
        default_domain=section.maturity_domain,
    )
    columns = list(rendered_df.columns)
    domain_second = len(columns) >= 2 and columns[1] == "Maturity Domain"
    missing_values = (
        0 if rendered_df.empty else
        int(rendered_df["Maturity Domain"].astype(str).str.strip().eq("").sum())
        if "Maturity Domain" in rendered_df.columns else len(rendered_df)
    )
    validation_rows.append({
        "Report Section": section.title,
        "Maturity Domain": section.maturity_domain,
        "Second Column": domain_second,
        "Missing Domain Values": missing_values,
        "Status": "PASS" if domain_second and missing_values == 0 else "FAIL",
    })

MATURITY_DOMAIN_VALIDATION_DF = pd.DataFrame(validation_rows)
display(MATURITY_DOMAIN_VALIDATION_DF)
failed = MATURITY_DOMAIN_VALIDATION_DF[MATURITY_DOMAIN_VALIDATION_DF["Status"] == "FAIL"]
if not failed.empty:
    raise RuntimeError(
        "Maturity Domain validation failed for: "
        + ", ".join(failed["Report Section"].astype(str))
    )

In [ ]:
print("Privilege Zone PDF renderer: v1.5.43 wholesale fixed-coordinate replacement")

# ---------------------------------------------------------------------------
# Final deliverable export and verification
# ---------------------------------------------------------------------------

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_basename = report_output_stem()
final_pdf_path = (OUTPUT_DIR / f"{output_basename}.pdf").resolve()
final_xlsx_path = (OUTPUT_DIR / f"{output_basename}_evidence.xlsx").resolve()

# PDF is the default report deliverable. No HTML report is generated.
build_plain_pdf_report(final_pdf_path)

# Preserve the evidence workbook as the authoritative technical validation artifact.
with pd.ExcelWriter(final_xlsx_path, engine="openpyxl") as writer:
    run_summary_df.to_excel(writer, sheet_name="Run Summary", index=False)
    risk_theme_df.to_excel(writer, sheet_name="Risk Themes", index=False)

    if globals().get("ATTACK_PATH_METRICS_AVAILABLE", False):
        pd.DataFrame([{
            "Metric": "Total Enterprise Tier Zero Attack Paths",
            "Value": int(globals().get("TOTAL_ENTERPRISE_TIER_ZERO_ATTACK_PATHS", globals().get("TOTAL_ENTERPRISE_ATTACK_PATHS", 0))),
            "Snapshot Date": globals().get("ATTACK_PATH_SNAPSHOT_DATE", ""),
        }]).to_excel(writer, sheet_name="Enterprise Attack Paths", index=False)
        globals().get("ATTACK_PATH_PLATFORM_DF", pd.DataFrame()).to_excel(
            writer, sheet_name="Attack Paths by Platform", index=False
        )
        globals().get("ATTACK_PATH_ENVIRONMENT_DF", pd.DataFrame()).to_excel(
            writer, sheet_name="Attack Paths by Environment", index=False
        )

    if "NATIVE_FINDING_INVENTORY_DF" in globals():
        NATIVE_FINDING_INVENTORY_DF.to_excel(writer, sheet_name="Native Finding Inventory", index=False)
    if "NATIVE_FINDING_STATUS_DF" in globals():
        NATIVE_FINDING_STATUS_DF.to_excel(writer, sheet_name="Native Finding Status", index=False)
    if "PRIVILEGE_ZONE_FINDINGS_DF" in globals() and isinstance(PRIVILEGE_ZONE_FINDINGS_DF, pd.DataFrame) and not PRIVILEGE_ZONE_FINDINGS_DF.empty:
        PRIVILEGE_ZONE_FINDINGS_DF.to_excel(writer, sheet_name="PZ Active Findings", index=False)
    if "PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF" in globals() and isinstance(PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF, pd.DataFrame) and not PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF.empty:
        PRIVILEGE_ZONE_REPRESENTATIVE_EXAMPLES_DF.to_excel(writer, sheet_name="PZ Path Examples", index=False)
    if "DOMAIN_DETAILS_CALLS_DF" in globals() and isinstance(DOMAIN_DETAILS_CALLS_DF, pd.DataFrame):
        DOMAIN_DETAILS_CALLS_DF.to_excel(writer, sheet_name="PZ Domain Detail Calls", index=False)
    if "PZ_DISCOVERED_ENVIRONMENTS_DF" in globals() and isinstance(PZ_DISCOVERED_ENVIRONMENTS_DF, pd.DataFrame):
        PZ_DISCOVERED_ENVIRONMENTS_DF.to_excel(writer, sheet_name="PZ Environments", index=False)
    if "PZ_PLATFORM_COLLECTION_VALIDATION_DF" in globals() and isinstance(PZ_PLATFORM_COLLECTION_VALIDATION_DF, pd.DataFrame):
        PZ_PLATFORM_COLLECTION_VALIDATION_DF.to_excel(writer, sheet_name="PZ Platform Validation", index=False)
    if "PZ_NATIVE_DETAIL_VALIDATION_DF" in globals() and isinstance(PZ_NATIVE_DETAIL_VALIDATION_DF, pd.DataFrame):
        PZ_NATIVE_DETAIL_VALIDATION_DF.to_excel(writer, sheet_name="PZ Native Detail Validation", index=False)
    if "OBJECT_LEVEL_FINDINGS_DF" in globals() and isinstance(OBJECT_LEVEL_FINDINGS_DF, pd.DataFrame):
        OBJECT_LEVEL_FINDINGS_DF.to_excel(writer, sheet_name="PZ Object Findings", index=False)
    if "REPRESENTATIVE_RELATIONSHIPS_DF" in globals() and isinstance(REPRESENTATIVE_RELATIONSHIPS_DF, pd.DataFrame):
        REPRESENTATIVE_RELATIONSHIPS_DF.to_excel(writer, sheet_name="PZ Representative", index=False)
    if "FINDING_COLLECTION_VALIDATION_DF" in globals() and isinstance(FINDING_COLLECTION_VALIDATION_DF, pd.DataFrame):
        FINDING_COLLECTION_VALIDATION_DF.to_excel(writer, sheet_name="PZ Finding Validation", index=False)

    underlying_bhe_findings_df.to_excel(writer, sheet_name="BHE Finding Mapping", index=False)
    platform_coverage_df.to_excel(writer, sheet_name="Platform Coverage", index=False)

    collection_diagnostics_df = pd.DataFrame([
        {
            "Report Section": result.section.title,
            "Evidence Source": error.source.label,
            "Detail": error.error,
        }
        for result in section_results.values()
        for error in result.errors
    ])
    if collection_diagnostics_df.empty:
        collection_diagnostics_df = pd.DataFrame([{
            "Report Section": "All",
            "Evidence Source": "Collection",
            "Detail": "No collection errors were recorded.",
        }])
    collection_diagnostics_df.to_excel(writer, sheet_name="Collection Diagnostics", index=False)

    if "NATIVE_ACTIONABLE_PATHS_DF" in globals():
        NATIVE_ACTIONABLE_PATHS_DF.to_excel(writer, sheet_name="Actionable Paths", index=False)
    if "NATIVE_PATH_STATUS_DF" in globals():
        NATIVE_PATH_STATUS_DF.to_excel(writer, sheet_name="Actionable Path Status", index=False)

    risk_context_df = pd.DataFrame([
        {
            "Report Section": section.title,
            "Maturity Domain": section.maturity_domain,
            "Risk Theme": section.risk_theme,
            "Understanding the Risk": risk_context_for_section(section),
        }
        for section in SECTIONS
    ])
    risk_context_df.to_excel(writer, sheet_name="Risk Context", index=False)

    for extension in extension_registry:
        extension_id = extension["id"]
        extension_result = extension_manager._results.get(extension_id)
        if extension_result is None:
            continue
        platform = extension.get("platform", extension_id.title())
        extension_result.status_df.to_excel(
            writer, sheet_name=safe_sheet_name(f"{platform} Status"), index=False
        )
        if not extension_result.technical_df.empty:
            extension_result.technical_df.to_excel(
                writer, sheet_name=safe_sheet_name(f"{platform} Technical"), index=False
            )

    for result in section_results.values():
        dataframe = ensure_maturity_domain_second(
            clean_final_dataframe(result.combined_evidence_df),
            default_domain=result.section.maturity_domain,
        )
        if dataframe.empty:
            if result.errors:
                dataframe = pd.DataFrame([
                    {
                        "status": "collection note",
                        "evidence_source": error.source.label,
                        "detail": error.error,
                    }
                    for error in result.errors
                ])
            else:
                dataframe = pd.DataFrame([{"status": "No rows returned"}])
        dataframe.to_excel(writer, sheet_name=safe_sheet_name(result.section.title), index=False)

    format_evidence_workbook(writer)

export_rows = [
    {
        "Deliverable": "PDF Report",
        "Path": str(final_pdf_path),
        "Exists": final_pdf_path.exists(),
        "Size Bytes": final_pdf_path.stat().st_size if final_pdf_path.exists() else 0,
    },
    {
        "Deliverable": "Evidence Workbook",
        "Path": str(final_xlsx_path),
        "Exists": final_xlsx_path.exists(),
        "Size Bytes": final_xlsx_path.stat().st_size if final_xlsx_path.exists() else 0,
    },
]

export_status_df = pd.DataFrame(export_rows)
display(Markdown("## Final Deliverables"))
display(export_status_df)

missing_outputs = [row["Path"] for row in export_rows if not row["Exists"] or row["Size Bytes"] == 0]
if missing_outputs:
    raise RuntimeError("Final export failed for: " + ", ".join(missing_outputs))

display(Markdown(
    f"PDF report written to `{final_pdf_path}`  \n"
    f"Evidence workbook written to `{final_xlsx_path}`"
))


In [ ]:
# Evidence stabilization helper

def _stabilize_evidence(df):
    if df is None or getattr(df, "empty", True):
        return df

    clean = df.copy()

    cols = {str(c).strip().lower(): c for c in clean.columns}
    src = cols.get("source")
    tgt = cols.get("target")

    if src is not None and tgt is not None:
        clean = clean[
            clean[src].astype(str).str.strip().ne("") &
            clean[tgt].astype(str).str.strip().ne("")
        ]

    clean = clean.drop_duplicates()

    return clean.reset_index(drop=True)

print("v2.5.0-14 stabilization loaded.")